<a href="https://colab.research.google.com/github/hmghaly/hparser/blob/master/Parsing_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Start

In [1]:
import os
from google.colab import drive
drive.mount('/content/drive/')
cwd="/content/drive/My Drive/projects/parsing/code"
os.chdir(cwd)

Mounted at /content/drive/


In [ ]:
!rm -r code_utils #first delete the code_utils folder, then clone from the codebase (very crude till we figure out using github from colab)
!git clone https://github.com/hmghaly/word_align.git code_utils

Cloning into 'code_utils'...
remote: Enumerating objects: 4289, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 4289 (delta 7), reused 0 (delta 0), pack-reused 4274 (from 3)
Receiving objects: 100% (4289/4289), 29.65 MiB | 4.88 MiB/s, done.
Resolving deltas: 100% (2432/2432), done.
Updating files: 100% (65/65), done.


#EMNLP Process - May 2026


##Load rule workbook
process rules, word features, and XPOS features

In [31]:
#14 May 2026
from code_utils.pandas_utils import *
from code_utils.parsing_lib import *

from itertools import groupby
import json, copy


file_id="21_may_2026"
info_fname=f"rules_word_features_{file_id}.json"

rules_dir="../data/rules"
if not os.path.exists(rules_dir): os.makedirs(rules_dir)
info_fpath=os.path.join(rules_dir,info_fname)


#categories that we replace XP - we may need to update them
#categories that can be conjugated with AND and comma sequences
xp_cat_list=["V","VP","NP",'NP-DT',"NP-U","S","S-U","AdvP","AP","V-PAS","PP",'VP-O','TO-VP','VP-2O',"CP"]

rules_workbook_url="https://docs.google.com/spreadsheets/d/e/2PACX-1vQcnggjmm9lpsURmsKal23AAUfUEA-6H5VQ6X42T9nUfeipUq5miqWPRZUoUsslixyRz0U6U3qV9v1i/pub?output=xlsx"
rw_data_dict=get_wb_data_dict(rules_workbook_url)

rw_data_dict.keys()

phrase_rules_sheet=rw_data_dict["phrase_rules"]
terminal_rules_sheet=rw_data_dict["terminal_rules"]
#lexical_info_sheet=rw_data_dict["lexical_info"]
lexical_sheet=rw_data_dict["lexical_tags"]


# rules_sheet=rw_data_dict["rule_list"]
# word_features_list=rw_data_dict["words-features"]
# xpos_sheet_rows=rw_data_dict["xpos"]
# upos_sheet_rows=rw_data_dict["upos"]

final_combined_rules_dict={}
sheet_names=["phrase_rules","terminal_rules"]
for sh0 in sheet_names:
  final_processed_rules=[]
  cur_sheet=rw_data_dict[sh0]

  xp_rule_dict_list=[]
  all_cat_list=[]
  for r0 in cur_sheet:
    cur_rule0=r0["rule"].strip()
    if not cur_rule0: continue
    cur_wt_str=r0.get("weight","").strip()
    cur_deprel=r0.get("deprel","").strip()
    cur_description=r0.get("description","").strip()
    cur_example=r0.get("example","").strip()

    if cur_deprel=="": cur_deprel="dep"
    if cur_wt_str=="": cur_wt=1
    else: cur_wt=float(cur_wt_str)
    r0["rule_wt"]=cur_wt
    r0["deprel"]=cur_deprel
    #print(r0)
    cur_rule_str_list=[]
    #replace {XP} with relevant categories till we improve the algorithm with regex style synamic matching
    if "{XP}" in cur_rule0:
      for repl_cat0 in xp_cat_list:
        replaced_rule0=cur_rule0.replace("{XP}",repl_cat0)
        cur_rule_str_list.append(replaced_rule0)
    else: cur_rule_str_list.append(cur_rule0)

    for cur_rule1 in cur_rule_str_list:
      #print(cur_rule1)


      processed_rule=process_rule(cur_rule1)
      if processed_rule=={}: continue
      processed_rule_parent=processed_rule.get("parent",{})
      processed_rule_parent_cat=processed_rule_parent.get("cat","")
      if processed_rule_parent_cat=="": continue
      #final_rules.append(cur_rule1)
      # print(sh0,cur_rule1)
      # print(processed_rule)

      processed_rule["rule_wt"]=cur_wt
      processed_rule["deprel"]=cur_deprel
      processed_rule["description"]=cur_description
      processed_rule["example"]=cur_example

      #print(processed_rule)
      final_processed_rules.append(processed_rule)

  final_combined_rules_dict[sh0]=final_processed_rules
  print(sh0,len(final_processed_rules))

#lexical_info
lexical_tags_dict={}

for r0 in lexical_sheet:
  word0=r0["word"].strip() #word or token, punctuation .. etc
  category0=r0["category"].strip()
  if category0=="": continue
  features0=r0["features"].strip()
  ft_list=features0.split()


  if word0.startswith("\\") and word0!="\\": word0=word0.replace("\\","") #account for lexical categories with escape slashes
  #print("word0",word0)

  cur_wt_str=r0.get("weight","").strip()
  if cur_wt_str=="": cur_wt=1
  else: cur_wt=float(cur_wt_str)

  cur_obj={"category":category0,"weight":cur_wt,"features":ft_list}
  lexical_tags_dict[word0]=lexical_tags_dict.get(word0,[])+[cur_obj]


print("lexical_tags_dict",len(lexical_tags_dict.items()))
# for a,b in lexical_custom_tag_dict.items():
#   print(a,b)

final_combined_rules_dict["lexical_tags"]=lexical_tags_dict

with open(info_fpath,"w") as info_fopen:
  json.dump(final_combined_rules_dict,info_fopen)

#print("final_rules", len(final_rules))
print("data saved to:",info_fpath)


phrase_rules 220
terminal_rules 22
lexical_tags_dict 92
data saved to: ../data/rules/rules_word_features_21_may_2026.json


##Start Parsing - May 2026

In [34]:
#17 May 2026
import time, json, re
from itertools import groupby
from code_utils import general
from code_utils.parsing_lib import *


file_id="21_may_2026"
info_fname=f"rules_word_features_{file_id}.json"
rules_dir="../data/rules"
info_fpath=os.path.join(rules_dir,info_fname)
with open(info_fpath) as fopen:
  rule_info_dict=json.load(fopen)

rnn_model_fpath="../models/pos_models/H64_L3_LR001_updated_xpos_loading.model"
params0={"pos_model_path":rnn_model_fpath,"rule_info_dict":rule_info_dict}
params0["max_n_phrases"]=20
params0["debug"]=False

class Parser:
  def __init__(self,params={}) -> None:
    self.unknown_tags=params.get("unknown_tags",["N","V","JJ","RB"]) #maybe get the actual distribution of these tags from corpora
    self.rule_info_dict=params.get("rule_info_dict",{})
    self.pos_model_path=params.get("pos_model_path")
    self.min_pos_wt=params.get("min_pos_wt",0.5)
    self.max_n_phrases=params.get("max_n_phrases",5) #maximum number of phrases (keys) per span
    self.max_dist=params.get("max_dist",2) #maximum distance between parsed phrases

    self.phrase_rules=self.rule_info_dict.get("phrase_rules",[])

    self.terminal_rules=self.rule_info_dict.get("terminal_rules",[])
    self.lexical_tags_dict=self.rule_info_dict.get("lexical_tags",{})


    #self.word_rules_inv_index={} #inverted index for rules corresponding to certain word (from lexical rules)
    self.cat_rules_inv_index={} #inverted index for rules corresponding to a certain category
    self.feat_rules_inv_index={} #inverted index for rules corresponding to a certain feature

    self.relevant_phrase_features_dict={} #features that are relevant for parsing
    temp_child_feat_list=[]
    # for lex_rule0 in self.lexical_rules:
    #   word0=lex_rule0["children"][-1]["cat"]
    #   self.word_rules_inv_index[word0]=self.word_rules_inv_index.get(word0,[])+[lex_rule0] #index the rules corresponding to this word
    used_rule_dict={}
    for phrase_rule0 in self.phrase_rules+self.terminal_rules:
      children=phrase_rule0["children"]
      rule_str0=phrase_rule0["rule_str"]
      print("rule_str0",rule_str0)
      if used_rule_dict.get(rule_str0,False)==True: continue
      used_rule_dict[rule_str0]=True
      last_child=children[-1]
      last_child_cat,last_child_feat=last_child["cat"],last_child["feat"]
      self.cat_rules_inv_index[last_child_cat]=self.cat_rules_inv_index.get(last_child_cat,[])+[phrase_rule0] #index the rules corresponding to this category
      for ft0 in last_child_feat:
        self.feat_rules_inv_index[ft0]=self.feat_rules_inv_index.get(ft0,[])+[phrase_rule0] #index the rules corresponding to this feature
      for ch0 in children: temp_child_feat_list.extend(ch0["feat"])
    temp_child_feat_list=list(set(temp_child_feat_list))
    for ft0 in temp_child_feat_list: self.relevant_phrase_features_dict[ft0]=True

    if self.pos_model_path!=None: self.pos_tagger=POS(self.pos_model_path)
    else: self.pos_tagger=None

  def parse(self,tokens):
    #self.phrase_list=[] #list of all phrase objects, with spans, weights, children
    self.phrase_cat_counter_dict={} #counting number of instances of each category to use for generating phrase ID

    #self.end_cat_ft_start_index={} #dict[end][cat|ft][start]=[phrase_key1,phrase_key2 ...]
    self.end_cat_ft_index={} #dict[end][cat|ft]=[phrase_key1,phrase_key2 ...]
    self.span_key_dict={} #dict[(start,end)]=[phrase_key1,phrase_key2 ...]
    self.span_key_phrase_id_wt_dict={} #dict[phrase_key1]=[(phrase_id1,phrase_wt1),(phrase_id2,phrase_wt2) ...]
    self.phrase_id_obj_dict={} #mapping the ID of each phrase to its phrase object

    self.n_projections=0 #calculating number of projections for the whole parse

    self.tokens_pos_list=self.pos_tagger.tag_words(tokens,min_wt=self.min_pos_wt)
    for i0,item0 in enumerate(self.tokens_pos_list):
      print(i0,item0)
      start0,end0=i0,i0
      word0,upos_list0,xpos_list0=item0["word"],item0["upos"],item0["xpos"]
      custom_tags=self.lexical_tags_dict.get(word0.lower(),[])
      all_applicable_tags_ft_wt=[]
      for ct0 in custom_tags:
        #print(ct0)
        all_applicable_tags_ft_wt.append([ct0["category"],ct0["features"],ct0["weight"]])
      for u0,u_wt0 in upos_list0:
        #print(u0,u_wt0)
        all_applicable_tags_ft_wt.append([u0,[],u_wt0])
      for x0,x_wt0 in xpos_list0:
        #print(x0,x_wt0)
        all_applicable_tags_ft_wt.append([x0,[],x_wt0])
        #xpos_rules=self.cat_rules_inv_index.get(x0,[])
      all_applicable_tags_ft_wt.sort(key=lambda x:(-x[-1],-len(x[1])))
      used_cat_dict={}
      for cat0,ft0,wt0 in all_applicable_tags_ft_wt:
        if used_cat_dict.get(cat0,False)==True: continue
        cur_phrase_obj={"wt":wt0,"span":1,"start":start0,"end":end0,"cat":cat0,"feat":ft0,"head_loc":start0,"children":[],"level":0,"dep":"dep"}
        used_cat_dict[cat0]=True
        #print(cur_phrase_obj)
        self.project_phrase(cur_phrase_obj)
      print("=================")



    #if self.debug: print(tokens_pos_list)


  def project_phrase(self,phrase_obj):
    #print(phrase_obj)
    added_phrase=self.add_phrase(phrase_obj)
    if added_phrase==None: return
    print(added_phrase)
    #now scanning applicable rules
    rules=self.scan_rules(added_phrase)
    for rule_obj in rules:
      print(rule_obj["rule_str"])
      rule_children=rule_obj["children"]
      rule_children_reversed=list(reversed(rule_children))
      last_child=rule_children_reversed[0]
      match_last=match_rule(added_phrase,last_child) #check if the last child of the retrieved rule matches the current phrase
      if not match_last: continue
      print("match_last",match_last, rule_obj["rule_str"])
      new_combined_phrases=[]
      if len(rule_children)>1:
        first_child= rule_children_reversed[1]
        found_preceding_phrases=self.scan_phrases(first_child,added_phrase)
        print("found_preceding_phrases",found_preceding_phrases)
        #find previous phrases
      else:
        unary_combined_phrase=combine_phrases([phrase_obj],rule_obj)
        print("unary_combined_phrase",unary_combined_phrase)

    print("----")

  def add_phrase(self,phrase_obj):
    cat0=phrase_obj["cat"]
    ft_list0=phrase_obj["feat"]
    start0=phrase_obj["start"]
    end0=phrase_obj["end"]
    wt0=phrase_obj["wt"]
    cat_count=self.phrase_cat_counter_dict.get(cat0,0)
    phrase_id0=f"{cat0}_{cat_count}"
    self.phrase_cat_counter_dict[cat0]=cat_count+1
    phrase_key0=get_phrase_key(phrase_obj)
    phrase_obj["id"]=phrase_id0
    #print(phrase_id0, phrase_key0)
    key_ft_list=[cat0]
    for ft0 in ft_list0:
      if self.relevant_phrase_features_dict.get(ft0,False)==True: key_ft_list.append(f"ft:{ft0}")
    #print("key_ft_list",key_ft_list)
    #found_end_dict=self.end_cat_ft_start_index.get(end0,{}) #populate the main index - first retrieve the first key (end)

    found_end_dict=self.end_cat_ft_index.get(end0,{}) #populate the main index - first retrieve the first key (end)
    for kf0 in key_ft_list: #for each category or valid feature, update the following keys
      found_phrase_key_list=found_end_dict.get(kf0,[])+[phrase_key0]  #from temporary found_end_dict, update it with the current cat/ft - update it with current key
      found_end_dict[kf0]=found_phrase_key_list
    self.end_cat_ft_index[end0]=found_end_dict
    self.phrase_id_obj_dict[phrase_id0]=phrase_obj

    self.span_key_dict[(start0,end0)]=self.span_key_dict.get((start0,end0),[])+[phrase_key0]

    found_phrase_ids_wts=self.span_key_phrase_id_wt_dict.get(phrase_key0,[])+[(phrase_id0,wt0)]
    found_phrase_ids_wts.sort(key=lambda x:-x[-1])
    self.span_key_phrase_id_wt_dict[phrase_key0]=found_phrase_ids_wts[:self.max_n_phrases]

    return phrase_obj

  def scan_rules(self,cur_phrase_obj):
    #given a token or phrase with cat/feat pair, identify the rules that apply
    all_rules=[]
    phrase_obj_cat,phrase_obj_feat=cur_phrase_obj["cat"],cur_phrase_obj["feat"]
    cur_cat_rules=self.cat_rules_inv_index.get(phrase_obj_cat,[])
    all_rules.extend(cur_cat_rules)
    for s_feat0 in phrase_obj_feat:
      if self.relevant_phrase_features_dict.get(s_feat0,False)==True: continue
      corr_rules=self.feat_rules_inv_index.get(s_feat0,[])
      for cr0 in corr_rules:
        if not cr0 in all_rules: all_rules.append(cr0)
      #all_rules.extend(corr_rules)
    #return list(set(all_rules))
    return all_rules




  def scan_phrases(self,rule_child,phrase_object): #scan preceding phrases for matching the first child for binary rules

    phrase_start=phrase_object["start"]
    found_preceding_phrases=[]
    child_cat=rule_child["cat"]
    child_feat_list=rule_child["feat"]

    #create the list of keys for cat/feat for the current child
    cat_ft_list=[child_cat]
    for ft0 in child_feat_list:
      if self.relevant_phrase_features_dict.get(ft0,False)==True: cat_ft_list.append(f"ft:{ft0}")

    last_phrase_end=max(0,phrase_start-self.max_dist)
    all_found_phrase_keys=[]
    for end0 in range(last_phrase_end,phrase_start):
      found_end_items=self.end_cat_ft_index.get(end0,{})
      for cat_ft0 in cat_ft_list: all_found_phrase_keys.extend(found_end_items.get(cat_ft0,[]))
    all_found_phrase_keys=list(set(all_found_phrase_keys))
    for ph_key0 in all_found_phrase_keys:
      corr_top_phrase_ids_wts=self.span_key_phrase_id_wt_dict.get(ph_key0,[])
      if corr_top_phrase_ids_wts==[]: continue
      top_phrase_id,top_phrase_wt=corr_top_phrase_ids_wts[0]
      top_phrase_obj=self.phrase_id_obj_dict[top_phrase_id]
      phrase_matches_child=match_rule(top_phrase_obj,rule_child)
      if phrase_matches_child: found_preceding_phrases.append(top_phrase_obj)
    return found_preceding_phrases


def get_phrase_key(phrase_obj0): #check an obj for corresponding key
  start0,end0,cat0,feat_split0=phrase_obj0["start"],phrase_obj0["end"],phrase_obj0["cat"],phrase_obj0["feat"]
  ft_str0=" ".join(feat_split0)
  cur_phrase_key=(start0,end0,cat0,ft_str0)
  return cur_phrase_key


def combine_phrases(phrase_obj_list,applied_rule={}):
  parent_phrase_obj={}
  #cur_child_keys=[v["i"] for v in phrase_obj_list]
  cur_child_keys=[get_phrase_key(v) for v in phrase_obj_list]

  levels=[vw["level"] for vw in phrase_obj_list]

  parent_phrase_obj["wt"]=sum([vw["wt"] for vw in phrase_obj_list])
  phrase_start0=min([vw["start"] for vw in phrase_obj_list])
  phrase_end0=max([vw["end"] for vw in phrase_obj_list])

  parent_phrase_obj["span"]=phrase_end0-phrase_start0+1
  parent_phrase_obj["start"]=phrase_start0
  parent_phrase_obj["end"]=phrase_end0

  if applied_rule!={}:
    parent_phrase_obj["cat"]=applied_rule["parent"]["cat"]
    parent_phrase_obj["feat"]=applied_rule["parent"]["feat"]
    rule_head_i=applied_rule["head_i"]
  else:
    parent_phrase_obj["cat"]=""
    parent_phrase_obj["feat"]=[]
    rule_head_i=0


  parent_phrase_obj["children"]=cur_child_keys
  head_phrase_index=cur_child_keys[rule_head_i] #which of the child phrases is the head

  parent_phrase_obj["head_phrase"]=head_phrase_index
  # head_phrase_key=self.phrase_key_list[head_phrase_index]
  head_phrase_obj=phrase_obj_list[rule_head_i]  #phrase object of the head child

  #11 Feb 2026
  if applied_rule!={}:
    head_child_cat0=head_phrase_obj["cat"]
    applied_rule_parent_obj=applied_rule["parent"]
    percolate=applied_rule_parent_obj.get("percolate",False)
    apply_cat=applied_rule_parent_obj.get("apply_cat",False)
    if percolate: parent_phrase_obj["feat"]=head_phrase_obj["feat"]
    if apply_cat: parent_phrase_obj["feat"]+=[f"cat={head_child_cat0}"] #we may need to adjust later, to accommdate multiple features + applied category of head child

  parent_phrase_obj["head_loc"]=head_phrase_obj["head_loc"]
  parent_phrase_obj["rule"]=applied_rule
  parent_phrase_obj["level"]=max(levels)+1
  return parent_phrase_obj


parser_obj=Parser(params0)

sent="he is happy that they all arrived safely"
sent_words=re.findall(r"\w+",sent.lower())
parser_obj.parse(sent_words)

for a,b in parser_obj.span_key_phrase_id_wt_dict.items():
  print(a,b)
# for a in parser_obj.tokens_pos_list:
#   print(a)
# for a,b in parser_obj.feat_rules_index.items():
#   print(a,[v["rule_str"] for v in b])

# for a,b in parser_obj.relevant_phrase_features_dict.items():
#   print(a,b)

rule_str0 DT^ --> X[det]
rule_str0 NP-U^ --> X[noun]
rule_str0 AP^ --> X[adj]
rule_str0 MD^ --> X[modal]
rule_str0 AdvP^ --> X[adv]
rule_str0 NP-DT --> PPRON
rule_str0 NP-DT^ --> DT NP-U^
rule_str0 NP-U^ --> AP NP-U^
rule_str0 NP-U[adv]^ --> AP[adv] NP-U^
rule_str0 NP --> NP-U
rule_str0 NP --> NP-DT
rule_str0 NP --> NP^ PP
rule_str0 PP --> IN NP^
rule_str0 DT --> NP^ APO
rule_str0 NP-CP --> NP^ CP
rule_str0 NP --> NP-CP
rule_str0 NP-U2 --> NP-U NP-U^
rule_str0 NP-U3 --> NP-U NP-U2^
rule_str0 NP-DT --> DT NP-U2^
rule_str0 NP-DT --> DT NP-U3^
rule_str0 NP --> NP-U2
rule_str0 NP --> NP-U3
rule_str0 NP --> NP^ TO-VP
rule_str0 NP --> WP VP^
rule_str0 NP --> WP S^
rule_str0 NP --> AdvP NP^
rule_str0 NP-DASH --> NP^ HYPHEN
rule_str0 NP --> NP-DASH NP^
rule_str0 AP^ --> AP^ TO-VP
rule_str0 AP^ --> AdvP AP^
rule_str0 AdvP --> PP
rule_str0 AdvP --> NP-U[adv]
rule_str0 AdvP --> AdvP^ COMMA
rule_str0 AdvP --> COMMA AdvP^
rule_str0 AdvP --> ^NP DOT
rule_str0 AdvP --> ^NP HYPHEN
rule_str0 AdvP --> N

In [6]:
for a,b_dict in parser_obj.end_cat_ft_index.items():
  print(a,b_dict)
  # for b,c in b_dict.items(): print(b,c)
  # print("---")

0 {'PRON': [(0, 0, 'PRON', 'noun 3rd masc sentient|human')], 'ft:noun': [(0, 0, 'PRON', 'noun 3rd masc sentient|human')], 'PRP': [(0, 0, 'PRP', '')]}
1 {'AUX': [(1, 1, 'AUX', 'be 3rd singular present')], 'ft:be': [(1, 1, 'AUX', 'be 3rd singular present')], 'VBZ': [(1, 1, 'VBZ', '')], 'VERB': [(1, 1, 'VERB', '')]}
2 {'ADJ': [(2, 2, 'ADJ', '')], 'JJ': [(2, 2, 'JJ', '')], 'NOUN': [(2, 2, 'NOUN', '')], 'NN': [(2, 2, 'NN', '')]}
3 {'DT': [(3, 3, 'DT', '')], 'SCONJ': [(3, 3, 'SCONJ', '')], 'PRON': [(3, 3, 'PRON', '')], 'IN': [(3, 3, 'IN', '')], 'DET': [(3, 3, 'DET', '')], 'WDT': [(3, 3, 'WDT', '')], 'NN': [(3, 3, 'NN', '')]}
4 {'PRON': [(4, 4, 'PRON', 'noun 3rd plural')], 'ft:noun': [(4, 4, 'PRON', 'noun 3rd plural')], 'PRP': [(4, 4, 'PRP', '')]}
5 {'RB': [(5, 5, 'RB', '')], 'DT': [(5, 5, 'DT', '')], 'DET': [(5, 5, 'DET', '')], 'ADV': [(5, 5, 'ADV', '')]}
6 {'VERB': [(6, 6, 'VERB', '')], 'VBD': [(6, 6, 'VBD', '')]}
7 {'ADV': [(7, 7, 'ADV', '')], 'RB': [(7, 7, 'RB', '')]}


In [17]:
start0=4
max_dist=5
last_end=max(0,start0-max_dist)
list(range(last_end,start0))

[0, 1, 2, 3]

In [ ]:
from collections import defaultdict
nested_dict = defaultdict(dict)
nested_dict["a"]["b"]["c"]=123


KeyError: 'b'

In [ ]:
tree = lambda: defaultdict(tree)
d = tree()
d["a"]["b"]["c"]=123

In [ ]:
d

defaultdict(<function __main__.<lambda>()>,
            {'a': defaultdict(<function __main__.<lambda>()>,
                         {'b': defaultdict(<function __main__.<lambda>()>,
                                      {'c': 123})})})

In [ ]:
defaultdict(lambda: defaultdict(int))

In [ ]:
nested_dict["a"]["b"]["c"]=123

TypeError: 'int' object is not subscriptable

In [ ]:
from code_utils.parsing_lib import *
from code_utils import general

file_id="17_may_2026"
info_fname=f"rules_word_features_{file_id}.json"
rules_dir="../data/rules"
info_fpath=os.path.join(rules_dir,info_fname)
with open(info_fpath) as fopen:
  rules_word_features_info=json.load(fopen)

rules_word_features_info

# final_word_features_dict=rules_word_features_info["final_word_features_dict"]
# xpos_ft_dict=rules_word_features_info["xpos_ft_dict"]
# rules=rules_word_features_info["final_rules"]



# sent="Our commitment is also reflected in the efforts made by the Ecuadorian presidency of the Committee in charge of developing a legally binding instrument to eliminate plastics pollution."

# sent=re.sub(r"(\W)",r" \1 ",sent)
# sent_toks=sent.strip().split()



# sent_toks=['5', 'star', 'detail', 'job']
# sent_toks=['"', 'Further', ',', 'I', 'find', 'it', 'even', 'more', 'disturbing', 'that', 'he', 'made', 'these', 'comments', 'about', 'a', 'nation', 'that', 'just', 'last', 'year', 'tested', 'nuclear', 'weapons', '--', 'shortly', 'after', 'voicing', 'his', 'public', 'opposition', 'to', 'the', 'Comprehensive', 'Test', 'Ban', 'Treaty', '.']
# # sent_toks=['A', 'spokesman', 'for', 'President', 'Clinton', 'also', 'criticized', 'Bush', "'s", 'comments', '.']
# sent_toks=['Further', ',', 'I', 'find', 'it', 'even', 'more', 'disturbing']


# unknown_tags=["N","V","JJ","RB"]
# #sent_toks=sent.split()
# print(sent_toks)

# rnn_model_fpath="../models/pos_models/H64_L3_LR001_updated_xpos_loading.model"
# params0={"pos_model_path":rnn_model_fpath,"final_word_features_dict":final_word_features_dict,"xpos_ft_dict":xpos_ft_dict}
# params0["max_n_phrases"]=20
# params0["debug"]=False

# parser_obj=Parser(rules_list=rules, params=params0)

# parser_output=parser_obj.parse(sent_toks)
# final_parses=parser_output["parses"]
# final_pos=parser_output["pos"]
# #print(final_pos)
# for a in final_pos: print(a)
# final_parses.sort(key=lambda x:-x[0]["wt"])
# for ph_obj0,dep0,const0 in final_parses[:3]:
#   print(ph_obj0.get("wt",0))
#   for d0 in dep0:
#     print(d0)
#   print("---")
#   for c0 in const0:
#     print(c0)
#   print("=========")

{'phrase_rules': [{'parent': {'cat': 'DT',
    'feat': [],
    'percolate': True,
    'apply_cat': False},
   'children': [{'cat': 'X', 'is_head': False, 'feat': ['det']}],
   'head_i': 0,
   'rule_wt': 1,
   'deprel': 'dep',
   'description': 'determiner, possessive pronoun',
   'example': '[the] car | [my] car'},
  {'parent': {'cat': 'NP-U',
    'feat': [],
    'percolate': True,
    'apply_cat': False},
   'children': [{'cat': 'X', 'is_head': False, 'feat': ['noun']}],
   'head_i': 0,
   'rule_wt': 1,
   'deprel': 'dep',
   'description': 'NP without determiner',
   'example': ''},
  {'parent': {'cat': 'AP', 'feat': [], 'percolate': True, 'apply_cat': False},
   'children': [{'cat': 'X', 'is_head': False, 'feat': ['adj']}],
   'head_i': 0,
   'rule_wt': 1,
   'deprel': 'dep',
   'description': 'Adjectival phrase',
   'example': ''},
  {'parent': {'cat': 'MD', 'feat': [], 'percolate': True, 'apply_cat': False},
   'children': [{'cat': 'X', 'is_head': False, 'feat': ['modal']}],
   'h

##Parser Algorithm/Code May 2026

In [ ]:
#31 December 2025
class Parser:
  def __init__(self,rules_list=[],word_features_list=[],params={}) -> None:
    self.unknown_tags=params.get("unknown_tags",["N","V","JJ","RB"]) #maybe get the actual distribution of these tags from corpora
    self.sent_padding=params.get("sent_padding",False)
    self.default_wt=params.get("default_wt",0.5) #to apply on unknown tags
    self.min_pos_wt=params.get("min_pos_wt",0.2)
    self.pos_model_path=params.get("pos_model_path")
    self.final_word_features_dict=params.get("final_word_features_dict",{})
    self.xpos_ft_dict=params.get("xpos_ft_dict",{})
    self.max_n_phrases=params.get("max_n_phrases",10) #max n phrases per key
    self.djk_max_span_distance=params.get("djk_max_span_distance",3) #max difference between two spans when applying djekstra's algorithm
    self.djk_distance_penalty_ratio=params.get("djk_distance_penalty_ratio",0.5) #penalty applied dpending on the distance between two consecutive spans on djk path
    self.djk_use_negative_wt=params.get("djk_use_negative_wt",False) #whether to use actual weight or negative weight to get the optimum path

    self.max_skip_distance=params.get("max_skip_distance",3) #max number of tokens to skip between phrases
    self.debug=params.get("debug",False)

    self.combined_fragmented=params.get("combined_fragmented",False) #combine fragmented phrases



    self.rerank_params=params.get("rerank_params",{}) #rerank/parse scoring parameters, including reranking neural model

    self.max_cat_proj=params.get("max_cat_proj",100) #maximum number of times to project a category for a particular head token
    self.pos_tagger=POS(self.pos_model_path)

    self.unknown_token_tags=[("N","noun"),("V","verb"),("JJ","adj"),("RB","adv")]
    self.unknown_token_tags_wt_dict={}
    #process all word features

    #process all rules
    self.all_processed_rules=[]
    self.cat_fwd_index=[]
    self.feat_fwd_index=[]
    for i0,r0 in enumerate(rules_list):
      processed_rule=process_rule(r0)
      if processed_rule==None: continue
      processed_rule["rule_i"]=i0
      self.all_processed_rules.append(processed_rule)

    for i0, processed_rule in enumerate(self.all_processed_rules):
      last_child=processed_rule["children"][-1]
      last_child_cat=last_child["cat"]
      last_child_features=last_child["feat"]
      self.cat_fwd_index.append((last_child_cat,i0))
      for ft0 in last_child_features: self.feat_fwd_index.append((ft0,i0))
    self.cat_fwd_index.sort()
    self.feat_fwd_index.sort()
    self.cat_inv_index=dict(iter([(key,[v[1] for v in list(group)]) for key,group in groupby(self.cat_fwd_index,lambda x:x[0])]))
    self.feat_inv_index=dict(iter([(key,[v[1] for v in list(group)]) for key,group in groupby(self.feat_fwd_index,lambda x:x[0])]))
    #self.phrase_list=[]


  def parse(self,tokens):
    self.phrase_list=[] #list of all phrase objects, with spans, weights, children
    self.cat_phrase_index={}
    self.feat_phrase_index={}

    self.end_start_phrase_dict={} #dict[end][start][cat]=[{phrase_obj1},{phrase_obj2},{phrase_obj3}]
    self.span_wt_dict={} #dict[(start,end)]=wt
    self.span_phrase_dict={} #dict[(start,end)]={phrase_obj}
    self.phrase_key_obj_dict={} #dict[phrase_key]=phrase_obj


    self.projection_counter={}
    self.head_loc_cat_projection_counter={} #counting how many times this category was projected from the same head token dict[(head_loc,cat)]=count


    tokens_pos_list=self.pos_tagger.tag_words(tokens,min_wt=self.min_pos_wt)
    self.cur_tokens_pos_list=tokens_pos_list

    self.tag_wt_list=[]

    if self.debug: print(tokens_pos_list)
    for token_i,a in enumerate(tokens_pos_list): #process each token
      token0=a["word"]
      if self.debug: print(token_i,token0,len(self.phrase_key_list))

      xpos0=a.get("xpos",[])
      start0,end0=token_i,token_i
      wt=self.default_wt
      outcome=self.final_word_features_dict.get(token0.lower(),[]) #outcome is multiple tag/cat options, with their features
      outcome_wt=[(cat0,feat0,1) for cat0,feat0 in outcome] #use a weight of 1 for the specific closed class words
      temp_used_cat_list=[v[0] for v in outcome_wt]
      for xpos_tag0,xpos_wt0 in xpos0:
        if xpos_tag0 in temp_used_cat_list: continue
        xpos_ft0=self.xpos_ft_dict.get(xpos_tag0,[])
        outcome_wt.append((xpos_tag0,xpos_ft0,xpos_wt0))
      #if nothing found
      if outcome_wt==[]:
        for cat0,feat0 in self.unknown_token_tags: outcome_wt.append((cat0,feat0,self.default_wt))

      self.tag_wt_list.append((token0,outcome_wt))
      #for the current token, identify all the outcomes with their weights
      #print("token0",token0)
      all_token_new_phrases=[]
      for cat0,feat_split,wt0 in outcome_wt:
        if cat0=="": continue
        #only one phrase object per cat/features pair
        cur_phrase_obj={"wt":wt0,"span":1,"start":start0,"end":end0,"cat":cat0,"feat":feat_split,"head_loc":start0,"children":[],"level":0}
        check_added_phrases=self.add_phrase3(cur_phrase_obj)

        #print(check_added_phrases,"Projecting:",cur_phrase_obj)
        new_phrases=self.project_phrase3(cur_phrase_obj)
        # for a in new_phrases:
        #   print("NEW:",a)
        all_token_new_phrases.extend(new_phrases)
      while all_token_new_phrases:
        all_token_new_phrases_copy=copy.deepcopy(all_token_new_phrases)
        all_token_new_phrases=[]
        for ph0 in all_token_new_phrases_copy:
          check_added_phrases=self.add_phrase3(ph0)
          #print("recursive check/project:", check_added_phrases,ph0)
          if check_added_phrases==False: continue
          projected_new_phrases=self.project_phrase3(ph0)

          all_token_new_phrases.extend(projected_new_phrases)



    final_parse_phrases=[]
    final_raw_parses=[]
    end_of_sentence_dict=self.end_start_phrase_dict.get(len(tokens)-1,{})

    full_span_sub_dict=end_of_sentence_dict.get(0,{}) #first getting the parses that span the full sentence
    for cat0,phrases0 in full_span_sub_dict.items():
      final_parse_phrases.extend(phrases0)
      for ph0 in phrases0:
        dep0,const0=self.export_parse2(tokens,ph0,True)
        final_raw_parses.append((ph0,dep0,const0))

    #full_span_phrases=end_of_sentence_dict.get(0,[])

    #combine_exported_deps(words,dep_list)
    djk_phrases=self.optimum_path(tokens,self.span_phrase_dict) #then get djk optimum path phrases
    if djk_phrases:
      temp_dep_list=[]
      combined_const=[]
      for djk_ph0 in djk_phrases:
        dep0,const0=self.export_parse2(tokens,djk_ph0,False)
        temp_dep_list.append(dep0)
        combined_const.extend(const0)
      combined_djk_phrase=self.combine_phrases(djk_phrases,{}) #combine all phrases on djk path into one phrase
      combined_djk_phrase["cat"]="DJK-combined"
      self.combined_dep=combine_exported_deps(tokens,temp_dep_list)
      final_raw_parses.append((combined_djk_phrase,self.combined_dep,combined_const))

    #finally include additional top phrases based on weight and span
    span_wt_items=list(self.span_wt_dict.items())
    span_wt_items.sort(key=lambda x:-x[-1])

    #create a parse of multiple phrases by combining the highest phrases with non-overlapping spans
    #basically combining fragmented phrases
    if self.combined_fragmented:
      valid_sorted_span_phrases=[]
      used_indexes=[]
      for span0,wt0 in span_wt_items:
        i0,i1=span0
        if i0==i1: continue
        cur_span_indexes=list(range(i0,i1+1))
        overlap_with_used=list(set(used_indexes).intersection(set(cur_span_indexes)))
        valid=False
        if overlap_with_used==[]:
          used_indexes.extend(cur_span_indexes)
          valid=True
        if valid:
          ph0=self.span_phrase_dict.get(span0)
          if ph0!=None: valid_sorted_span_phrases.append(ph0)

      if valid_sorted_span_phrases:
        combined_phrase=self.combine_phrases(valid_sorted_span_phrases)
        combined_phrase["cat"]="COMBINED"
        dep0,const0=self.export_parse2(tokens,combined_phrase,True)
        final_raw_parses.append((ph0,dep0,const0))

    cur_max=max(0,self.max_n_phrases-len(final_raw_parses))  #remaining number of parses till we get to the maximum n_parses/prases per span
    for span0,wt0 in span_wt_items[:cur_max]:
      ph0=self.span_phrase_dict.get(span0)
      if ph0==None: continue
      dep0,const0=self.export_parse2(tokens,ph0,True)
      final_raw_parses.append((ph0,dep0,const0))

    temp_wt_score_parse_list=[]
    for ph0,dep0,const0 in final_raw_parses:
      ph_wt0=ph0["wt"]
      dep_score0=self.score_dep_obj(dep0)
      temp_wt_score_parse_list.append((ph0,dep0,const0,ph_wt0,dep_score0))

    temp_wt_score_parse_list.sort(key=lambda x:(-x[-1],-x[-2])) #sort by the dependency score then the top phrase weight
    final_sorted_parses=[v[:3] for v in temp_wt_score_parse_list]

    outcome_dict={}
    outcome_dict["pos"]=self.tag_wt_list
    outcome_dict["parses"]=final_sorted_parses

    return outcome_dict

  def score_dep_obj(self,dep_obj,rerank_params=None):
    if rerank_params==None: cur_rerank_params=rerank_params
    else: cur_rerank_params=self.rerank_params
    n_heads=0
    n_unparsed=0
    for d0 in dep_obj:
      cur_head=d0["head"]
      if cur_head=="0": n_heads+=1
      if cur_head=='-' or cur_head=="": n_unparsed+=1
    multi_head_penalty=n_heads-1 #if we have only one head, penalty is zero
    multi_head_unparsed_score=(len(dep_obj)-multi_head_penalty-n_unparsed) /len(dep_obj)
    dep_score0=multi_head_unparsed_score
    return dep_score0




  def project_phrase3(self,phrase_obj0): #only for maximum binary branching
    #print("projecting:",phrase_obj0)
    new_level=phrase_obj0.get("level",0)+1
    wt_inc=new_level*0.00001 #weight increment - to give a higher weight for the higher level projection phrases
    #wt_inc=0

    #phrase_obj0["level"]=phrase_obj0.get("level",0)+1
    cur_phrase_key=self.get_phrase_key(phrase_obj0)
    self.projection_counter[cur_phrase_key]=self.projection_counter.get(cur_phrase_key,0)+1
    if self.projection_counter.get(cur_phrase_key,0)>10: return []
    cur_start0=phrase_obj0["start"]


    cur_rules_numbers=self.get_rules(phrase_obj0)
    cur_rules_obj_list=[self.all_processed_rules[r_i] for r_i in cur_rules_numbers]
    cur_rules_obj_list.sort(key=lambda x:len(x["children"]))
    all_combined_phrases=[]
    for cur_rule_obj in cur_rules_obj_list:

      cur_rule_children=cur_rule_obj["children"]
      first_child,last_child=cur_rule_children[0],cur_rule_children[-1] #unary or binary rules
      first_child_cat=first_child["cat"] #we will only use categories for first cat now, but later we can do features

      match_last=match_rule(phrase_obj0,last_child)
      if not match_last: continue
      if len(cur_rule_children)==1:
        new_combined_phrase=self.combine_phrases([phrase_obj0],cur_rule_obj)
        new_combined_phrase["level"]=new_level
        new_combined_phrase["wt"]+=wt_inc
        all_combined_phrases.append(new_combined_phrase)
        #print("new_combined_phrase",new_combined_phrase)
      else:
        min_prev_end=max(0,cur_start0-self.max_skip_distance)
        for temp_end0 in range(min_prev_end,cur_start0):
          corr_start_dict=self.end_start_phrase_dict.get(temp_end0,{})
          for temp_start0,temp_start_sub_dict0 in corr_start_dict.items():
            temp_phrases=temp_start_sub_dict0.get(first_child_cat,[])
            for tph0 in temp_phrases:
              new_combined_phrase=self.combine_phrases([tph0,phrase_obj0],cur_rule_obj)
              #we want to monitor projection from same rule e.g. VP --> VP^ PP or NP --> NP NP^
              #and limit having so many projections from the same head phrase according to the same rule
              # new_combined_phrase_head_key=new_combined_phrase["head_phrase"] #checking the key/phrase of the new combined phrase
              # new_combined_phrase_head_phrase=self.phrase_key_obj_dict.get(new_combined_phrase_head_key)
              # if new_combined_phrase_head_phrase==None: continue
              # new_combined_phrase_head_phrase_rule=new_combined_phrase_head_phrase.get("rule")
              # same_rule_proj_level=new_combined_phrase.get("same_rule_proj_level",0)
              # if new_combined_phrase_head_phrase_rule==cur_rule_obj:  same_rule_proj_level+=1
              # new_combined_phrase["same_rule_proj_level"]=same_rule_proj_level

              new_combined_phrase["level"]=new_level
              new_combined_phrase["wt"]+=wt_inc
              all_combined_phrases.append(new_combined_phrase)

    return all_combined_phrases



  def export_parse2(self,words,cur_phrase,export_full_sent=True):
    start0,end0=cur_phrase["start"],cur_phrase["end"]
    phrase_key0=self.get_phrase_key(cur_phrase)
    # print(cur_phrase)
    # return
    #phrase_head_loc_dict={cur_phrase["i"]: cur_phrase["head_loc"]} #identify the head location for each phrase obj/key is phrase number/phrase index in the phrase list
    phrase_head_loc_dict={phrase_key0: cur_phrase["head_loc"]} #identify the head location for each phrase obj/key is phrase number/phrase index in the phrase list

    dep_dict={} #identify which phrase head loc depends on which phrase head loc
    root_index=cur_phrase["head_loc"] #the root of the current phrase

    cur_tokens_xpos_dict={}
    #unsorted_phrases=sorted(final_phrases,key=lambda x:x["i"]) #they should be sorted anyway by their original order of being added to the list
    parse_phrases=[cur_phrase] #these are the phrases to be used for constituency parsing
    children=cur_phrase.get("children",[])
    new_children=[]

    while children: #go on a semi-recursive way to identify children and subchildren of current main phrase
      for ch0 in children:
        # ch_phrase_key=self.phrase_key_list[ch0]
        # child_phrase=self.main_phrase_dict[ch_phrase_key][0]

        ch_phrase_key=ch0
        #child_phrase=self.phrase_key_obj_dict[ch_phrase_key]
        child_phrase=self.phrase_key_obj_dict.get(ch_phrase_key)
        if child_phrase==None:
          start,end,cat,feat=ch_phrase_key
          span=end-start+1
          child_phrase={"wt":0, "start":start,"end":end,"cat":cat, "feat":feat.split(), "head_loc":start,"children":[],"span":span}
        phrase_head_loc_dict[ch0]=child_phrase["head_loc"]
        if child_phrase["span"]==1 and child_phrase["children"]==[]: #identify the matching XPOS for terminal phrases
          cur_tokens_xpos_dict[child_phrase["start"]]=child_phrase["cat"]
        parse_phrases.append(child_phrase)
        sub_children=child_phrase.get("children",())
        sub_children=list(sub_children)
        for sc in sub_children:
          if sc==ch0: continue
          new_children.append(sc) #print("sc",sc)
      children=new_children
      new_children=[]

    for a in parse_phrases:
      cur_children=a["children"]
      cur_phrase_head_loc=a["head_loc"]
      for ch0 in cur_children: #for each child of the current phrase, identify its head_loc
        ch_head_loc=phrase_head_loc_dict[ch0]
        if ch_head_loc==cur_phrase_head_loc: continue #skip if the head_loc for current child is the same as of the main phrase
        dep_dict[ch_head_loc]=cur_phrase_head_loc #update dependency dict between heads of two phrases

    final_dep_list=[]
    for i0,w0 in enumerate(words): #create dependency info for each word
      cur_id0=i0+1
      if not export_full_sent: #export only the tokens and dependency info within the span of the exported phrase
        if i0<start0 or i0>end0: continue

      #identify head index
      if i0==root_index: #root word
        cur_dep="0"
        head_word="^"
        offset=0
      else:
        cur_dep=dep_dict.get(i0)
        if cur_dep==None:  #word without attachment
          cur_dep="-"
          head_word=""
          offset=100 #any random number for no offset
        else: #any other word
          head_word=words[cur_dep]
          offset=i0-cur_dep
          cur_dep=str(cur_dep+1)

      cur_xpos0=cur_tokens_xpos_dict.get(i0,"-")
      final_dep_list.append({"id":str(cur_id0),"word": w0,"head":cur_dep,"xpos":cur_xpos0,"head_word":head_word,"offset":offset})
    return final_dep_list, parse_phrases

  def add_phrase3(self,phrase_obj0): #only for maximum binary branching
    span0=start0,end0=phrase_obj0["start"],phrase_obj0["end"]
    wt0=phrase_obj0["wt"]
    found_span_wt0=self.span_wt_dict.get(span0,0)
    # phrase_has_top_span_wt=False

    cat0,children0=phrase_obj0["cat"],phrase_obj0["children"]

    phrase_key0=self.get_phrase_key(phrase_obj0)
    self.phrase_key_obj_dict[phrase_key0]=phrase_obj0


    used_phrase_key_dict={}


    temp_end_sub_dict=self.end_start_phrase_dict.get(end0,{}) #get the sub dict for phrases having this end loc
    temp_start_sub_dict2=temp_end_sub_dict.get(start0,{}) #get the subdict with the start loc
    cur_phrases=temp_start_sub_dict2.get(cat0,[]) #get the list of phrases with the [end][start][cat] keys
    if cur_phrases==[]:
      cur_phrases=[phrase_obj0]
    else:
      cur_top_phrase,cur_bottom_phrase=cur_phrases[0],cur_phrases[-1]
      if len(cur_phrases)>=self.max_n_phrases and wt0<cur_bottom_phrase["wt"]:
        #print("1","cur_phrases",len(cur_phrases))
        return False
      for cur_ph0 in cur_phrases:
        cur_ph0_cat,cur_ph0_children=cur_ph0["cat"],cur_ph0["children"]


        # if cur_ph0_cat==cat0 and cur_ph0_children==children0 and len(children0)>1:
        #   #print("2","cur_ph0_cat",cur_ph0_cat,"cat0",cat0)
        #   return False #and wt0<=cur_top_phrase["wt"]: return False


      cur_phrases+=[phrase_obj0]
      cur_phrases.sort(key=lambda x:-x["wt"])

    temp_start_sub_dict2[cat0]=cur_phrases[:self.max_n_phrases]
    temp_end_sub_dict[start0]=temp_start_sub_dict2
    self.end_start_phrase_dict[end0]=temp_end_sub_dict

    if cur_phrases[0]["wt"]>found_span_wt0:
      self.span_wt_dict[span0]=cur_phrases[0]["wt"]
      self.span_phrase_dict[span0]=cur_phrases[0]


    head_loc0=phrase_obj0["head_loc"]
    head_loc_cat_pair=(head_loc0,cat0)
    updated_head_loc_cat_proj_level=self.head_loc_cat_projection_counter.get(head_loc_cat_pair,0)+1
    #if updated_head_loc_cat_proj_level>self.max
    self.head_loc_cat_projection_counter[head_loc_cat_pair]=updated_head_loc_cat_proj_level


    return True


  def combine_phrases(self,phrase_obj_list,applied_rule={}):
    parent_phrase_obj={}
    #cur_child_keys=[v["i"] for v in phrase_obj_list]
    cur_child_keys=[self.get_phrase_key(v) for v in phrase_obj_list]

    levels=[vw["level"] for vw in phrase_obj_list]

    parent_phrase_obj["wt"]=sum([vw["wt"] for vw in phrase_obj_list])
    phrase_start0=min([vw["start"] for vw in phrase_obj_list])
    phrase_end0=max([vw["end"] for vw in phrase_obj_list])

    parent_phrase_obj["span"]=phrase_end0-phrase_start0+1
    parent_phrase_obj["start"]=phrase_start0
    parent_phrase_obj["end"]=phrase_end0

    if applied_rule!={}:
      parent_phrase_obj["cat"]=applied_rule["parent"]["cat"]
      parent_phrase_obj["feat"]=applied_rule["parent"]["feat"]
      rule_head_i=applied_rule["head_i"]
    else:
      parent_phrase_obj["cat"]=""
      parent_phrase_obj["feat"]=[]
      rule_head_i=0




    parent_phrase_obj["children"]=cur_child_keys
    head_phrase_index=cur_child_keys[rule_head_i] #which of the child phrases is the head

    parent_phrase_obj["head_phrase"]=head_phrase_index
    # head_phrase_key=self.phrase_key_list[head_phrase_index]
    head_phrase_obj=phrase_obj_list[rule_head_i]  #phrase object of the head child


    #11 Feb 2026
    if applied_rule!={}:
      head_child_cat0=head_phrase_obj["cat"]
      applied_rule_parent_obj=applied_rule["parent"]
      percolate=applied_rule_parent_obj.get("percolate",False)
      apply_cat=applied_rule_parent_obj.get("apply_cat",False)
      if percolate: parent_phrase_obj["feat"]=head_phrase_obj["feat"]
      if apply_cat: parent_phrase_obj["feat"]+=[f"cat={head_child_cat0}"] #we may need to adjust later, to accommdate multiple features + applied category of head child



    parent_phrase_obj["head_loc"]=head_phrase_obj["head_loc"]

    parent_phrase_obj["rule"]=applied_rule

    parent_phrase_obj["level"]=max(levels)+1

    return parent_phrase_obj
    #parent_phrase_obj["level"]=new_level



  def get_phrase_key(self,phrase_obj0): #check an obj for corresponding key
    start0,end0,cat0,feat_split0=phrase_obj0["start"],phrase_obj0["end"],phrase_obj0["cat"],phrase_obj0["feat"]
    ft_str0=" ".join(feat_split0)
    cur_phrase_key=(start0,end0,cat0,ft_str0)
    return cur_phrase_key


  def get_rules(self,cur_phrase_obj):
    #given a token or phrase with cat/feat pair, identify the rules that apply
    all_rules=[]
    phrase_obj_cat,phrase_obj_feat=cur_phrase_obj["cat"],cur_phrase_obj["feat"]
    cur_cat_rules=self.cat_inv_index.get(phrase_obj_cat,[])
    all_rules.extend(cur_cat_rules)
    for s_feat0 in phrase_obj_feat:
      corr_rules=self.feat_inv_index.get(s_feat0,[])
      all_rules.extend(corr_rules)
    return list(set(all_rules))

  def get_children(self,phrase_obj):
    all_phrases=[phrase_obj]
    children=phrase_obj.get("children",[])
    new_children=[]
    while children: #go on a semi-recursive way to identify children and subchildren of current main phrase
      for ch0 in children:
        ch_phrase_key=self.phrase_key_list[ch0]
        child_phrase=self.main_phrase_dict[ch_phrase_key][0]
        all_phrases.append(child_phrase)
        sub_children=child_phrase.get("children",())
        sub_children=list(sub_children)
        for sc in sub_children:
          if sc==ch0: continue
          new_children.append(sc) #print("sc",sc)
      children=new_children
      new_children=[]
    return all_phrases
  def inspect_span(self,start,end,cat=None):
    end_temp_dict=self.end_start_phrase_dict.get(end,{})
    final_temp_dict=end_temp_dict.get(start,{})
    if cat!=None: final_temp_dict=final_temp_dict.get(cat,{})
    return final_temp_dict


  def optimum_path(self,words,span_phrase_dict): #apply Djkestra's algorithm to get the optimum path of phrases
    inc0=self.djk_max_span_distance
    dist_penalty0=self.djk_distance_penalty_ratio
    use_negative=self.djk_use_negative_wt

    span_list=sorted(list(span_phrase_dict.keys()))
    span_list=[v for v in span_list if (v[1]-v[0])>0] #only choose multi word phrases
    first_pt0=(-1,-1)
    last_pt0=(len(words),len(words))
    span_list=[first_pt0]+span_list+[last_pt0]
    grouped=[(key,list(group)) for key,group in groupby(span_list,lambda x:x[0])]
    grouped_dict=dict(iter(grouped))


    transition_dict={}
    for span0 in span_list:
      span0_start,span0_end=span0
      transition_dict[span0]={}
      all_corr_spans=[]
      for span1_start in range(span0_end+1,span0_end+inc0+1):
        corr_spans=grouped_dict.get(span1_start,[])
        span_dist=span1_start-span0_end-1
        cur_dist_penalty=span_dist*dist_penalty0
        for span1 in corr_spans:
          cur_phrase=span_phrase_dict.get(span1,{})
          span1_wt=cur_phrase.get("wt",0)
          adj_wt=span1_wt-cur_dist_penalty
          if use_negative: adj_wt=-adj_wt
          transition_dict[span0][span1]=adj_wt
      if transition_dict[span0]=={}: transition_dict[span0]={last_pt0:0}

    path0,path_wt0=general.djk(first_pt0,last_pt0,transition_dict,pt_list0=span_list)

    final_top_phrases=[]
    for span0 in path0:
      corr_phrase=span_phrase_dict.get(span0)
      if corr_phrase!=None: final_top_phrases.append(corr_phrase)
    return final_top_phrases

TypeError: list indices must be integers or slices, not str

In [ ]:
ph_obj0

'p'

In [ ]:
final_parses

{'pos': [('5', [('CD', ['noun'], 0.9987455606460571)]),
  ('star',
   [('NN', ['noun'], 0.9999191761016846),
    ('VB', ['verb'], 0.9997699856758118)]),
  ('detail',
   [('NN', ['noun'], 0.9999998807907104),
    ('VB', ['verb'], 0.9999982118606567)]),
  ('job', [('NN', ['noun'], 0.9999992847442627)])],
 'parses': [({'wt': 4.000863902282715,
    'span': 4,
    'start': 0,
    'end': 3,
    'cat': 'S_Q/BE',
    'feat': [],
    'children': [(0, 2, 'NP', ''), (3, 3, 'NP', '')],
    'head_phrase': (3, 3, 'NP', ''),
    'head_loc': 3,
    'rule': {'parent': {'cat': 'S_Q/BE',
      'feat': [],
      'percolate': False,
      'apply_cat': False},
     'children': [{'cat': 'NP', 'is_head': False, 'feat': []},
      {'cat': 'NP', 'is_head': True, 'feat': []}],
     'head_i': 1,
     'rule_i': 185},
    'level': 12},
   [{'id': '1',
     'word': '5',
     'head': '4',
     'xpos': '-',
     'head_word': 'job',
     'offset': -3},
    {'id': '2',
     'word': 'star',
     'head': '3',
     'xpos':

In [ ]:
final_processed_rules

[{'parent': {'cat': 'DT', 'feat': [], 'percolate': True, 'apply_cat': False},
  'children': [{'cat': 'X', 'is_head': False, 'feat': ['det']}],
  'head_i': 0,
  'rule_wt': 1,
  'deprel': 'dep',
  'description': 'determiner, possessive pronoun',
  'example': '[the] car | [my] car'},
 {'parent': {'cat': 'NP-U', 'feat': [], 'percolate': True, 'apply_cat': False},
  'children': [{'cat': 'X', 'is_head': False, 'feat': ['noun']}],
  'head_i': 0,
  'rule_wt': 1,
  'deprel': 'dep',
  'description': 'NP without determiner',
  'example': ''},
 {'parent': {'cat': 'AP', 'feat': [], 'percolate': True, 'apply_cat': False},
  'children': [{'cat': 'X', 'is_head': False, 'feat': ['adj']}],
  'head_i': 0,
  'rule_wt': 1,
  'deprel': 'dep',
  'description': 'Adjectival phrase',
  'example': ''},
 {'parent': {'cat': 'MD', 'feat': [], 'percolate': True, 'apply_cat': False},
  'children': [{'cat': 'X', 'is_head': False, 'feat': ['modal']}],
  'head_i': 0,
  'rule_wt': 1,
  'deprel': 'dep',
  'description': '

In [ ]:
from code_utils.parsing_lib import *

#categories that we replace XP - we may need to update them
#categories that can be conjugated with AND and comma sequences
xp_cat_list=["V","VP","NP",'NP-DT',"NP-U","S","S-U","AdvP","AP","V-PAS","PP",'VP-O','TO-VP','VP-2O',"CP"]

rule0='S-U^ --> NP VP^'
rule0='S_Q_WH/WH^[apply_cat] --> BE S_Q_WH/BE^'
process_rule(rule0)
final_rules=[]
xp_rule_dict_list=[]
all_cat_list=[]
for r0 in rules_sheet:
  cur_rule0=r0["rule"].strip()
  if not cur_rule0: continue
  cur_wt_str=r0["weight"].strip()
  cur_deprel=r0["deprel"].strip()
  cur_description=r0["description"].strip()
  cur_example=r0["example"].strip()

  if cur_deprel=="": cur_deprel="dep"
  if cur_wt_str=="": cur_wt=1
  else: cur_wt=float(cur_wt_str)
  r0["rule_wt"]=cur_wt
  r0["deprel"]=cur_deprel
  #print(r0)
  cur_rule_str_list=[]
  #replace {XP} with relevant categories till we improve the algorithm with regex style synamic matching
  if "{XP}" in cur_rule0:
    for repl_cat0 in xp_cat_list:
      replaced_rule0=cur_rule0.replace("{XP}",repl_cat0)
      cur_rule_str_list.append(replaced_rule0)
  else: cur_rule_str_list.append(cur_rule0)

  #if "{XP}" in cur_rule0: xp_rule_dict_list.append(r0)
  for cur_rule1 in cur_rule_str_list:
    print(cur_rule1)
    processed_rule=process_rule(cur_rule1)
    if processed_rule=={}: continue
    processed_rule["rule_wt"]=cur_wt
    processed_rule["deprel"]=cur_deprel
    processed_rule["description"]=cur_description
    processed_rule["example"]=cur_example

    print(processed_rule)
    final_rules.append(processed_rule)

    parent_cat=processed_rule["parent"]["cat"]
    rule_children_cat=[v["cat"] for v in  processed_rule["children"]]
    rule_children_is_head=[v["is_head"] for v in  processed_rule["children"]]
    #check if any rule has more than or less than one head
    if len(rule_children_is_head)>1 and rule_children_is_head.count(True)!=1: print("<<<<<<<<<<<<<<<")
    all_rule_cat=rule_children_cat+[parent_cat]
    all_cat_list.extend(all_rule_cat)
    print("--------")

  #print(cur_rule0)
all_cat_list=list(set(all_cat_list))

#cur_cat_list=['VBP', 'CP', 'NP-U', 'V-PAS', 'AdvP', 'NP-U3', 'S_Q_WH/BE', 'S/NP', 'NP-CP', 'S_Q_WH/DO_MOD', 'IN', 'S_Q', 'VB', 'WP', 'NP-DT', 'S_Q/BE', 'COND1', 'LRB_NP_U', 'S-U', 'V-PAS-1O', 'VP-2O/NP', 'MD', 'APO', 'X', 'V', 'NP-DASH', 'VBG', 'VBN', 'S_Q_WH/WH', 'NP-U2', 'MOD', '-RRB-', 'SCONJ', 'VP-O/NP', 'BEEN', 'AUX', 'DOT', 'VP-O', 'PP/NP', 'S', 'HAVE', 'VBD', 'AP', 'COMMA', 'NP', 'NP-BR', 'VP/NP', 'RB', 'Q_MARK', 'S_Q_YN', 'PP', 'HYPHEN', 'VBZ', 'WRB', 'NP-LB', 'BE', 'S_COMMA', 'TO', 'COLON', 'DO', 'TO-VP', 'LRB_S', 'VP-2O', '-LRB-', 'DT', 'VP']

#and_cat_list=["V","VP","NP",'NP-DT',"NP-U","S","S-U","AdvP","AP","V-PAS","PP",'VP-O','TO-VP','VP-2O',"CP"]
print("final_rules",len(final_rules))


word_features_list=rw_data_dict["words-features"]
upos_sheet_rows=rw_data_dict["upos"]
xpos_sheet_rows=rw_data_dict["xpos"]

upos_ft_dict={}
xpos_ft_dict={}
for a in  upos_sheet_rows:
  tag0,ft0=a["tag"],a["features"]
  upos_ft_dict[tag0]=ft0.strip().split()
upos_list=sorted(list(upos_ft_dict.keys()))

for a in  xpos_sheet_rows:
  #print(a)
  tag0,ft0,freq0=a["tag"],a["features"],int(a["freq"])
  if "-" in tag0 and not tag0.startswith("-"): continue
  if freq0<10: continue
  xpos_ft_dict[tag0]=ft0.strip().split()
xpos_list=sorted(list(xpos_ft_dict.keys()))

print("upos_list",len(upos_list), upos_list)
print("xpos_list",len(xpos_list), xpos_list)
word_features_list.sort(key=lambda x:x["word"])
word_features_list_filtered=[]
for a in word_features_list:
  if a["category"]=="": continue
  obj_copy0=copy.deepcopy(a)
  word0=a["word"].strip()
  cat0=a["category"]
  if word0.startswith("\\") and word0!="\\": word0=word0.replace("\\","")
  obj_copy0["word"]=word0
  word_features_list_filtered.append(obj_copy0)
  #print(word0,cat0,word0.startswith("\\"))
word_features_list_grouped=[(key,[(v["category"],v["features"].strip().split()) for v in list(group)]) for key,group in groupby(word_features_list_filtered,lambda x:x["word"])]
final_word_features_dict=dict(iter(word_features_list_grouped))


final_parsing_data_dict={}
final_parsing_data_dict["final_word_features_dict"]=final_word_features_dict
final_parsing_data_dict["upos_list"]=upos_list
final_parsing_data_dict["xpos_list"]=xpos_list
final_parsing_data_dict["xpos_ft_dict"]=xpos_ft_dict
final_parsing_data_dict["upos_ft_dict"]=upos_ft_dict
final_parsing_data_dict["final_rules"]=final_rules

with open(info_fpath,"w") as info_fopen:
  json.dump(final_parsing_data_dict,info_fopen)

print("final_rules", len(final_rules))



DT^ --> X[det]
{'parent': {'cat': 'DT', 'feat': [], 'percolate': True, 'apply_cat': False}, 'children': [{'cat': 'X', 'is_head': False, 'feat': ['det']}], 'head_i': 0, 'rule_wt': 1, 'deprel': 'dep', 'description': 'determiner, possessive pronoun', 'example': '[the] car | [my] car'}
--------
NP-U^ --> X[noun]
{'parent': {'cat': 'NP-U', 'feat': [], 'percolate': True, 'apply_cat': False}, 'children': [{'cat': 'X', 'is_head': False, 'feat': ['noun']}], 'head_i': 0, 'rule_wt': 1, 'deprel': 'dep', 'description': 'NP without determiner', 'example': ''}
--------
AP^ --> X[adj]
{'parent': {'cat': 'AP', 'feat': [], 'percolate': True, 'apply_cat': False}, 'children': [{'cat': 'X', 'is_head': False, 'feat': ['adj']}], 'head_i': 0, 'rule_wt': 1, 'deprel': 'dep', 'description': 'Adjectival phrase', 'example': ''}
--------
MD^ --> X[modal]
{'parent': {'cat': 'MD', 'feat': [], 'percolate': True, 'apply_cat': False}, 'children': [{'cat': 'X', 'is_head': False, 'feat': ['modal']}], 'head_i': 0, 'rule_wt

#Data Processing

##Load Parsing Data
From [spreadsheet](https://docs.google.com/spreadsheets/d/1ejt5G9_XgrCBRZFz_8J9mSOjk8UPnzmsmaG6FiWuJsQ/edit?usp=sharing) of rules, tags, and word features

In [ ]:
from code_utils.pandas_utils import *
from itertools import groupby
import json, copy

#file_id="feb_2026.pp_brackets"
#file_id="feb_2026.pp_brackets_adv"
#file_id="feb_2026.s_comma.1"
#file_id="feb_2026.s_comma.1.be"
#file_id="feb_2026.s_comma.1.be0" #also NP/AP
#file_id="feb_2026.s_comma.1.be_rb" #also NP/AP - RB
#file_id="feb_2026.s_comma.1.be_rb_do" #also NP/AP - RB - VB with DO
# file_id="feb_2026.s_comma.1.be_rb_do_wp" #also NP/AP - RB - VB with DO - WP
# file_id="feb_2026.s_comma.1.be_rb_do_wp_single_quote" #also NP/AP - RB - VB with DO - WP- single quote

#file_id="feb_2026.s_comma.1.be_rb_do_wp_single_quote_fixed" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.s_comma.2.np_colon" #also NP/AP - RB - VB with DO - WP- single quote

#file_id="feb_2026.s_comma.2.np_colon_md_be" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.s_comma.2.np_colon_md_be_have_vbn" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.s_comma.2.np_colon_md_be_have_vbn_ap_to" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.s_comma.2.np_colon_md_be_have_vbn_ap_to_double_quote" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.s_comma.2.np_colon_md_be_have_vbn_ap_to_double_quote_vb_rb" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.s_comma.2.np_colon_md_be_have_vbn_ap_to_double_quote_vb_rb_s_init" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.3.questions" #also NP/AP - RB - VB with DO - WP- single quote

#file_id="feb_2026.3.TEST" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.3.TEST1" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.3.TEST2" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.3.existing_questions" #also NP/AP - RB - VB with DO - WP- single quote

#file_id="feb_2026.3.existing_questions_v_pas" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.3.existing_questions_v_pas_vbn" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.3.existing_questions_v_pas_vbn_q" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.3.existing_questions_v_pas_vbn_q_inv_ap" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.3.existing_questions_v_pas_vbn_vbg_q_test" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.3.existing_questions_v_pas_vbn_vbg_q_test2" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.3.existing_questions_v_pas_vbn_vbg_vb_q" #also NP/AP - RB - VB with DO - WP- single quote

#file_id="feb_2026.3.existing_questions_v_pas_vbn_vbg_vb_q_be_ap" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.3.existing_questions_v_pas_vbn_vbg_vb_q_be_np_ap_pp" #also NP/AP - RB - VB with DO - WP- single quote

file_id="feb_2026.3.existing_questions_v_pas_vbn_vbg_vb_q_be_np_ap_pp_np_dash_advp_vp_ap" #also NP/AP - RB - VB with DO - WP- single quote

file_id="submission_model_feb_17"

file_id="syntactic_gap_experiment"

# file_id="jan_2026.pp"
# file_id="jan_2026.0"
# file_id="jan_2026"
# file_id="feb_2026"
info_fname=f"rules_word_features_{file_id}.json"

rules_dir="../data/rules"
if not os.path.exists(rules_dir): os.makedirs(rules_dir)
info_fpath=os.path.join(rules_dir,info_fname)

rules_workbook_url="https://docs.google.com/spreadsheets/d/e/2PACX-1vQWwi-tdS4z1oXN3fELGqBSh6elDa2OtTIh7zFLZfufOON2PYIg2QBd9avPdXz2g3CwpmCj-tNV76Qn/pub?output=xlsx"
rw_data_dict=get_wb_data_dict(rules_workbook_url)

rw_data_dict.keys()
#rules_sheet=rw_data_dict["short-list"]
rules_sheet=rw_data_dict["rule-list-new"]

rules_sheet_v0=rw_data_dict["rules_v0"]

word_features_list=rw_data_dict["words-features"]
upos_sheet_rows=rw_data_dict["upos"]
xpos_sheet_rows=rw_data_dict["xpos"]

final_rules=[]
for r0 in rules_sheet:
  cur_rule=r0["Rule"].strip()
  if not cur_rule: continue
  final_rules.append(cur_rule)

final_rules_v0=[]
for r0 in rules_sheet_v0:
  cur_rule=r0["Rule"].strip()
  if not cur_rule: continue
  final_rules_v0.append(cur_rule)


upos_ft_dict={}
xpos_ft_dict={}
for a in  upos_sheet_rows:
  tag0,ft0=a["tag"],a["features"]
  upos_ft_dict[tag0]=ft0.strip().split()
upos_list=sorted(list(upos_ft_dict.keys()))

for a in  xpos_sheet_rows:
  #print(a)
  tag0,ft0,freq0=a["tag"],a["features"],int(a["freq"])
  if "-" in tag0 and not tag0.startswith("-"): continue
  if freq0<10: continue
  xpos_ft_dict[tag0]=ft0.strip().split()
xpos_list=sorted(list(xpos_ft_dict.keys()))

print("upos_list",len(upos_list), upos_list)
print("xpos_list",len(xpos_list), xpos_list)
word_features_list.sort(key=lambda x:x["word"])
word_features_list_filtered=[]
for a in word_features_list:
  if a["category"]=="": continue
  obj_copy0=copy.deepcopy(a)
  word0=a["word"].strip()
  cat0=a["category"]
  if word0.startswith("\\") and word0!="\\": word0=word0.replace("\\","")
  obj_copy0["word"]=word0
  word_features_list_filtered.append(obj_copy0)
  #print(word0,cat0,word0.startswith("\\"))
word_features_list_grouped=[(key,[(v["category"],v["features"].strip().split()) for v in list(group)]) for key,group in groupby(word_features_list_filtered,lambda x:x["word"])]
final_word_features_dict=dict(iter(word_features_list_grouped))


final_parsing_data_dict={}
final_parsing_data_dict["final_word_features_dict"]=final_word_features_dict
final_parsing_data_dict["upos_list"]=upos_list
final_parsing_data_dict["xpos_list"]=xpos_list
final_parsing_data_dict["xpos_ft_dict"]=xpos_ft_dict
final_parsing_data_dict["upos_ft_dict"]=upos_ft_dict
final_parsing_data_dict["final_rules"]=final_rules
final_parsing_data_dict["final_rules_v0"]=final_rules_v0

with open(info_fpath,"w") as info_fopen:
  json.dump(final_parsing_data_dict,info_fopen)

print("final_rules", len(final_rules))


upos_list 17 ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']
xpos_list 117 ['!', '"', '$', "'", ',', '-LRB-', '-RRB-', '.', ':', '?', 'A', 'ADD', 'ADJ', 'AFX', 'AP', 'B', 'BN', 'CC', 'CD', 'CMP', 'CS', 'Colon', 'Comma', 'DD', 'DEF', 'DI', 'DT', 'Dash', 'Dots', 'E', 'EX', 'ExclMark', 'FB', 'FC', 'FF', 'FS', 'FW', 'GEN', 'GW', 'HYPH', 'ID', 'IMP', 'IN', 'IND', 'INF', 'ING', 'JJ', 'JJR', 'JJS', 'LS', 'LeftParenthesis', 'MD', 'N', 'NEG', 'NFP', 'NN', 'NNP', 'NNPS', 'NNS', 'NO', 'NOM', 'ORD', 'PART', 'PASS', 'PAST', 'PD', 'PDT', 'PE', 'PERF', 'PI', 'PL', 'POS', 'PQ', 'PR', 'PRES', 'PRP', 'PRP$', 'Period', 'QuestionMark', 'Quote', 'RB', 'RBR', 'RBS', 'RD', 'REL', 'RI', 'RP', 'RightParenthesis', 'S', 'SBJ', 'SG', 'SP', 'SPL', 'SW', 'SYM', 'SemiColon', 'T', 'TO', 'TOT', 'UH', 'V', 'VA', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'VM', 'WDT', 'WH', 'WP', 'WP$', 'WRB', 'X', '_', '``']
final_rules 148


In [ ]:
from code_utils.parsing_lib import *
from pprint import pprint
# rule_i=0
# cur_rule0=final_rules [0]
cur_rule0='DT[apply_cat noun] --> X[det]'
for cur_rule0 in final_rules:
  print(cur_rule0)
  processed0=process_rule(cur_rule0)
  pprint(processed0)
  print("------")


DT --> X[det]
{'children': [{'cat': 'X', 'feat': ['det'], 'is_head': False}],
 'head_i': 0,
 'parent': {'apply_cat': False, 'cat': 'DT', 'feat': [], 'percolate': False}}
------
NP-U --> X[noun]
{'children': [{'cat': 'X', 'feat': ['noun'], 'is_head': False}],
 'head_i': 0,
 'parent': {'apply_cat': False, 'cat': 'NP-U', 'feat': [], 'percolate': False}}
------
AP --> X[adj]
{'children': [{'cat': 'X', 'feat': ['adj'], 'is_head': False}],
 'head_i': 0,
 'parent': {'apply_cat': False, 'cat': 'AP', 'feat': [], 'percolate': False}}
------
V[apply_cat] --> X[verb]
{'children': [{'cat': 'X', 'feat': ['verb'], 'is_head': False}],
 'head_i': 0,
 'parent': {'apply_cat': True, 'cat': 'V', 'feat': [], 'percolate': False}}
------
MD --> X[modal]
{'children': [{'cat': 'X', 'feat': ['modal'], 'is_head': False}],
 'head_i': 0,
 'parent': {'apply_cat': False, 'cat': 'MD', 'feat': [], 'percolate': False}}
------
AdvP --> RB
{'children': [{'cat': 'RB', 'feat': [], 'is_head': False}],
 'head_i': 0,
 'parent'

In [ ]:
final_word_features_dict

{'(': [('-LRB-', [])],
 ')': [('-RRB-', [])],
 ',': [('COMMA', [])],
 '-': [('HYPHEN', [])],
 ':': [('COLON', [])],
 '?': [('Q_MARK', [])],
 'I': [('PRON', ['noun', '1st', 'singular', 'subj'])],
 '[': [('-LRB-', [])],
 '\\': [('SLASH', [])],
 '"': [('-LRB-', []), ('-RRB-', [])],
 "'": [('-LRB-', []), ('-RRB-', [])],
 "'d": [('AUX', ['modal'])],
 "'ll": [('AUX', ['modal'])],
 "'m": [('AUX', ['be', '1st', 'present'])],
 "'re": [('AUX', ['be', '3rd', 'plural', 'present'])],
 "'s": [('AUX', ['be', '3rd', 'singular', 'present']),
  ('AUX', ['have']),
  ('APO', [])],
 ']': [('-RRB-', [])],
 'a': [('DT', [])],
 'all': [('RB', []), ('DT', [])],
 'am': [('AUX', ['be', '1st', 'present'])],
 'an': [('DT', [])],
 'and': [('CC', [])],
 'are': [('AUX', ['be', '3rd', 'plural', 'present']),
  ('AUX', ['be', '2nd', 'present'])],
 'as': [('IN', [])],
 'at': [('IN', [])],
 'be': [('AUX', ['be', 'base', 'be_base'])],
 'because': [('SCONJ', [])],
 'been': [('AUX', ['been'])],
 'can': [('AUX', ['modal'])],


In [ ]:
#final_word_features_dict
final_rules
#final_rules_v0

['DT --> X[det]',
 'NP-U --> X[noun]',
 'AP --> X[adj]',
 'V --> X[verb]',
 'MD --> X[modal]',
 'AdvP --> RB',
 'V --> V^ X[adv]',
 'V --> X[adv] V^',
 'VP-O --> V^ NP',
 'VP-2O --> VP-O^ NP',
 'VP --> VP-O',
 'VP --> VP-2O',
 'VP --> V',
 'BE --> AUX[be]',
 'VP --> BE NP^',
 'VP --> BE AP^',
 'VP --> VP^ PP',
 'PP --> IN NP^',
 'NP-DT --> DT NP-U^',
 'NP-U --> AP NP-U^',
 'NP --> NP-U',
 'NP --> NP-DT',
 'NP --> NP^ PP',
 'S-U --> NP VP^',
 'S --> S-U^ .',
 'AND-NP --> CC NP^',
 'NP-CC --> NP^ AND-NP',
 'NP --> NP-CC',
 'NP-COMMA --> NP^ COMMA',
 'NP-CC --> NP-COMMA^ NP-CC',
 'NP-CC --> NP-COMMA^ AND-NP',
 'BEEN --> AUX[been]',
 'HAVE --> AUX[have]',
 'BE --> HAVE BEEN^',
 'BE --> MD AUX[be_base]^',
 'V-PAS --> BE VBN^',
 'VP --> V-PAS',
 'VBN --> AP',
 'VBN --> RB VBN^',
 'COND1 --> WRB S^',
 'COND1 --> COND1^ COMMA',
 'VBG --> RB VBG^',
 'VP-G --> VBG',
 'VP-G-1O --> VP-G^ NP',
 'VP-G --> VP-G-1O',
 'NP-U --> VP-G',
 'V --> BE VBG^',
 'V --> MD VB^',
 'VP-INF --> VB',
 'VP-INF-1O --

In [ ]:
process_rule('S_COMMA --> S-U^ COMMA')

{'parent': {'cat': 'S_COMMA', 'feat': []},
 'children': [{'cat': 'S-U', 'is_head': True, 'feat': []},
  {'cat': 'COMMA', 'is_head': False, 'feat': []}],
 'head_i': 0}

In [ ]:
final_word_features_dict["that"]

[('DT', []), ('C', [])]

##Load UD Treebanks

Download data from [Universal Dependecies](https://universaldependencies.org/download.html)

In [ ]:
#process treebanks for POS info
#also extract dependency pairs to use for reranking
#also extract features
#build terminal rules based on standard/universal POS tags
#adjust the features in our rule tables, according to universal features

def iter_treebank(treebank_fpath):
  fopen=open(cur_train_file)
  cur_conll_str=""
  for i0,line0 in enumerate(fopen):
    if line0.startswith("#"): continue
    if line0.strip()=="":
      yield cur_conll_str
      cur_conll_str=""
    else:
      line_split0=line0.strip().split("\t")
      id0,form0,lemma0,upos0,xpos0,feats0,head0,deprel0,deps0,misc0=line_split0[:10]
      if not id0.isdigit(): continue
      cur_conll_str+=line0
  fopen.close()

word_upos_dict={}
word_xpos_dict={}
word_feat_dict={}

all_xpos_dict={}
all_upos_dict={}

all_xpos_upos_dict={}

ud_dir="../data/UD"
excluded_tb_list=['UD_English-GENTLE']
treebank_dir_list=os.listdir(ud_dir)
for tb0 in treebank_dir_list:
  #if tb0 in excluded_tb_list: continue
  tb_full_path=os.path.join(ud_dir,tb0)
  #print(tb_full_path)
  tb_files=os.listdir(tb_full_path)
  for tbf0 in tb_files:
    if "train" in tbf0.lower() and "conllu" in tbf0.lower():
      cur_train_file=os.path.join(tb_full_path,tbf0)
      print(cur_train_file)
      #
      fopen=open(cur_train_file)
      cur_conll_str=""
      for i0,line0 in enumerate(fopen):
        if line0.startswith("#"): continue
        if line0.strip()=="":
          #print(cur_conll_str)
          #print("======")
          cur_conll_str=""
        else:
          cur_conll_str+=line0
          line_split0=line0.strip().split("\t")
          id0,form0,lemma0,upos0,xpos0,feats0,head0,deprel0,deps0,misc0=line_split0[:10]
          if not id0.isdigit(): continue
          temp_word_upos_dict=word_upos_dict.get(form0,{})
          temp_word_upos_dict[upos0]=temp_word_upos_dict.get(upos0,0)+1
          word_upos_dict[form0]=temp_word_upos_dict

          temp_word_xpos_dict=word_xpos_dict.get(form0,{})
          temp_word_xpos_dict[xpos0]=temp_word_xpos_dict.get(xpos0,0)+1
          word_xpos_dict[form0]=temp_word_xpos_dict

          all_xpos_dict[xpos0]=all_xpos_dict.get(xpos0,0)+1
          all_upos_dict[upos0]=all_upos_dict.get(upos0,0)+1


          temp_all_xpos_upos_dict=all_xpos_upos_dict.get(xpos0,{})
          temp_all_xpos_upos_dict[upos0]=temp_all_xpos_upos_dict.get(upos0,0)+1
          all_xpos_upos_dict[xpos0]=temp_all_xpos_upos_dict

          temp_word_feat_dict=word_feat_dict.get(form0,{})
          temp_word_feat_dict[feats0]=temp_word_feat_dict.get(feats0,0)+1
          word_feat_dict[form0]=temp_word_feat_dict

          #word_feat_dict


          #print(line_split0)
        #print(line0.strip())
        #if i0>300: break
      fopen.close()

# items=list(word_upos_dict.items())
# items.sort(key=lambda x:-sum(x[1].values()))
# for a,b in items[:500]:
#   print(a,b)
#   print(word_xpos_dict[a])
#   print(word_feat_dict[a])
#   print("------")


all_upos_dict_items=list(all_upos_dict.items())
# for a in all_upos_dict_items: print(a)
# print("--------")

import csv
upos_file_path="upos.tsv"
upos_data=[]
#fopen=open("xpos.tsv","w")
# all_xpos_dict_items=list(all_xpos_dict.items())
# all_xpos_dict_items.sort()
for a,b in all_upos_dict_items:
  upos_data.append([a,b])


with open(upos_file_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    # Write all rows at once
    writer.writerows(upos_data)

print(f"CSV file '{upos_file_path}' created successfully using csv.writer().")

#writing xpos data
file_path="xpos.tsv"
xpos_data=[]
#fopen=open("xpos.tsv","w")
all_xpos_dict_items=list(all_xpos_dict.items())
all_xpos_dict_items.sort()
for a,b in all_xpos_dict_items:
  xpos_data.append([a,b])


with open(file_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    # Write all rows at once
    writer.writerows(xpos_data)

print(f"CSV file '{file_path}' created successfully using csv.writer().")

../data/UD/UD_English-EWT/en_ewt-ud-train.conllu
../data/UD/UD_English-CHILDES/en_childes-ud-train.conllu
../data/UD/UD_English-Atis/en_atis-ud-train.conllu
../data/UD/UD_English-ESLSpok/en_eslspok-ud-train.conllu
../data/UD/UD_English-ParTUT/en_partut-ud-train.conllu
../data/UD/UD_English-LinES/en_lines-ud-train.conllu
../data/UD/UD_English-GUMReddit/en_gumreddit-ud-train.conllu
../data/UD/UD_English-GUM/en_gum-ud-train.conllu
CSV file 'upos.tsv' created successfully using csv.writer().
CSV file 'xpos.tsv' created successfully using csv.writer().


##Learn rules from training data

In [ ]:
from code_utils.parsing_lib import *
from code_utils.dep_lib import *

train_files,dev_files,test_files=[],[],[]
ud_dir="../data/UD"
#excluded_tb_list=['UD_English-GENTLE']
treebank_dir_list=os.listdir(ud_dir)
for tb0 in treebank_dir_list:
  tb_full_path=os.path.join(ud_dir,tb0)
  tb_files=os.listdir(tb_full_path)
  for tbf0 in tb_files:
    cur_file=os.path.join(tb_full_path,tbf0)
    if "train" in tbf0.lower() and "conllu" in tbf0.lower(): train_files.append(cur_file)
    elif "dev" in tbf0.lower() and "conllu" in tbf0.lower(): dev_files.append(cur_file)
    elif "test" in tbf0.lower() and "conllu" in tbf0.lower(): test_files.append(cur_file)

###Test on any given sentence

In [ ]:
sent_i=27
file_i=3

n_items=50

cur_fpath=train_files[file_i]
conll_list=[]
for i0,item0 in enumerate(iter_conll_treebank(cur_fpath)):
  conll_list.append(item0)
  if i0>n_items: break
print("cur_fpath",cur_fpath)

cur_conll=conll_list[sent_i]
dep_obj0=conll2obj(cur_conll)

dep_const_obj=dep_const(dep_obj0)
ph_list=dep_const_obj.all_phrases

for ph0 in ph_list:
  if ph0["rule"]=="": continue
  print(ph0["rule"],ph0.get("joined_phrase",""))

print(dep_const_obj.ptb)


cur_fpath ../data/UD/UD_English-ESLSpok/en_eslspok-ud-train.conllu
NP[deprel=amod] --> AP[word=main xpos=JJ] NP[word=point xpos=NN]^ main | ^ point
PRP[deprel=advmod] --> RB[word=just xpos=RB] PRP[word=we xpos=PRP]^ just | ^ we
NP[deprel=det] --> DT[word=the xpos=DT] NP[word=students xpos=NNS]^ the | ^ students
NP[deprel=amod] --> AP[word=other xpos=JJ] NP[word=class xpos=NN]^ other | ^ class
NP[deprel=det] --> DT[word=the xpos=DT] NP[word=point xpos=NN]^ the | ^ main point
PRP[deprel=advmod] --> RB[word=not xpos=RB] PRP[word=we xpos=PRP]^ not | ^ just we
NP[deprel=det] --> DT[word=the xpos=DT] NP[word=class xpos=NN]^ the | ^ other class
VP[deprel=obj] --> VP[word='s xpos=VBZ]^ PRP[word=we xpos=PRP] ^ 's | not just we
NP[deprel=case] --> TO[word=to xpos=TO] NP[word=class xpos=NN]^ to | ^ the other class
VP[deprel=obl] --> VP[word=move xpos=VBP]^ NP[word=class xpos=NN] ^ move | to the other class
S[deprel=nsubj] --> NP[word=students xpos=NNS] VP[word=move xpos=VBP]^ the students | ^ mov

###Extract the rules from all treebanks

In [ ]:
learned_rules_fname="exported_dep_rules.txt"
models_dir="../models/exported_dep"
if not os.path.exists(models_dir): os.makedirs(models_dir)
# cur_fpath=train_files[file_i]
# conll_list=[]
for train_fpath in train_files:
  print(train_fpath)
  path0,train_fname=os.path.split(train_fpath)
  exported_fname=train_fname.replace(".conllu",".txt")
  exported_path=os.path.join(models_dir,exported_fname)
  print(exported_path)
  exported_fopen=open(exported_path,"w")
  for i0,cur_conll in enumerate(iter_conll_treebank(train_fpath)):
    #if i0>10: break
    #conll_list.append(item0)
    dep_obj0=conll2obj(cur_conll)
    dep_const_obj=dep_const(dep_obj0)
    ph_list=dep_const_obj.all_phrases

    for ph0 in ph_list:
      if ph0["rule"]=="": continue
      exported_fopen.write(json.dumps(ph0)+"\n")
    #   print(ph0["rule"],ph0.get("joined_phrase",""))
    # print("------")


  exported_fopen.close()

  #print(train_fname)
  # for i0,item0 in enumerate(iter_conll_treebank(cur_fpath)):
  #   conll_list.append(item0)


# learned_rules_fpath=os.path.join(models_dir,learned_rules_fname)
# print(learned_rules_fpath)

../data/UD/UD_English-EWT/en_ewt-ud-train.conllu
../models/exported_dep/en_ewt-ud-train.txt
../data/UD/UD_English-CHILDES/en_childes-ud-train.conllu
../models/exported_dep/en_childes-ud-train.txt
../data/UD/UD_English-Atis/en_atis-ud-train.conllu
../models/exported_dep/en_atis-ud-train.txt
../data/UD/UD_English-ESLSpok/en_eslspok-ud-train.conllu
../models/exported_dep/en_eslspok-ud-train.txt
../data/UD/UD_English-ParTUT/en_partut-ud-train.conllu
../models/exported_dep/en_partut-ud-train.txt
../data/UD/UD_English-LinES/en_lines-ud-train.conllu
../models/exported_dep/en_lines-ud-train.txt
../data/UD/UD_English-GUMReddit/en_gumreddit-ud-train.conllu
../models/exported_dep/en_gumreddit-ud-train.txt
../data/UD/UD_English-GUM/en_gum-ud-train.conllu
../models/exported_dep/en_gum-ud-train.txt


###Processing rules

In [ ]:
import json, re
import pandas as pd
from code_utils.pandas_utils import *
models_dir="../models/exported_dep"

max_n_examples=5

rule_count_dict={}
rule_example_dict={}
total_rules=0
for fname in os.listdir(models_dir):
  if "en_gumreddit" in fname: continue

  fpath=os.path.join(models_dir,fname)
  print(fpath)
  fopen=open(fpath)
  for i0,line0 in enumerate(fopen):
    #if i0>10: break
    line_dict0=json.loads(line0)
    rule0=line_dict0["rule"]
    joined_phrase0=line_dict0["joined_phrase"]
    #rule0=re.sub(r'\[word=.+?\]',"",rule0)
    rule0=re.sub(r'word=\S+\s',"",rule0)
    rule_count_dict[rule0]=rule_count_dict.get(rule0,0)+1
    cur_examples=rule_example_dict.get(rule0,[])
    cur_examples+=[joined_phrase0+f" |{fname}"]
    cur_examples=list(set(cur_examples))
    #cur_examples.sort(key=lambda x:len(x))
    rule_example_dict[rule0]=cur_examples[:max_n_examples]

    total_rules+=1
    #print(rule0)
  fopen.close()

rule_data=[]
rule_count_dict_items=list(rule_count_dict.items())
rule_count_dict_items.sort(key=lambda x:-x[-1])
for rule0,count0 in rule_count_dict_items[:100]:
  #print(rule0,count0)
  examples=rule_example_dict.get(rule0,[])
  examples_str="\n".join(examples)
  rule_data.append([rule0,count0,examples_str])
  #for ex0 in examples: print(ex0)
  #print("----")
print("rule_count_dict_items",len(rule_count_dict_items))

print("total_rules",total_rules)

write2excel(rule_data,"test_rules_output0.xlsx")

../models/exported_dep/en_ewt-ud-train.txt
../models/exported_dep/en_childes-ud-train.txt
../models/exported_dep/en_eslspok-ud-train.txt
../models/exported_dep/en_atis-ud-train.txt
../models/exported_dep/en_partut-ud-train.txt
../models/exported_dep/en_lines-ud-train.txt
../models/exported_dep/en_gum-ud-train.txt
rule_count_dict_items 17629
total_rules 707726


True

In [ ]:
def write2excel(data_list,excel_fpath,headers=None,sheet_name=None):
  # Convert to DataFrame
  df = pd.DataFrame(data_list, columns=headers)

  # Save to Excel
  if sheet_name!=None: df.to_excel(excel_fpath, index=False,sheet_name=sheet_name)
  else: df.to_excel(excel_fpath, index=False)
  return True

excel_fpath="test1.xlsx"
data_list=[["A","B"],["C","D"]]
write2excel(data_list,excel_fpath,headers=["H1","H2"],sheet_name="Test1")

True

In [ ]:
import pandas as pd

# Sample data
data = ['apple', 'banana', 'orange']

# Convert to DataFrame
df = pd.DataFrame(data, columns=['Fruits'])

# Save to Excel
df.to_excel('fruits.xlsx', index=False)

In [ ]:
df.to_excel('fruits1.xlsx', index=False,sheet_name=None)

IndexError: At least one sheet must be visible

In [ ]:
df.to_excel(sheet_name=)

In [ ]:
line_dict0

{'cat': 'NNP',
 'word': 'Zaman',
 'upos': 'PROPN',
 'xpos': 'NNP',
 'deprel': 'flat',
 'children': ['2'],
 'id': '3',
 'start': 1,
 'end': 2,
 'span': 2,
 'rule': 'NNP[deprel=punct] --> HYPH[word=- xpos=HYPH] NNP[word=Zaman xpos=NNP]^',
 'ptb': '(NNP (HYPH -) (NNP Zaman))',
 'phrase': '- Zaman',
 'joined_phrase': '- | ^ Zaman',
 'level': 1}

In [ ]:
rule_count_dict_items=list(rule_count_dict.items())
rule_count_dict_items.sort(key=lambda x:-x[-1])
for a in rule_count_dict_items[:100]:
  print(a)
print("rule_count_dict_items",len(rule_count_dict_items))

('NP[deprel=det] --> DT[xpos=DT] NP[xpos=NN]^', 34218)
('PP[deprel=case] --> IN[xpos=IN] NP[xpos=NN]^', 20957)
('NP[deprel=amod] --> AP[xpos=JJ] NP[xpos=NN]^', 12134)
('VP[deprel=punct] --> VP[xpos=VB]^ .[xpos=.]', 12123)
('S[deprel=nsubj] --> NP[xpos=PRP] VP[xpos=VB]^', 10112)
('S[deprel=nsubj] --> NP[xpos=PRP] VP[xpos=VBP]^', 8351)
('VP[deprel=mark] --> TO[xpos=TO] VP[xpos=VB]^', 8310)
('_[deprel=case] --> _[xpos=_] _[xpos=_]^', 7948)
('NP[deprel=punct] --> NP[xpos=NN]^ .[xpos=.]', 7335)
('PP[deprel=case] --> IN[xpos=IN] NP[xpos=NNS]^', 7037)
('NP[deprel=nmod:poss] --> PRP$[xpos=PRP$] NP[xpos=NN]^', 6683)
('VP[deprel=obj] --> VP[xpos=VB]^ NP[xpos=NN]', 6674)
('NP[deprel=compound] --> NP[xpos=NN] NP[xpos=NN]^', 6491)
('VP[deprel=advmod] --> RB[xpos=RB] VP[xpos=VB]^', 5862)
('VP[deprel=aux] --> MD[xpos=MD] VP[xpos=VB]^', 5820)
('S[deprel=nsubj] --> NP[xpos=PRP] VP[xpos=VBD]^', 5735)
('NNP[deprel=case] --> IN[xpos=IN] NNP[xpos=NNP]^', 5711)
('NP[deprel=nmod] --> NP[xpos=_]^ _[xpos=_]', 

In [ ]:
#dep_obj0=obj_list[1]
sent_i=27
file_i=1

n_items=50

#converting dependency object to constituency structure (with multiple options for projections and final format)
class dep_const:
  def __init__(self,dep_obj,params={}) -> None:
    self.dep_obj=dep_obj #obtaining basic info from the dependency object
    self.id_list,self.heads,self.words=[],[],[]
    self.id_info_dict={}
    self.head_children_dict={}
    self.id_projection_dict={}

    for it0 in dep_obj:
      id0,head0,word0=it0["id"],it0["head"],it0["form"]
      self.id_list.append(id0)
      self.heads.append(head0)
      self.words.append(word0)
      self.id_info_dict[id0]=it0
      self.head_children_dict[head0]=self.head_children_dict.get(head0,[])+[id0] #identify the ids of children corresponding to current head
    #print(self.words)

    childless_ids=list(set(self.id_list).difference(set(self.head_children_dict))) #tokens that are not heads/have no children
    childless_ids.sort(key=lambda x:int(x))

    for ch_id in childless_ids: #creating projections for tokens without children
      cur_info=self.id_info_dict[ch_id]
      loc_i=int(ch_id)-1
      phrase_obj={"cat": cur_info["xpos"],"word": cur_info["form"], "deprel":cur_info["deprel"],"upos":cur_info["upos"], "xpos":cur_info["xpos"],"children":[],"id":ch_id,"start":loc_i,"end":loc_i,"span":1, "rule":"","level":0}
      phrase_obj["ptb"]=f'({phrase_obj["cat"]} {phrase_obj["word"]})'
      cur_proj_list=[phrase_obj]

      phrase_obj1=copy.deepcopy(phrase_obj)
      phrase_cat=None
      if cur_info["upos"].lower()=="noun" or cur_info["deprel"].lower().startswith(("nsubj","nobj")): phrase_cat="NP"
      if cur_info["upos"].lower()=="verb": phrase_cat="VP"
      if cur_info["upos"].lower()=="adj": phrase_cat="AP"
      if phrase_cat!=None:
        phrase_obj1["cat"]=phrase_cat
        phrase_obj["ptb"]=f'({phrase_cat} {phrase_obj["word"]})'
        phrase_obj1["level"]=1
        cur_proj_list.append(phrase_obj1)
      self.id_projection_dict[ch_id]=cur_proj_list

    #iterating over remaining tokens that are heads, stop when all are projected
    while len(self.id_projection_dict)<len(self.id_list):
      for head_id,ch_id_list in self.head_children_dict.items():
        if self.id_projection_dict.get(head_id)!=None: continue #already projected - skip
        head_info=self.id_info_dict.get(head_id)
        if head_info==None: continue #mainly zero/root heads

        head_id_int=int(head_id)
        head_level=head_info.get("level",0)
        child_projection_list=[]
        child_sorting_list=[]
        all_children_projected=True
        #identifying and sorting children
        for ch0 in ch_id_list:
          ch_id_int=int(ch0)
          offset0=head_id_int-ch_id_int
          #offset0=ch_id_int-head_id_int
          ch_proj0=self.id_projection_dict.get(ch0)
          if ch_proj0==None: #if a child is not projected yet, we skip this head
            all_children_projected=False
            break
          highest_ch_proj=ch_proj0[-1]
          ch_ch_upos=highest_ch_proj["upos"]
          ch_ch_cat=highest_ch_proj["cat"]
          ch_ch_deprel=highest_ch_proj["deprel"]
          abs_offset=abs(offset0)
          direction0=int(offset0/abs_offset)
          ch_sorting_obj={"id": ch0,"offset":abs_offset,"direction":direction0,"upos":ch_ch_upos,"cat":ch_ch_cat,"deprel":ch_ch_deprel}
          child_sorting_list.append(ch_sorting_obj)

          child_projection_list.append(ch_proj0)

        if not all_children_projected: continue

        head_loc_i=head_id_int-1
        #head_level+=1
        head_phrase_obj={"cat":head_info["xpos"],"word": head_info["form"],"upos":head_info["upos"],"xpos":head_info["xpos"],"deprel":head_info["deprel"],"children":[],"id":head_id,"start":head_loc_i,"end":head_loc_i,"span":1,"rule":""}
        head_phrase_obj["ptb"]=f'({head_phrase_obj["cat"]} {head_phrase_obj["word"]})'
        cur_head_projections=[head_phrase_obj]
        head_phrase_obj1=copy.deepcopy(head_phrase_obj)
        phrase_cat=None
        if head_phrase_obj["upos"].lower()=="noun" or head_phrase_obj["deprel"].lower().startswith(("nsubj","nobj")): phrase_cat="NP"
        if head_phrase_obj["upos"].lower()=="verb": phrase_cat="VP"
        if head_phrase_obj["upos"].lower()=="adj": phrase_cat="AP"
        if phrase_cat!=None:
          head_phrase_obj1["cat"]=phrase_cat
          head_phrase_obj1["ptb"]=f'({phrase_cat} {head_phrase_obj1["word"]})'
          head_phrase_obj1["level"]=1
          cur_head_projections.append(head_phrase_obj1)


        child_sorting_list.sort(key=lambda x:(x["direction"],x["offset"])) #we sort children IDs based on direction and distance from the head, we go right firest, from closer to farther, and then go left the same way


        for ch_sort0 in child_sorting_list:
          ch_proj=self.id_projection_dict[ch_sort0["id"]][-1]
          hightest_cur_proj=cur_head_projections[-1] #we get the hightest projection for each new child/dependency
          new_combined_projection=copy.deepcopy(hightest_cur_proj)
          new_combined_projection["children"]+=[ch_sort0["id"]]

          head_cat0=hightest_cur_proj["cat"]
          head_word0=hightest_cur_proj["word"]
          head_ptb0=hightest_cur_proj["ptb"]
          original_head_cat=str(head_cat0)
          ch_cat0=ch_proj["cat"]
          ch_word0=ch_proj["word"]
          ch_deprel0=ch_proj["deprel"]
          ch_ptb0=ch_proj["ptb"]

          direction0=ch_sort0["direction"]

          head_xpos=hightest_cur_proj["xpos"]
          ch_xpos=ch_proj["xpos"]

          ch_start0,ch_end0=ch_proj["start"],ch_proj["end"]
          highest_proj_start0,highest_proj_end0=hightest_cur_proj["start"],hightest_cur_proj["end"]

          new_start=min(hightest_cur_proj["start"],ch_proj["start"])
          new_end=max(hightest_cur_proj["end"],ch_proj["end"])
          new_span=new_end-new_start+1

          #direction0=ch_sort0["direction"]
          if direction0==1 and ch_sort0["cat"]=="NP" and head_cat0=="VP": head_cat0="S"
          if ch_sort0["deprel"].lower().startswith("nsubj") and head_cat0=="VP": head_cat0="S"
          if direction0==1 and ch_sort0["upos"]=="ADP" and head_cat0=="NP": head_cat0="PP"
          if direction0==1 and ch_sort0["cat"]=="CC": head_cat0="CC_"+head_cat0

          new_combined_projection["cat"]=head_cat0
          if direction0>0: #1 is left, -1 is right
            rule=f'{head_cat0}[deprel={ch_deprel0}] --> {ch_cat0}[word={ch_word0} xpos={ch_xpos}] {original_head_cat}[word={head_word0} xpos={head_xpos}]^'
            ptb0=f'({head_cat0} {ch_ptb0} {head_ptb0})'
            phrase_items=self.words[ch_start0:ch_end0+1]+["| ^"]+self.words[highest_proj_start0:highest_proj_end0+1]
            joined_phrase=" ".join(phrase_items)
          else:
            rule=f'{head_cat0}[deprel={ch_deprel0}] --> {original_head_cat}[word={head_word0} xpos={head_xpos}]^ {ch_cat0}[word={ch_word0} xpos={ch_xpos}]'
            ptb0=f'({head_cat0} {head_ptb0} {ch_ptb0})'
            phrase_items=["^"]+self.words[highest_proj_start0:highest_proj_end0+1]+["|"]+self.words[ch_start0:ch_end0+1]
            joined_phrase=" ".join(phrase_items)

          new_combined_projection["rule"]=rule
          new_combined_projection["start"]=new_start
          new_combined_projection["end"]=new_end
          new_combined_projection["span"]=new_span
          phrase0=" ".join(self.words[new_start:new_end+1])
          new_combined_projection["phrase"]=phrase0
          new_combined_projection["joined_phrase"]=joined_phrase
          head_level+=1
          new_combined_projection["level"]=head_level
          new_combined_projection["ptb"]=ptb0
          cur_head_projections.append(new_combined_projection)


        self.id_projection_dict[head_id]=cur_head_projections
    self.all_phrases=[]
    for id0 in self.id_list:
      projections=self.id_projection_dict[id0]
      for pr0 in projections:
        self.all_phrases.append(pr0)
    self.all_phrases.sort(key=lambda x:x["span"])
    self.ptb=self.all_phrases[-1].get("ptb","") #the PTB string for the phrase with the largest span



cur_fpath=train_files[file_i]
conll_list=[]
for i0,item0 in enumerate(iter_conll_treebank(cur_fpath)):
  conll_list.append(item0)
  if i0>n_items: break
print("cur_fpath",cur_fpath)


cur_conll=conll_list[sent_i]
dep_obj0=conll2obj(cur_conll)

dep_const_obj=dep_const(dep_obj0)
ph_list=dep_const_obj.all_phrases

# ph_list=project_dep(dep_obj0)
#ph_list.sort(key=lambda x:x["span"])
for ph0 in ph_list:
  if ph0["rule"]=="": continue
  print(ph0["rule"],ph0.get("joined_phrase",""))

print(dep_const_obj.ptb)

cur_fpath ../data/UD/UD_English-CHILDES/en_childes-ud-train.conllu
VP[deprel=iobj] --> VP[word=get xpos=VB]^ PRP[word=you xpos=PRP] ^ get | you
NP[deprel=amod] --> AP[word=small xpos=JJ] NP[word=surprise xpos=NN]^ small | ^ surprise
NP[deprel=det] --> DT[word=a xpos=DT] NP[word=surprise xpos=NN]^ a | ^ small surprise
VP[deprel=obj] --> VP[word=get xpos=VB]^ NP[word=surprise xpos=NN] ^ get you | a small surprise
VP[deprel=discourse] --> VP[word=get xpos=VB]^ RB[word=okay xpos=RB] ^ get you a small surprise | okay
VP[deprel=punct] --> VP[word=get xpos=VB]^ ?[word=? xpos=?] ^ get you a small surprise okay | ?
VP[deprel=aux] --> MD[word='ll xpos=MD] VP[word=get xpos=VB]^ 'll | ^ get you a small surprise okay ?
S[deprel=nsubj] --> NP[word=I xpos=PRP] VP[word=get xpos=VB]^ I | ^ 'll get you a small surprise okay ?
CC_S[deprel=cc] --> CC[word=And xpos=CC] S[word=get xpos=VB]^ And | ^ I 'll get you a small surprise okay ?
(CC_S (CC And) (S (PRP I) (VP (MD 'll) (VP (VP (VP (VP (VP get) (PRP you

In [ ]:
ph0["ptb"]

'(S (NP (DT The) (NN Commission)) (VP (VP (VBD said) (S (PRP it) (VP (VBD had) (NP (DT no) (NP (JJ legal) (NP (NNS grounds) (PP (IN for) (NP (PDT such) (NP (DT an) (NN exclusion)))))))))) (. .)))'

In [ ]:
a='nsubj:pass'
a.lower().startswith(("nsubj","nobj"))

True

In [ ]:
cur_fpath=train_files[0]

obj_list=[]
conll_list=[]
for i0,item0 in enumerate(iter_conll_treebank(cur_fpath)):

  print(item0)
  dep_obj0=conll2obj(item0)
  #obj_list.append(dep_obj0)
  conll_list.append(item0)
  #for a in dep_obj0: print(a)
  print("========")
  if i0>10: break

print("cur_fpath",cur_fpath)


1	Al	Al	PROPN	NNP	Number=Sing	0	root	0:root	SpaceAfter=No
2	-	-	PUNCT	HYPH	_	3	punct	3:punct	SpaceAfter=No
3	Zaman	Zaman	PROPN	NNP	Number=Sing	1	flat	1:flat	_
4	:	:	PUNCT	:	_	7	punct	7:punct	_
5	American	American	ADJ	JJ	Degree=Pos	6	amod	6:amod	_
6	forces	force	NOUN	NNS	Number=Plur	7	nsubj	7:nsubj	_
7	killed	kill	VERB	VBD	Mood=Ind|Number=Plur|Person=3|Tense=Past|VerbForm=Fin	1	parataxis	1:parataxis	_
8	Shaikh	Shaikh	PROPN	NNP	Number=Sing	7	obj	7:obj	_
9	Abdullah	Abdullah	PROPN	NNP	Number=Sing	8	flat	8:flat	_
10	al	al	PROPN	NNP	Number=Sing	8	flat	8:flat	SpaceAfter=No
11	-	-	PUNCT	HYPH	_	12	punct	12:punct	SpaceAfter=No
12	Ani	Ani	PROPN	NNP	Number=Sing	8	flat	8:flat	SpaceAfter=No
13	,	,	PUNCT	,	_	15	punct	15:punct	_
14	the	the	DET	DT	Definite=Def|PronType=Art	15	det	15:det	_
15	preacher	preacher	NOUN	NN	Number=Sing	8	appos	8:appos	_
16	at	at	ADP	IN	_	18	case	18:case	_
17	the	the	DET	DT	Definite=Def|PronType=Art	18	det	18:det	_
18	mosque	mosque	NOUN	NN	Number=Sing	15	nmod	15:nmod:at	_
19	i

In [ ]:
dep_obj0[0]

{'id': '1',
 'form': 'The',
 'lemma': 'the',
 'upos': 'DET',
 'xpos': 'DT',
 'feats': 'Definite=Def|PronType=Art',
 'head': '3',
 'deprel': 'det',
 'deps': '3:det',
 'misc': '_'}

In [ ]:
def obj2conll(dep_obj,params={}):
    final_conll_str=""
    ignored_cols=params.get("ignored_cols",["deps"])
    col_headers=["id","form","lemma","upos","xpos","feats","head","deprel","deps","misc"]
    for obj0 in dep_obj:
        cur_cells=[]
        for col0 in col_headers:
            cur_val=obj0.get(col0,"_")
            if col0 in ignored_cols: cur_val="_"
            cur_cells.append(cur_val)
        line="\t".join(cur_cells)+"\n"
        final_conll_str+=line
    return final_conll_str

conll0=obj2conll(dep_obj0)
print(conll0)

1	The	the	DET	DT	Definite=Def|PronType=Art	3	det	_	_
2	police	police	NOUN	NN	Number=Sing	3	compound	_	_
3	commander	commander	NOUN	NN	Number=Sing	7	nsubj	_	_
4	of	of	ADP	IN	_	6	case	_	_
5	Ninevah	Ninevah	PROPN	NNP	Number=Sing	6	compound	_	_
6	Province	Province	PROPN	NNP	Number=Sing	3	nmod	_	_
7	announced	announce	VERB	VBD	Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin	0	root	_	_
8	that	that	SCONJ	IN	_	11	mark	_	_
9	bombings	bombing	NOUN	NNS	Number=Plur	11	nsubj	_	_
10	had	have	AUX	VBD	Mood=Ind|Number=Plur|Person=3|Tense=Past|VerbForm=Fin	11	aux	_	_
11	declined	decline	VERB	VBN	Tense=Past|VerbForm=Part	7	ccomp	_	_
12	80	80	NUM	CD	NumForm=Digit|NumType=Card	13	nummod	_	_
13	percent	percent	NOUN	NN	Number=Sing	11	obj	_	_
14	in	in	ADP	IN	_	15	case	_	_
15	Mosul	Mosul	PROPN	NNP	Number=Sing	11	obl	_	SpaceAfter=No
16	,	,	PUNCT	,	_	20	punct	_	_
17	whereas	whereas	SCONJ	IN	_	20	mark	_	_
18	there	there	PRON	EX	PronType=Dem	20	expl	_	_
19	had	have	AUX	VBD	Mood=Ind|Number=Sing|Person=3|Tense

#Parsing flow
connect to rules and features tables - build parsing algorithm

##Start Parsing

In [ ]:
from code_utils.parsing_lib import *
from code_utils import general

#djk(pt0,pt1,transition_dict0,pt_list0=None)



rules=["NP --> X[noun]",
       "AP --> JJ",
       "VP --> X^[be] AP",
       "VP --> X^[be] JJ",
       "VP --> X^[verb] NP",
       "V[verb] --> RB X^[verb]",
       "VP --> VP^ NP",
       "S --> NP VP^",
       "V → V[be] V^[ing]",
       'AUX[neg] → AUX^ RB[neg]',
       "NP --> DT NP^",
       "NP --> AP NP^",
       "PP --> IN^ NP",
       "NP --> NP^ PP",
       "NP --> NP NP^",
       "NP --> NP POS NP^",
       "S --> S^ .",
       ]

#       "NP --> NP^ PP",
#rules=final_rules
#rules+=["NP --> DT NP^"]


sent="he is happy"
sent="the man * is happy"

sent="the man * is happy"
sent="the man is happy"
sent="the old man in the kitchen"
sent="the old man in the kitchen is not happy"
sent="what also happened in Germany"
sent="the old man also asked the same question"
sent="the old man has also asked the same questions"
sent="however , we didn't have any good answers"
sent="I will probably ask him to come some time later"
#sent="I will probably ask him to come some time later"
#sent="Ecuador also supports initiatives for the protection, conservation and sustainable use of the ocean, such as the prompt entry into force"
sent="Ecuador also supports initiatives for the protection, conservation and sustainable use of the ocean, such as the prompt entry into force of the Agreement under the United Nations Convention on the Law of the Sea on the Conservation and Sustainable Use of Marine Biological Diversity of Areas beyond National Jurisdiction, and the holding of the third United Nations Conference on Oceans in 2025."
#sent="Our commitment is also reflected in the efforts made by the Ecuadorian presidency of the Committee in charge of developing a legally binding instrument to eliminate plastics pollution."

sent=re.sub(r"(\W)",r" \1 ",sent)
sent_toks=sent.strip().split()



sent_toks=['5', 'star', 'detail', 'job']
sent_toks=['"', 'Further', ',', 'I', 'find', 'it', 'even', 'more', 'disturbing', 'that', 'he', 'made', 'these', 'comments', 'about', 'a', 'nation', 'that', 'just', 'last', 'year', 'tested', 'nuclear', 'weapons', '--', 'shortly', 'after', 'voicing', 'his', 'public', 'opposition', 'to', 'the', 'Comprehensive', 'Test', 'Ban', 'Treaty', '.']
sent_toks=['A', 'spokesman', 'for', 'President', 'Clinton', 'also', 'criticized', 'Bush', "'s", 'comments', '.']



unknown_tags=["N","V","JJ","RB"]
#sent_toks=sent.split()
print(sent_toks)

rnn_model_fpath="../models/pos_models/H64_L3_LR001_updated_xpos_loading.model"
params0={"pos_model_path":rnn_model_fpath,"final_word_features_dict":final_word_features_dict,"xpos_ft_dict":xpos_ft_dict}
params0["max_n_phrases"]=20
params0["debug"]=False

parser_obj=Parser(rules_list=rules, params=params0)
final_parses=parser_obj.parse(sent_toks)
for ph_obj0,dep0,const0 in final_parses:
  print(ph_obj0["wt"])
  for d0 in dep0:
    print(d0)
  print("---")

# for fp0 in final_phrases:
#   if fp0["span"]<len(sent_toks): continue
#   dep_info,const_info=export_parse(sent_toks,fp0,final_phrases)
#   print(fp0)
#   for di in dep_info:
#     print(di)
#   print("---")

print("========")
# for a in parser_obj.phrase_list:
#   cur_words=sent_toks[a["start"]:a["end"]+1]
#   print(cur_words,a)
# print(len(parser_obj.phrase_list))


['A', 'spokesman', 'for', 'President', 'Clinton', 'also', 'criticized', 'Bush', "'s", 'comments', '.']
9.999394178390503
{'id': '1', 'word': 'A', 'head': '2', 'xpos': 'DT', 'head_word': 'spokesman', 'offset': -1}
{'id': '2', 'word': 'spokesman', 'head': '5', 'xpos': 'NN', 'head_word': 'Clinton', 'offset': -3}
{'id': '3', 'word': 'for', 'head': '2', 'xpos': 'IN', 'head_word': 'spokesman', 'offset': 1}
{'id': '4', 'word': 'President', 'head': '3', 'xpos': 'NNP', 'head_word': 'for', 'offset': 1}
{'id': '5', 'word': 'Clinton', 'head': '7', 'xpos': 'NNP', 'head_word': 'criticized', 'offset': -2}
{'id': '6', 'word': 'also', 'head': '7', 'xpos': 'RB', 'head_word': 'criticized', 'offset': -1}
{'id': '7', 'word': 'criticized', 'head': '0', 'xpos': 'VBD', 'head_word': '^', 'offset': 0}
{'id': '8', 'word': 'Bush', 'head': '10', 'xpos': 'NNP', 'head_word': 'comments', 'offset': -2}
{'id': '9', 'word': "'s", 'head': '10', 'xpos': 'POS', 'head_word': 'comments', 'offset': -1}
{'id': '10', 'word': 'c

In [ ]:
for a in parser_obj.tag_wt_list: print(a)

('A', [('DT', ['det'], 0.9999443292617798)])
('spokesman', [('NN', ['noun'], 0.9999995231628418)])
('for', [('IN', [], 1)])
('President', [('NNP', ['noun'], 0.9999995231628418), ('NN', ['noun'], 0.9999890327453613)])
('Clinton', [('NNP', ['noun'], 0.9999979734420776), ('NN', ['noun'], 0.9999967813491821)])
('also', [('', [], 1), ('RB', ['adv'], 1.0)])
('criticized', [('VBD', ['verb'], 0.9996175765991211), ('VBN', ['verb'], 0.9810216426849365)])
('Bush', [('NNP', ['noun'], 0.9999872446060181)])
("'s", [('POS', [], 0.9999498128890991), ('VBZ', ['verb'], 0.8467115163803101)])
('comments', [('NNS', ['noun'], 0.9998594522476196), ('NN', ['noun'], 0.18591611087322235)])
('.', [('.', ['punc'], 0.9999885559082031)])


In [ ]:
tokens=['5', 'star', 'detail', 'job']
tokens=['The', 'other', 'problem', '?']
tokens=['The', 'interim', 'Governing', 'Council', 'issued', 'a', 'communique', 'saying', ',', '"', 'It', 'is', 'a', 'proof', 'of', 'the', 'emptiness', 'of', 'the', 'Israeli', 'authority', 'and', 'a', 'destruction', 'of', 'the', 'peace', 'endeavors', 'in', 'the', 'region', '.', '.', '.']

#parser_obj=Parser(rules_list=rules, params=params0)
parses=parser_obj.parse(tokens)
for ph_obj0,dep0,const0 in parses[:1]:
  for d0 in dep0: print(d0)
  print("----")

{'id': '1', 'word': 'The', 'head': '2', 'xpos': 'DT', 'head_word': 'interim', 'offset': -1}
{'id': '2', 'word': 'interim', 'head': '3', 'xpos': 'NN', 'head_word': 'Governing', 'offset': -1}
{'id': '3', 'word': 'Governing', 'head': '4', 'xpos': 'NN', 'head_word': 'Council', 'offset': -1}
{'id': '4', 'word': 'Council', 'head': '14', 'xpos': 'NNP', 'head_word': 'proof', 'offset': -10}
{'id': '5', 'word': 'issued', 'head': '-', 'xpos': '-', 'head_word': '', 'offset': 100}
{'id': '6', 'word': 'a', 'head': '8', 'xpos': 'DT', 'head_word': 'saying', 'offset': -2}
{'id': '7', 'word': 'communique', 'head': '8', 'xpos': 'JJ', 'head_word': 'saying', 'offset': -1}
{'id': '8', 'word': 'saying', 'head': '14', 'xpos': 'NN', 'head_word': 'proof', 'offset': -6}
{'id': '9', 'word': ',', 'head': '-', 'xpos': '-', 'head_word': '', 'offset': 100}
{'id': '10', 'word': '"', 'head': '14', 'xpos': 'JJ', 'head_word': 'proof', 'offset': -4}
{'id': '11', 'word': 'It', 'head': '-', 'xpos': '-', 'head_word': '', 'of

In [ ]:
for wtag0 in parser_obj.tag_wt_list:
  print(wtag0)

('Ecuador', [('NN', ['noun'], 0.9999328851699829), ('NNP', ['noun'], 0.5867447853088379)])
('also', [('', [], 1), ('RB', ['adv'], 1.0)])
('supports', [('VBZ', ['verb'], 0.999607264995575), ('NNS', ['noun'], 0.8740209341049194)])
('initiatives', [('NNS', ['noun'], 0.9999417066574097)])
('for', [('IN', [], 1)])
('the', [('DT', [], 1)])
('protection', [('NN', ['noun'], 0.9999990463256836)])
(',', [('CCONJ', [], 1), (',', ['punc'], 0.9999215602874756)])
('conservation', [('NN', ['noun'], 0.9999998807907104), ('VB', ['verb'], 0.1614929586648941)])
('and', [('CCONJ', [], 1), ('CC', [], 0.9999985694885254)])
('sustainable', [('JJ', ['adj'], 1.0), ('NN', ['noun'], 0.9999717473983765)])
('use', [('NN', ['noun'], 1.0), ('VB', ['verb'], 0.874775230884552), ('VBP', ['verb'], 0.5029446482658386)])
('of', [('IN', [], 1)])
('the', [('DT', [], 1)])
('ocean', [('NN', ['noun'], 1.0), ('JJ', ['adj'], 0.9997401833534241)])
(',', [('CCONJ', [], 1), (',', ['punc'], 0.999927282333374)])
('such', [('', [], 1)

In [ ]:
phrase_key_items=list(parser_obj.main_phrase_wt_dict.items())
phrase_key_items.sort(key=lambda x:-x[-1])
for a,b in phrase_key_items[:200]:
  start,end=a[:2]
  phrase=sent_toks[start:end+1]
  print(a,b,phrase)


(10, 30, 'S', '') 8.557272672653198 ['sustainable', 'use', 'of', 'the', 'ocean', ',', 'such', 'as', 'the', 'prompt', 'entry', 'into', 'force', 'of', 'the', 'Agreement', 'under', 'the', 'United', 'Nations', 'Convention']
(10, 29, 'S', '') 8.557270526885986 ['sustainable', 'use', 'of', 'the', 'ocean', ',', 'such', 'as', 'the', 'prompt', 'entry', 'into', 'force', 'of', 'the', 'Agreement', 'under', 'the', 'United', 'Nations']
(10, 28, 'S', '') 7.798563599586487 ['sustainable', 'use', 'of', 'the', 'ocean', ',', 'such', 'as', 'the', 'prompt', 'entry', 'into', 'force', 'of', 'the', 'Agreement', 'under', 'the', 'United']
(14, 30, 'S', '') 7.557272672653198 ['ocean', ',', 'such', 'as', 'the', 'prompt', 'entry', 'into', 'force', 'of', 'the', 'Agreement', 'under', 'the', 'United', 'Nations', 'Convention']
(14, 29, 'S', '') 7.557270526885986 ['ocean', ',', 'such', 'as', 'the', 'prompt', 'entry', 'into', 'force', 'of', 'the', 'Agreement', 'under', 'the', 'United', 'Nations']
(10, 25, 'S', '') 6.967

In [ ]:
parser_obj.main_phrase_dict[(10, 14, 'NP', '')]

[{'wt': 2.0,
  'span': 5,
  'start': 10,
  'end': 14,
  'cat': 'NP',
  'feat': [],
  'children': (28, 43),
  'head_phrase': 43,
  'head_loc': 14,
  'rule_i': 9,
  'i': None}]

In [ ]:
import time

max_phrases_per_span=5

cur_rule_i=0
cur_rule=parser_obj.all_processed_rules[cur_rule_i]
rule_children=cur_rule["children"]
cur_child=rule_children[1]
print(cur_rule)
print(cur_child)
cur_phrase_ids=parser_obj.scan_phrases(cur_child)
t0=time.time()
span_counter={}
cur_phrases=[parser_obj.phrase_list[p_i] for p_i in cur_phrase_ids]
cur_phrases.sort(key=lambda x:x["wt"])
for ph0 in cur_phrases:
  cur_span= ph0["start"],ph0["end"]
  cur_span_count=span_counter.get(cur_span,0)
  continue
t1=time.time()
#cur_phrases.sort(key=lambda x:x["wt"])
print("cur_phrases",len(cur_phrases),t1-t0)

{'parent': {'cat': 'S', 'feat': ['declarative']}, 'children': [{'cat': 'NP', 'is_head': False, 'feat': []}, {'cat': 'VP', 'is_head': True, 'feat': []}], 'head_i': 1}
{'cat': 'VP', 'is_head': True, 'feat': []}
cur_phrases 336 0.0005002021789550781


In [ ]:
temp_phrases=list(parser_obj.phrase_list)
sorting_func=lambda x:(x["cat"],x["start"],x["end"])
temp_phrases.sort(key=sorting_func)

grouped=[(key,list(group)) for key,group in groupby(temp_phrases,sorting_func)]
for key0,grp0 in grouped:
  print(key0, len(grp0))
  grp0.sort(key=lambda x:x["wt"])
  for g0 in grp0[:5]: print(g0)
  print("--------")

# for a in temp_phrases:
#   #if a["cat"]!="NP": continue #print(a)
#   cur_words=sent_toks[a["start"]:a["end"]+1]
#   cur_words_str=" ".join(cur_words)
#   print(a["cat"],(a["start"],a["end"]),cur_words_str, a["wt"])

('AP', 2, 9) 10
{'wt': 1.756718635559082, 'span': 8, 'start': 2, 'end': 9, 'cat': 'AP', 'feat': [], 'children': (6, 543), 'head_phrase': 543, 'head_loc': 9, 'rule_i': 32, 'i': 544}
{'wt': 1.9231658577919006, 'span': 8, 'start': 2, 'end': 9, 'cat': 'AP', 'feat': [], 'children': (6, 534), 'head_phrase': 534, 'head_loc': 9, 'rule_i': 32, 'i': 535}
{'wt': 3.756718635559082, 'span': 8, 'start': 2, 'end': 9, 'cat': 'AP', 'feat': [], 'children': (103, 543), 'head_phrase': 543, 'head_loc': 9, 'rule_i': 32, 'i': 548}
{'wt': 3.756718635559082, 'span': 8, 'start': 2, 'end': 9, 'cat': 'AP', 'feat': [], 'children': (6, 546), 'head_phrase': 546, 'head_loc': 9, 'rule_i': 32, 'i': 549}
{'wt': 3.9231658577919006, 'span': 8, 'start': 2, 'end': 9, 'cat': 'AP', 'feat': [], 'children': (103, 534), 'head_phrase': 534, 'head_loc': 9, 'rule_i': 32, 'i': 539}
--------
('AP', 5, 9) 4
{'wt': 2.7567187547683716, 'span': 5, 'start': 5, 'end': 9, 'cat': 'AP', 'feat': [], 'children': (57, 543), 'head_phrase': 543, '

In [ ]:
for i0,pr0 in enumerate(parser_obj.all_processed_rules):
  print(i0,pr0)

0 {'parent': {'cat': 'S', 'feat': ['declarative']}, 'children': [{'cat': 'NP', 'is_head': False, 'feat': []}, {'cat': 'VP', 'is_head': True, 'feat': []}], 'head_i': 1}
1 {'parent': {'cat': 'S-YNQ', 'feat': []}, 'children': [{'cat': 'Aux', 'is_head': True, 'feat': []}, {'cat': 'S/Aux', 'is_head': False, 'feat': []}], 'head_i': 0}
2 {'parent': {'cat': 'S', 'feat': []}, 'children': [{'cat': 'WH', 'is_head': False, 'feat': []}, {'cat': 'VP', 'is_head': True, 'feat': []}], 'head_i': 1}
3 {'parent': {'cat': 'S-WHQ', 'feat': []}, 'children': [{'cat': 'WH', 'is_head': True, 'feat': []}, {'cat': '+', 'is_head': False, 'feat': []}, {'cat': 'Aux', 'is_head': False, 'feat': []}, {'cat': '+', 'is_head': False, 'feat': []}, {'cat': 'S/Aux/PP', 'is_head': False, 'feat': []}], 'head_i': 0}
4 {'parent': {'cat': 'S-IMP', 'feat': []}, 'children': [{'cat': 'VP', 'is_head': False, 'feat': ['+base']}], 'head_i': 0}
5 {'parent': {'cat': 'S', 'feat': ['+YNQ']}, 'children': [{'cat': 'Aux', 'is_head': False, 'f

In [ ]:
final_phrases=parser_obj.parse(sent_toks)

for fp0 in final_phrases:
  if fp0["span"]<len(sent_toks): continue
  dep_info,const_info=export_parse(sent_toks,fp0,final_phrases)
  print(fp0)
  for di in dep_info:
    print(di)
  print("---")

print("========")
for a in parser_obj.phrase_list:
  cur_words=sent_toks[a["start"]:a["end"]+1]
  print(cur_words,a)
print(len(parser_obj.phrase_list))

{'wt': 3.9999996423721313, 'span': 4, 'start': 0, 'end': 3, 'cat': 'S', 'feat': ['declarative'], 'children': (3, 14), 'head_phrase': 14, 'head_loc': 2, 'rule_i': 0, 'i': 16}
{'id': '1', 'word': 'the', 'head': '2', 'xpos': 'DT', 'head_word': 'man', 'offset': -1}
{'id': '2', 'word': 'man', 'head': '3', 'xpos': 'NN', 'head_word': 'is', 'offset': -1}
{'id': '3', 'word': 'is', 'head': '0', 'xpos': 'AUX', 'head_word': '^', 'offset': 0}
{'id': '4', 'word': 'happy', 'head': '3', 'xpos': 'JJ', 'head_word': 'is', 'offset': 1}
---
{'wt': 3.9999996423721313, 'span': 4, 'start': 0, 'end': 3, 'cat': 'S', 'feat': ['declarative'], 'children': (4, 14), 'head_phrase': 14, 'head_loc': 2, 'rule_i': 0, 'i': 17}
{'id': '1', 'word': 'the', 'head': '2', 'xpos': 'DT', 'head_word': 'man', 'offset': -1}
{'id': '2', 'word': 'man', 'head': '3', 'xpos': 'NN', 'head_word': 'is', 'offset': -1}
{'id': '3', 'word': 'is', 'head': '0', 'xpos': 'AUX', 'head_word': '^', 'offset': 0}
{'id': '4', 'word': 'happy', 'head': '3'

In [ ]:
parser_obj.pos_tagger.tag_words(sent_toks,min_wt=0.1)

[{'word': 'the',
  'upos': [('DET', 0.999991774559021)],
  'xpos': [('DT', 0.9991812109947205)]},
 {'word': 'man',
  'upos': [('NOUN', 1.0)],
  'xpos': [('NN', 0.9999997615814209)]},
 {'word': 'is',
  'upos': [('AUX', 0.9999998807907104)],
  'xpos': [('VBZ', 0.9999758005142212)]},
 {'word': 'happy',
  'upos': [('ADJ', 1.0), ('NOUN', 0.9999998807907104)],
  'xpos': [('JJ', 0.9999998807907104), ('NN', 0.9999996423721313)]}]

In [ ]:
for a in parser_obj.phrase_list:
  print(a)
print(len(parser_obj.phrase_list))

{'wt': 1, 'span': 1, 'start': 0, 'end': 0, 'cat': 'DT', 'feat': [], 'head_loc': 0, 'head_phrase': 0, 'children': [], 'i': 0}
{'wt': 0.9991812109947205, 'span': 1, 'start': 0, 'end': 0, 'cat': 'DT', 'feat': ['det'], 'head_loc': 0, 'head_phrase': 1, 'children': [], 'i': 1}
{'wt': 0.9999953508377075, 'span': 1, 'start': 1, 'end': 1, 'cat': 'JJ', 'feat': [], 'head_loc': 1, 'head_phrase': 2, 'children': [], 'i': 2}
{'wt': 0.9999953508377075, 'span': 1, 'start': 1, 'end': 1, 'cat': 'AP', 'feat': [], 'children': (2,), 'head_phrase': 2, 'head_loc': 1, 'i': 3}
{'wt': 0.9999562501907349, 'span': 1, 'start': 1, 'end': 1, 'cat': 'NN', 'feat': ['noun'], 'head_loc': 1, 'head_phrase': 4, 'children': [], 'i': 4}
{'wt': 0.9999562501907349, 'span': 1, 'start': 1, 'end': 1, 'cat': 'NP', 'feat': [], 'children': (4,), 'head_phrase': 4, 'head_loc': 1, 'i': 5}
{'wt': 1.9999562501907349, 'span': 2, 'start': 0, 'end': 1, 'cat': 'NP', 'feat': [], 'children': (0, 5), 'head_phrase': 5, 'head_loc': 1, 'i': 6}
{'wt

In [ ]:
phrase0=parser_obj.phrase_list[-1]
rules=parser_obj.get_rules(phrase0)
for r0 in rules:
  print(r0, parser_obj.all_processed_rules[r0])

14 {'parent': {'cat': 'NP', 'feat': ['det']}, 'children': [{'cat': 'X', 'is_head': False, 'feat': ['det']}, {'cat': 'NP', 'is_head': True, 'feat': ['-det']}], 'head_i': 1}
16 {'parent': {'cat': 'NP', 'feat': []}, 'children': [{'cat': 'NP', 'is_head': False, 'feat': []}, {'cat': 'NP', 'is_head': True, 'feat': []}], 'head_i': 1}
17 {'parent': {'cat': 'NP', 'feat': []}, 'children': [{'cat': 'NP', 'is_head': False, 'feat': []}, {'cat': 'POSS', 'is_head': False, 'feat': []}, {'cat': 'NP', 'is_head': True, 'feat': []}], 'head_i': 2}
18 {'parent': {'cat': 'NP', 'feat': []}, 'children': [{'cat': 'AP', 'is_head': False, 'feat': []}, {'cat': 'NP', 'is_head': True, 'feat': []}], 'head_i': 1}
21 {'parent': {'cat': 'NP', 'feat': []}, 'children': [{'cat': 'AP', 'is_head': False, 'feat': []}, {'cat': 'NP', 'is_head': True, 'feat': ['-det']}], 'head_i': 1}
33 {'parent': {'cat': 'PP', 'feat': []}, 'children': [{'cat': 'P', 'is_head': True, 'feat': []}, {'cat': 'NP', 'is_head': False, 'feat': []}], 'hea

In [ ]:
parser_obj.all_processed_rules

[{'parent': {'cat': 'S', 'feat': ['+declarative']},
  'children': [{'cat': 'NP', 'is_head': False, 'feat': []},
   {'cat': 'VP', 'is_head': False, 'feat': []}],
  'head_i': 0},
 {'parent': {'cat': 'S', 'feat': []},
  'children': [{'cat': 'NP', 'is_head': False, 'feat': []},
   {'cat': 'VP', 'is_head': True, 'feat': []}],
  'head_i': 1},
 {'parent': {'cat': 'S-YNQ', 'feat': []},
  'children': [{'cat': 'Aux', 'is_head': True, 'feat': []},
   {'cat': 'S/Aux', 'is_head': False, 'feat': []}],
  'head_i': 0},
 {'parent': {'cat': 'S', 'feat': []},
  'children': [{'cat': 'WH', 'is_head': False, 'feat': []},
   {'cat': 'VP', 'is_head': True, 'feat': []}],
  'head_i': 1},
 {'parent': {'cat': 'S-WHQ', 'feat': []},
  'children': [{'cat': 'WH', 'is_head': True, 'feat': []},
   {'cat': '+', 'is_head': False, 'feat': []},
   {'cat': 'Aux', 'is_head': False, 'feat': []},
   {'cat': '+', 'is_head': False, 'feat': []},
   {'cat': 'S/Aux/PP', 'is_head': False, 'feat': []}],
  'head_i': 0},
 {'parent': {'

In [ ]:
def export_parse(words,main_phrase,final_phrases):

  cur_phrase=main_phrase
  start0,end0=cur_phrase["start"],cur_phrase["end"]
  phrase_head_loc_dict={cur_phrase["i"]: cur_phrase["head_loc"]} #identify the head location for each phrase obj/key is phrase number/phrase index in the phrase list

  dep_dict={} #identify which phrase head loc depends on which phrase head loc
  root_index=cur_phrase["head_loc"] #the root of the current phrase

  cur_tokens_xpos_dict={}
  unsorted_phrases=sorted(final_phrases,key=lambda x:x["i"]) #they should be sorted anyway by their original order of being added to the list
  parse_phrases=[cur_phrase] #these are the phrases to be used for constituency parsing
  children=cur_phrase.get("children",[])
  new_children=[]

  while children: #go on a semi-recursive way to identify children and subchildren of current main phrase
    for ch0 in children:
      child_phrase=unsorted_phrases[ch0]
      phrase_head_loc_dict[ch0]=child_phrase["head_loc"]
      if child_phrase["span"]==1 and child_phrase["children"]==[]: #identify the matching XPOS for terminal phrases
        cur_tokens_xpos_dict[child_phrase["start"]]=child_phrase["cat"]
      parse_phrases.append(child_phrase)
      sub_children=child_phrase.get("children",())
      sub_children=list(sub_children)
      for sc in sub_children:
        if sc==ch0: continue
        new_children.append(sc) #print("sc",sc)
    children=new_children
    new_children=[]

  for a in parse_phrases:
    cur_children=a["children"]
    cur_phrase_head_loc=a["head_loc"]
    for ch0 in cur_children: #for each child of the current phrase, identify its head_loc
      ch_head_loc=phrase_head_loc_dict[ch0]
      if ch_head_loc==cur_phrase_head_loc: continue #skip if the head_loc for current child is the same as of the main phrase
      dep_dict[ch_head_loc]=cur_phrase_head_loc #update dependency dict between heads of two phrases

  final_dep_list=[]
  for i0,w0 in enumerate(words): #create dependency info for each word
    cur_id0=i0+1

    #identify head index
    if i0==root_index: #root word
      cur_dep="0"
      head_word="^"
      offset=0
    else:
      cur_dep=dep_dict.get(i0)
      if cur_dep==None:  #word without attachment
        cur_dep="-"
        head_word=""
        offset=100 #any random number for no offset
      else: #any other word
        head_word=words[cur_dep]
        offset=i0-cur_dep
        cur_dep=str(cur_dep+1)

    cur_xpos0=cur_tokens_xpos_dict.get(i0,"-")
    final_dep_list.append({"id":str(cur_id0),"word": w0,"head":cur_dep,"xpos":cur_xpos0,"head_word":head_word,"offset":offset})
  return final_dep_list, parse_phrases

cur_phrase0=final_phrases[0]

final_dep_list0, parse_phrases0=export_parse(sent_toks,cur_phrase0,final_phrases)

for a in final_dep_list0: print(a)

for a in parse_phrases0: print(a)

  #print(i0,dep_dict[i0],cur_tokens_xpos_dict[i0])
#for a in final_dep_list: print(a)

{'id': '1', 'word': 'the', 'head': '2', 'xpos': 'DET', 'head_word': 'man', 'offset': -1}
{'id': '2', 'word': 'man', 'head': '4', 'xpos': 'NN', 'head_word': 'is', 'offset': -2}
{'id': '3', 'word': '*', 'head': '-', 'xpos': '-', 'head_word': '', 'offset': 100}
{'id': '4', 'word': 'is', 'head': '0', 'xpos': 'AUX', 'head_word': '^', 'offset': 0}
{'id': '5', 'word': 'happy', 'head': '4', 'xpos': 'JJ', 'head_word': 'is', 'offset': 1}
{'wt': 3.9999979734420776, 'span': 5, 'start': 0, 'end': 4, 'cat': 'S', 'feat': [], 'children': (4, 14), 'head_phrase': 14, 'head_loc': 3, 'i': 17}
{'wt': 1.999999761581421, 'span': 2, 'start': 0, 'end': 1, 'cat': 'NP', 'feat': [], 'children': (0, 3), 'head_phrase': 3, 'head_loc': 1, 'i': 4}
{'wt': 1.9999982118606567, 'span': 2, 'start': 3, 'end': 4, 'cat': 'VP', 'feat': [], 'children': (10, 12), 'head_phrase': 10, 'head_loc': 3, 'i': 14}
{'wt': 1, 'span': 1, 'start': 0, 'end': 0, 'cat': 'DET', 'feat': [], 'head_loc': 0, 'head_phrase': 0, 'children': [], 'i': 0}

##Parser Obj Definition

In [ ]:
import re, copy
import itertools
from itertools import product
from code_utils.parsing_lib import *

In [ ]:
#31 December 2025
class Parser:
  def __init__(self,rules_list=[],word_features_list=[],params={}) -> None:
    self.unknown_tags=params.get("unknown_tags",["N","V","JJ","RB"]) #maybe get the actual distribution of these tags from corpora
    self.sent_padding=params.get("sent_padding",False)
    self.default_wt=params.get("default_wt",0.5) #to apply on unknown tags
    self.min_pos_wt=params.get("min_pos_wt",0.2)
    self.pos_model_path=params.get("pos_model_path")
    self.final_word_features_dict=params.get("final_word_features_dict",{})
    self.xpos_ft_dict=params.get("xpos_ft_dict",{})
    self.max_n_phrases=params.get("max_n_phrases",10) #max n phrases per key
    self.djk_max_span_distance=params.get("djk_max_span_distance",3) #max difference between two spans when applying djekstra's algorithm
    self.djk_distance_penalty_ratio=params.get("djk_distance_penalty_ratio",0.5) #penalty applied dpending on the distance between two consecutive spans on djk path
    self.djk_use_negative_wt=params.get("djk_use_negative_wt",False) #whether to use actual weight or negative weight to get the optimum path

    self.max_skip_distance=params.get("max_skip_distance",3) #max number of tokens to skip between phrases
    self.debug=params.get("debug",False)

    self.max_cat_proj=params.get("max_cat_proj",100) #maximum number of times to project a category for a particular head token
    self.pos_tagger=POS(self.pos_model_path)

    self.unknown_token_tags=[("N","noun"),("V","verb"),("JJ","adj"),("RB","adv")]
    self.unknown_token_tags_wt_dict={}
    #process all word features

    #process all rules
    self.all_processed_rules=[]
    self.cat_fwd_index=[]
    self.feat_fwd_index=[]
    for i0,r0 in enumerate(rules_list):
      processed_rule=process_rule(r0)
      if processed_rule==None: continue
      processed_rule["rule_i"]=i0
      self.all_processed_rules.append(processed_rule)

    for i0, processed_rule in enumerate(self.all_processed_rules):
      last_child=processed_rule["children"][-1]
      last_child_cat=last_child["cat"]
      last_child_features=last_child["feat"]
      self.cat_fwd_index.append((last_child_cat,i0))
      for ft0 in last_child_features: self.feat_fwd_index.append((ft0,i0))
    self.cat_fwd_index.sort()
    self.feat_fwd_index.sort()
    self.cat_inv_index=dict(iter([(key,[v[1] for v in list(group)]) for key,group in groupby(self.cat_fwd_index,lambda x:x[0])]))
    self.feat_inv_index=dict(iter([(key,[v[1] for v in list(group)]) for key,group in groupby(self.feat_fwd_index,lambda x:x[0])]))
    #self.phrase_list=[]


  def parse2(self,tokens):
    self.phrase_list=[] #list of all phrase objects, with spans, weights, children
    self.cat_phrase_index={}
    self.feat_phrase_index={}

    self.end_start_phrase_dict={} #dict[end][start][cat]=[{phrase_obj1},{phrase_obj2},{phrase_obj3}]
    self.span_wt_dict={} #dict[(start,end)]=wt
    self.span_phrase_dict={} #dict[(start,end)]={phrase_obj}
    self.phrase_key_obj_dict={} #dict[phrase_key]=phrase_obj


    self.projection_counter={}
    self.head_loc_cat_projection_counter={} #counting how many times this category was projected from the same head token dict[(head_loc,cat)]=count


    tokens_pos_list=self.pos_tagger.tag_words(tokens,min_wt=self.min_pos_wt)
    self.cur_tokens_pos_list=tokens_pos_list

    self.tag_wt_list=[]

    if self.debug: print(tokens_pos_list)
    for token_i,a in enumerate(tokens_pos_list): #process each token
      token0=a["word"]
      if self.debug: print(token_i,token0,len(self.phrase_key_list))

      xpos0=a.get("xpos",[])
      start0,end0=token_i,token_i
      wt=self.default_wt
      outcome=self.final_word_features_dict.get(token0.lower(),[]) #outcome is multiple tag/cat options, with their features
      outcome_wt=[(cat0,feat0,1) for cat0,feat0 in outcome] #use a weight of 1 for the specific closed class words
      temp_used_cat_list=[v[0] for v in outcome_wt]
      for xpos_tag0,xpos_wt0 in xpos0:
        if xpos_tag0 in temp_used_cat_list: continue
        xpos_ft0=self.xpos_ft_dict.get(xpos_tag0,[])
        outcome_wt.append((xpos_tag0,xpos_ft0,xpos_wt0))
      #if nothing found
      if outcome_wt==[]:
        for cat0,feat0 in self.unknown_token_tags: outcome_wt.append((cat0,feat0,self.default_wt))

      self.tag_wt_list.append((token0,outcome_wt))
      #for the current token, identify all the outcomes with their weights
      #print("token0",token0)
      all_token_new_phrases=[]
      for cat0,feat_split,wt0 in outcome_wt:
        if cat0=="": continue
        #only one phrase object per cat/features pair
        cur_phrase_obj={"wt":wt0,"span":1,"start":start0,"end":end0,"cat":cat0,"feat":feat_split,"head_loc":start0,"children":[],"level":0}
        check_added_phrases=self.add_phrase3(cur_phrase_obj)

        #print(check_added_phrases,"Projecting:",cur_phrase_obj)
        new_phrases=self.project_phrase3(cur_phrase_obj)
        # for a in new_phrases:
        #   print("NEW:",a)
        all_token_new_phrases.extend(new_phrases)
      while all_token_new_phrases:
        all_token_new_phrases_copy=copy.deepcopy(all_token_new_phrases)
        all_token_new_phrases=[]
        for ph0 in all_token_new_phrases_copy:
          check_added_phrases=self.add_phrase3(ph0)
          #print("recursive check/project:", check_added_phrases,ph0)
          if check_added_phrases==False: continue
          projected_new_phrases=self.project_phrase3(ph0)

          all_token_new_phrases.extend(projected_new_phrases)



    final_parse_phrases=[]
    end_of_sentence_dict=self.end_start_phrase_dict.get(len(tokens)-1,{})
    full_span_sub_dict=end_of_sentence_dict.get(0,{})
    for cat0,phrases0 in full_span_sub_dict.items():
      final_parse_phrases.extend(phrases0)

    #full_span_phrases=end_of_sentence_dict.get(0,[])


    if final_parse_phrases==[]:
      span_wt_items=list(self.span_wt_dict.items())
      span_wt_items.sort(key=lambda x:-x[-1])
      for span0,wt0 in span_wt_items[:5]:
        start0,end0=span0
        cat_phrases_subdict=self.end_start_phrase_dict[end0][start0]
        for cat0,phrases0 in cat_phrases_subdict.items():
          for ph0 in phrases0: final_parse_phrases.append(ph0) #print(ph0)


    #return self.tag_wt_list
    final_raw_parses=[]
    final_parse_phrases.sort(key=lambda x:(-x["span"],-x["wt"]))
    for fp0 in final_parse_phrases[:self.max_n_phrases]:
      dep0,const0=self.export_parse2(tokens,fp0)
      final_raw_parses.append((fp0,dep0,const0))
    return final_raw_parses

  def project_phrase3(self,phrase_obj0): #only for maximum binary branching
    #print("projecting:",phrase_obj0)
    new_level=phrase_obj0.get("level",0)+1
    #wt_inc=new_level*0.001 #weight increment - to give a higher weight for the higher level projection phrases
    wt_inc=0

    #phrase_obj0["level"]=phrase_obj0.get("level",0)+1
    cur_phrase_key=self.get_phrase_key(phrase_obj0)
    self.projection_counter[cur_phrase_key]=self.projection_counter.get(cur_phrase_key,0)+1
    if self.projection_counter.get(cur_phrase_key,0)>10: return []
    cur_start0=phrase_obj0["start"]


    cur_rules_numbers=self.get_rules(phrase_obj0)
    cur_rules_obj_list=[self.all_processed_rules[r_i] for r_i in cur_rules_numbers]
    cur_rules_obj_list.sort(key=lambda x:len(x["children"]))
    all_combined_phrases=[]
    for cur_rule_obj in cur_rules_obj_list:

      cur_rule_children=cur_rule_obj["children"]
      first_child,last_child=cur_rule_children[0],cur_rule_children[-1] #unary or binary rules
      first_child_cat=first_child["cat"] #we will only use categories for first cat now, but later we can do features

      match_last=match_rule(phrase_obj0,last_child)
      if not match_last: continue
      if len(cur_rule_children)==1:
        new_combined_phrase=self.combine_phrases([phrase_obj0],cur_rule_obj)
        new_combined_phrase["level"]=new_level
        new_combined_phrase["wt"]+=wt_inc
        all_combined_phrases.append(new_combined_phrase)
        #print("new_combined_phrase",new_combined_phrase)
      else:
        min_prev_end=max(0,cur_start0-self.max_skip_distance)
        for temp_end0 in range(min_prev_end,cur_start0):
          corr_start_dict=self.end_start_phrase_dict.get(temp_end0,{})
          for temp_start0,temp_start_sub_dict0 in corr_start_dict.items():
            temp_phrases=temp_start_sub_dict0.get(first_child_cat,[])
            for tph0 in temp_phrases:
              new_combined_phrase=self.combine_phrases([tph0,phrase_obj0],cur_rule_obj)
              #we want to monitor projection from same rule e.g. VP --> VP^ PP or NP --> NP NP^
              #and limit having so many projections from the same head phrase according to the same rule
              # new_combined_phrase_head_key=new_combined_phrase["head_phrase"] #checking the key/phrase of the new combined phrase
              # new_combined_phrase_head_phrase=self.phrase_key_obj_dict.get(new_combined_phrase_head_key)
              # if new_combined_phrase_head_phrase==None: continue
              # new_combined_phrase_head_phrase_rule=new_combined_phrase_head_phrase.get("rule")
              # same_rule_proj_level=new_combined_phrase.get("same_rule_proj_level",0)
              # if new_combined_phrase_head_phrase_rule==cur_rule_obj:  same_rule_proj_level+=1
              # new_combined_phrase["same_rule_proj_level"]=same_rule_proj_level

              new_combined_phrase["level"]=new_level
              new_combined_phrase["wt"]+=wt_inc
              all_combined_phrases.append(new_combined_phrase)

    return all_combined_phrases



  def export_parse2(self,words,cur_phrase):
    start0,end0=cur_phrase["start"],cur_phrase["end"]
    phrase_key0=self.get_phrase_key(cur_phrase)
    # print(cur_phrase)
    # return
    #phrase_head_loc_dict={cur_phrase["i"]: cur_phrase["head_loc"]} #identify the head location for each phrase obj/key is phrase number/phrase index in the phrase list
    phrase_head_loc_dict={phrase_key0: cur_phrase["head_loc"]} #identify the head location for each phrase obj/key is phrase number/phrase index in the phrase list

    dep_dict={} #identify which phrase head loc depends on which phrase head loc
    root_index=cur_phrase["head_loc"] #the root of the current phrase

    cur_tokens_xpos_dict={}
    #unsorted_phrases=sorted(final_phrases,key=lambda x:x["i"]) #they should be sorted anyway by their original order of being added to the list
    parse_phrases=[cur_phrase] #these are the phrases to be used for constituency parsing
    children=cur_phrase.get("children",[])
    new_children=[]

    while children: #go on a semi-recursive way to identify children and subchildren of current main phrase
      for ch0 in children:
        # ch_phrase_key=self.phrase_key_list[ch0]
        # child_phrase=self.main_phrase_dict[ch_phrase_key][0]

        ch_phrase_key=ch0
        #child_phrase=self.phrase_key_obj_dict[ch_phrase_key]
        child_phrase=self.phrase_key_obj_dict.get(ch_phrase_key)
        if child_phrase==None:
          start,end,cat,feat=ch_phrase_key
          span=end-start+1
          child_phrase={"wt":0, "start":start,"end":end,"cat":cat, "feat":feat.split(), "head_loc":start,"children":[],"span":span}
        phrase_head_loc_dict[ch0]=child_phrase["head_loc"]
        if child_phrase["span"]==1 and child_phrase["children"]==[]: #identify the matching XPOS for terminal phrases
          cur_tokens_xpos_dict[child_phrase["start"]]=child_phrase["cat"]
        parse_phrases.append(child_phrase)
        sub_children=child_phrase.get("children",())
        sub_children=list(sub_children)
        for sc in sub_children:
          if sc==ch0: continue
          new_children.append(sc) #print("sc",sc)
      children=new_children
      new_children=[]

    for a in parse_phrases:
      cur_children=a["children"]
      cur_phrase_head_loc=a["head_loc"]
      for ch0 in cur_children: #for each child of the current phrase, identify its head_loc
        ch_head_loc=phrase_head_loc_dict[ch0]
        if ch_head_loc==cur_phrase_head_loc: continue #skip if the head_loc for current child is the same as of the main phrase
        dep_dict[ch_head_loc]=cur_phrase_head_loc #update dependency dict between heads of two phrases

    final_dep_list=[]
    for i0,w0 in enumerate(words): #create dependency info for each word
      cur_id0=i0+1

      #identify head index
      if i0==root_index: #root word
        cur_dep="0"
        head_word="^"
        offset=0
      else:
        cur_dep=dep_dict.get(i0)
        if cur_dep==None:  #word without attachment
          cur_dep="-"
          head_word=""
          offset=100 #any random number for no offset
        else: #any other word
          head_word=words[cur_dep]
          offset=i0-cur_dep
          cur_dep=str(cur_dep+1)

      cur_xpos0=cur_tokens_xpos_dict.get(i0,"-")
      final_dep_list.append({"id":str(cur_id0),"word": w0,"head":cur_dep,"xpos":cur_xpos0,"head_word":head_word,"offset":offset})
    return final_dep_list, parse_phrases

  def add_phrase3(self,phrase_obj0): #only for maximum binary branching
    span0=start0,end0=phrase_obj0["start"],phrase_obj0["end"]
    wt0=phrase_obj0["wt"]
    found_span_wt0=self.span_wt_dict.get(span0,0)
    # phrase_has_top_span_wt=False

    cat0,children0=phrase_obj0["cat"],phrase_obj0["children"]

    phrase_key0=self.get_phrase_key(phrase_obj0)
    self.phrase_key_obj_dict[phrase_key0]=phrase_obj0


    used_phrase_key_dict={}


    temp_end_sub_dict=self.end_start_phrase_dict.get(end0,{}) #get the sub dict for phrases having this end loc
    temp_start_sub_dict2=temp_end_sub_dict.get(start0,{}) #get the subdict with the start loc
    cur_phrases=temp_start_sub_dict2.get(cat0,[]) #get the list of phrases with the [end][start][cat] keys
    if cur_phrases==[]:
      cur_phrases=[phrase_obj0]
    else:
      cur_top_phrase,cur_bottom_phrase=cur_phrases[0],cur_phrases[-1]
      if len(cur_phrases)>=self.max_n_phrases and wt0<cur_bottom_phrase["wt"]:
        #print("1","cur_phrases",len(cur_phrases))
        return False
      for cur_ph0 in cur_phrases:
        cur_ph0_cat,cur_ph0_children=cur_ph0["cat"],cur_ph0["children"]


        # if cur_ph0_cat==cat0 and cur_ph0_children==children0 and len(children0)>1:
        #   #print("2","cur_ph0_cat",cur_ph0_cat,"cat0",cat0)
        #   return False #and wt0<=cur_top_phrase["wt"]: return False


      cur_phrases+=[phrase_obj0]
      cur_phrases.sort(key=lambda x:-x["wt"])

    temp_start_sub_dict2[cat0]=cur_phrases[:self.max_n_phrases]
    temp_end_sub_dict[start0]=temp_start_sub_dict2
    self.end_start_phrase_dict[end0]=temp_end_sub_dict

    if cur_phrases[0]["wt"]>found_span_wt0:
      self.span_wt_dict[span0]=cur_phrases[0]["wt"]
      self.span_phrase_dict[span0]=cur_phrases[0]


    head_loc0=phrase_obj0["head_loc"]
    head_loc_cat_pair=(head_loc0,cat0)
    updated_head_loc_cat_proj_level=self.head_loc_cat_projection_counter.get(head_loc_cat_pair,0)+1
    #if updated_head_loc_cat_proj_level>self.max
    self.head_loc_cat_projection_counter[head_loc_cat_pair]=updated_head_loc_cat_proj_level


    return True


  def combine_phrases(self,phrase_obj_list,applied_rule):
    parent_phrase_obj={}
    #cur_child_keys=[v["i"] for v in phrase_obj_list]
    cur_child_keys=[self.get_phrase_key(v) for v in phrase_obj_list]

    parent_phrase_obj["wt"]=sum([vw["wt"] for vw in phrase_obj_list])
    phrase_start0=min([vw["start"] for vw in phrase_obj_list])
    phrase_end0=max([vw["end"] for vw in phrase_obj_list])

    parent_phrase_obj["span"]=phrase_end0-phrase_start0+1
    parent_phrase_obj["start"]=phrase_start0
    parent_phrase_obj["end"]=phrase_end0

    parent_phrase_obj["cat"]=applied_rule["parent"]["cat"]
    parent_phrase_obj["feat"]=applied_rule["parent"]["feat"]
    rule_head_i=applied_rule["head_i"]
    parent_phrase_obj["children"]=cur_child_keys
    head_phrase_index=cur_child_keys[rule_head_i] #which of the child phrases is the head

    parent_phrase_obj["head_phrase"]=head_phrase_index
    # head_phrase_key=self.phrase_key_list[head_phrase_index]
    head_phrase_obj=phrase_obj_list[rule_head_i]  #self.main_phrase_dict[head_phrase_key][0]


    parent_phrase_obj["head_loc"]=head_phrase_obj["head_loc"]

    parent_phrase_obj["rule"]=applied_rule
    return parent_phrase_obj
    #parent_phrase_obj["level"]=new_level



  def get_phrase_key(self,phrase_obj0): #check an obj for corresponding key
    start0,end0,cat0,feat_split0=phrase_obj0["start"],phrase_obj0["end"],phrase_obj0["cat"],phrase_obj0["feat"]
    ft_str0=" ".join(feat_split0)
    cur_phrase_key=(start0,end0,cat0,ft_str0)
    return cur_phrase_key


  def get_rules(self,cur_phrase_obj):
    #given a token or phrase with cat/feat pair, identify the rules that apply
    all_rules=[]
    phrase_obj_cat,phrase_obj_feat=cur_phrase_obj["cat"],cur_phrase_obj["feat"]
    cur_cat_rules=self.cat_inv_index.get(phrase_obj_cat,[])
    all_rules.extend(cur_cat_rules)
    for s_feat0 in phrase_obj_feat:
      corr_rules=self.feat_inv_index.get(s_feat0,[])
      all_rules.extend(corr_rules)
    return list(set(all_rules))

  def get_children(self,phrase_obj):
    all_phrases=[phrase_obj]
    children=phrase_obj.get("children",[])
    new_children=[]
    while children: #go on a semi-recursive way to identify children and subchildren of current main phrase
      for ch0 in children:
        ch_phrase_key=self.phrase_key_list[ch0]
        child_phrase=self.main_phrase_dict[ch_phrase_key][0]
        all_phrases.append(child_phrase)
        sub_children=child_phrase.get("children",())
        sub_children=list(sub_children)
        for sc in sub_children:
          if sc==ch0: continue
          new_children.append(sc) #print("sc",sc)
      children=new_children
      new_children=[]
    return all_phrases
  def inspect_span(self,start,end,cat=None):
    end_temp_dict=self.end_start_phrase_dict.get(end,{})
    final_temp_dict=end_temp_dict.get(start,{})
    if cat!=None: final_temp_dict=final_temp_dict.get(cat,{})
    return final_temp_dict


  def optimum_path(self,words,span_phrase_dict): #apply Djkestra's algorithm to get the optimum path of phrases
    inc0=self.djk_max_span_distance
    dist_penalty0=self.djk_distance_penalty_ratio
    use_negative=self.djk_use_negative_wt

    span_list=sorted(list(span_phrase_dict.keys()))
    span_list=[v for v in span_list if (v[1]-v[0])>0] #only choose multi word phrases
    first_pt0=(-1,-1)
    last_pt0=(len(words),len(words))
    span_list=[first_pt0]+span_list+[last_pt0]
    grouped=[(key,list(group)) for key,group in groupby(span_list,lambda x:x[0])]
    grouped_dict=dict(iter(grouped))


    transition_dict={}
    for span0 in span_list:
      span0_start,span0_end=span0
      transition_dict[span0]={}
      all_corr_spans=[]
      for span1_start in range(span0_end+1,span0_end+inc0+1):
        corr_spans=grouped_dict.get(span1_start,[])
        span_dist=span1_start-span0_end-1
        cur_dist_penalty=span_dist*dist_penalty0
        for span1 in corr_spans:
          cur_phrase=span_phrase_dict.get(span1,{})
          span1_wt=cur_phrase.get("wt",0)
          adj_wt=span1_wt-cur_dist_penalty
          if use_negative: adj_wt=-adj_wt
          transition_dict[span0][span1]=adj_wt
      if transition_dict[span0]=={}: transition_dict[span0]={last_pt0:0}

    path0,path_wt0=general.djk(first_pt0,last_pt0,transition_dict,pt_list0=span_list)

    final_top_phrases=[]
    for span0 in path0:
      corr_phrase=span_phrase_dict.get(span0)
      if corr_phrase!=None: final_top_phrases.append(corr_phrase)
    return final_top_phrases



#End of new parser def

##Run Parser ★

In [ ]:
import re, copy
import itertools
from itertools import product
from code_utils.parsing_lib import *

file_id="jan_2026"
info_fname=f"rules_word_features_{file_id}.json"
rules_dir="../data/rules"
info_fpath=os.path.join(rules_dir,info_fname)
with open(info_fpath) as fopen:
  rules_word_features_info=json.load(fopen)

final_word_features_dict=rules_word_features_info["final_word_features_dict"]
xpos_ft_dict=rules_word_features_info["xpos_ft_dict"]
rules=rules_word_features_info["final_rules"]


sent="he is happy"
sent="the man * is happy"

sent="the man * is happy"
sent="the man is happy"
sent="the old man in the kitchen"
sent="the old man in the kitchen is not happy"
sent="what also happened in Germany"
sent="the old man also asked the same question"
sent="the old man has also asked the same questions"
sent="however , we didn't have any good answers"
sent="I will probably ask him to come some time later"
#sent="I will probably ask him to come some time later"
sent="Ecuador also supports initiatives for the protection, conservation and sustainable use of the ocean, such as the prompt entry into force"
#sent="Ecuador also supports initiatives for the protection, conservation and sustainable use of the ocean, such as the prompt entry into force of the Agreement under the United Nations Convention on the Law of the Sea on the Conservation and Sustainable Use of Marine Biological Diversity of Areas beyond National Jurisdiction, and the holding of the third United Nations Conference on Oceans in 2025."

#sent="Ecuador , , , , also supports initiatives for the protection, conservation and sustainable use of the ocean, such as the prompt entry into force of the Agreement under the United Nations Convention on the Law of the Sea on the Conservation and Sustainable Use of Marine Biological Diversity of Areas beyond National Jurisdiction, and the holding of the third United Nations Conference on Oceans in 2025."
sent="Our commitment is also reflected in the efforts made by the Ecuadorian presidency of the Committee in charge of developing a legally binding instrument to eliminate plastics pollution."
#sent="he is happy"
#sent="I will probably ask him to come some time later"
# sent="I will probably ask him to come some time later"
#sent="the old man is happy"
#sent="the old XXXX ?  + man is happy"
# sent = "the man in the kitchen on the table is happy"
# sent = "the man is happy"

#sent = "the man sings"
#sent = "the man in the kitchen sings a song"
#sent = "the man and the woman in the kitchen sing a song"
#sent = "the old evil man"

sent=re.sub(r"(\W)",r" \1 ",sent)
sent_toks=sent.strip().split()


#rules=final_rules

print(sent_toks)

rnn_model_fpath="../models/pos_models/H64_L3_LR001_updated_xpos_loading.model"
#rnn_model_fpath=None
params0={"pos_model_path":rnn_model_fpath,"final_word_features_dict":final_word_features_dict,"xpos_ft_dict":xpos_ft_dict}
params0["debug"]=False
params0["max_n_phrases"]=3
params0["max_skip_distance"]=2
import time
#parser_obj=Parser(rules_list=rules,params=params0)
parser_obj=Parser(rules_list=rules,params=params0)
t0=time.time()

parser_outcome_dict=parser_obj.parse(sent_toks)
final_parses=parser_outcome_dict["parses"]
pos_tags=parser_outcome_dict["pos"]
t1=time.time()

from IPython.display import HTML, display
fp0,dep0,const0=final_parses[0]
html_output=viz_parse(dep0,const0)
display(HTML(html_output))

# for fp0,dep0,const0 in final_parses:
#   print("main parse phrase:", fp0)
#   #dep0,const0=parser_obj.export_parse2(sent_toks,fp0)
#   for d0 in dep0:
#     print(d0)
#   print("---")
#   for c0 in const0: print(c0)
#   print("=======")
print("final_parses",len(final_parses))
print(t1-t0)

rules_word_features_info {'final_word_features_dict': {'': [['', []], ['', []]], '(': [['-LRB-', []]], ')': [['-RRB-', []]], ',': [['COMMA', []]], '-': [['HYPHEN', []]], ':': [['COLON', []]], 'I': [['PRON', ['noun', '1st', 'singular', 'subj']]], '[': [['-LRB-', []]], ']': [['-RRB-', []]], 'a': [['DT', []]], 'a lot': [['', []]], 'a lot of': [['', []]], 'about': [['', []]], 'across': [['', []]], 'after': [['', []]], 'against': [['', []]], 'ago': [['', []]], 'all': [['RB', []], ['DT', []], ['', []]], 'almost': [['', []]], 'already': [['', []]], 'also': [['', []]], 'although': [['', []]], 'always': [['', []]], 'am': [['AUX', ['be', '1st', 'present']]], 'among': [['', []]], 'amongst': [['', []]], 'an': [['DT', []]], 'and': [['CC', []]], 'another': [['', []]], 'any': [['det|noun', []]], 'are': [['AUX', ['be', '3rd', 'plural', 'present']], ['AUX', ['be', '2nd', 'present']]], 'as': [['IN', []]], 'as well': [['', []]], 'as well as': [['', []]], 'at': [['IN', []]], 'be': [['AUX', ['be', 'base']]

final_parses 4
1.9834258556365967


In [ ]:
rules_word_features_info.keys()

dict_keys(['final_word_features_dict', 'upos_list', 'xpos_list', 'xpos_ft_dict', 'upos_ft_dict', 'final_rules', 'final_rules_v0'])

In [ ]:
span_wt_items=list(parser_obj.span_wt_dict.items())
span_wt_items.sort(key=lambda x:-x[-1])
for a in span_wt_items[:10]: print(a)

((0, 28), 26.748527075023656)
((0, 27), 25.748517313442235)
((1, 28), 25.748487075023654)
((0, 26), 25.74817673250199)
((1, 27), 24.748477313442233)
((0, 25), 24.748467977390295)
((2, 27), 24.74821638845444)
((1, 26), 24.748136732501987)
((1, 25), 23.748427977390293)
((2, 26), 23.748136892452244)


In [ ]:
for a in pos_tags: print(a)

('Our', [('PRON', ['poss', 'det', '1st', 'plural'], 1), ('PRP$', [], 0.9997376799583435)])
('commitment', [('NN', ['noun'], 1.0), ('JJ', ['adj'], 0.9969983100891113)])
('is', [('AUX', ['be', '3rd', 'singular', 'present'], 1), ('VBZ', ['verb'], 0.9999921321868896)])
('also', [('', [], 1), ('RB', ['adv'], 0.9999996423721313)])
('reflected', [('VBN', ['verb'], 0.9992351531982422), ('VB', ['verb'], 0.8263813853263855), ('VBD', ['verb'], 0.28955093026161194)])
('in', [('IN', ['loc'], 1), ('RP', ['adv'], 0.9533300995826721), ('RB', ['adv'], 0.9520432353019714)])
('the', [('DT', [], 1)])
('efforts', [('NNS', ['noun'], 0.9999934434890747)])
('made', [('VBD', ['verb'], 0.9993301630020142), ('VBN', ['verb'], 0.9339714050292969)])
('by', [('', [], 1), ('IN', [], 0.9999997615814209)])
('the', [('DT', [], 1)])
('Ecuadorian', [('NNP', ['noun'], 0.9999934434890747), ('JJ', ['adj'], 0.9998353719711304), ('NN', ['noun'], 0.9959347248077393)])
('presidency', [('NN', ['noun'], 1.0)])
('of', [('IN', [], 1

In [ ]:
parser_obj.inspect_span(24,27)

{'PP': [{'wt': 3.9997578934860227,
   'span': 4,
   'start': 24,
   'end': 27,
   'cat': 'PP',
   'feat': [],
   'children': [(24, 24, 'IN', 'loc'), (25, 27, 'NP', '')],
   'head_phrase': (24, 24, 'IN', 'loc'),
   'head_loc': 24,
   'rule': {'parent': {'cat': 'PP', 'feat': []},
    'children': [{'cat': 'IN', 'is_head': True, 'feat': []},
     {'cat': 'NP', 'is_head': False, 'feat': []}],
    'head_i': 0,
    'rule_i': 16},
   'level': 5},
  {'wt': 3.973998654499054,
   'span': 4,
   'start': 24,
   'end': 27,
   'cat': 'PP',
   'feat': [],
   'children': [(24, 24, 'IN', 'loc'), (25, 27, 'NP', '')],
   'head_phrase': (24, 24, 'IN', 'loc'),
   'head_loc': 24,
   'rule': {'parent': {'cat': 'PP', 'feat': []},
    'children': [{'cat': 'IN', 'is_head': True, 'feat': []},
     {'cat': 'NP', 'is_head': False, 'feat': []}],
    'head_i': 0,
    'rule_i': 16},
   'level': 4},
  {'wt': 3.4863546566200254,
   'span': 4,
   'start': 24,
   'end': 27,
   'cat': 'PP',
   'feat': [],
   'children': [(

In [ ]:
from IPython.display import HTML, display
fp0,dep0,const0=final_parses[0]
html_output=viz_parse(dep0,const0)
#display(HTML('<h1>Hello, World!</h1><p>This is a custom HTML snippet in Google Colab.</p>'))
display(HTML(html_output))


In [ ]:
def viz_parse(dep,const):
  table_items=[]
  ids=[d0["id"] for d0 in dep]
  tokens=[d0["word"] for d0 in dep]
  heads=[d0["head"] for d0 in dep]
  ids_labels=["IDs"]+ids
  tokens_labels=["Words"]+tokens
  heads_labels=["Heads"]+heads

  border_style='style="border: 2px solid black;"'
  ids_row="<tr>"+"".join([f"<td {border_style}>{v}</td>" for v in ids_labels])+"</tr>"
  tokens_row="<tr>"+"".join([f"<td {border_style}>{v}</td>" for v in tokens_labels])+"</tr>"
  head_row="<tr>"+"".join([f"<td {border_style}>{v}</td>" for v in heads_labels])+"</tr>"
  table_items+=[ids_row,tokens_row,head_row]
  head_phrase_keys_dict={}
  child_check_dict={}
  max_level=0
  min_updated_level=0
  for c0 in const:
    if c0.get('head_phrase')!=None: head_phrase_keys_dict[c0["head_phrase"]]=True
    for ch0 in c0.get("children",[]): child_check_dict[ch0]=True
    if c0["level"]>max_level: max_level=c0["level"]

  reverse_sorted_const_by_span=sorted(const,key=lambda x:-x.get("span",100))
  adj_const=[]
  adj_level_dict={}
  for c_i in range(len(reverse_sorted_const_by_span)):
    cur_const_obj=reverse_sorted_const_by_span[c_i]
    cur_const_obj_key=parser_obj.get_phrase_key(cur_const_obj)
    found_adj_level=adj_level_dict.get(cur_const_obj_key)
    if child_check_dict.get(cur_const_obj_key,False): #if a phrase is not a child of something, assign maximum level
      cur_const_obj["level"]=max_level
    if found_adj_level!=None:
      #print("original level",cur_const_obj["level"],"adj level",found_adj_level)
      cur_const_obj["level"]=found_adj_level

    cur_const_obj_level=cur_const_obj["level"]
    cur_obj_children=cur_const_obj["children"]

    for ch_key0 in cur_obj_children:
      updated_level=cur_const_obj["level"]-1
      if updated_level<min_updated_level: min_updated_level=updated_level
      adj_level_dict[ch_key0]=updated_level
    #if len(cur_obj_children)==0: cur_const_obj["level"]=0
    adj_const.append(cur_const_obj)
  adj_const1=[]
  for i0 in range(len(adj_const)):
    cur_c0=adj_const[i0]
    if len(cur_c0["children"])==0: cur_c0["level"]=min_updated_level-1
    adj_const1.append(cur_c0)
  # for a,b in head_phrase_keys_dict.items():
  #   print(a,b)
  #sorted_const_by_level=sorted(const,key=lambda x:x.get("level",100))
  sorted_const_by_level=sorted(adj_const1,key=lambda x:x.get("level",100))
  grouped_by_level=[(key,list(group)) for key,group in groupby(sorted_const_by_level,lambda x:x.get("level",100))]
  level_counter=0
  for key0,gl0 in grouped_by_level:
    #print(key0,gl0) #phrases at this level
    level_phrases_start_dict={}
    for g0 in gl0:
      start0=g0["start"]
      level_phrases_start_dict[start0]=g0
    cell_i=0
    first_cell=f'<td></td>'
    if level_counter==0: first_cell=f'<td>Tags</td>'
    level_counter+=1
    cur_row_items=[first_cell]
    while cell_i<len(dep):
      found_phrase=level_phrases_start_dict.get(cell_i)
      span0=1
      content=""
      if found_phrase!=None:
        phrase_key0=parser_obj.get_phrase_key(found_phrase)
        is_head=head_phrase_keys_dict.get(phrase_key0,False)
        #print("cell_i", cell_i,"is_head",is_head,"found_phrase",found_phrase)


        cat0=found_phrase["cat"]
        span0=found_phrase["span"]
        head_loc=found_phrase["head_loc"]
        head_word=tokens[head_loc]

        content=f"{cat0}"
        if span0>1: content=f"{cat0} | {head_word}"
        if is_head: content=f'<b><i>{content}</i></b>'
        #content=f"{cat0} | {head_word}"
        #print("content",content)
      cell_html=f'<td colspan="{span0}"  {border_style}>{content}</td>'
      if content=="": cell_html=f'<td colspan="{span0}">{content}</td>'

      cur_row_items.append(cell_html)
      cell_i+=span0
    combined_row_items_str="".join(cur_row_items)
    cur_row=f'<tr>{combined_row_items_str}</tr>'
    table_items.append(cur_row)
  table_rows_str="\n".join(table_items)
  table_str=f'<table style="width: 100%;text-align: center;">{table_rows_str}</table>'
  return table_str




fp0,dep0,const0=final_parses[0]
viz_parse(dep0,const0)

'<table style="width: 100%;text-align: center;"><tr><td style="border: 2px solid black;">IDs</td><td style="border: 2px solid black;">1</td><td style="border: 2px solid black;">2</td><td style="border: 2px solid black;">3</td><td style="border: 2px solid black;">4</td><td style="border: 2px solid black;">5</td><td style="border: 2px solid black;">6</td><td style="border: 2px solid black;">7</td><td style="border: 2px solid black;">8</td><td style="border: 2px solid black;">9</td><td style="border: 2px solid black;">10</td><td style="border: 2px solid black;">11</td></tr>\n<tr><td style="border: 2px solid black;">Words</td><td style="border: 2px solid black;">the</td><td style="border: 2px solid black;">man</td><td style="border: 2px solid black;">and</td><td style="border: 2px solid black;">the</td><td style="border: 2px solid black;">woman</td><td style="border: 2px solid black;">in</td><td style="border: 2px solid black;">the</td><td style="border: 2px solid black;">kitchen</td><td s

In [ ]:
for a in const0:
  print(a)

{'wt': 1, 'span': 1, 'start': 1, 'end': 1, 'cat': 'IN', 'feat': [], 'head_loc': 1, 'children': [], 'level': 0}
{'wt': 0.9999722242355347, 'span': 1, 'start': 0, 'end': 0, 'cat': 'VB', 'feat': ['verb'], 'head_loc': 0, 'children': [], 'level': 0}
{'wt': 1.0, 'span': 1, 'start': 2, 'end': 2, 'cat': 'NN', 'feat': ['noun'], 'head_loc': 2, 'children': [], 'level': 0}
{'wt': 0.9999822242355346, 'span': 1, 'start': 0, 'end': 0, 'cat': 'V', 'feat': [], 'children': [(0, 0, 'VB', 'verb')], 'head_phrase': (0, 0, 'VB', 'verb'), 'head_loc': 0, 'rule': {'parent': {'cat': 'V', 'feat': []}, 'children': [{'cat': 'X', 'is_head': False, 'feat': ['verb']}], 'head_i': 0, 'rule_i': 3}, 'level': 1}
{'wt': 1.00001, 'span': 1, 'start': 2, 'end': 2, 'cat': 'NP-U', 'feat': [], 'children': [(2, 2, 'NN', 'noun')], 'head_phrase': (2, 2, 'NN', 'noun'), 'head_loc': 2, 'rule': {'parent': {'cat': 'NP-U', 'feat': []}, 'children': [{'cat': 'X', 'is_head': False, 'feat': ['noun']}], 'head_i': 0, 'rule_i': 1}, 'level': 1}
{

In [ ]:
for fp0,dep0,const0 in final_parses:
  cur_dep_score=parser_obj.score_dep_obj(dep0)
  print("cur_dep_score",cur_dep_score)
  print(fp0)
  print("----")

cur_dep_score 0.8620689655172413
{'wt': 26.383968417491918, 'span': 29, 'start': 0, 'end': 28, 'cat': 'S', 'feat': [], 'children': [(0, 27, 'S', ''), (28, 28, '.', 'punc')], 'head_phrase': (0, 27, 'S', ''), 'head_loc': 4, 'rule': {'parent': {'cat': 'S', 'feat': []}, 'children': [{'cat': 'S', 'is_head': True, 'feat': []}, {'cat': '.', 'is_head': False, 'feat': []}], 'head_i': 0, 'rule_i': 24}, 'level': 1}
----
cur_dep_score 0.8620689655172413
{'wt': 26.38362783655167, 'span': 29, 'start': 0, 'end': 28, 'cat': 'S', 'feat': [], 'children': [(0, 26, 'S', ''), (28, 28, '.', 'punc')], 'head_phrase': (0, 26, 'S', ''), 'head_loc': 4, 'rule': {'parent': {'cat': 'S', 'feat': []}, 'children': [{'cat': 'S', 'is_head': True, 'feat': []}, {'cat': '.', 'is_head': False, 'feat': []}], 'head_i': 0, 'rule_i': 24}, 'level': 1}
----
cur_dep_score 0.8620689655172413
{'wt': 26.35823917850495, 'span': 29, 'start': 0, 'end': 28, 'cat': 'S', 'feat': [], 'children': [(0, 27, 'S', ''), (28, 28, '.', 'punc')], 'h

In [ ]:
ph0,dep0,const0=final_parses[0]

for c0 in const0:
  print(c0)

{'wt': 62.001795995254525, 'span': 67, 'start': 0, 'end': 66, 'cat': 'S', 'feat': [], 'children': [(0, 65, 'S', ''), (66, 66, '.', 'punc')], 'head_phrase': (0, 65, 'S', ''), 'head_loc': 2, 'rule': {'parent': {'cat': 'S', 'feat': []}, 'children': [{'cat': 'S', 'is_head': True, 'feat': []}, {'cat': '.', 'is_head': False, 'feat': []}], 'head_i': 0, 'rule_i': 24}, 'level': 1}
{'wt': 60.78805084863664, 'span': 66, 'start': 0, 'end': 65, 'cat': 'S', 'feat': [], 'children': [(0, 60, 'NP', ''), (61, 65, 'VP', '')], 'head_phrase': (61, 65, 'VP', ''), 'head_loc': 61, 'rule': {'parent': {'cat': 'S', 'feat': []}, 'children': [{'cat': 'NP', 'is_head': False, 'feat': []}, {'cat': 'VP', 'is_head': True, 'feat': []}], 'head_i': 1, 'rule_i': 23}, 'level': 7}
{'wt': 0.9999988079071045, 'span': 1, 'start': 66, 'end': 66, 'cat': '.', 'feat': ['punc'], 'head_loc': 66, 'children': [], 'level': 0}
{'wt': 55.87562979942323, 'span': 61, 'start': 0, 'end': 60, 'cat': 'NP', 'feat': [], 'children': [(0, 16, 'NP',

In [ ]:
parser_obj.optimum_path(sent_toks,parser_obj.span_phrase_dict)

[{'wt': 1.9987336993217468,
  'span': 2,
  'start': 6,
  'end': 7,
  'cat': 'VP-O',
  'feat': [],
  'children': [(6, 6, 'V', ''), (7, 7, 'NP', '')],
  'head_phrase': (6, 6, 'V', ''),
  'head_loc': 6,
  'rule': {'parent': {'cat': 'VP-O', 'feat': []},
   'children': [{'cat': 'V', 'is_head': True, 'feat': []},
    {'cat': 'NP', 'is_head': False, 'feat': []}],
   'head_i': 0,
   'rule_i': 6},
  'level': 3},
 {'wt': 13.999997735023499,
  'span': 16,
  'start': 8,
  'end': 23,
  'cat': 'PP',
  'feat': [],
  'children': [(8, 8, 'IN', ''), (9, 23, 'NP', '')],
  'head_phrase': (8, 8, 'IN', ''),
  'head_loc': 8,
  'rule': {'parent': {'cat': 'PP', 'feat': []},
   'children': [{'cat': 'IN', 'is_head': True, 'feat': []},
    {'cat': 'NP', 'is_head': False, 'feat': []}],
   'head_i': 0,
   'rule_i': 15},
  'level': 6},
 {'wt': 45.91094374656677,
  'span': 47,
  'start': 24,
  'end': 70,
  'cat': 'S',
  'feat': [],
  'children': [(24, 69, 'S', ''), (70, 70, '.', 'punc')],
  'head_phrase': (24, 69, 'S

In [ ]:
parser_obj.inspect_span(2,2)

{'COMMA': [{'wt': 1,
   'span': 1,
   'start': 2,
   'end': 2,
   'cat': 'COMMA',
   'feat': [],
   'head_loc': 2,
   'children': [],
   'level': 0}],
 ',': [{'wt': 0.9989338517189026,
   'span': 1,
   'start': 2,
   'end': 2,
   'cat': ',',
   'feat': ['punc'],
   'head_loc': 2,
   'children': [],
   'level': 0}]}

In [ ]:
from code_utils import general
span_wt_items=list(parser_obj.span_wt_dict.items())
span_wt_items.sort(key=lambda x:-x[-1])
span_wt_items.sort()

span_list=sorted(list(parser_obj.span_wt_dict.keys()))
span_list=[v for v in span_list if (v[1]-v[0])>0]
first_pt0=(-1,-1)
last_pt0=(len(sent_toks),len(sent_toks))
span_list=[first_pt0]+span_list+[last_pt0]
grouped=[(key,list(group)) for key,group in groupby(span_list,lambda x:x[0])]
grouped_dict=dict(iter(grouped))
inc0=3

transition_dict={}
for span0 in span_list:
  span0_start,span0_end=span0
  transition_dict[span0]={}
  all_corr_spans=[]
  for span1_start in range(span0_end+1,span0_end+inc0+1):
    corr_spans=grouped_dict.get(span1_start,[])
    span_dist=span1_start-span0_end-1
    cur_dist_penalty=span_dist*0.1
    for span1 in corr_spans:
      span1_wt=parser_obj.span_wt_dict.get(span1,0)
      adj_wt=span1_wt-cur_dist_penalty
      transition_dict[span0][span1]=adj_wt
  if transition_dict[span0]=={}: transition_dict[span0]={last_pt0:0}

path0,path_wt0=general.djk(first_pt0,last_pt0,transition_dict,pt_list0=span_list)

final_top_phrases=[]
for a in path0:
  print(a,parser_obj.span_phrase_dict.get(a))

print("path_wt0",path_wt0)
print("span_wt_items",len(span_wt_items))

path_wt0 59.91094148159027
(6, 7) {'wt': 1.9987336993217468, 'span': 2, 'start': 6, 'end': 7, 'cat': 'VP-O', 'feat': [], 'children': [(6, 6, 'V', ''), (7, 7, 'NP', '')], 'head_phrase': (6, 6, 'V', ''), 'head_loc': 6, 'rule': {'parent': {'cat': 'VP-O', 'feat': []}, 'children': [{'cat': 'V', 'is_head': True, 'feat': []}, {'cat': 'NP', 'is_head': False, 'feat': []}], 'head_i': 0, 'rule_i': 6}, 'level': 3}
(8, 23) {'wt': 13.999997735023499, 'span': 16, 'start': 8, 'end': 23, 'cat': 'PP', 'feat': [], 'children': [(8, 8, 'IN', ''), (9, 23, 'NP', '')], 'head_phrase': (8, 8, 'IN', ''), 'head_loc': 8, 'rule': {'parent': {'cat': 'PP', 'feat': []}, 'children': [{'cat': 'IN', 'is_head': True, 'feat': []}, {'cat': 'NP', 'is_head': False, 'feat': []}], 'head_i': 0, 'rule_i': 15}, 'level': 6}
(24, 70) {'wt': 45.91094374656677, 'span': 47, 'start': 24, 'end': 70, 'cat': 'S', 'feat': [], 'children': [(24, 69, 'S', ''), (70, 70, '.', 'punc')], 'head_phrase': (24, 69, 'S', ''), 'head_loc': 65, 'rule': {'

In [ ]:
items=list(parser_obj.span_phrase_dict.items())
for a in items[:5]: print(a)

((0, 0), [{'wt': 0.9999328851699829, 'span': 1, 'start': 0, 'end': 0, 'cat': 'NN', 'feat': ['noun'], 'head_loc': 0, 'children': [], 'level': 0}])
((1, 1), [{'wt': 1.0, 'span': 1, 'start': 1, 'end': 1, 'cat': 'RB', 'feat': ['adv'], 'head_loc': 1, 'children': [], 'level': 0}])
((2, 2), [{'wt': 0.999607264995575, 'span': 1, 'start': 2, 'end': 2, 'cat': 'VBZ', 'feat': ['verb'], 'head_loc': 2, 'children': [], 'level': 0}])
((0, 2), [{'wt': 1.9995401501655579, 'span': 3, 'start': 0, 'end': 2, 'cat': 'S', 'feat': [], 'children': [(0, 0, 'NP', ''), (2, 2, 'VP', '')], 'head_phrase': (2, 2, 'VP', ''), 'head_loc': 2, 'rule': {'parent': {'cat': 'S', 'feat': []}, 'children': [{'cat': 'NP', 'is_head': False, 'feat': []}, {'cat': 'VP', 'is_head': True, 'feat': []}], 'head_i': 1, 'rule_i': 22}, 'level': 3}])
((3, 3), [{'wt': 0.9999417066574097, 'span': 1, 'start': 3, 'end': 3, 'cat': 'NNS', 'feat': ['noun'], 'head_loc': 3, 'children': [], 'level': 0}])


In [ ]:
span_list[:10], span_list[-10:]

([(-1, -1),
  (0, 0),
  (0, 2),
  (0, 3),
  (0, 6),
  (0, 8),
  (0, 10),
  (0, 11),
  (0, 14),
  (0, 16)],
 [(62, 63),
  (62, 65),
  (63, 63),
  (63, 65),
  (64, 64),
  (64, 65),
  (65, 65),
  (66, 66),
  67,
  67])

##Run Parser 2

In [ ]:
rules=["DT --> X[det]",
        "NP-U --> X[noun]",
       "AP --> X[adj]",
       "V --> X[verb]",
       "V --> V^ X[adv]",
       "V --> X[adv] V^",
       "VP-O --> V^ NP",
       "VP-2O --> VP-O^ NP",
       "VP --> VP-O",
       "VP --> VP-2O",
       "VP --> V",
       "BE --> AUX[be]",
       "VP --> BE^ NP",
       "VP --> BE^ AP",
       "VP --> VP^ PP",
       "PP --> IN^ NP",
       "NP-DT --> DT NP-U^",
       "NP-U --> AP NP-U^",
       "NP-U --> NP-U NP-U^",
       "NP --> NP-U",
       "NP --> NP-DT",
       "NP --> NP^ PP",
       "S --> NP VP^",
       "S --> S^ .",
       "AND-NP --> CC NP^",
       "NP --> NP^ AND-NP",
       "BEEN --> AUX[been]",
       "HAVE --> AUX[have]",
       "BE --> HAVE BEEN^",
       "V-PAS --> BE VBN^",
       "VP --> V-PAS",
       "VBN --> AP",




]

unknown_tags=["N","V","JJ","RB"]
rnn_model_fpath="../models/pos_models/H64_L3_LR001_updated_xpos_loading.model"
# params0={"pos_model_path":rnn_model_fpath,"final_word_features_dict":final_word_features_dict,"xpos_ft_dict":xpos_ft_dict}
# params0["max_n_phrases"]=20
# params0["debug"]=False

params0={"pos_model_path":rnn_model_fpath,"final_word_features_dict":final_word_features_dict,"xpos_ft_dict":xpos_ft_dict}
params0["max_n_phrases"]=5
params0["debug"]=False


parser_obj=Parser(rules_list=rules, params=params0)
tokens=['5', 'star', 'detail', 'job']
tokens=['Two', 'other', 'Dulaim', 'leaders', 'have', 'been', 'killed', 'in', 'the', 'past', 'week', 'and', 'a', 'half', '.']
tokens=['[', 'This', 'killing', 'of', 'a', 'respected', 'cleric', 'will', 'be', 'causing', 'us', 'trouble', 'for', 'years', 'to', 'come', '.', ']']
tokens=['This', 'killing', 'of', 'a', 'respected', 'cleric']
#['His', 'mother', 'was', 'also', 'killed', 'in', 'the', 'attack', '.']
tokens=['a', 'respected', 'cleric']
tokens=['His', 'mother', 'was', 'also', 'killed', 'in', 'the', 'attack', '.']



#tokens=['have', 'been', 'killed']
#tokens=['have', 'been', 'killed',"in","Canada"]
#tokens=['in', 'the', 'past', 'week', 'and', 'a', 'half']

parses=parser_obj.parse2(tokens)
for ph_obj0,dep0,const0 in parses[:5]:
  print(ph_obj0)
  for d0 in dep0: print(d0)
  print("--")
  for c0 in const0: print(c0)
  print("====")

{'wt': 7.9976776242256165, 'span': 9, 'start': 0, 'end': 8, 'cat': 'S', 'feat': [], 'children': [(0, 7, 'S', ''), (8, 8, '.', 'punc')], 'head_phrase': (0, 7, 'S', ''), 'head_loc': 2, 'rule': {'parent': {'cat': 'S', 'feat': []}, 'children': [{'cat': 'S', 'is_head': True, 'feat': []}, {'cat': '.', 'is_head': False, 'feat': []}], 'head_i': 0, 'rule_i': 23}, 'level': 1}
{'id': '1', 'word': 'His', 'head': '2', 'xpos': 'PRON', 'head_word': 'mother', 'offset': -1}
{'id': '2', 'word': 'mother', 'head': '5', 'xpos': 'NN', 'head_word': 'killed', 'offset': -3}
{'id': '3', 'word': 'was', 'head': '0', 'xpos': '-', 'head_word': '^', 'offset': 0}
{'id': '4', 'word': 'also', 'head': '-', 'xpos': '-', 'head_word': '', 'offset': 100}
{'id': '5', 'word': 'killed', 'head': '3', 'xpos': 'VB', 'head_word': 'was', 'offset': 2}
{'id': '6', 'word': 'in', 'head': '5', 'xpos': 'IN', 'head_word': 'killed', 'offset': 1}
{'id': '7', 'word': 'the', 'head': '8', 'xpos': 'DT', 'head_word': 'attack', 'offset': -1}
{'id

In [ ]:
#check the phrases within a certain span
parser_obj.end_start_phrase_dict[2][0]

{'NP-DT': [{'wt': 2.3075806498527527,
   'span': 3,
   'start': 0,
   'end': 2,
   'cat': 'NP-DT',
   'feat': [],
   'children': [(0, 0, 'DT', ''), (1, 2, 'NP-U', '')],
   'head_phrase': (1, 2, 'NP-U', ''),
   'head_loc': 2,
   'rule': {'parent': {'cat': 'NP-DT', 'feat': []},
    'children': [{'cat': 'DT', 'is_head': False, 'feat': []},
     {'cat': 'NP-U', 'is_head': True, 'feat': []}],
    'head_i': 1,
    'rule_i': 16},
   'level': 3},
  {'wt': 1.999987006187439,
   'span': 3,
   'start': 0,
   'end': 2,
   'cat': 'NP-DT',
   'feat': [],
   'children': [(0, 0, 'DT', ''), (2, 2, 'NP-U', '')],
   'head_phrase': (2, 2, 'NP-U', ''),
   'head_loc': 2,
   'rule': {'parent': {'cat': 'NP-DT', 'feat': []},
    'children': [{'cat': 'DT', 'is_head': False, 'feat': []},
     {'cat': 'NP-U', 'is_head': True, 'feat': []}],
    'head_i': 1,
    'rule_i': 16},
   'level': 2}],
 'NP': [{'wt': 2.3075806498527527,
   'span': 3,
   'start': 0,
   'end': 2,
   'cat': 'NP',
   'feat': [],
   'children': 

In [ ]:

ph0={'wt': 2.3075806498527527,
   'span': 3,
   'start': 0,
   'end': 2,
   'cat': 'NP-DT',
   'feat': [],
   'children': [(0, 0, 'DT', ''), (1, 2, 'NP-U', '')],
   'head_phrase': (1, 2, 'NP-U', ''),
   'head_loc': 2,
   'rule': {'parent': {'cat': 'NP-DT', 'feat': []},
    'children': [{'cat': 'DT', 'is_head': False, 'feat': []},
     {'cat': 'NP-U', 'is_head': True, 'feat': []}],
    'head_i': 1,
    'rule_i': 16},
   'level': 3}
projected_phrases=parser_obj.project_phrase3(ph0)
for a in projected_phrases:
  check0=parser_obj.add_phrase3(a)
  print(check0,a)

True {'wt': 2.3075806498527527, 'span': 3, 'start': 0, 'end': 2, 'cat': 'NP', 'feat': [], 'children': [(0, 2, 'NP-DT', '')], 'head_phrase': (0, 2, 'NP-DT', ''), 'head_loc': 2, 'rule': {'parent': {'cat': 'NP', 'feat': []}, 'children': [{'cat': 'NP-DT', 'is_head': False, 'feat': []}], 'head_i': 0, 'rule_i': 20}, 'level': 4}


In [ ]:
items=list(parser_obj.head_loc_cat_projection_counter.items())
items.sort(key=lambda x:-x[-1])
for a in items[:50]: print(a)

((22, 'S'), 831)
((61, 'S'), 740)
((3, 'NP'), 568)
((11, 'S'), 482)
((30, 'NP'), 426)
((20, 'NP'), 422)
((8, 'VP-2O'), 406)
((16, 'S'), 404)
((2, 'VP-2O'), 394)
((8, 'NP'), 389)
((11, 'NP'), 350)
((16, 'NP'), 323)
((25, 'NP'), 237)
((2, 'VP'), 232)
((8, 'VP'), 220)
((33, 'NP'), 213)
((48, 'NP-U'), 211)
((16, 'VP-2O'), 194)
((11, 'VP-2O'), 193)
((36, 'NP'), 193)
((45, 'NP-U'), 190)
((46, 'NP-U'), 189)
((51, 'NP-U'), 188)
((16, 'VP-O'), 187)
((50, 'NP-U'), 184)
((39, 'NP'), 182)
((11, 'VP'), 181)
((42, 'NP-U'), 180)
((22, 'VP-2O'), 175)
((14, 'NP'), 170)
((16, 'VP'), 170)
((6, 'NP'), 159)
((44, 'NP-U'), 157)
((22, 'VP'), 153)
((8, 'S'), 147)
((41, 'NP-U'), 146)
((46, 'NP'), 144)
((22, 'NP'), 143)
((33, 'NP-U'), 143)
((17, 'PP'), 134)
((26, 'PP'), 126)
((30, 'NP-U'), 120)
((42, 'NP'), 110)
((28, 'NP-U'), 97)
((25, 'NP-U'), 95)
((22, 'NP-U'), 94)
((36, 'NP-U'), 93)
((29, 'NP-U'), 92)
((48, 'NP'), 92)
((51, 'NP'), 79)


In [ ]:
ph0={'wt': 8.04562282180786, 'span': 11, 'start': 0, 'end': 10, 'cat': 'S', 'feat': [], 'children': [(0, 7, 'NP', ''), (8, 10, 'VP', '')], 'head_phrase': (8, 10, 'VP', ''), 'head_loc': 8, 'rule': {'parent': {'cat': 'S', 'feat': []}, 'children': [{'cat': 'NP', 'is_head': False, 'feat': []}, {'cat': 'VP', 'is_head': True, 'feat': []}], 'head_i': 1}, 'level': 5}

dep0,const0=parser_obj.export_parse2(sent_toks,ph0)
for d0 in dep0:
  print(d0)

const0

{'id': '1', 'word': 'the', 'head': '2', 'xpos': 'DT', 'head_word': 'man', 'offset': -1}
{'id': '2', 'word': 'man', 'head': '9', 'xpos': 'NN', 'head_word': 'sings', 'offset': -7}
{'id': '3', 'word': 'and', 'head': '5', 'xpos': 'CC', 'head_word': 'woman', 'offset': -2}
{'id': '4', 'word': 'the', 'head': '5', 'xpos': 'DT', 'head_word': 'woman', 'offset': -1}
{'id': '5', 'word': 'woman', 'head': '2', 'xpos': 'NN', 'head_word': 'man', 'offset': 3}
{'id': '6', 'word': 'in', 'head': '5', 'xpos': 'IN', 'head_word': 'woman', 'offset': 1}
{'id': '7', 'word': 'the', 'head': '8', 'xpos': 'DT', 'head_word': 'kitchen', 'offset': -1}
{'id': '8', 'word': 'kitchen', 'head': '6', 'xpos': 'NN', 'head_word': 'in', 'offset': 2}
{'id': '9', 'word': 'sings', 'head': '0', 'xpos': 'VBZ', 'head_word': '^', 'offset': 0}
{'id': '10', 'word': 'a', 'head': '11', 'xpos': 'DT', 'head_word': 'song', 'offset': -1}
{'id': '11', 'word': 'song', 'head': '9', 'xpos': 'NN', 'head_word': 'sings', 'offset': 2}


In [ ]:
for a in parser_obj.tag_wt_list:
  print(a)

('Ecuador', [('NN', ['noun'], 0.9999328851699829), ('NNP', ['noun'], 0.5867447853088379)])
('also', [('', [], 1), ('RB', ['adv'], 1.0)])
('supports', [('VBZ', ['verb'], 0.999607264995575), ('NNS', ['noun'], 0.8740209341049194)])
('initiatives', [('NNS', ['noun'], 0.9999417066574097)])
('for', [('IN', [], 1)])
('the', [('DT', [], 1)])
('protection', [('NN', ['noun'], 0.9999990463256836)])
(',', [('CCONJ', [], 1), (',', ['punc'], 0.9999215602874756)])
('conservation', [('NN', ['noun'], 0.9999998807907104), ('VB', ['verb'], 0.1614929586648941)])
('and', [('CCONJ', [], 1), ('CC', [], 0.9999985694885254)])
('sustainable', [('JJ', ['adj'], 1.0), ('NN', ['noun'], 0.9999717473983765)])
('use', [('NN', ['noun'], 1.0), ('VB', ['verb'], 0.874775230884552), ('VBP', ['verb'], 0.5029446482658386)])
('of', [('IN', [], 1)])
('the', [('DT', [], 1)])
('ocean', [('NN', ['noun'], 1.0), ('JJ', ['adj'], 0.9997401833534241)])
(',', [('CCONJ', [], 1), (',', ['punc'], 0.999927282333374)])
('such', [('', [], 1)

In [ ]:
items0=list(parser_obj.end_start_phrase_dict.items())
for a,b in items0:
  print(a)
  #print(b)
  for a0,b0 in b.items():
    print(a0)
    for a1,b1 in b0.items():
      print(a1,b1)
  print("----")

0
0
DT [{'wt': 1, 'span': 1, 'start': 0, 'end': 0, 'cat': 'DT', 'feat': [], 'head_loc': 0, 'children': [], 'level': 0}]
----
1
1
NN [{'wt': 0.9999997615814209, 'span': 1, 'start': 1, 'end': 1, 'cat': 'NN', 'feat': ['noun'], 'head_loc': 1, 'children': [], 'level': 0}]
NP-U [{'wt': 1.0009997615814208, 'span': 1, 'start': 1, 'end': 1, 'cat': 'NP-U', 'feat': [], 'children': [(1, 1, 'NN', 'noun')], 'head_phrase': (1, 1, 'NN', 'noun'), 'head_loc': 1, 'rule': {'parent': {'cat': 'NP-U', 'feat': []}, 'children': [{'cat': 'X', 'is_head': False, 'feat': ['noun']}], 'head_i': 0}, 'level': 1}]
NP [{'wt': 1.0029997615814208, 'span': 1, 'start': 1, 'end': 1, 'cat': 'NP', 'feat': [], 'children': [(1, 1, 'NP-U', '')], 'head_phrase': (1, 1, 'NP-U', ''), 'head_loc': 1, 'rule': {'parent': {'cat': 'NP', 'feat': []}, 'children': [{'cat': 'NP-U', 'is_head': False, 'feat': []}], 'head_i': 0}, 'level': 2}]
0
NP-DT [{'wt': 2.0029997615814206, 'span': 2, 'start': 0, 'end': 1, 'cat': 'NP-DT', 'feat': [], 'childre

In [ ]:
#test_phrase={'wt': 1, 'span': 1, 'start': 0, 'end': 0, 'cat': 'DT', 'feat': [], 'head_loc': 0, 'children': [], 'level': 0}
test_phrase={'wt': 1.3, 'span': 3, 'start': 0, 'end': 2, 'cat': 'NP', 'feat': [], 'head_loc': 0, 'children': [], 'level': 0}
parser_obj.add_phrase3(test_phrase)

False

In [ ]:
for a,b in parser_obj.phrase_key_obj_dict.items():
  print(b)

{'wt': 1, 'span': 1, 'start': 0, 'end': 0, 'cat': 'DT', 'feat': [], 'head_loc': 0, 'children': [], 'level': 0}
{'wt': 0.9999997615814209, 'span': 1, 'start': 1, 'end': 1, 'cat': 'NN', 'feat': ['noun'], 'head_loc': 1, 'children': [], 'level': 0}
{'wt': 0.9996987581253052, 'span': 1, 'start': 2, 'end': 2, 'cat': 'VBZ', 'feat': ['verb'], 'head_loc': 2, 'children': [], 'level': 0}
{'wt': 0.9952141046524048, 'span': 1, 'start': 2, 'end': 2, 'cat': 'NNS', 'feat': ['noun'], 'head_loc': 2, 'children': [], 'level': 0}


In [ ]:
import re, copy
import itertools
from itertools import product
from code_utils.parsing_lib import *

class Parser:
  def __init__(self,rules_list=[],word_features_list=[],params={}) -> None:
    self.unknown_tags=params.get("unknown_tags",["N","V","JJ","RB"]) #maybe get the actual distribution of these tags from corpora
    self.sent_padding=params.get("sent_padding",False)
    self.default_wt=params.get("default_wt",0.5) #to apply on unknown tags
    self.min_pos_wt=params.get("min_pos_wt",0.1)
    self.pos_model_path=params.get("pos_model_path")
    self.final_word_features_dict=params.get("final_word_features_dict",{})
    self.xpos_ft_dict=params.get("xpos_ft_dict",{})
    self.max_n_phrases=params.get("max_n_phrases",10) #max n phrases per key
    self.max_skip_distance=params.get("max_skip_distance",3) #max number of tokens to skip between phrases
    self.debug=params.get("debug",False)


    self.pos_tagger=POS(self.pos_model_path)

    self.unknown_token_tags=[("N","noun"),("V","verb"),("JJ","adj"),("RB","adv")]
    self.unknown_token_tags_wt_dict={}
    #process all word features

    #process all rules
    self.all_processed_rules=[]
    self.cat_fwd_index=[]
    self.feat_fwd_index=[]
    for i0,r0 in enumerate(rules_list):
      processed_rule=process_rule(r0)
      if processed_rule==None: continue
      self.all_processed_rules.append(processed_rule)

    for i0, processed_rule in enumerate(self.all_processed_rules):
      last_child=processed_rule["children"][-1]
      last_child_cat=last_child["cat"]
      last_child_features=last_child["feat"]
      self.cat_fwd_index.append((last_child_cat,i0))
      for ft0 in last_child_features: self.feat_fwd_index.append((ft0,i0))
    self.cat_fwd_index.sort()
    self.feat_fwd_index.sort()
    self.cat_inv_index=dict(iter([(key,[v[1] for v in list(group)]) for key,group in groupby(self.cat_fwd_index,lambda x:x[0])]))
    self.feat_inv_index=dict(iter([(key,[v[1] for v in list(group)]) for key,group in groupby(self.feat_fwd_index,lambda x:x[0])]))
    #self.phrase_list=[]


  def parse2(self,tokens):
    self.phrase_list=[] #list of all phrase objects, with spans, weights, children
    self.cat_phrase_index={}
    self.feat_phrase_index={}

    # self.phrase_key_list=[] #phrase key is (start,end,cat,feat)
    # self.phrase_key_index_dict={} #mapping the number of each key to the corresponding key
    # self.ft_phrase_key_map_dict={} #an index to map each feature to the corresponding keys
    # self.cat_phrase_key_map_dict={} #an index to map each category to the corresponding keys
    # self.span_phrase_key_map_dict={} #an index to map each span (start,end) to the corresponding keys
    # self.main_phrase_dict={} #mapping each key to the actual phrases corresponding to this key - sorting phrases by weight
    # self.main_phrase_wt_dict={} #mapping each key to the heighest weight for any of its phrases

    self.end_start_phrase_dict={} #dict[end][start]=[{phrase_obj1},{phrase_obj2},{phrase_obj3}]
    self.span_wt_dict={} #dict[(start,end)]=wt
    self.phrase_key_obj_dict={} #dict[phrase_key]=phrase_obj


    self.projection_counter={}


    tokens_pos_list=self.pos_tagger.tag_words(tokens,min_wt=self.min_pos_wt)
    self.cur_tokens_pos_list=tokens_pos_list

    self.tag_wt_list=[]

    if self.debug: print(tokens_pos_list)
    for token_i,a in enumerate(tokens_pos_list): #process each token
      token0=a["word"]
      if self.debug: print(token_i,token0,len(self.phrase_key_list))
      # if self.debug:
      #   proj_items=list(self.projection_counter.items())
      #   proj_items.sort(key=lambda x:-x[-1])
      #   first_item=None
      #   if proj_items: first_item=proj_items[0]
      #   print(token_i,token0,len(self.phrase_list),len(self.phrase_key_list),"proj_items",len(proj_items),"first_item",proj_items[:3])

      xpos0=a.get("xpos",[])
      start0,end0=token_i,token_i
      wt=self.default_wt
      outcome=self.final_word_features_dict.get(token0.lower(),[]) #outcome is multiple tag/cat options, with their features
      outcome_wt=[(cat0,feat0,1) for cat0,feat0 in outcome] #use a weight of 1 for the specific closed class words
      temp_used_cat_list=[v[0] for v in outcome_wt]
      for xpos_tag0,xpos_wt0 in xpos0:
        if xpos_tag0 in temp_used_cat_list: continue
        xpos_ft0=self.xpos_ft_dict.get(xpos_tag0,[])
        outcome_wt.append((xpos_tag0,xpos_ft0,xpos_wt0))
      #if nothing found
      if outcome_wt==[]:
        for cat0,feat0 in self.unknown_token_tags: outcome_wt.append((cat0,feat0,self.default_wt))

      self.tag_wt_list.append((token0,outcome_wt))
      #for the current token, identify all the outcomes with their weights
      #print("token0",token0)
      all_token_new_phrases=[]
      for cat0,feat_split,wt0 in outcome_wt:
        if cat0=="": continue
        #only one phrase object per cat/features pair
        cur_phrase_obj={"wt":wt0,"span":1,"start":start0,"end":end0,"cat":cat0,"feat":feat_split,"head_loc":start0,"children":[],"level":0}
        self.add_phrase2(cur_phrase_obj)
        new_phrases=self.project_phrase3(cur_phrase_obj)
        all_token_new_phrases.extend(new_phrases)
      while all_token_new_phrases:
        all_token_new_phrases_copy=copy.deepcopy(all_token_new_phrases)
        all_token_new_phrases=[]
        for ph0 in all_token_new_phrases_copy:
          check=self.add_phrase2(ph0)
          print("check:", check,ph0)
          projected_new_phrases=self.project_phrase3(ph0)
          # if ph0["cat"]=="VP":
          #   print(ph0)
          #   for pp0 in projected_new_phrases: print("projected", pp0)
          all_token_new_phrases.extend(projected_new_phrases)


        # if cur_phrase_obj["cat"]=="VP":
        #   print("Projecting:",cur_phrase_obj)
        #   for np0 in new_phrases: print(np0)
        #   print("---")
        # #for np0 in new_phrases: self.add_phrase2(np0)
        # while new_phrases:

        #   new_phrases_copy=copy.deepcopy(new_phrases)
        #   new_phrases=[]
        #   for ph0 in new_phrases_copy:
        #     self.add_phrase2(ph0)


        #     projected_new_phrases=self.project_phrase3(ph0)
        #     if ph0["cat"]=="VP":
        #       print("projecting:", ph0)
        #       for pp0 in projected_new_phrases: print("new projected phrases:",pp0)
        #       print("-----")

        #     new_phrases.extend(projected_new_phrases)
        #     for proj0 in projected_new_phrases:
        #       #further_projected=self.project_phrase3(proj0)
        #       #new_phrases.extend(further_projected)
        #       if proj0["cat"]=="VP": print("projected:", proj0)
        #   print("new_phrases",new_phrases)
        #   print("==========")
    #print("new_phrases",new_phrases)
    final_parse_phrases=[]
    end_of_sentence_dict=self.end_start_phrase_dict.get(len(tokens)-1,{})
    full_span_phrases=end_of_sentence_dict.get(0,[])
    final_parse_phrases.extend(full_span_phrases)

    if final_parse_phrases==[]:
      span_wt_items=list(self.span_wt_dict.items())
      span_wt_items.sort(key=lambda x:-x[-1])
      for span0,wt0 in span_wt_items[:5]:
        start0,end0=span0
        phrases=parser_obj.end_start_phrase_dict[end0][start0]
        for ph0 in phrases: final_parse_phrases.append(ph0) #print(ph0)


    #return self.tag_wt_list
    return final_parse_phrases

  def project_phrase3(self,phrase_obj0): #only for maximum binary branching
    print("projecting:",phrase_obj0)
    cur_level=phrase_obj0.get("level",0)
    #phrase_obj0["level"]=phrase_obj0.get("level",0)+1
    cur_phrase_key=self.get_phrase_key(phrase_obj0)
    self.projection_counter[cur_phrase_key]=self.projection_counter.get(cur_phrase_key,0)+1
    if self.projection_counter.get(cur_phrase_key,0)>10: return []
    cur_start0=phrase_obj0["start"]


    cur_rules_numbers=self.get_rules(phrase_obj0)
    cur_rules_obj_list=[self.all_processed_rules[r_i] for r_i in cur_rules_numbers]
    cur_rules_obj_list.sort(key=lambda x:len(x["children"]))
    all_combined_phrases=[]
    for cur_rule_obj in cur_rules_obj_list:
      #if phrase_obj0["cat"]=="VP": print("getting corresponding rules:", phrase_obj0,cur_rule_obj)
      #print("getting corresponding rules:", phrase_obj0,cur_rule_obj)
    # for r_i in cur_rules_numbers:
    #   cur_rule_obj=self.all_processed_rules[r_i]
      cur_rule_children=cur_rule_obj["children"]
      #print("Rule:", r_i,"Children:", cur_rule_children)
      first_child,last_child=cur_rule_children[0],cur_rule_children[-1]
      match_last=match_rule(phrase_obj0,last_child)
      if not match_last: continue
      if len(cur_rule_children)==1:
        new_combined_phrase=self.combine_phrases([phrase_obj0],cur_rule_obj)
        new_combined_phrase["level"]=cur_level+1
        all_combined_phrases.append(new_combined_phrase)
        #print("new_combined_phrase",new_combined_phrase)
      else:
        min_prev_end=max(0,cur_start0-self.max_skip_distance)
        for temp_end0 in range(min_prev_end,cur_start0):
          print("temp_end0",temp_end0)
          corr_start_dict=self.end_start_phrase_dict.get(temp_end0,{})
          used_cat_dict={}
          for temp_start0,temp_start_phrases0 in corr_start_dict.items():
            #print("temp_start0",temp_start0)
            for tph0 in temp_start_phrases0:

              temp_cat0=tph0["cat"]
              if used_cat_dict.get(temp_cat0,False): continue #avoid processing all phrases of the same category
              used_cat_dict[temp_cat0]=True
              matched_phrase=match_rule(tph0,first_child)
              print(matched_phrase,"phrase:",tph0)
              if matched_phrase:
                new_combined_phrase=self.combine_phrases([tph0,phrase_obj0],cur_rule_obj)
                new_combined_phrase["level"]=cur_level+1
                all_combined_phrases.append(new_combined_phrase)
    print("all_combined_phrases",all_combined_phrases)
    print("---------")
    return all_combined_phrases



  def export_parse2(self,words,cur_phrase):
    start0,end0=cur_phrase["start"],cur_phrase["end"]
    print(cur_phrase)
    return
    phrase_head_loc_dict={cur_phrase["i"]: cur_phrase["head_loc"]} #identify the head location for each phrase obj/key is phrase number/phrase index in the phrase list

    dep_dict={} #identify which phrase head loc depends on which phrase head loc
    root_index=cur_phrase["head_loc"] #the root of the current phrase

    cur_tokens_xpos_dict={}
    #unsorted_phrases=sorted(final_phrases,key=lambda x:x["i"]) #they should be sorted anyway by their original order of being added to the list
    parse_phrases=[cur_phrase] #these are the phrases to be used for constituency parsing
    children=cur_phrase.get("children",[])
    new_children=[]

    while children: #go on a semi-recursive way to identify children and subchildren of current main phrase
      for ch0 in children:
        ch_phrase_key=self.phrase_key_list[ch0]
        child_phrase=self.main_phrase_dict[ch_phrase_key][0]
        phrase_head_loc_dict[ch0]=child_phrase["head_loc"]
        if child_phrase["span"]==1 and child_phrase["children"]==[]: #identify the matching XPOS for terminal phrases
          cur_tokens_xpos_dict[child_phrase["start"]]=child_phrase["cat"]
        parse_phrases.append(child_phrase)
        sub_children=child_phrase.get("children",())
        sub_children=list(sub_children)
        for sc in sub_children:
          if sc==ch0: continue
          new_children.append(sc) #print("sc",sc)
      children=new_children
      new_children=[]

    for a in parse_phrases:
      cur_children=a["children"]
      cur_phrase_head_loc=a["head_loc"]
      for ch0 in cur_children: #for each child of the current phrase, identify its head_loc
        ch_head_loc=phrase_head_loc_dict[ch0]
        if ch_head_loc==cur_phrase_head_loc: continue #skip if the head_loc for current child is the same as of the main phrase
        dep_dict[ch_head_loc]=cur_phrase_head_loc #update dependency dict between heads of two phrases

    final_dep_list=[]
    for i0,w0 in enumerate(words): #create dependency info for each word
      cur_id0=i0+1

      #identify head index
      if i0==root_index: #root word
        cur_dep="0"
        head_word="^"
        offset=0
      else:
        cur_dep=dep_dict.get(i0)
        if cur_dep==None:  #word without attachment
          cur_dep="-"
          head_word=""
          offset=100 #any random number for no offset
        else: #any other word
          head_word=words[cur_dep]
          offset=i0-cur_dep
          cur_dep=str(cur_dep+1)

      cur_xpos0=cur_tokens_xpos_dict.get(i0,"-")
      final_dep_list.append({"id":str(cur_id0),"word": w0,"head":cur_dep,"xpos":cur_xpos0,"head_word":head_word,"offset":offset})
    return final_dep_list, parse_phrases







  def add_phrase2(self,phrase_obj0): #only for maximum binary branching
    span0=start0,end0=phrase_obj0["start"],phrase_obj0["end"]
    wt0=phrase_obj0["wt"]
    found_wt0=self.span_wt_dict.get(span0,0)
    phrase_has_top_span_wt=False

    # if wt0>found_wt0:
    #   phrase_has_top_span_wt=True

    cat0,children0=phrase_obj0["cat"],phrase_obj0["children"]
    phrase_key0=self.get_phrase_key(phrase_obj0)
    used_phrase_key_dict={}


    temp_end_sub_dict=self.end_start_phrase_dict.get(end0,{})
    cur_phrases=temp_end_sub_dict.get(start0,[])

    #for ph0 in cur_phrases
    if cur_phrases: #check the top phrase if it is the same as the one we're adding
      top_cur_phrase=cur_phrases[0]
      top_cat0,top_children0=top_cur_phrase["cat"],top_cur_phrase["children"]
      if top_cat0==cat0 and top_children0==children0: return False
      lowest_cur_phrase=cur_phrases[-1] #check if the weight of the current phrase is less than the lowest weight of current phrases in this span
      lowest_phrase_wt=lowest_cur_phrase["wt"]
      if len(cur_phrases)==self.max_n_phrases and wt0<lowest_phrase_wt: return False
      key0=self.get_phrase_key(phrase_obj0)


    cur_phrases=[phrase_obj0]+cur_phrases
    cur_phrases.sort(key=lambda x:-x["wt"])
    temp_end_sub_dict[start0]=cur_phrases[:self.max_n_phrases]
    self.end_start_phrase_dict[end0]=temp_end_sub_dict

    self.span_wt_dict[span0]=wt0
    phrase_key0=self.get_phrase_key(phrase_obj0)
    self.phrase_key_obj_dict[phrase_key0]=phrase_obj0

    return True


  def combine_phrases(self,phrase_obj_list,applied_rule):
    parent_phrase_obj={}
    #cur_child_keys=[v["i"] for v in phrase_obj_list]
    cur_child_keys=[self.get_phrase_key(v) for v in phrase_obj_list]

    parent_phrase_obj["wt"]=sum([vw["wt"] for vw in phrase_obj_list])
    phrase_start0=min([vw["start"] for vw in phrase_obj_list])
    phrase_end0=max([vw["end"] for vw in phrase_obj_list])

    parent_phrase_obj["span"]=phrase_end0-phrase_start0+1
    parent_phrase_obj["start"]=phrase_start0
    parent_phrase_obj["end"]=phrase_end0

    parent_phrase_obj["cat"]=applied_rule["parent"]["cat"]
    parent_phrase_obj["feat"]=applied_rule["parent"]["feat"]
    rule_head_i=applied_rule["head_i"]
    parent_phrase_obj["children"]=cur_child_keys
    head_phrase_index=cur_child_keys[rule_head_i] #which of the child phrases is the head

    parent_phrase_obj["head_phrase"]=head_phrase_index
    # head_phrase_key=self.phrase_key_list[head_phrase_index]
    head_phrase_obj=phrase_obj_list[rule_head_i]  #self.main_phrase_dict[head_phrase_key][0]


    parent_phrase_obj["head_loc"]=head_phrase_obj["head_loc"]

    parent_phrase_obj["rule"]=applied_rule
    return parent_phrase_obj
    #parent_phrase_obj["level"]=new_level


  # def check_phrase_obj(self,phrase_obj0): #check an obj for corresponding key - if not, add the
  #   cur_phrase_key=self.get_phrase_key(phrase_obj0)
  #   cur_phrase_key_i=self.phrase_key_index_dict.get(cur_phrase_key)
  #   new_key=False
  #   if cur_phrase_key_i==None:
  #     cur_phrase_key_i=len(self.phrase_key_list)
  #     self.phrase_key_list.append(cur_phrase_key)
  #     new_key=True

  #   phrase_obj0["i"]=cur_phrase_key_i
  #   phrase_obj0["head_phrase"]=cur_phrase_key_i
  #   if new_key: self.update_phrase_indexes(cur_phrase_key,cur_phrase_key_i)
  #   self.update_phrase_mappings(phrase_obj0,cur_phrase_key)
  #   return phrase_obj0

  def get_phrase_key(self,phrase_obj0): #check an obj for corresponding key
    start0,end0,cat0,feat_split0=phrase_obj0["start"],phrase_obj0["end"],phrase_obj0["cat"],phrase_obj0["feat"]
    ft_str0=" ".join(feat_split0)
    cur_phrase_key=(start0,end0,cat0,ft_str0)
    return cur_phrase_key


  # def update_phrase_mappings(self,phrase_obj0,phrase_key0): #update lists and dictionaies organizing phrases
  #   start0,end0,phrase_obj_cat,feat_split0,phrase_key_i=phrase_obj0["start"],phrase_obj0["end"],phrase_obj0["cat"],phrase_obj0["feat"],phrase_obj0["i"]
  #   phrase_obj_wt=phrase_obj0["wt"]
  #   cur_children=phrase_obj0["children"]
  #   found_phrases=self.main_phrase_dict.get(phrase_key0,[])
  #   found_phrases_children=[v["children"] for v in found_phrases] #exclude
  #   if cur_children in found_phrases_children: return
  #   #adding the actual phrase to the corresponding key at the main phrase dict
  #   found_wt=self.main_phrase_wt_dict.get(phrase_key0,0) #update the top weight of the current key
  #   if phrase_obj_wt>found_wt: self.main_phrase_wt_dict[phrase_key0]=phrase_obj_wt
  #   updated_found_phrases=found_phrases+[phrase_obj0]
  #   updated_found_phrases.sort(key=lambda x:-x["wt"])
  #   updated_found_phrases=updated_found_phrases[:self.max_n_phrases] #use only top n phrases for each key
  #   self.main_phrase_dict[phrase_key0]=updated_found_phrases


  # def update_phrase_indexes(self,phrase_key0,phrase_key_i0): #features/Cat/Span to phrase keys
  #   start0,end0,cat0,feat_str0=phrase_key0
  #   feat_split0=feat_str0.split()
  #   cur_phrase_span=(start0,end0)
  #   for ft0 in feat_split0: #update the list of phrase keys associated with each feature
  #     temp_ft_key_indexes=self.ft_phrase_key_map_dict.get(ft0,[])
  #     updated_temp_ft_key_indexes=temp_ft_key_indexes+[phrase_key_i0]
  #     self.ft_phrase_key_map_dict[ft0]=list(set(updated_temp_ft_key_indexes))
  #   temp_cat_key_indexes=self.cat_phrase_key_map_dict.get(cat0,[]) #update list of keys corresponding to current phrase category
  #   updated_temp_cat_key_indexes=temp_cat_key_indexes+[phrase_key_i0]
  #   self.cat_phrase_key_map_dict[cat0]=list(set(updated_temp_cat_key_indexes))
  #   temp_span_key_indexes=self.span_phrase_key_map_dict.get(cur_phrase_span,[]) #update list of keys corresponding to current phrase span
  #   updated_temp_span_key_indexes=temp_span_key_indexes+[phrase_key_i0]
  #   self.span_phrase_key_map_dict[cur_phrase_span]=list(set(updated_temp_span_key_indexes))

  def get_rules(self,cur_phrase_obj):
    #given a token or phrase with cat/feat pair, identify the rules that apply
    all_rules=[]
    phrase_obj_cat,phrase_obj_feat=cur_phrase_obj["cat"],cur_phrase_obj["feat"]
    cur_cat_rules=self.cat_inv_index.get(phrase_obj_cat,[])
    all_rules.extend(cur_cat_rules)
    for s_feat0 in phrase_obj_feat:
      corr_rules=self.feat_inv_index.get(s_feat0,[])
      all_rules.extend(corr_rules)
    return list(set(all_rules))

  def get_children(self,phrase_obj):
    all_phrases=[phrase_obj]
    children=phrase_obj.get("children",[])
    new_children=[]
    while children: #go on a semi-recursive way to identify children and subchildren of current main phrase
      for ch0 in children:
        ch_phrase_key=self.phrase_key_list[ch0]
        child_phrase=self.main_phrase_dict[ch_phrase_key][0]
        all_phrases.append(child_phrase)
        sub_children=child_phrase.get("children",())
        sub_children=list(sub_children)
        for sc in sub_children:
          if sc==ch0: continue
          new_children.append(sc) #print("sc",sc)
      children=new_children
      new_children=[]
    return all_phrases

  # def scan_phrases2(self,rule_child,cur_phrase_start=None):
  #   #for any child of a given rule, scan all the phrases that match this child (both cat/feat)
  #   phrases_i_list=[]
  #   rule_child_info_cat,rule_child_info_feat=rule_child["cat"],rule_child["feat"]

  #   negative_feat_corr_indexes=[] #identify the keys of phrases marked with a negative feature (e.g. -DET)
  #   positive_feat_corr_indexes=[] #identify the keys of all other features
  #   cat_corr_indexes=[] #identify keys corresponding to current category

  #   #identify the keys for phrases with positive feature, while the rule specifies that it needs to be negative
  #   for feat0 in rule_child_info_feat:
  #     is_negative=False
  #     if feat0.startswith("-"): is_negative=True
  #     feat0=feat0.strip("-")
  #     corr_indexes=self.ft_phrase_key_map_dict.get(feat0,[])
  #     if is_negative: negative_feat_corr_indexes.extend(corr_indexes)
  #     else: positive_feat_corr_indexes.extend(corr_indexes)
  #   #subtract the set of phrases upon which the negative feature applies from the phrases upon which positive features apply
  #   combined_key_indexes=list(set(positive_feat_corr_indexes).difference(set(negative_feat_corr_indexes)))
  #   if rule_child_info_cat!="X": #if category is specified
  #     cat_corr_indexes=self.cat_phrase_key_map_dict.get(rule_child_info_cat,[])
  #     if rule_child_info_feat==[]: combined_key_indexes=cat_corr_indexes
  #     else: combined_key_indexes=list(set(cat_corr_indexes).intersection(set(combined_key_indexes)))

  #   #combined_key_indexes is a set of indexes corresponding to phrase keys, based on feature and cat of the current child

  #   final_applicable_keys=[]
  #   for key_i in combined_key_indexes:
  #     cur_key0=self.phrase_key_list[key_i]
  #     ph_start,ph_end,ph_cat,ph_feat=cur_key0
  #     if cur_phrase_start!=None and ph_end>=cur_phrase_start: continue
  #     final_applicable_keys.append(key_i)

  #   return final_applicable_keys



  # def parse(self,tokens):
  #   self.phrase_list=[] #list of all phrase objects, with spans, weights, children
  #   self.cat_phrase_index={}
  #   self.feat_phrase_index={}

  #   self.phrase_key_list=[] #phrase key is (start,end,cat,feat)
  #   self.phrase_key_index_dict={} #mapping the number of each key to the corresponding key
  #   self.ft_phrase_key_map_dict={} #an index to map each feature to the corresponding keys
  #   self.cat_phrase_key_map_dict={} #an index to map each category to the corresponding keys
  #   self.span_phrase_key_map_dict={} #an index to map each span (start,end) to the corresponding keys
  #   self.main_phrase_dict={} #mapping each key to the actual phrases corresponding to this key - sorting phrases by weight
  #   self.main_phrase_wt_dict={} #mapping each key to the heighest weight for any of its phrases

  #   self.projection_counter={}


  #   tokens_pos_list=self.pos_tagger.tag_words(tokens,min_wt=self.min_pos_wt)
  #   self.cur_tokens_pos_list=tokens_pos_list

  #   self.tag_wt_list=[]

  #   if self.debug: print(tokens_pos_list)
  #   for i,a in enumerate(tokens_pos_list): #process each token
  #     token0=a["word"]
  #     if self.debug:
  #       proj_items=list(self.projection_counter.items())
  #       proj_items.sort(key=lambda x:-x[-1])
  #       first_item=None
  #       if proj_items: first_item=proj_items[0]
  #       print(i,token0,len(self.phrase_list),len(self.phrase_key_list),"proj_items",len(proj_items),"first_item",proj_items[:3])

  #     xpos0=a.get("xpos",[])
  #     start,end=i,i
  #     wt=self.default_wt
  #     outcome=self.final_word_features_dict.get(token0,[]) #outcome is multiple tag/cat options, with their features
  #     outcome_wt=[(cat0,feat0,1) for cat0,feat0 in outcome] #use a weight of 1 for the specific closed class words
  #     temp_used_cat_list=[v[0] for v in outcome_wt]
  #     for xpos_tag0,xpos_wt0 in xpos0:
  #       if xpos_tag0 in temp_used_cat_list: continue
  #       xpos_ft0=self.xpos_ft_dict.get(xpos_tag0,[])
  #       outcome_wt.append((xpos_tag0,xpos_ft0,xpos_wt0))
  #     #if nothing found
  #     if outcome_wt==[]:
  #       for cat0,feat0 in self.unknown_token_tags: outcome_wt.append((cat0,feat0,self.default_wt))

  #     self.tag_wt_list.append((token0,outcome_wt))


  #     for cat0,feat0,wt0 in outcome_wt:
  #       feat_split=feat0#.split()
  #       #only one phrase object per cat/features pair
  #       cur_token_rules=[] #retrieved rules that apply to current token

  #       ft_str=" ".join(feat_split)
  #       cur_phrase_key=(start,end,cat0,ft_str)
  #       cur_phrase_key_i=self.phrase_key_index_dict.get(cur_phrase_key)

  #       cur_phrase_obj={"wt":wt0,"span":1,"start":start,"end":end,"cat":cat0,"feat":feat_split,"head_loc":start,"head_phrase":cur_phrase_key_i,"children":[],"i":cur_phrase_key_i,"level":0}
  #       self.add_phrase(cur_phrase_obj)
  #       new_phrases=self.project_phrase(cur_phrase_obj)
  #       #if self.debug: print("main terminal projection", len(new_phrases),cur_phrase_obj)
  #       for a in new_phrases: self.add_phrase(a)
  #       while new_phrases:
  #         new_phrases_copy=copy.deepcopy(new_phrases)
  #         new_phrases=[]
  #         for ph0 in new_phrases_copy:
  #           projected_phrases=self.project_phrase(ph0)
  #           #if self.debug: print("non-terminal projection", len(projected_phrases),ph0)
  #           new_phrases.extend(projected_phrases)
  #         for a in new_phrases: self.add_phrase(a)

  #   full_sent_span=(0,len(tokens)-1)
  #   full_span_keys_numbers=self.span_phrase_key_map_dict.get(full_sent_span,[])
  #   full_span_keys=[self.phrase_key_list[key_i] for key_i in full_span_keys_numbers]

  #   if full_span_keys==[]: #if no full span keys, we just get the top phrase key ordered by weight
  #     phrase_key_wt=list(self.main_phrase_wt_dict.items())
  #     phrase_key_wt.sort(key=lambda x:-x[-1])
  #     full_span_keys.append(phrase_key_wt[0][0])

  #   top_phrase_objects=[]
  #   for cur_key_tuple in full_span_keys:
  #     cur_key_phrases=self.main_phrase_dict[cur_key_tuple]
  #     top_phrase_objects.extend(cur_key_phrases)

  #   top_phrase_objects.sort(key=lambda x:-x["wt"])
  #   final_raw_parses=[]
  #   for phrase_obj0 in top_phrase_objects[:self.max_n_phrases]:
  #     dep0,const0=self.export_parse(tokens,phrase_obj0)
  #     final_raw_parses.append((phrase_obj0,dep0,const0))
  #   return final_raw_parses #sorted_phrase_list

  # def project_phrase(self,cur_phrase_obj):
  #   #if self.debug: print("projecting:",cur_phrase_obj)
  #   new_phrases=[]
  #   new_phrases2=[]
  #   new_level=cur_phrase_obj.get("level",0)+1

  #   ft_str=" ".join(cur_phrase_obj["feat"])
  #   cur_phrase_key=(cur_phrase_obj["start"],cur_phrase_obj["end"],cur_phrase_obj["cat"],ft_str)

  #   self.projection_counter[cur_phrase_key]=self.projection_counter.get(cur_phrase_key,0)+1

  #   if self.projection_counter.get(cur_phrase_key,0)>10: return []

  #   cur_phrase_key_i=self.phrase_key_index_dict.get(cur_phrase_key)

  #   rules=self.get_rules(cur_phrase_obj)
  #   for r_i in rules:
  #     rule_obj=self.all_processed_rules[r_i]
  #     rule_children=rule_obj["children"]
  #     rule_children_reversed=list(reversed(rule_children))
  #     last_child=rule_children_reversed[0]
  #     match_last=match_rule(cur_phrase_obj,last_child)

  #     if not match_last: continue

  #     rule_children_corr_phrases=[]

  #     rule_children_corr_phrase_keys=[]
  #     rule_children_corr_phrase_keys.append([cur_phrase_key_i])
  #     for child0 in rule_children_reversed[1:]:
  #       corr_phrases2=self.scan_phrases2(child0,cur_phrase_start=cur_phrase_obj["start"])
  #       if corr_phrases2==[]: break
  #       rule_children_corr_phrase_keys.append(corr_phrases2)

  #     if len(rule_children_corr_phrase_keys)!=len(rule_children): continue
  #     rule_children_corr_phrase_keys=list(reversed(rule_children_corr_phrase_keys))
  #     phrase_combinations2 = list(product(*rule_children_corr_phrase_keys))
  #     for pc in phrase_combinations2: #combinations of phrase ids
  #       parent_phrase_obj={}
  #       cur_child_keys=[self.phrase_key_list[vi] for vi in pc]
  #       phrase_obj_list=[self.main_phrase_dict[key0][0] for key0 in cur_child_keys] #get the top phrase object corresponding to each key
  #       phrase_obj_pairs=list(zip(phrase_obj_list,phrase_obj_list[1:]))
  #       skip_combination=False
  #       for p0,p1 in phrase_obj_pairs:
  #         distance=p1["start"]-p0["end"]-1
  #         if distance<0 or distance>self.max_skip_distance: skip_combination=True
  #         if skip_combination: break
  #       if skip_combination: continue

  #       parent_phrase_obj["wt"]=sum([vw["wt"] for vw in phrase_obj_list])
  #       phrase_start0=min([vw["start"] for vw in phrase_obj_list])
  #       phrase_end0=max([vw["end"] for vw in phrase_obj_list])

  #       parent_phrase_obj["span"]=phrase_end0-phrase_start0+1
  #       parent_phrase_obj["start"]=phrase_start0
  #       parent_phrase_obj["end"]=phrase_end0

  #       parent_phrase_obj["cat"]=rule_obj["parent"]["cat"]
  #       parent_phrase_obj["feat"]=rule_obj["parent"]["feat"]
  #       rule_head_i=rule_obj["head_i"]
  #       parent_phrase_obj["children"]=pc
  #       head_phrase_index=pc[rule_head_i]
  #       parent_phrase_obj["head_phrase"]=head_phrase_index
  #       head_phrase_key=self.phrase_key_list[head_phrase_index]

  #       head_phrase_obj=self.main_phrase_dict[head_phrase_key][0]
  #       parent_phrase_obj["head_loc"]=head_phrase_obj["head_loc"]

  #       parent_phrase_obj["rule_i"]=r_i
  #       parent_phrase_obj["level"]=new_level

  #       new_phrases2.append(parent_phrase_obj)


  #   return new_phrases2

  # def add_phrase(self,cur_phrase_obj):
  #   phrase_obj_cat,phrase_obj_feat=cur_phrase_obj["cat"],cur_phrase_obj["feat"]
  #   phrase_obj_start,phrase_obj_end=cur_phrase_obj["start"],cur_phrase_obj["end"]
  #   phrase_obj_wt=cur_phrase_obj["wt"]
  #   cur_phrase_span=(phrase_obj_start,phrase_obj_end)
  #   cur_children=cur_phrase_obj["children"]

  #   phrase_key=(phrase_obj_start,phrase_obj_end,phrase_obj_cat," ".join(phrase_obj_feat))
  #   phrase_key_i=self.phrase_key_index_dict.get(phrase_key)

  #   found_phrases=self.main_phrase_dict.get(phrase_key,[])
  #   found_phrases_children=[v["children"] for v in found_phrases] #exclude
  #   if cur_children in found_phrases_children: return

  #   cur_phrase_obj["i"]=phrase_key_i
  #   if phrase_key_i==None: #if no key exists, we create a key, and index the

  #     phrase_key_i=len(self.phrase_key_list)
  #     cur_phrase_obj["i"]=phrase_key_i
  #     self.phrase_key_index_dict[phrase_key]=phrase_key_i #map the phrase key to its number in the phrase key list
  #     self.phrase_key_list.append(phrase_key)
  #     for ft0 in phrase_obj_feat: #update the list of phrase keys associated with each feature
  #       temp_ft_key_indexes=self.ft_phrase_key_map_dict.get(ft0,[])
  #       updated_temp_ft_key_indexes=temp_ft_key_indexes+[phrase_key_i]
  #       self.ft_phrase_key_map_dict[ft0]=list(set(updated_temp_ft_key_indexes))

  #     temp_cat_key_indexes=self.cat_phrase_key_map_dict.get(phrase_obj_cat,[]) #update list of keys corresponding to current phrase category
  #     updated_temp_cat_key_indexes=temp_cat_key_indexes+[phrase_key_i]
  #     self.cat_phrase_key_map_dict[phrase_obj_cat]=list(set(updated_temp_cat_key_indexes))
  #     temp_span_key_indexes=self.span_phrase_key_map_dict.get(cur_phrase_span,[]) #update list of keys corresponding to current phrase span
  #     updated_temp_span_key_indexes=temp_span_key_indexes+[phrase_key_i]
  #     self.span_phrase_key_map_dict[cur_phrase_span]=list(set(updated_temp_span_key_indexes))

  #   #adding the actual phrase to the corresponding key at the main phrase dict
  #   found_wt=self.main_phrase_wt_dict.get(phrase_key,0) #update the top weight of the current key
  #   if phrase_obj_wt>found_wt: self.main_phrase_wt_dict[phrase_key]=phrase_obj_wt

  #   updated_found_phrases=found_phrases+[cur_phrase_obj]
  #   updated_found_phrases.sort(key=lambda x:-x["wt"])
  #   updated_found_phrases=updated_found_phrases[:self.max_n_phrases] #use only top n phrases for each key
  #   self.main_phrase_dict[phrase_key]=updated_found_phrases

  #   self.phrase_list.append(cur_phrase_obj)









  # def export_parse(self,words,phrase_obj):
  #   cur_phrase=phrase_obj
  #   start0,end0=cur_phrase["start"],cur_phrase["end"]
  #   phrase_head_loc_dict={cur_phrase["i"]: cur_phrase["head_loc"]} #identify the head location for each phrase obj/key is phrase number/phrase index in the phrase list

  #   dep_dict={} #identify which phrase head loc depends on which phrase head loc
  #   root_index=cur_phrase["head_loc"] #the root of the current phrase

  #   cur_tokens_xpos_dict={}
  #   #unsorted_phrases=sorted(final_phrases,key=lambda x:x["i"]) #they should be sorted anyway by their original order of being added to the list
  #   parse_phrases=[cur_phrase] #these are the phrases to be used for constituency parsing
  #   children=cur_phrase.get("children",[])
  #   new_children=[]

  #   while children: #go on a semi-recursive way to identify children and subchildren of current main phrase
  #     for ch0 in children:
  #       ch_phrase_key=self.phrase_key_list[ch0]
  #       child_phrase=self.main_phrase_dict[ch_phrase_key][0]
  #       phrase_head_loc_dict[ch0]=child_phrase["head_loc"]
  #       if child_phrase["span"]==1 and child_phrase["children"]==[]: #identify the matching XPOS for terminal phrases
  #         cur_tokens_xpos_dict[child_phrase["start"]]=child_phrase["cat"]
  #       parse_phrases.append(child_phrase)
  #       sub_children=child_phrase.get("children",())
  #       sub_children=list(sub_children)
  #       for sc in sub_children:
  #         if sc==ch0: continue
  #         new_children.append(sc) #print("sc",sc)
  #     children=new_children
  #     new_children=[]

  #   for a in parse_phrases:
  #     cur_children=a["children"]
  #     cur_phrase_head_loc=a["head_loc"]
  #     for ch0 in cur_children: #for each child of the current phrase, identify its head_loc
  #       ch_head_loc=phrase_head_loc_dict[ch0]
  #       if ch_head_loc==cur_phrase_head_loc: continue #skip if the head_loc for current child is the same as of the main phrase
  #       dep_dict[ch_head_loc]=cur_phrase_head_loc #update dependency dict between heads of two phrases

  #   final_dep_list=[]
  #   for i0,w0 in enumerate(words): #create dependency info for each word
  #     cur_id0=i0+1

  #     #identify head index
  #     if i0==root_index: #root word
  #       cur_dep="0"
  #       head_word="^"
  #       offset=0
  #     else:
  #       cur_dep=dep_dict.get(i0)
  #       if cur_dep==None:  #word without attachment
  #         cur_dep="-"
  #         head_word=""
  #         offset=100 #any random number for no offset
  #       else: #any other word
  #         head_word=words[cur_dep]
  #         offset=i0-cur_dep
  #         cur_dep=str(cur_dep+1)

  #     cur_xpos0=cur_tokens_xpos_dict.get(i0,"-")
  #     final_dep_list.append({"id":str(cur_id0),"word": w0,"head":cur_dep,"xpos":cur_xpos0,"head_word":head_word,"offset":offset})
  #   return final_dep_list, parse_phrases





In [ ]:
cur_ph0={'wt': 0.9996987581253052, 'span': 1, 'start': 2, 'end': 2, 'cat': 'VP', 'feat': [], 'children': [(2, 2, 'V', '')], 'head_phrase': (2, 2, 'V', ''), 'head_loc': 2, 'rule': {'parent': {'cat': 'VP', 'feat': []}, 'children': [{'cat': 'V', 'is_head': False, 'feat': []}], 'head_i': 0}}

parser_obj.project_phrase3(cur_ph0)

[{'wt': 1.999698519706726,
  'span': 2,
  'start': 1,
  'end': 2,
  'cat': 'S',
  'feat': [],
  'children': [(1, 1, 'NP', ''), (2, 2, 'VP', '')],
  'head_phrase': (2, 2, 'VP', ''),
  'head_loc': 2,
  'rule': {'parent': {'cat': 'S', 'feat': []},
   'children': [{'cat': 'NP', 'is_head': False, 'feat': []},
    {'cat': 'VP', 'is_head': True, 'feat': []}],
   'head_i': 1}}]

In [ ]:
span_wt_items=list(parser_obj.span_wt_dict.items())
span_wt_items.sort(key=lambda x:-x[-1])
for span0,wt0 in span_wt_items[:200]:
  start0,end0=span0
  phrases=parser_obj.end_start_phrase_dict[end0][start0]
  print(span0,wt0)
  for ph0 in phrases: print(ph0)
  print("----")

(0, 7) 4.983922779560089
{'wt': 4.983922779560089, 'span': 8, 'start': 0, 'end': 7, 'cat': 'NP', 'feat': [], 'children': [(0, 7, 'NP-DT', '')], 'head_phrase': (0, 7, 'NP-DT', ''), 'head_loc': 7, 'rule': {'parent': {'cat': 'NP', 'feat': []}, 'children': [{'cat': 'NP-DT', 'is_head': False, 'feat': []}], 'head_i': 0}}
{'wt': 4.983922779560089, 'span': 8, 'start': 0, 'end': 7, 'cat': 'NP-DT', 'feat': [], 'children': [(0, 0, 'DT', ''), (1, 7, 'NP-U', '')], 'head_phrase': (1, 7, 'NP-U', ''), 'head_loc': 7, 'rule': {'parent': {'cat': 'NP-DT', 'feat': []}, 'children': [{'cat': 'DT', 'is_head': False, 'feat': []}, {'cat': 'NP-U', 'is_head': True, 'feat': []}], 'head_i': 1}}
{'wt': 3.9999996423721313, 'span': 8, 'start': 0, 'end': 7, 'cat': 'NP', 'feat': [], 'children': [(0, 7, 'NP-DT', '')], 'head_phrase': (0, 7, 'NP-DT', ''), 'head_loc': 7, 'rule': {'parent': {'cat': 'NP', 'feat': []}, 'children': [{'cat': 'NP-DT', 'is_head': False, 'feat': []}], 'head_i': 0}}
{'wt': 3.9999996423721313, 'span'

In [ ]:
cur_items=list(parser_obj.end_start_phrase_dict.items())
for a0,b0 in cur_items:
  print(a0)
  for a1,b1 in b0.items():
    print(a1)
    for b1_item in b1: print(b1_item)
  print("------")

0
0
{'wt': 1, 'span': 1, 'start': 0, 'end': 0, 'cat': 'PRON', 'feat': ['poss', 'det', '1st', 'plural'], 'head_loc': 0, 'children': [], 'level': 0}
------
1
1
{'wt': 1.0, 'span': 1, 'start': 1, 'end': 1, 'cat': 'NP', 'feat': [], 'children': [(1, 1, 'NN', 'noun')], 'head_phrase': (1, 1, 'NN', 'noun'), 'head_loc': 1, 'rule': {'parent': {'cat': 'NP', 'feat': []}, 'children': [{'cat': 'X', 'is_head': False, 'feat': ['noun']}], 'head_i': 0}}
{'wt': 1.0, 'span': 1, 'start': 1, 'end': 1, 'cat': 'NN', 'feat': ['noun'], 'head_loc': 1, 'children': [], 'level': 0}
------
2
2
{'wt': 1, 'span': 1, 'start': 2, 'end': 2, 'cat': 'AUX', 'feat': ['be', '3rd', 'singular', 'present'], 'head_loc': 2, 'children': [], 'level': 0}
1
{'wt': 1.9999921321868896, 'span': 2, 'start': 1, 'end': 2, 'cat': 'S', 'feat': [], 'children': [(1, 1, 'NP', ''), (2, 2, 'VP', '')], 'head_phrase': (2, 2, 'VP', ''), 'head_loc': 2, 'rule': {'parent': {'cat': 'S', 'feat': []}, 'children': [{'cat': 'NP', 'is_head': False, 'feat': []

In [ ]:
items0=list(parser_obj.main_phrase_wt_dict.items())
print("items0",len(items0))
for a,b in items0[1000:1100]:
  phrase=sent_toks[a[0]:a[1]+1]
  print(a,b, phrase)

items0 133


In [ ]:
items0=list(parser_obj.main_phrase_dict.items())
for a,b in items0: print(a,b)

(0, 0, 'PRON', 'noun 3rd masc sentient|human') [{'wt': 1, 'span': 1, 'start': 0, 'end': 0, 'cat': 'PRON', 'feat': ['noun', '3rd', 'masc', 'sentient|human'], 'head_loc': 0, 'children': [], 'level': 0, 'i': 0, 'head_phrase': 0}]
(0, 0, 'NP', '') [{'wt': 1, 'span': 1, 'start': 0, 'end': 0, 'cat': 'NP', 'feat': [], 'children': [0], 'head_phrase': 1, 'head_loc': 0, 'rule': {'parent': {'cat': 'NP', 'feat': []}, 'children': [{'cat': 'X', 'is_head': False, 'feat': ['noun']}], 'head_i': 0}, 'i': 1}]
(0, 0, 'PRP', '') [{'wt': 0.9996054768562317, 'span': 1, 'start': 0, 'end': 0, 'cat': 'PRP', 'feat': [], 'head_loc': 0, 'children': [], 'level': 0, 'i': 2, 'head_phrase': 2}]
(1, 1, 'AUX', 'be 3rd singular present') [{'wt': 1, 'span': 1, 'start': 1, 'end': 1, 'cat': 'AUX', 'feat': ['be', '3rd', 'singular', 'present'], 'head_loc': 1, 'children': [], 'level': 0, 'i': 3, 'head_phrase': 3}]
(1, 1, 'VBZ', 'verb') [{'wt': 0.9999963045120239, 'span': 1, 'start': 1, 'end': 1, 'cat': 'VBZ', 'feat': ['verb'],

In [ ]:
sent="Ecuador also supports initiatives for the protection, conservation and sustainable use of the ocean, such as the prompt entry into force of the Agreement under the United Nations Convention on the Law of the Sea on the Conservation and Sustainable Use of Marine Biological Diversity of Areas beyond National Jurisdiction, and the holding of the third United Nations Conference on Oceans in 2025."
#sent="Our commitment is also reflected in the efforts made by the Ecuadorian presidency of the Committee in charge of developing a legally binding instrument to eliminate plastics pollution."
#sent="he is happy"
#sent="I will probably ask him to come some time later"
# sent="I will probably ask him to come some time later"
#sent="the old man is happy"
#sent="the old XXXX ?  + man is happy"
#sent = "the man in the kitchen on the table is happy"
sent=re.sub(r"(\W)",r" \1 ",sent)
sent_toks=sent.strip().split()

len(sent_toks)
all_spans=[]
for i0 in range(len(sent_toks)):
  for i1 in range(i0,len(sent_toks)):
    #print(i0,i1)
    all_spans.append((i0,i1))
print(len(all_spans))

2278


##Test parser

In [ ]:
sent="he is happy"
sent="the man * is happy"

sent="the man * is happy"
sent="the man is happy"
sent="the old man in the kitchen"
sent="the old man in the kitchen is not happy"
sent="what also happened in Germany"
sent="the old man also asked the same question"
sent="the old man has also asked the same questions"
sent="however , we didn't have any good answers"
sent="I will probably ask him to come some time later"
#sent="I will probably ask him to come some time later"
#sent="Ecuador also supports initiatives for the protection, conservation and sustainable use of the ocean, such as the prompt entry into force"
sent="Ecuador also supports initiatives for the protection, conservation and sustainable use of the ocean, such as the prompt entry into force of the Agreement under the United Nations Convention on the Law of the Sea on the Conservation and Sustainable Use of Marine Biological Diversity of Areas beyond National Jurisdiction, and the holding of the third United Nations Conference on Oceans in 2025."
sent="Our commitment is also reflected in the efforts made by the Ecuadorian presidency of the Committee in charge of developing a legally binding instrument to eliminate plastics pollution."
sent="he is happy"
sent="I will probably ask him to come some time later"
# sent="I will probably ask him to come some time later"
# sent="the man is happy"
sent="the old XXXX ?  + man is happy"
#sent = "the man in the kitchen on the table is happy"
sent=re.sub(r"(\W)",r" \1 ",sent)
sent_toks=sent.strip().split()


rules=["NP --> X[noun]",
       "AP --> JJ",
       "VP --> X^[be] AP",
       "VP --> X^[be] JJ",
       "VP --> X^[verb] NP",
       "S --> NP VP^",
       "V → V[be] V^[ing]",
       'AUX[neg] → AUX^ RB[neg]',
       "NP --> DT NP^",
       "NP --> AP NP^",
       "PP --> IN^ NP",
       "NP --> NP^ PP"
       ]

# rules=["NP --> X[noun]",
#        "AP --> JJ",
#        "VP --> X^[be] AP",
#        "VP --> X^[be] JJ",
#        "VP --> X^[verb] NP",
#        "S --> NP VP^",
#        "V → V[be] V^[ing]",
#        'AUX[neg] → AUX^ RB[neg]',
#        "NP --> DT NP^",
#        "NP --> AP NP^",
#        "PP --> IN^ NP",
#        ]


# rules=["NP --> X[noun]",
#        "AP --> JJ",
#        "VP --> X^[be] AP",
#        "VP --> X^[be] JJ",
#        "S --> NP VP^",
#        "V → V[be] V^[ing]",
#        'AUX[neg] → AUX^ RB[neg]',
#        "NP --> DT NP^",
#        "NP --> AP NP^",
#        ]



unknown_tags=["N","V","JJ","RB"]
#sent_toks=sent.split()
print(sent_toks)

rnn_model_fpath="../models/pos_models/H64_L3_LR001_updated_xpos_loading.model"
#rnn_model_fpath=None
params0={"pos_model_path":rnn_model_fpath,"final_word_features_dict":final_word_features_dict,"xpos_ft_dict":xpos_ft_dict}
params0["debug"]=True
params0["max_n_phrases"]=20
import time
parser_obj=Parser(rules_list=rules,params=params0)
t0=time.time()
final_parses=parser_obj.parse(sent_toks)
t1=time.time()

for fp0 in final_parses: print(fp0)
print(t1-t0)


['the', 'old', 'XXXX', '?', '+', 'man', 'is', 'happy']
[{'word': 'the', 'upos': [('DET', 0.999991774559021)], 'xpos': [('DT', 0.9991812109947205)]}, {'word': 'old', 'upos': [('ADJ', 1.0), ('NOUN', 1.0)], 'xpos': [('JJ', 0.9999953508377075), ('NN', 0.9999562501907349)]}, {'word': 'XXXX', 'upos': [('PROPN', 0.9999997615814209), ('NOUN', 0.999208390712738)], 'xpos': [('NNP', 0.9999916553497314), ('NN', 0.9977214932441711)]}, {'word': '?', 'upos': [('PUNCT', 1.0)], 'xpos': [('.', 1.0)]}, {'word': '+', 'upos': [('PUNCT', 0.9999868869781494)], 'xpos': []}, {'word': 'man', 'upos': [('NOUN', 1.0), ('ADJ', 0.11133862286806107)], 'xpos': [('NN', 1.0), ('JJ', 0.15407252311706543)]}, {'word': 'is', 'upos': [('AUX', 0.9999994039535522)], 'xpos': [('VBZ', 0.9999940395355225)]}, {'word': 'happy', 'upos': [('ADJ', 1.0), ('NOUN', 1.0)], 'xpos': [('JJ', 1.0), ('NN', 0.9999997615814209)]}]
0 the 0 0 proj_items 0 first_item []
1 old 1 1 proj_items 1 first_item [((0, 0, 'DT', ''), 1)]
2 XXXX 6 6 proj_items

In [ ]:
items=list(parser_obj.main_phrase_wt_dict.items())
items.sort(key=lambda x:-x[-1])
for a,b in items[:100]:
  print(a,round(b,4))
print(len(items))

(0, 63, 'S', '') 43.7568
(3, 63, 'S', '') 43.6371
(2, 63, 'S', '') 43.5112
(2, 63, 'VP', '') 42.7568
(5, 63, 'S', '') 42.6372
(3, 63, 'NP', '') 42.4757
(0, 63, 'NP', '') 42.4756
(2, 63, 'NP', '') 42.3497
(6, 63, 'S', '') 41.6372
(4, 63, 'PP', '') 41.4757
(0, 61, 'S', '') 41.4753
(3, 61, 'S', '') 40.9188
(2, 61, 'S', '') 40.7929
(10, 63, 'NP', '') 40.7573
(5, 63, 'NP', '') 40.7573
(8, 63, 'VP', '') 40.6372
(2, 61, 'VP', '') 40.4754
(0, 58, 'S', '') 40.4753
(0, 60, 'S', '') 40.4753
(0, 59, 'S', '') 40.4753
(5, 61, 'S', '') 39.9189
(3, 58, 'S', '') 39.9188
(3, 60, 'S', '') 39.9188
(3, 59, 'S', '') 39.9188
(2, 58, 'S', '') 39.7929
(2, 60, 'S', '') 39.7929
(2, 59, 'S', '') 39.7929
(3, 61, 'NP', '') 39.7574
(0, 61, 'NP', '') 39.7573
(11, 63, 'NP', '') 39.7573
(8, 63, 'NP', '') 39.7573
(8, 63, 'S', '') 39.6321
(10, 63, 'S', '') 39.632
(2, 61, 'NP', '') 39.6314
(2, 58, 'VP', '') 39.4754
(2, 60, 'VP', '') 39.4754
(2, 59, 'VP', '') 39.4753
(5, 58, 'S', '') 38.9189
(6, 61, 'S', '') 38.9189
(5, 60

##Apply Parser to data

##On Dev (Test) Data

In [ ]:
from code_utils.rnn_utils import *
from code_utils.parsing_lib import *
from code_utils.dep_lib import *


# res_file_fpath="res_dev_50rules-reranked.tsv"
# res_file_fpath="res_dev_expanded_rules_training-reranked.tsv"
#res_file_fpath="res_dev_expanded_rules_dev-test-reranked_fragments_15_jan_0.tsv"

#file_id="feb_2026.3.existing_questions_v_pas_vbn_q" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.3.existing_questions_v_pas_vbn_q_inv_ap" #also NP/AP - RB - VB with DO - WP- single quote

#file_id="feb_2026.3.existing_questions_v_pas_vbn_vbg_q" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.3.existing_questions_v_pas_vbn_vbg_q_test" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.3.existing_questions_v_pas_vbn_vbg_q_test2" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.3.existing_questions_v_pas_vbn_vbg_vb_q" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.3.existing_questions_v_pas_vbn_vbg_vb_q_be_np_ap_pp" #also NP/AP - RB - VB with DO - WP- single quote

file_id="feb_2026.3.existing_questions_v_pas_vbn_vbg_vb_q_be_np_ap_pp_np_dash_advp_vp_ap" #also NP/AP - RB - VB with DO - WP- single quote

#file_id="feb_2026.3.existing_questions_v_pas" #also NP/AP - RB - VB with DO - WP- single quote


# file_id="jan_2026.pp"
# file_id="jan_2026.0"
# file_id="jan_2026"
# file_id="feb_2026"

res_file_fpath="res_dev_expanded_rules_dev_08_feb_2026_be_rb_do.tsv"
#res_file_fpath="res_dev_expanded_rules_dev_07_feb_2026.tsv"
#res_file_fpath="res_dev_expanded_rules_dev_feb_2026.tsv"
#res_file_fpath="res_dev_expanded_rules_dev_3_feb.tsv"

info_fname=f"rules_word_features_{file_id}.json"
rules_dir="../data/rules"
info_fpath=os.path.join(rules_dir,info_fname)
with open(info_fpath) as fopen:
  rules_word_features_info=json.load(fopen)

final_word_features_dict=rules_word_features_info["final_word_features_dict"]
xpos_ft_dict=rules_word_features_info["xpos_ft_dict"]
rules=rules_word_features_info["final_rules"]




max_n_sents_per_dataset=None

#rules=final_rules

unknown_tags=["N","V","JJ","RB"]
rnn_model_fpath="../models/pos_models/H64_L3_LR001_updated_xpos_loading.model"
# params0={"pos_model_path":rnn_model_fpath,"final_word_features_dict":final_word_features_dict,"xpos_ft_dict":xpos_ft_dict}
# params0["max_n_phrases"]=20
# params0["debug"]=False

params0={"pos_model_path":rnn_model_fpath,"final_word_features_dict":final_word_features_dict,"xpos_ft_dict":xpos_ft_dict}
params0["max_n_phrases"]=3
params0["max_skip_distance"]=1
params0["debug"]=False
params0["djk_use_negative_wt"]=False


parser_obj1=Parser(rules_list=rules, params=params0)


train_files,dev_files,test_files=[],[],[]
ud_dir="../data/UD"
#excluded_tb_list=['UD_English-GENTLE']
treebank_dir_list=os.listdir(ud_dir)
for tb0 in treebank_dir_list:
  tb_full_path=os.path.join(ud_dir,tb0)
  tb_files=os.listdir(tb_full_path)
  for tbf0 in tb_files:
    cur_file=os.path.join(tb_full_path,tbf0)
    if "train" in tbf0.lower() and "conllu" in tbf0.lower(): train_files.append(cur_file)
    elif "dev" in tbf0.lower() and "conllu" in tbf0.lower(): dev_files.append(cur_file)
    elif "test" in tbf0.lower() and "conllu" in tbf0.lower(): test_files.append(cur_file)


#print("dev_files",dev_files[0])
cur_dev_file=dev_files[0]
cur_train_file=train_files[0]

cur_file=dev_files[0]
cur_file=dev_files[5]


#res_file_fpath="res_dev_50rules-TEST.tsv"
res_file_fopen=open(res_file_fpath,"w")

#all_cur_files=dev_files+test_files

#all_cur_files=dev_files

all_cur_files=test_files


#all_cur_files=train_files
final_content=""

for cur_file in all_cur_files:
  print(cur_file)
  if "en_gumreddit" in cur_file: continue

  # cur_train_file=train_files[1]
  # cur_train_file=train_files[2]
  # cur_train_file=train_files[3]

  cur_iter0=iter_conll_treebank(cur_file)

  total_correct,total_tokens=0,0
  n_sents=0
  for i0,item0 in enumerate(cur_iter0):
    n_sents+=1
    if i0%200==0: print(i0)
    if i0==max_n_sents_per_dataset: break

    cur_obj0=conll2obj(item0)
    cur_words=[v["form"] for v in cur_obj0]
    gold_heads=[v["head"] for v in cur_obj0]
    # print(cur_obj0[0])
    # print(len(cur_words),cur_words)
    #parser_obj1=Parser(rules_list=rules, params=params0)
    #final_parses=parser_obj1.parse(cur_words)
    parsing_outcome=parser_obj1.parse(cur_words)
    final_parses=parsing_outcome["parses"]
    ph_obj1,dep1,const1=final_parses[0]

    #dep1=parser_obj1.combined_dep

    parser_heads=[v["head"] for v in dep1]

    head_matching=[1 if v==w else 0 for v,w in zip(gold_heads,parser_heads)]

    total_tokens+=len(head_matching)
    local_correct=len([v for v in head_matching if v==1])
    total_correct+=local_correct
    local_uas=local_correct/len(head_matching)
    # if local_uas<0.2:
    #   print("local_uas",local_uas)
    #   print(cur_words)
    #   print("gold_heads",gold_heads)
    #   print("parser_heads",parser_heads)
    #   pos_list0=parser_obj1.cur_tokens_pos_list
    #   print("pos_list0",pos_list0)
    #   print("--------")


  avg_sent_n_tokens=total_tokens/n_sents
  print(cur_file)
  print("n_sents",n_sents,"avg_sent_n_tokens",round(avg_sent_n_tokens,1))
  print("total_correct",total_correct)
  print("total_tokens",total_tokens)
  uas_score=total_correct/total_tokens
  print("uas_score",uas_score)
  line0=f"{cur_file}\t{n_sents}\t{round(avg_sent_n_tokens,1)}\t{total_tokens}\t{total_correct}\t{round(uas_score,2)}\n"
  final_content+=line0
  res_file_fopen.write(line0)
res_file_fopen.close()

print(params0)
print("results saved to:", res_file_fpath)
print(final_content)

../data/UD/UD_English-GENTLE/en_gentle-ud-test.conllu
0
200
400
600
800
1000
1200
../data/UD/UD_English-GENTLE/en_gentle-ud-test.conllu
n_sents 1334 avg_sent_n_tokens 13.3
total_correct 8733
total_tokens 17799
uas_score 0.49064554188437554
../data/UD/UD_English-EWT/en_ewt-ud-test.conllu
0
200
400
600
800
1000
1200
1400
1600
1800
2000
../data/UD/UD_English-EWT/en_ewt-ud-test.conllu
n_sents 2077 avg_sent_n_tokens 12.1
total_correct 13027
total_tokens 25094
uas_score 0.5191280784251215
../data/UD/UD_English-CTeTex/en_ctetex-ud-test.conllu
0
200
../data/UD/UD_English-CTeTex/en_ctetex-ud-test.conllu
n_sents 276 avg_sent_n_tokens 33.6
total_correct 3942
total_tokens 9273
uas_score 0.4251051439663539
../data/UD/UD_English-CHILDES/en_childes-ud-test.conllu
0
200
400
600
800
1000
1200
1400
1600
1800
2000
2200
2400
2600
2800
3000
3200
3400
3600
3800
4000
4200
4400
4600
4800
5000
5200
5400
5600
5800
6000
6200
6400
6600
6800
7000
7200
7400
7600
7800
8000
8200
8400
8600
8800
9000
9200
9400
../data/

In [ ]:
#!cat res_dev_expanded_rules_dev-test-reranked_14_jan_0.tsv
#!cat res_dev_expanded_rules_dev-test-reranked_fragments_15_jan_0.tsv
!cat res_dev_expanded_rules_dev_jan_2026.tsv

../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu	2001	12.6	25147	9038	0.36
../data/UD/UD_English-CHILDES/en_childes-ud-dev.conllu	3860	6.3	24310	10898	0.45
../data/UD/UD_English-Atis/en_atis-ud-dev.conllu	572	11.6	6644	2305	0.35
../data/UD/UD_English-ESLSpok/en_eslspok-ud-dev.conllu	232	9.3	2167	894	0.41
../data/UD/UD_English-ParTUT/en_partut-ud-dev.conllu	156	17.4	2716	984	0.36
../data/UD/UD_English-LinES/en_lines-ud-dev.conllu	1118	19.4	21637	7844	0.36
../data/UD/UD_English-GUM/en_gum-ud-dev.conllu	1575	17.9	28119	9416	0.33


In [ ]:
all_cur_files

['../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu',
 '../data/UD/UD_English-CHILDES/en_childes-ud-dev.conllu',
 '../data/UD/UD_English-Atis/en_atis-ud-dev.conllu',
 '../data/UD/UD_English-ESLSpok/en_eslspok-ud-dev.conllu',
 '../data/UD/UD_English-ParTUT/en_partut-ud-dev.conllu',
 '../data/UD/UD_English-LinES/en_lines-ud-dev.conllu',
 '../data/UD/UD_English-GUMReddit/en_gumreddit-ud-dev.conllu',
 '../data/UD/UD_English-GUM/en_gum-ud-dev.conllu']

##On training data

In [ ]:
from code_utils.rnn_utils import *
from code_utils.parsing_lib import *
from code_utils.dep_lib import *

# file_id="feb_2026.pp_brackets"
#file_id="feb_2026.s_comma.1.be_rb" #also NP/AP - RB

#file_id="feb_2026.s_comma.1.be_rb_do" #also NP/AP - RB - VB with DO
#file_id="feb_2026.s_comma.1.be_rb_do_wp_single_quote_fixed" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.s_comma.2.np_colon_md_be_have_vbn" #also NP/AP - RB - VB with DO - WP- single quote

#file_id="feb_2026.3.existing_questions" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.3.existing_questions_v_pas_vbn_vbg_q" #also NP/AP - RB - VB with DO - WP- single quote
#file_id="feb_2026.3.existing_questions_v_pas_vbn_vbg_vb_q" #also NP/AP - RB - VB with DO - WP- single quote
file_id="feb_2026.3.existing_questions_v_pas_vbn_vbg_vb_q_be_np_ap_pp" #also NP/AP - RB - VB with DO - WP- single quote

# file_id="jan_2026.pp"
# file_id="jan_2026"

max_n_sents_per_dataset=100 #None

#res_file_fpath="res_dev_expanded_rules_dev-test-reranked_14_jan.tsv"
res_file_fpath="res_dev_expanded_rules_dev_4_feb.tsv"

info_fname=f"rules_word_features_{file_id}.json"
rules_dir="../data/rules"
info_fpath=os.path.join(rules_dir,info_fname)
with open(info_fpath) as fopen:
  rules_word_features_info=json.load(fopen)

final_word_features_dict=rules_word_features_info["final_word_features_dict"]
xpos_ft_dict=rules_word_features_info["xpos_ft_dict"]
rules=rules_word_features_info["final_rules"]


# res_file_fpath="res_dev_50rules-reranked.tsv"
# res_file_fpath="res_dev_expanded_rules_training-reranked.tsv"




#rules=final_rules

unknown_tags=["N","V","JJ","RB"]
rnn_model_fpath="../models/pos_models/H64_L3_LR001_updated_xpos_loading.model"
# params0={"pos_model_path":rnn_model_fpath,"final_word_features_dict":final_word_features_dict,"xpos_ft_dict":xpos_ft_dict}
# params0["max_n_phrases"]=20
# params0["debug"]=False

params0={"pos_model_path":rnn_model_fpath,"final_word_features_dict":final_word_features_dict,"xpos_ft_dict":xpos_ft_dict}
params0["max_n_phrases"]=5
params0["max_skip_distance"]=1
params0["debug"]=False
params0["djk_use_negative_wt"]=False


parser_obj1=Parser(rules_list=rules, params=params0)


train_files,dev_files,test_files=[],[],[]
ud_dir="../data/UD"
#excluded_tb_list=['UD_English-GENTLE']
treebank_dir_list=os.listdir(ud_dir)
for tb0 in treebank_dir_list:
  tb_full_path=os.path.join(ud_dir,tb0)
  tb_files=os.listdir(tb_full_path)
  for tbf0 in tb_files:
    cur_file=os.path.join(tb_full_path,tbf0)
    if "train" in tbf0.lower() and "conllu" in tbf0.lower(): train_files.append(cur_file)
    elif "dev" in tbf0.lower() and "conllu" in tbf0.lower(): dev_files.append(cur_file)
    elif "test" in tbf0.lower() and "conllu" in tbf0.lower(): test_files.append(cur_file)


#print("dev_files",dev_files[0])
cur_dev_file=dev_files[0]
cur_train_file=train_files[0]

cur_file=dev_files[0]
cur_file=dev_files[5]


#res_file_fpath="res_dev_50rules-TEST.tsv"
res_file_fopen=open(res_file_fpath,"w")
res_file_content=""


all_cur_files=dev_files+test_files


all_cur_files=train_files

sent_error_list=[]

for file_i,cur_file in enumerate(all_cur_files):
  print(file_i,cur_file)
  if "en_gumreddit" in cur_file: continue

  # cur_train_file=train_files[1]
  # cur_train_file=train_files[2]
  # cur_train_file=train_files[3]

  cur_iter0=iter_conll_treebank(cur_file)

  total_correct,total_tokens=0,0
  n_sents=0
  for i0,item0 in enumerate(cur_iter0):
    n_sents+=1
    if i0%100==0: print(i0)
    if i0==max_n_sents_per_dataset: break

    cur_obj0=conll2obj(item0)
    cur_words=[v["form"] for v in cur_obj0]
    gold_heads=[v["head"] for v in cur_obj0]
    # print(cur_obj0[0])
    # print(len(cur_words),cur_words)
    #parser_obj1=Parser(rules_list=rules, params=params0)
    #final_parses=parser_obj1.parse(cur_words)
    parsing_outcome=parser_obj1.parse(cur_words)
    final_parses=parsing_outcome["parses"]
    ph_obj1,dep1,const1=final_parses[0]

    #dep1=parser_obj1.combined_dep

    parser_heads=[v["head"] for v in dep1]

    head_matching=[1 if v==w else 0 for v,w in zip(gold_heads,parser_heads)]

    total_tokens+=len(head_matching)
    local_correct=len([v for v in head_matching if v==1])
    total_correct+=local_correct
    local_uas=local_correct/len(head_matching)
    if local_uas<0.2:
      sent_error_obj={}
      sent_error_obj["uas"]=local_uas
      sent_error_obj["size"]=len(cur_words)
      sent_error_obj["words"]=cur_words
      sent_error_obj["file"]=cur_file
      sent_error_obj["sent_i"]=i0
      sent_error_obj["final_parses"]=final_parses
      sent_error_obj["parse_pos_list"]=parser_obj1.tag_wt_list #parser_obj1.cur_tokens_pos_list
      sent_error_obj["gold_conll"]=item0
      sent_error_obj["gold_heads"]=gold_heads
      sent_error_obj["parser_heads"]=parser_heads
      sent_error_obj["obj"]=parser_obj1
      sent_error_list.append(sent_error_obj)



      #print("local_uas",local_uas)
      # print([file_i, i0],"local_uas",local_uas)
      # print(cur_words)

    # if local_uas<0.2:
    #   print("local_uas",local_uas)
    #   print(cur_words)
    #   print("gold_heads",gold_heads)
    #   print("parser_heads",parser_heads)
    #   pos_list0=parser_obj1.cur_tokens_pos_list
    #   print("pos_list0",pos_list0)
    #   print("--------")


  avg_sent_n_tokens=total_tokens/n_sents
  print(cur_file)
  print("n_sents",n_sents,"avg_sent_n_tokens",round(avg_sent_n_tokens,1))
  print("total_correct",total_correct)
  print("total_tokens",total_tokens)
  uas_score=total_correct/total_tokens
  print("uas_score",uas_score)
  line0=f"{cur_file}\t{n_sents}\t{round(avg_sent_n_tokens,1)}\t{total_tokens}\t{total_correct}\t{round(uas_score,2)}\n"
  res_file_fopen.write(line0)
  res_file_content+=line0
res_file_fopen.close()

print(params0)
print("results saved to:", res_file_fpath)
print(res_file_content)

0 ../data/UD/UD_English-EWT/en_ewt-ud-train.conllu
0
100
../data/UD/UD_English-EWT/en_ewt-ud-train.conllu
n_sents 101 avg_sent_n_tokens 22.9
total_correct 1154
total_tokens 2311
uas_score 0.4993509303331891
1 ../data/UD/UD_English-CHILDES/en_childes-ud-train.conllu
0
100
../data/UD/UD_English-CHILDES/en_childes-ud-train.conllu
n_sents 101 avg_sent_n_tokens 5.8
total_correct 341
total_tokens 586
uas_score 0.5819112627986348
2 ../data/UD/UD_English-Atis/en_atis-ud-train.conllu
0
100
../data/UD/UD_English-Atis/en_atis-ud-train.conllu
n_sents 101 avg_sent_n_tokens 14.1
total_correct 731
total_tokens 1427
uas_score 0.5122634898388227
3 ../data/UD/UD_English-ESLSpok/en_eslspok-ud-train.conllu
0
100
../data/UD/UD_English-ESLSpok/en_eslspok-ud-train.conllu
n_sents 101 avg_sent_n_tokens 9.9
total_correct 583
total_tokens 996
uas_score 0.5853413654618473
4 ../data/UD/UD_English-ParTUT/en_partut-ud-train.conllu
0
100
../data/UD/UD_English-ParTUT/en_partut-ud-train.conllu
n_sents 101 avg_sent_n_to

In [ ]:
#res_file_fpath
!cat res_dev_expanded_rules_dev-test-reranked.tsv
#!cat res_dev_expanded_rules_training-reranked.tsv
#! cat res_dev_50rules-reranked.tsv
#!cat res_dev_50rules.tsv
#!cat res_dev_50rules-5ph.tsv

###Debugging issues with low UAS sentences

In [ ]:
cur_i=21
sent_error_list1=sorted(sent_error_list,key=lambda x:len(x["words"]))
cur_obj0=sent_error_list1[cur_i]
print(cur_obj0["uas"])
final_parses=cur_obj0["final_parses"]
parse_obj=cur_obj0["obj"]
words=cur_obj0["words"]
print(words)
print("Correct:", cur_obj0["gold_heads"])
print("Predicted:", cur_obj0["parser_heads"])
print("file",cur_obj0["file"])
#print(cur_obj0['parse_pos_list'])
# for a,b in cur_obj0.items():
#   print(a,b)
for a in cur_obj0['parse_pos_list']: print(a)

from IPython.display import HTML, display
fp0,dep0,const0=final_parses[0]
html_output=viz_parse(dep0,const0)
display(HTML(html_output))

0.0
['No', 'not', 'now', 'you', 'just', 'ate', '.']
Correct: ['3', '3', '0', '6', '6', '3', '3']
Predicted: ['-', '4', '4', '0', '-', '0', '6']
file ../data/UD/UD_English-CHILDES/en_childes-ud-train.conllu
('No', [('UH', [], 0.9999921321868896), ('DT', ['det'], 0.9995734095573425)])
('not', [('RB', ['neg'], 1)])
('now', [('RB', ['adv'], 0.9999997615814209)])
('you', [('PRON', ['noun', '2nd', 'singular|plural', 'subj|obj'], 1), ('PRP', [], 0.9999938011169434)])
('just', [('RB', ['adv'], 0.999994158744812)])
('ate', [('VBD', ['verb'], 0.9990653395652771), ('VBP', ['verb'], 0.8357948660850525)])
('.', [('.', ['punc'], 0.9999998807907104)])


In [ ]:
parse_obj.

In [ ]:
cur_obj0.keys()
cur_obj0['parse_pos_list']

[{'word': 'where',
  'upos': [('ADV', 0.9992625117301941), ('PRON', 0.7847432494163513)],
  'xpos': [('_', 0.8003944158554077), ('WRB', 0.6022530794143677)]},
 {'word': ':', 'upos': [('PUNCT', 0.9999985694885254)], 'xpos': []}]

##★ Debug rules on training data

In [ ]:
import re, copy
import itertools
from itertools import product
from code_utils.parsing_lib import *
from IPython.display import HTML, display



file_id="syntactic_gap_experiment"


info_fname=f"rules_word_features_{file_id}.json"
rules_dir="../data/rules"
info_fpath=os.path.join(rules_dir,info_fname)
with open(info_fpath) as fopen:
  rules_word_features_info=json.load(fopen)

final_word_features_dict=rules_word_features_info["final_word_features_dict"]
xpos_ft_dict=rules_word_features_info["xpos_ft_dict"]
rules=rules_word_features_info["final_rules"]



sent_toks=['The', 'situation', 'in', 'Iraq', 'is', 'only', 'going', 'to', 'get', 'better', 'this', 'way', '.']
sent_toks=['Our', 'experience', 'of', 'art', 'develops', 'from', 'the', 'interaction', 'of', 'several', 'cognitive', 'and', 'affective', 'processes', ';', 'the', 'beginning', 'of', 'which', 'is', 'a', 'visual', 'scan', 'of', 'the', 'artwork', '.']
sent_toks=['3', '.', 'License', 'Grant', '.']
sent_toks=['3', ':', 'License', 'Grant', '.']

sent_toks=['WASHINGTON', '(', 'CNN', ')', '--']
sent_toks=['Even', 'the', 'IP', 'and', 'ICDC', 'have', 'abandoned', 'the', 'neighbourhood', ',', 'and', 'those', 'are', 'trained', 'and', 'armed', ',', 'so', 'do', "n't", 'expect', 'scared', 'civilians', 'to', 'do', 'anything', 'except', 'to', 'hide', 'inside', 'and', 'pray', 'a', 'helicopter', 'or', 'a', 'tank', 'does', "n't", 'bomb', 'them', ',', 'and', 'also', 'how', 'are', 'American', 'soldiers', 'going', 'to', 'distinguish', 'the', 'brave', 'and', 'valiant', 'civilians', 'from', 'the', 'Fedayeen', '?']

sent_toks=['Baghdadis', 'do', "n't", 'venture', 'much', 'out', 'of', 'their', 'neighbourhoods', 'any', 'more', ',', 'you', 'never', 'know', 'where', 'you', 'might', 'get', 'stuck', '.']
sent_toks=['Baghdadis', 'do', "n't", 'venture', 'much', 'out', 'of', 'their', 'neighbourhoods', 'any', 'more']

sent_toks=['There', 'has', 'been', 'talk', 'that', 'the', 'night', 'curfew', 'might', 'be', 'implemented', 'again', '.']
sent_toks=['What', 'irritates', 'me', 'is', 'this', 'sudden', 'false', "'", 'solidarity', "'", 'between', 'Sunni', 'and', "Shi'ite", 'clerics', ',', 'we', 'all', 'know', 'that', 'they', 'would', 'be', 'glad', 'to', 'get', 'at', 'each', 'other', 's', 'throats', 'when', 'they', 'have', 'the', 'chance', ',', 'and', 'Shia', 'clerics', 'were', 'describing', 'Fallujan', 'insurgents', 'as', "'", "Ba'athists", "'", ',', "'", 'Saddamites', "'", ',', "'", 'Wahhabis', "'", ',', 'and', "'", 'terrorists', "'", 'just', 'a', 'few', 'days', 'ago', '.']
sent_toks=['radiation', ':']
sent_toks=['[', 'This', 'killing', 'of', 'a', 'respected', 'cleric', 'will', 'be', 'causing', 'us', 'trouble', 'for', 'years', 'to', 'come', '.', ']']
#sent_toks=['His', 'military', 'intelligence', 'has', 'captured', 'major', 'figures', 'like', 'Abu', 'Zubayda', 'and', 'Khalid', 'Shaykh', 'Muhammad', ',', 'as', 'well', 'as', 'nearly', '500', 'other', 'al', '-', 'Qaeda', 'operatives', ',', 'over', '400', 'of', 'whom', 'the', 'Pakistanis', 'have', 'turned', 'over', 'to', 'the', 'US', '.']
sent_toks=['His', 'military', 'intelligence', 'has', 'captured', 'major', 'figures', 'like', 'Abu', 'Zubayda']
sent_toks=['Texas', 'Gov.', 'George', 'W.', 'Bush', 'is', 'enduring', 'sharp', 'criticism', 'for', 'being', 'unable', 'to', 'name', 'the', 'leaders', 'of', 'four', 'current', 'world', 'hot', 'spots', ',', 'but', 'President', 'Bill', 'Clinton', 'says', 'Bush', '"', 'should', ',', 'and', 'probably', 'will', ',', 'pick', 'up', '"', 'those', 'names', '.']
sent_toks=['Show', 'All']
sent_toks=['where', ':']
sent_toks=['Why', 'has', 'no', 'air', 'quality', 'test', 'been', 'done']
sent_toks=['What', 'is', 'it', '?']
sent_toks=['Are', 'you', 'sure', '?']
sent_toks=['Insights', 'from', 'Eye', '-', 'Tracking']
sent_toks=['Oh', 'that', "'s", 'very', 'good', '.']
sent_toks=['No', 'not', 'now', 'you', 'just', 'ate', '.']




#sent_toks=['Mosques', 'are', 'calling', 'for', 'donating', 'blood', ',', 'food', ',', 'and', 'medicine', 'for', 'Fallujah', ',', 'and', 'several', 'convoys', 'have', 'already', 'headed', 'out', 'for', 'Fallujah', ',', 'most', 'of', 'them', 'returned', 'later', 'though', '.']
#sent_toks=['do', "n't", 'expect', 'scared', 'civilians', 'to', 'do', 'anything']
#sent_toks=['blood', ',', 'food', ',', 'and', 'medicine']
#sent_toks=['those', 'are', 'trained', 'and', 'armed']
#sent_toks=['Even', 'the', 'IP', 'and', 'ICDC', 'have', 'abandoned', 'the', 'neighbourhood']

# sent = "the man and the woman in the kitchen sing a song"
sent="I am going to tell him, and he will tell me"

sent="these are some difficult problems"
#sent="the man my friend saw"
#sent="going , to tell him"
sent="smart and beautiful"
sent="will be causing us trouble"

#sent="he is smart"
sent="he is smart,"
#sent="he is unable to learn"
sent="what is this?"

sent="he has been working in the factory"
sent="the big man will be working in the factory"

sent="it has been done"

sent="he has been given a task"
sent="he has been already given a task"
sent="have they mistaken the task?"
sent="where is he taking us?"
sent="what is he doing?"

sent="does he understand English?"
sent="the man my friend saw"
sent="the place we stayed in was good"

sent=re.sub(r"(\W)",r" \1 ",sent)
sent_toks=sent.strip().split()



#rules=final_rules

rnn_model_fpath="../models/pos_models/H64_L3_LR001_updated_xpos_loading.model"
#rnn_model_fpath=None
params0={"pos_model_path":rnn_model_fpath,"final_word_features_dict":final_word_features_dict,"xpos_ft_dict":xpos_ft_dict}
params0["debug"]=False
params0["max_n_phrases"]=10
params0["max_skip_distance"]=1
import time
#parser_obj=Parser(rules_list=rules,params=params0)
parser_obj=Parser(rules_list=rules,params=params0)
t0=time.time()
parsing_outcome=parser_obj.parse(sent_toks)
final_parses=parsing_outcome["parses"]
pos_tags=parsing_outcome["pos"]
t1=time.time()


fp0,dep0,const0=final_parses[0]
html_output=viz_parse(dep0,const0)
#display(HTML('<h1>Hello, World!</h1><p>This is a custom HTML snippet in Google Colab.</p>'))
display(HTML(html_output))
print()
#print(pos_tags)
#for ps0 in pos_tags: print(ps0)

In [ ]:
out0=parser_obj.inspect_span(6,7)
for a,b in out0.items():
  print(a)
  print(b)

VBN
[{'wt': 1.999944421310425, 'span': 2, 'start': 6, 'end': 7, 'cat': 'VBN', 'feat': [], 'children': [(6, 6, 'BEEN', ''), (7, 7, 'VBN', 'verb')], 'head_phrase': (6, 6, 'BEEN', ''), 'head_loc': 6, 'rule': {'parent': {'cat': 'VBN', 'feat': [], 'percolate': False, 'apply_cat': False}, 'children': [{'cat': 'BEEN', 'is_head': False, 'feat': []}, {'cat': 'VBN', 'is_head': False, 'feat': []}], 'head_i': 0, 'rule_i': 42}, 'level': 1}]
AP
[{'wt': 1.9999644213104248, 'span': 2, 'start': 6, 'end': 7, 'cat': 'AP', 'feat': [], 'children': [(6, 7, 'VBN', '')], 'head_phrase': (6, 7, 'VBN', ''), 'head_loc': 6, 'rule': {'parent': {'cat': 'AP', 'feat': [], 'percolate': False, 'apply_cat': False}, 'children': [{'cat': 'VBN', 'is_head': False, 'feat': []}], 'head_i': 0, 'rule_i': 40}, 'level': 2}]
VP-VBN
[{'wt': 1.9999644213104248, 'span': 2, 'start': 6, 'end': 7, 'cat': 'VP-VBN', 'feat': [], 'children': [(6, 7, 'VBN', '')], 'head_phrase': (6, 7, 'VBN', ''), 'head_loc': 6, 'rule': {'parent': {'cat': 'VP-

In [ ]:
parser_obj.tag_wt_list

[('Show', [('VB', ['verb'], 0.9995513558387756)]),
 ('All', [('RB', [], 1), ('DT', [], 1)])]

In [ ]:
ord("'")

39

In [ ]:
ord("'")

39

In [ ]:
final_word_features_dict["'"]

KeyError: "'"

In [ ]:
!ls ../data/rules/

rules_word_features_feb_2026.json  rules_word_features_jan_2026.json


In [ ]:
info_fname

'rules_word_features_feb_2026.pp_brackets.json'

In [ ]:
info_fpath

'../data/rules/rules_word_features_feb_2026.pp_brackets.json'

In [ ]:
span_wt_items=list(parser_obj.span_wt_dict.items())
span_wt_items.sort(key=lambda x:-x[-1])


valid_sorted_span_phrases=[]
used_indexes=[]
for span0,wt0 in span_wt_items:
  i0,i1=span0
  if i0==i1: continue
  cur_span_indexes=list(range(i0,i1+1))
  overlap_with_used=list(set(used_indexes).intersection(set(cur_span_indexes)))
  valid=False
  if overlap_with_used==[]:
    used_indexes.extend(cur_span_indexes)
    valid=True
  if valid:
    ph0=parser_obj.span_phrase_dict.get(span0)
    if ph0!=None: valid_sorted_span_phrases.append(ph0)

combined_phrase=parser_obj.combine_phrases(valid_sorted_span_phrases)
combined_phrase["cat"]="COMBINED"
print("combined_phrase",combined_phrase)


{'wt': 15.9475012301445, 'span': 16, 'start': 0, 'end': 15, 'cat': 'S-U', 'feat': [], 'children': [(0, 8, 'S-U', ''), (9, 15, 'COMMA-CC-S', '')], 'head_phrase': (0, 8, 'S-U', ''), 'head_loc': 5, 'rule': {'parent': {'cat': 'S-U', 'feat': []}, 'children': [{'cat': 'S-U', 'is_head': True, 'feat': []}, {'cat': 'COMMA-CC-S', 'is_head': False, 'feat': []}], 'head_i': 0, 'rule_i': 74}, 'level': 10}
{'wt': 13.776766883487703, 'span': 14, 'start': 45, 'end': 58, 'cat': 'VP', 'feat': [], 'children': [(45, 58, 'VP-2O', '')], 'head_phrase': (45, 58, 'VP-2O', ''), 'head_loc': 45, 'rule': {'parent': {'cat': 'VP', 'feat': []}, 'children': [{'cat': 'VP-2O', 'is_head': False, 'feat': []}], 'head_i': 0, 'rule_i': 10}, 'level': 14}
{'wt': 9.00052791912079, 'span': 9, 'start': 32, 'end': 40, 'cat': 'S-U', 'feat': [], 'children': [(32, 36, 'NP', ''), (37, 40, 'VP', '')], 'head_phrase': (37, 40, 'VP', ''), 'head_loc': 37, 'rule': {'parent': {'cat': 'S-U', 'feat': []}, 'children': [{'cat': 'NP', 'is_head': F

In [ ]:
for a in const0: print(a)

{'wt': 46.72525385982514, 'span': 59, 'start': 0, 'end': 58, 'cat': 'COMBINED', 'feat': [], 'children': [(0, 15, 'S-U', ''), (45, 58, 'VP', ''), (32, 40, 'S-U', ''), (20, 25, 'VP', ''), (27, 28, 'VP-TO', '')], 'head_phrase': (0, 15, 'S-U', ''), 'head_loc': 5, 'rule': {}}
{'wt': 15.947404193782805, 'span': 16, 'start': 0, 'end': 15, 'cat': 'S-U', 'feat': [], 'children': [(0, 8, 'S-U', ''), (9, 15, 'COMMA-CC-S', '')], 'head_phrase': (0, 8, 'S-U', ''), 'head_loc': 5, 'rule': {'parent': {'cat': 'S-U', 'feat': []}, 'children': [{'cat': 'S-U', 'is_head': True, 'feat': []}, {'cat': 'COMMA-CC-S', 'is_head': False, 'feat': []}], 'head_i': 0, 'rule_i': 74}, 'level': 10}
{'wt': 13.776736883487702, 'span': 14, 'start': 45, 'end': 58, 'cat': 'VP', 'feat': [], 'children': [(45, 58, 'VP-O', '')], 'head_phrase': (45, 58, 'VP-O', ''), 'head_loc': 45, 'rule': {'parent': {'cat': 'VP', 'feat': []}, 'children': [{'cat': 'VP-O', 'is_head': False, 'feat': []}], 'head_i': 0, 'rule_i': 9}, 'level': 14}
{'wt': 

In [ ]:
parser_obj.inspect_span(27,28)

{'VP-TO': [{'wt': 2.00003,
   'span': 2,
   'start': 27,
   'end': 28,
   'cat': 'VP-TO',
   'feat': [],
   'children': [(27, 27, 'TO', ''), (28, 28, 'VP-INF', '')],
   'head_phrase': (28, 28, 'VP-INF', ''),
   'head_loc': 28,
   'rule': {'parent': {'cat': 'VP-TO', 'feat': []},
    'children': [{'cat': 'TO', 'is_head': False, 'feat': []},
     {'cat': 'VP-INF', 'is_head': True, 'feat': []}],
    'head_i': 1,
    'rule_i': 49},
   'level': 2}],
 'PP': [{'wt': 1.6936111231422424,
   'span': 2,
   'start': 27,
   'end': 28,
   'cat': 'PP',
   'feat': [],
   'children': [(27, 27, 'IN', 'loc'), (28, 28, 'NP', '')],
   'head_phrase': (27, 27, 'IN', 'loc'),
   'head_loc': 27,
   'rule': {'parent': {'cat': 'PP', 'feat': []},
    'children': [{'cat': 'IN', 'is_head': True, 'feat': []},
     {'cat': 'NP', 'is_head': False, 'feat': []}],
    'head_i': 0,
    'rule_i': 16},
   'level': 3}],
 'AdvP': [{'wt': 1.6936511231422424,
   'span': 2,
   'start': 27,
   'end': 28,
   'cat': 'AdvP',
   'feat'

In [ ]:
rules[-10:]

['NP --> NP^ S/NP',
 'NP --> NP^ VP-TO',
 'NP-LB --> -LRB- NP^',
 'NP-BR --> NP-LB^  -RRB-',
 'NP --> NP^ NP-BR',
 'NP --> RB NP^',
 'CC-S --> CC S-U^',
 'CC-S --> SCONJ S-U^',
 'COMMA-CC-S --> COMMA CC-S^',
 'S-U --> S-U^ COMMA-CC-S']

In [ ]:
!head ../data/UD/UD_English-GUMReddit/en_gumreddit-ud-dev.conllu


# newdoc id = GUM_reddit_macroeconomics
# global.Entity = GRP-etype-infstat-salience-centering-minspan-link-identity
# meta::author = Anjabe96, cchoe1
# meta::dateCollected = 2017-09-13
# meta::dateCreated = 2017-09-12
# meta::dateModified = 2017-09-12
# meta::genre = reddit
# meta::salientEntities = 12 (5*), 14 (5*), 145 (5*), 3 (4*), 7 (3*), 16 (3*), 23 (3), 81 (3), 95 (3*), 2 (2), 6 (2), 21 (2*), 31 (2), 97 (2*), 4 (1), 10 (1), 18 (1), 73 (1), 94 (1*), 99 (1*)
# meta::sourceURL = https://www.reddit.com/r/explainlikeimfive/comments/6zm74h/eli5_if_every_major_country_is_in_billions_if_not/
# meta::speakerCount = 2
# meta::summary1 = (human1) In a post answering the question how and to whom countries can be in debt, the author explains how money is a form of debt whose value depends on the amount of money in circulation, and how fiat currency developed as a substitute for gold reserves which formerly backed bank debt certificates.
# meta::summary2 = (human2) A user from an online forum

In [ ]:
params0={"pos_model_path":rnn_model_fpath,"final_word_features_dict":final_word_features_dict,"xpos_ft_dict":xpos_ft_dict}
params0["max_n_phrases"]=5
params0["debug"]=False
params0["djk_use_negative_wt"]=False
#parser_obj1.djk_use_negative_wt

parser_obj1=Parser(rules_list=rules, params=params0)


tokens=['5', 'star', 'detail', 'job']
tokens=['Two', 'other', 'Dulaim', 'leaders', 'have', 'been', 'killed', 'in', 'the', 'past', 'week', 'and', 'a', 'half', '.']
tokens=['[', 'This', 'killing', 'of', 'a', 'respected', 'cleric', 'will', 'be', 'causing', 'us', 'trouble', 'for', 'years', 'to', 'come', '.', ']']
tokens=['This', 'killing', 'of', 'a', 'respected', 'cleric']
tokens=['His', 'mother', 'was', 'also', 'killed', 'in', 'the', 'attack', '.']
tokens=['Two', 'other', 'Dulaim', 'leaders', 'have', 'been', 'killed', 'in', 'the', 'past', 'week', 'and', 'a', 'half', '.']
tokens=['When', 'a', 'connection', 'file', 'is', 'used']
tokens=['Using', 'a', 'connection', 'file', 'without', 'creating', 'a', 'link']
tokens=['Everyone', 'is', 'apprehensive', ',', 'there', 'is', 'some', 'talk', 'that', 'April', '9th', 'and', '10th', 'are', 'going', 'to', 'be', 'bloody', 'days', '.']
tokens=['some', 'talk', 'that', 'April', '9th', 'and', '10th', 'are', 'going', 'to', 'be', 'bloody', 'days']
tokens=['i', "'ll", 'need', 'to', 'rent', 'a', 'car', 'in', 'washington', 'dc']
tokens=['rental', 'cars', 'in', 'washington', 'dc']
tokens=['flights', 'from', 'newark', 'new', 'jersey', 'to', 'minneapolis']
tokens=['Because', 'I', 'had', 'the', 'receipt', '.']

# tokens=['there', 'is', 'some', 'talk']
# tokens=["said", 'that', 'April', '9th', 'and', '10th', 'are', 'going', 'to', 'be', 'bloody']



#tokens=['a', 'respected', 'cleric']
#tokens=['was', 'also', 'killed']
#tokens=['in', 'the', 'attack']
#tokens=['was', 'also', 'killed', 'in', 'the', 'attack']


#tokens=['have', 'been', 'killed']
#tokens=['have', 'been', 'killed',"in","Canada"]
#tokens=['in', 'the', 'past', 'week', 'and', 'a', 'half']

parses=parser_obj1.parse2(tokens)
# for ph_obj0,dep0,const0 in parses[:5]:
#   print(ph_obj0)
#   for d0 in dep0: print(d0)
#   print("--")
#   for c0 in const0: print(c0)
#   print("====")
# gold_heads ['4', '4', '4', '7', '7', '7', '0', '11', '11', '11', '7', '14', '14', '11', '7']
# parser_heads ['4', '4', '4', '7', '6', '7', '0', '0', '14', '14', '14', '-', '-', '8', '-']
# head_matching [1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]
# {"id":str(cur_id0),"word": w0,"head":cur_dep,"xpos":cur_xpos0,"head_word":head_word,"offset":offset}
for a in parser_obj1.combined_dep:
  print(a)



{'id': '1', 'word': 'Because', 'head': '0', 'xpos': 'IN', 'head_word': '^', 'offset': 0}
{'id': '2', 'word': 'I', 'head': '1', 'xpos': 'PRON', 'head_word': 'Because', 'offset': 1}
{'id': '3', 'word': 'had', 'head': '0', 'xpos': 'VBD', 'head_word': '^', 'offset': 0}
{'id': '4', 'word': 'the', 'head': '5', 'xpos': 'DT', 'head_word': 'receipt', 'offset': -1}
{'id': '5', 'word': 'receipt', 'head': '3', 'xpos': 'NN', 'head_word': 'had', 'offset': 2}
{'id': '6', 'word': '.', 'head': '-', 'xpos': '-', 'head_word': '', 'offset': 100}


In [ ]:
parser_obj1.cur_tokens_pos_list

[{'word': 'Because',
  'upos': [('SCONJ', 0.999998927116394), ('ADP', 0.9999864101409912)],
  'xpos': [('IN', 0.9999998807907104)]},
 {'word': 'I', 'upos': [('PRON', 1.0)], 'xpos': [('PRP', 0.999995231628418)]},
 {'word': 'had',
  'upos': [('VERB', 0.9999853372573853), ('AUX', 0.999875545501709)],
  'xpos': [('VBD', 0.9994557499885559)]},
 {'word': 'the',
  'upos': [('DET', 0.9999984502792358)],
  'xpos': [('DT', 0.9999988079071045)]},
 {'word': 'receipt',
  'upos': [('NOUN', 1.0),
   ('ADJ', 0.30192655324935913),
   ('NUM', 0.24441544711589813)],
  'xpos': [('NN', 0.9999721050262451)]},
 {'word': '.', 'upos': [('PUNCT', 1.0)], 'xpos': [('.', 0.9999964237213135)]}]

In [ ]:
parser_obj1.inspect_span(7,13)

{'PP': [{'wt': 6.9999988079071045,
   'span': 7,
   'start': 7,
   'end': 13,
   'cat': 'PP',
   'feat': [],
   'children': [(7, 7, 'IN', 'loc'), (8, 13, 'NP', '')],
   'head_phrase': (7, 7, 'IN', 'loc'),
   'head_loc': 7,
   'rule': {'parent': {'cat': 'PP', 'feat': []},
    'children': [{'cat': 'IN', 'is_head': True, 'feat': []},
     {'cat': 'NP', 'is_head': False, 'feat': []}],
    'head_i': 0,
    'rule_i': 15},
   'level': 6},
  {'wt': 6.9999964237213135,
   'span': 7,
   'start': 7,
   'end': 13,
   'cat': 'PP',
   'feat': [],
   'children': [(7, 7, 'IN', 'loc'), (8, 13, 'NP', '')],
   'head_phrase': (7, 7, 'IN', 'loc'),
   'head_loc': 7,
   'rule': {'parent': {'cat': 'PP', 'feat': []},
    'children': [{'cat': 'IN', 'is_head': True, 'feat': []},
     {'cat': 'NP', 'is_head': False, 'feat': []}],
    'head_i': 0,
    'rule_i': 15},
   'level': 6},
  {'wt': 5.999999761581421,
   'span': 7,
   'start': 7,
   'end': 13,
   'cat': 'PP',
   'feat': [],
   'children': [(7, 7, 'IN', 'lo

In [ ]:
def combine_exported_deps(words,dep_list):
  temp_dep_dict={}
  for dep1 in dep_list:
    for d1 in dep1:
      cur_id1=d1["id"]
      temp_dep_dict[d1["id"]]=d1
  final_dep_lines=[]
  for i0,w0 in enumerate(words):
    cur_id0=str(i0+1)
    found_line_obj=temp_dep_dict.get(cur_id0)
    temp_line_obj={"id":cur_id0,"word": w0,"head":'-',"xpos":'-',"head_word":"","offset":100}
    print(cur_id0, found_line_obj)
    if found_line_obj==None: found_line_obj=temp_line_obj
    final_dep_lines.append(found_line_obj)
  return final_dep_lines


djk_phrases=parser_obj1.optimum_path(tokens,parser_obj1.span_phrase_dict)
print(tokens)
temp_dep_list=[]
for a in djk_phrases:
  print(a)
  dep0,const0=parser_obj1.export_parse2(tokens,a,False)
  temp_dep_list.append(dep0)
  # for d0 in dep0:
  #   print(d0)
  # print("--")
  # for c0 in const0:
  #   print(c0)
  # print("=====")
combined_dep0=combine_exported_deps(tokens,temp_dep_list)
for cd0 in combined_dep0: print(cd0)





['Two', 'other', 'Dulaim', 'leaders', 'have', 'been', 'killed', 'in', 'the', 'past', 'week', 'and', 'a', 'half', '.']
{'wt': 6.999742865562439, 'span': 7, 'start': 0, 'end': 6, 'cat': 'S', 'feat': [], 'children': [(0, 3, 'NP', ''), (4, 6, 'VP', '')], 'head_phrase': (4, 6, 'VP', ''), 'head_loc': 6, 'rule': {'parent': {'cat': 'S', 'feat': []}, 'children': [{'cat': 'NP', 'is_head': False, 'feat': []}, {'cat': 'VP', 'is_head': True, 'feat': []}], 'head_i': 1, 'rule_i': 22}, 'level': 3}
{'wt': 6.9999988079071045, 'span': 7, 'start': 7, 'end': 13, 'cat': 'PP', 'feat': [], 'children': [(7, 7, 'IN', 'loc'), (8, 13, 'NP', '')], 'head_phrase': (7, 7, 'IN', 'loc'), 'head_loc': 7, 'rule': {'parent': {'cat': 'PP', 'feat': []}, 'children': [{'cat': 'IN', 'is_head': True, 'feat': []}, {'cat': 'NP', 'is_head': False, 'feat': []}], 'head_i': 0, 'rule_i': 15}, 'level': 6}
1 {'id': '1', 'word': 'Two', 'head': '4', 'xpos': 'CD', 'head_word': 'leaders', 'offset': -3}
2 {'id': '2', 'word': 'other', 'head': 

In [ ]:
parser_obj1.inspect_span(4,13)

{'VP-2O': [{'wt': 6.93721067905426,
   'span': 10,
   'start': 4,
   'end': 13,
   'cat': 'VP-2O',
   'feat': [],
   'children': [(4, 9, 'VP-O', ''), (10, 13, 'NP', '')],
   'head_phrase': (4, 9, 'VP-O', ''),
   'head_loc': 4,
   'rule': {'parent': {'cat': 'VP-2O', 'feat': []},
    'children': [{'cat': 'VP-O', 'is_head': True, 'feat': []},
     {'cat': 'NP', 'is_head': False, 'feat': []}],
    'head_i': 0,
    'rule_i': 7},
   'level': 6},
  {'wt': 6.749374806880951,
   'span': 10,
   'start': 4,
   'end': 13,
   'cat': 'VP-2O',
   'feat': [],
   'children': [(4, 9, 'VP-O', ''), (10, 13, 'NP', '')],
   'head_phrase': (4, 9, 'VP-O', ''),
   'head_loc': 4,
   'rule': {'parent': {'cat': 'VP-2O', 'feat': []},
    'children': [{'cat': 'VP-O', 'is_head': True, 'feat': []},
     {'cat': 'NP', 'is_head': False, 'feat': []}],
    'head_i': 0,
    'rule_i': 7},
   'level': 6},
  {'wt': 5.93721067905426,
   'span': 10,
   'start': 4,
   'end': 13,
   'cat': 'VP-2O',
   'feat': [],
   'children': 

In [ ]:
#parser_obj1.end_start_phrase_dict[2][0]
parser_obj1.inspect_span(7,13)

{'PP': [{'wt': 6.9999988079071045,
   'span': 7,
   'start': 7,
   'end': 13,
   'cat': 'PP',
   'feat': [],
   'children': [(7, 7, 'IN', 'loc'), (8, 13, 'NP', '')],
   'head_phrase': (7, 7, 'IN', 'loc'),
   'head_loc': 7,
   'rule': {'parent': {'cat': 'PP', 'feat': []},
    'children': [{'cat': 'IN', 'is_head': True, 'feat': []},
     {'cat': 'NP', 'is_head': False, 'feat': []}],
    'head_i': 0,
    'rule_i': 15},
   'level': 6},
  {'wt': 6.9999964237213135,
   'span': 7,
   'start': 7,
   'end': 13,
   'cat': 'PP',
   'feat': [],
   'children': [(7, 7, 'IN', 'loc'), (8, 13, 'NP', '')],
   'head_phrase': (7, 7, 'IN', 'loc'),
   'head_loc': 7,
   'rule': {'parent': {'cat': 'PP', 'feat': []},
    'children': [{'cat': 'IN', 'is_head': True, 'feat': []},
     {'cat': 'NP', 'is_head': False, 'feat': []}],
    'head_i': 0,
    'rule_i': 15},
   'level': 6},
  {'wt': 5.999999761581421,
   'span': 7,
   'start': 7,
   'end': 13,
   'cat': 'PP',
   'feat': [],
   'children': [(7, 7, 'IN', 'lo

In [ ]:
parser_obj1.inspect_span(4,13)

{'VP-2O': [{'wt': 6.93721067905426,
   'span': 10,
   'start': 4,
   'end': 13,
   'cat': 'VP-2O',
   'feat': [],
   'children': [(4, 9, 'VP-O', ''), (10, 13, 'NP', '')],
   'head_phrase': (4, 9, 'VP-O', ''),
   'head_loc': 4,
   'rule': {'parent': {'cat': 'VP-2O', 'feat': []},
    'children': [{'cat': 'VP-O', 'is_head': True, 'feat': []},
     {'cat': 'NP', 'is_head': False, 'feat': []}],
    'head_i': 0,
    'rule_i': 7},
   'level': 6},
  {'wt': 6.749374806880951,
   'span': 10,
   'start': 4,
   'end': 13,
   'cat': 'VP-2O',
   'feat': [],
   'children': [(4, 9, 'VP-O', ''), (10, 13, 'NP', '')],
   'head_phrase': (4, 9, 'VP-O', ''),
   'head_loc': 4,
   'rule': {'parent': {'cat': 'VP-2O', 'feat': []},
    'children': [{'cat': 'VP-O', 'is_head': True, 'feat': []},
     {'cat': 'NP', 'is_head': False, 'feat': []}],
    'head_i': 0,
    'rule_i': 7},
   'level': 6},
  {'wt': 5.93721067905426,
   'span': 10,
   'start': 4,
   'end': 13,
   'cat': 'VP-2O',
   'feat': [],
   'children': 

In [ ]:
ph0={'wt': 2.0,
   'span': 2,
   'start': 6,
   'end': 7,
   'cat': 'NP',
   'feat': [],
   'children': [(6, 7, 'NP-DT', '')],
   'head_phrase': (6, 7, 'NP-DT', ''),
   'head_loc': 7,
   'rule': {'parent': {'cat': 'NP', 'feat': []},
    'children': [{'cat': 'NP-DT', 'is_head': False, 'feat': []}],
    'head_i': 0,
    'rule_i': 20},
   'level': 3}
projected=parser_obj1.project_phrase3(ph0)
for p0 in projected:
  check0=parser_obj1.add_phrase3(p0)
  print(check0,p0)


False {'wt': 3.997677981853485, 'span': 6, 'start': 2, 'end': 7, 'cat': 'VP-O', 'feat': [], 'children': [(2, 3, 'V', ''), (6, 7, 'NP', '')], 'head_phrase': (2, 3, 'V', ''), 'head_loc': 2, 'rule': {'parent': {'cat': 'VP-O', 'feat': []}, 'children': [{'cat': 'V', 'is_head': True, 'feat': []}, {'cat': 'NP', 'is_head': False, 'feat': []}], 'head_i': 0, 'rule_i': 6}, 'level': 4}
False {'wt': 2.9999890327453613, 'span': 4, 'start': 4, 'end': 7, 'cat': 'VP-O', 'feat': [], 'children': [(4, 4, 'V', ''), (6, 7, 'NP', '')], 'head_phrase': (4, 4, 'V', ''), 'head_loc': 4, 'rule': {'parent': {'cat': 'VP-O', 'feat': []}, 'children': [{'cat': 'V', 'is_head': True, 'feat': []}, {'cat': 'NP', 'is_head': False, 'feat': []}], 'head_i': 0, 'rule_i': 6}, 'level': 4}
False {'wt': 2.2367586195468903, 'span': 4, 'start': 4, 'end': 7, 'cat': 'VP-O', 'feat': [], 'children': [(4, 4, 'V', ''), (6, 7, 'NP', '')], 'head_phrase': (4, 4, 'V', ''), 'head_loc': 4, 'rule': {'parent': {'cat': 'VP-O', 'feat': []}, 'childre

In [ ]:
parser_obj1.end_start_phrase_dict[2][1]

KeyError: 1

In [ ]:
parser_obj1.cur_tokens_pos_list

[{'word': 'was',
  'upos': [('AUX', 1.0), ('VERB', 1.0)],
  'xpos': [('VBD', 0.9744098782539368), ('VBG', 0.20264700055122375)]},
 {'word': 'also',
  'upos': [('ADV', 0.9999998807907104)],
  'xpos': [('RB', 0.9999914169311523)]},
 {'word': 'killed',
  'upos': [('VERB', 1.0), ('ADJ', 0.9982037544250488)],
  'xpos': [('VBN', 0.9999831914901733),
   ('JJ', 0.9915865659713745),
   ('VB', 0.48033279180526733)]},
 {'word': 'in',
  'upos': [('ADP', 0.9999996423721313), ('SCONJ', 0.9858422875404358)],
  'xpos': [('IN', 0.9999998807907104)]},
 {'word': 'the',
  'upos': [('DET', 0.9999986886978149)],
  'xpos': [('DT', 0.9999984502792358)]},
 {'word': 'attack', 'upos': [('NOUN', 1.0)], 'xpos': [('NN', 1.0)]}]

In [ ]:
test_ph0={'wt': 2.3075806498527527,
   'span': 3,
   'start': 0,
   'end': 2,
   'cat': 'NP-DT',
   'feat': [],
   'children': [(0, 0, 'DT', ''), (1, 2, 'NP-U', '')],
   'head_phrase': (1, 2, 'NP-U', ''),
   'head_loc': 2,
   'rule': {'parent': {'cat': 'NP-DT', 'feat': []},
    'children': [{'cat': 'DT', 'is_head': False, 'feat': []},
     {'cat': 'NP-U', 'is_head': True, 'feat': []}],
    'head_i': 1,
    'rule_i': 16},
   'level': 3}
projected_phrases=parser_obj1.project_phrase3(test_ph0)
for a in projected_phrases:
  check0=parser_obj1.add_phrase3(a)
  print(check0,a)

False {'wt': 2.3075806498527527, 'span': 3, 'start': 0, 'end': 2, 'cat': 'NP', 'feat': [], 'children': [(0, 2, 'NP-DT', '')], 'head_phrase': (0, 2, 'NP-DT', ''), 'head_loc': 2, 'rule': {'parent': {'cat': 'NP', 'feat': []}, 'children': [{'cat': 'NP-DT', 'is_head': False, 'feat': []}], 'head_i': 0, 'rule_i': 20}, 'level': 4}


In [ ]:
for a in parser_obj1.tag_wt_list: print(a)

('This', [('DT', [], 1)])
('killing', [('NN', ['noun'], 0.9999992847442627), ('VBG', ['verb'], 0.994005024433136), ('JJ', ['adj'], 0.5683211088180542)])
('of', [('IN', [], 1)])
('a', [('DT', [], 1)])
('respected', [('VBN', ['verb'], 0.9997286200523376), ('JJ', ['adj'], 0.9997045397758484), ('VBD', ['verb'], 0.8961520791053772), ('NN', ['noun'], 0.811540424823761)])
('cleric', [('JJ', ['adj'], 1.0), ('NN', ['noun'], 0.9999922513961792)])


In [ ]:
cur_dev_file

'../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu'

In [ ]:
key0=(0, 63, 'S', '')
cur_phrase_obj_list=parser_obj.main_phrase_dict[key0]
for a in cur_phrase_obj_list: print(a)

{'wt': 43.75677794218063, 'span': 64, 'start': 0, 'end': 63, 'cat': 'S', 'feat': [], 'children': (1, 1643), 'head_phrase': 1643, 'head_loc': 2, 'rule_i': 5, 'level': 5, 'i': 1610}
{'wt': 43.63713379204273, 'span': 64, 'start': 0, 'end': 63, 'cat': 'S', 'feat': [], 'children': (19, 1644), 'head_phrase': 1644, 'head_loc': 8, 'rule_i': 5, 'level': 5, 'i': 1610}
{'wt': 41.63198643922806, 'span': 64, 'start': 0, 'end': 63, 'cat': 'S', 'feat': [], 'children': (30, 1641), 'head_phrase': 1641, 'head_loc': 11, 'rule_i': 5, 'level': 5, 'i': 1610}
{'wt': 40.66937440633774, 'span': 64, 'start': 0, 'end': 63, 'cat': 'S', 'feat': [], 'children': (1184, 1546), 'head_phrase': 1546, 'head_loc': 61, 'rule_i': 5, 'level': 3, 'i': 1610}
{'wt': 40.66937035322189, 'span': 64, 'start': 0, 'end': 63, 'cat': 'S', 'feat': [], 'children': (1374, 1546), 'head_phrase': 1546, 'head_loc': 61, 'rule_i': 5, 'level': 3, 'i': 1610}
{'wt': 40.66934758424759, 'span': 64, 'start': 0, 'end': 63, 'cat': 'S', 'feat': [], 'chi

In [ ]:
obj0={'wt': 19.1611181050539, 'span': 23, 'start': 3, 'end': 25, 'cat': 'S', 'feat': [], 'children': (21, 254), 'head_phrase': 254, 'head_loc': 8, 'rule_i': 5, 'level': 10, 'i': 247}
obj0={'wt': 20.757439851760864, 'span': 30, 'start': 0, 'end': 29, 'cat': 'NP', 'feat': [], 'children': (1, 352), 'head_phrase': 1, 'head_loc': 0, 'rule_i': 11, 'level': 13, 'i': 329}
obj0={'wt': 43.75677794218063, 'span': 64, 'start': 0, 'end': 63, 'cat': 'S', 'feat': [], 'children': (1, 1643), 'head_phrase': 1643, 'head_loc': 2, 'rule_i': 5, 'level': 5, 'i': 1610}

ch_list=parser_obj.get_children(obj0)
for ch0 in ch_list:
  start0,end0=ch0["start"],ch0["end"]
  phrase_toks=sent_toks[start0:end0+1]
  print(ch0["cat"], "=", " ".join(phrase_toks),  ch0)
  print("---")


S = Ecuador also supports initiatives for the protection , conservation and sustainable use of the ocean , such as the prompt entry into force of the Agreement under the United Nations Convention on the Law of the Sea on the Conservation and Sustainable Use of Marine Biological Diversity of Areas beyond National Jurisdiction , and the holding of the third United Nations Conference on Oceans {'wt': 43.75677794218063, 'span': 64, 'start': 0, 'end': 63, 'cat': 'S', 'feat': [], 'children': (1, 1643), 'head_phrase': 1643, 'head_loc': 2, 'rule_i': 5, 'level': 5, 'i': 1610}
---
NP = Ecuador {'wt': 0.9999328851699829, 'span': 1, 'start': 0, 'end': 0, 'cat': 'NP', 'feat': [], 'children': (0,), 'head_phrase': 0, 'head_loc': 0, 'rule_i': 0, 'level': 1, 'i': 1}
---
VP = supports initiatives for the protection , conservation and sustainable use of the ocean , such as the prompt entry into force of the Agreement under the United Nations Convention on the Law of the Sea on the Conservation and Sustai

In [ ]:
obj0={'cat': 'X', 'is_head': True, 'feat': ['be']}
parser_obj.scan_phrases2(obj0)

[10]

In [ ]:
obj0={'cat': 'AP', 'is_head': False, 'feat': []}
parser_obj.scan_phrases2(obj0)

[2, 13]

In [ ]:
for a,phrase_list in parser_obj.main_phrase_dict.items():
  print(a,len(phrase_list),parser_obj.main_phrase_wt_dict[a])
  for p0 in phrase_list: print(p0)
  print("---")

(0, 0, 'DT', '') 1 1
{'wt': 1, 'span': 1, 'start': 0, 'end': 0, 'cat': 'DT', 'feat': [], 'head_loc': 0, 'head_phrase': 0, 'children': [], 'i': 0}
---
(1, 1, 'JJ', 'adj') 1 0.9999953508377075
{'wt': 0.9999953508377075, 'span': 1, 'start': 1, 'end': 1, 'cat': 'JJ', 'feat': ['adj'], 'head_loc': 1, 'head_phrase': 1, 'children': [], 'i': 1}
---
(1, 1, 'AP', '') 1 0.9999953508377075
{'wt': 0.9999953508377075, 'span': 1, 'start': 1, 'end': 1, 'cat': 'AP', 'feat': [], 'children': (1,), 'head_phrase': 1, 'head_loc': 1, 'rule_i': 1, 'i': 2}
---
(1, 1, 'NN', 'noun') 1 0.9999562501907349
{'wt': 0.9999562501907349, 'span': 1, 'start': 1, 'end': 1, 'cat': 'NN', 'feat': ['noun'], 'head_loc': 1, 'head_phrase': 3, 'children': [], 'i': 3}
---
(1, 1, 'NP', '') 1 0.9999562501907349
{'wt': 0.9999562501907349, 'span': 1, 'start': 1, 'end': 1, 'cat': 'NP', 'feat': [], 'children': (3,), 'head_phrase': 3, 'head_loc': 1, 'rule_i': 0, 'i': 4}
---
(0, 1, 'NP', '') 1 1.9999562501907349
{'wt': 1.9999562501907349, '

In [ ]:
for a in parser_obj.all_processed_rules:
  print(a)

{'parent': {'cat': 'NP', 'feat': []}, 'children': [{'cat': 'X', 'is_head': False, 'feat': ['noun']}], 'head_i': 0}
{'parent': {'cat': 'AP', 'feat': []}, 'children': [{'cat': 'JJ', 'is_head': False, 'feat': []}], 'head_i': 0}
{'parent': {'cat': 'VP', 'feat': []}, 'children': [{'cat': 'X', 'is_head': True, 'feat': ['be']}, {'cat': 'AP', 'is_head': False, 'feat': []}], 'head_i': 0}
{'parent': {'cat': 'VP', 'feat': []}, 'children': [{'cat': 'X', 'is_head': True, 'feat': ['be']}, {'cat': 'JJ', 'is_head': False, 'feat': []}], 'head_i': 0}
{'parent': {'cat': 'S', 'feat': []}, 'children': [{'cat': 'NP', 'is_head': False, 'feat': []}, {'cat': 'VP', 'is_head': True, 'feat': []}], 'head_i': 1}
{'parent': {'cat': 'V', 'feat': []}, 'children': [{'cat': 'V', 'is_head': False, 'feat': ['be']}, {'cat': 'V', 'is_head': True, 'feat': ['ing']}], 'head_i': 1}
{'parent': {'cat': 'AUX', 'feat': ['neg']}, 'children': [{'cat': 'AUX', 'is_head': True, 'feat': []}, {'cat': 'RB', 'is_head': False, 'feat': ['neg'

In [ ]:
#parse each rule - identify head of the rule, and feature application, including slash
#convert into a trie
#collect word features

#for an input sentence:
#tokenize the sentence with tok paradigm
#locate/retrieve the corresponding features for each word, and word clusters (multi token words)
#for each word, generate the possible labels/tags, with their likelihood, if available
#a simple POS tagger for any given word

#A simple POS tagger, based on training data from multiple treebanks
#we need to resolve POS/tags, which to use, and how it contributes to different features NNS, VBG .. etc


In [ ]:
all_xpos_dict_items=list(all_xpos_dict.items())
all_xpos_dict_items.sort(key=lambda x:-x[-1])
for a,b in all_xpos_dict_items:
  print(a,b,all_xpos_upos_dict[a])
print(len(all_xpos_dict_items))

NN 75554 {'NOUN': 73136, 'PRON': 1812, 'SYM': 104, 'X': 99, 'INTJ': 33, 'VERB': 126, 'PROPN': 62, 'ADJ': 54, 'ADV': 84, 'DET': 34, 'NUM': 6, 'AUX': 2, 'SCONJ': 1, 'PUNCT': 1}
_ 62455 {'X': 14, 'PRON': 3089, 'AUX': 1732, 'DET': 3820, 'NOUN': 8655, 'ADP': 17165, 'PROPN': 11657, 'VERB': 4597, 'NUM': 933, 'ADJ': 1648, 'CCONJ': 2789, 'ADV': 3525, 'PART': 1283, 'INTJ': 427, 'PUNCT': 5, 'SCONJ': 1112, 'SYM': 4}
IN 52597 {'ADP': 43463, 'SCONJ': 8673, 'SYM': 27, 'X': 7, 'ADV': 304, 'PRON': 70, 'DET': 31, 'PART': 12, 'NOUN': 1, 'VERB': 3, 'CCONJ': 1, 'INTJ': 4, 'ADJ': 1}
DT 52044 {'DET': 45640, 'PRON': 6298, 'NOUN': 32, 'ADV': 33, 'INTJ': 28, 'X': 2, 'NUM': 1, 'ADJ': 2, 'PART': 3, 'SCONJ': 5}
PRP 51769 {'PRON': 51745, 'X': 5, 'INTJ': 9, 'AUX': 3, 'NOUN': 6, 'PART': 1}
. 49881 {'PUNCT': 49801, 'X': 8, 'PRON': 44, 'DET': 12, 'ADV': 7, 'PART': 2, 'INTJ': 4, 'NOUN': 2, 'VERB': 1}
RB 34743 {'ADV': 27031, 'PART': 7500, 'X': 17, 'DET': 24, 'ADP': 41, 'ADJ': 48, 'PRON': 15, 'SCONJ': 17, 'VERB': 2, 'INTJ

#Word Feature Extraction

In [ ]:
import re, string, torch
import numpy as np

params0={"len":True,"caps":True,"n_chars":4,"norm_digit":True,"check_hyphen":True,"check_dot":True, "max_len":15}
#params0={"len":True,"caps":True,"n_chars":4,"norm_digit":False,"check_hyphen":True,"max_len":15}

class wd_ft:
  def __init__(self,params={}) -> None:
    self.params=params
    self.all_feature_slots=[]
    if self.params.get("len",True):
      max_len0=self.params.get("max_len",20)
      for len0 in range(1,max_len0+1): self.all_feature_slots.append(f"len:{len0}")
    if self.params.get("caps",True):
      self.all_feature_slots.append(f"caps_0")
      self.all_feature_slots.append(f"caps_all")
    if self.params.get("check_hyphen",True): self.all_feature_slots.append("hyphen")
    if self.params.get("dot",True): self.all_feature_slots.append("dot")
    all_alpha_chars=string.ascii_lowercase
    if params.get("norm_digit",True): all_alpha_chars+="5"
    else: all_alpha_chars+=string.digits

    n_pre_suf_chars=self.params.get("n_chars",4)
    for char_i in range(n_pre_suf_chars):
      rev_i=-char_i-1
      if char_i==0: #apply to only first/last char
        for punc0 in string.punctuation:
          self.all_feature_slots.append(f"{punc0}|{char_i}")
          self.all_feature_slots.append(f"{punc0}|{rev_i}")
      else: #empty chars - shorter words
        self.all_feature_slots.append(f"|{char_i}")
        self.all_feature_slots.append(f"|{rev_i}")
      for char0 in all_alpha_chars:
        self.all_feature_slots.append(f"{char0}|{char_i}")
        self.all_feature_slots.append(f"{char0}|{rev_i}")


    self.feature_idx_dict=dict(iter([(v,i) for i,v in enumerate(self.all_feature_slots)]))

    self.n_features=len(self.all_feature_slots)

  def extract(self,word):
    all_features=[]
    if self.params.get("len",True):
      len_word=min(len(word),self.params.get("max_len",20))
      all_features.append(f"len:{len_word}")
    if self.params.get("caps",True):
      if word[0].isupper(): all_features.append(f"caps_0")
      if word.isupper(): all_features.append(f"caps_all")
    word=word.lower()
    if len(word)>3:
      if self.params.get("check_hyphen",True) and len(word.split("-"))>1: all_features.append("hyphen")
      if self.params.get("check_dot",True) and len(word.split("."))>1: all_features.append("dot")
    if self.params.get("norm_digit",True): word=re.sub(r"\d",r"5",word)
    n_pre_suf_chars=self.params.get("n_chars",4)
    for char_i in range(n_pre_suf_chars):
      rev_i=-char_i-1
      if char_i>len(word)-1: cur_char,cur_rev_char="",""
      else:
        cur_char=word[char_i]
        cur_rev_char=word[rev_i]
      all_features.append(f"{cur_char}|{char_i}")
      all_features.append(f"{cur_rev_char}|{rev_i}")
    return all_features


  def to_tensor(self,word):
    features=self.extract(word)
    #ft_vector_list=[0.]*self.n_features #len(self.all_feature_slots)

    ft_vector = np.zeros(self.n_features)

    #ft_tensor0 = torch.zeros(self.n_features)
    #ft_vector_list=[0.]*len()
    #indexes=[]
    for ft0 in features:
      idx0=self.feature_idx_dict.get(ft0)
      if idx0!=None:
        #indexes.append((ft0,idx0))
        #ft_vector_list[idx0]=1.
        #ft_tensor0[idx0]=1.
        ft_vector[idx0]=1.
    #return indexes
    #return ft_vector #ft_tensor0 #torch.tensor(ft_vector_list) #ft_vector_list
    #return torch.tensor(ft_vector,dtype=torch.float32)
    return torch.tensor(ft_vector,dtype=torch.float32)


word0="inadverte.nt.ly123"
#word0="later"
#extract_word_features(word0,params0)

wd_ft_obj=wd_ft(params0)
wd_ft_obj.extract(word0)
#word_feat_list=create_word_feat_list(params0)


['len:15', 'dot', 'i|0', '5|-1', 'n|1', '5|-2', 'a|2', '5|-3', 'd|3', 'y|-4']

In [ ]:
import time
word0="absolutely"
#wd_ft_obj.extract(word0)
t0=time.time()
for _ in range(100000):
  out=wd_ft_obj.to_tensor(word0)
t1=time.time()
#out.sort(key=lambda x:x[0])
print(out)
print(t1-t0)
#wd_ft_obj.feature_idx_dict

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 

#Label Processing

In [ ]:
import numpy as np
import torch

#a class for processing multi-label inputs for classification tasks (e.g. both xpos & upos labels for each word)
class label_proc:
  def __init__(self,label_dict) -> None:
    self.label_dict=label_dict
    self.key_label_span_dict={}
    self.flattened_labels=[]
    self.label_loc_dict={}
    i0=0
    for key,cur_labels in label_dict.items():
      cur_loc_dict={}
      for inc0,label0 in enumerate(cur_labels):
        cur_loc_dict[label0]=i0+inc0
      self.label_loc_dict[key]=cur_loc_dict
      self.key_label_span_dict[key]=(i0,i0+len(cur_labels))
      i0+=len(cur_labels)
      self.flattened_labels.extend(cur_labels)
    self.n_labels=len(self.flattened_labels)
  def encode(self,input_labels): #input labels as dictionary {"xpos":"NN","upos":"NOUN"}
    cur_vec=np.zeros(self.n_labels)
    for key0,val0 in input_labels.items():
      found=self.label_loc_dict.get(key0,{})
      found_i=found.get(val0)
      if found_i!=None: cur_vec[found_i]=1.
    return torch.tensor(cur_vec,dtype=torch.float32)
  def decode(self,vec,sort_wt=True):
    try: vec=vec.tolist() #change numpy array/torch tensor to list - if not already a list
    except: pass
    out_dict={}
    for key,span in self.key_label_span_dict.items():
      start_i,end_i=span
      cur_labels= self.label_dict[key]
      cur_weights= vec[start_i:end_i]
      cur_label_wt_list=list(zip(cur_labels,cur_weights))
      if sort_wt: cur_label_wt_list.sort(key=lambda x:-x[-1])
      out_dict[key]=cur_label_wt_list
    return out_dict





test_dict={}
test_dict["a"]=["abba","ace","anaconda","araba"]
test_dict["b"]=["banana","boat"]
test_dict["c"]=["cara","cayman","coco"]

lp_obj=label_proc(test_dict)

In [ ]:
tensor0=lp_obj.encode({"a":"araba","c":"coco","b":"boat"})
tensor0

tensor([0., 0., 0., 1., 0., 1., 0., 0., 1.], dtype=torch.float64)

In [ ]:
test_input=torch.rand(lp_obj.n_labels)
lp_obj.decode(test_input)
#lp_obj.decode(tensor0)


{'a': [('abba', 0.8700562715530396),
  ('anaconda', 0.38907766342163086),
  ('araba', 0.28614532947540283),
  ('ace', 0.15671706199645996)],
 'b': [('banana', 0.6333872675895691), ('boat', 0.4177848696708679)],
 'c': [('cara', 0.8350324630737305),
  ('coco', 0.5850570201873779),
  ('cayman', 0.4866282343864441)]}

In [ ]:
tensor0.tolist()

[0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0]

In [ ]:
lp_obj.flattened_labels, lp_obj.n_labels, lp_obj.key_label_span_dict,lp_obj.label_loc_dict

(['abba',
  'ace',
  'anaconda',
  'araba',
  'banana',
  'boat',
  'cara',
  'cayman',
  'coco'],
 9,
 {'a': (0, 4), 'b': (4, 6), 'c': (6, 9)},
 {'a': {'abba': 0, 'ace': 1, 'anaconda': 2, 'araba': 3},
  'b': {'banana': 4, 'boat': 5},
  'c': {'cara': 6, 'cayman': 7, 'coco': 8}})

In [ ]:
import torch
#torch.tensor()
tensor_1d = torch.zeros(5)
tensor_1d[1]=1.
tensor_1d

tensor([0., 1., 0., 0., 0.])

In [ ]:
items=list(word_upos_dict.items())
items.sort(key=lambda x:-sum(x[1].values()))
word_feature_counter={}
xpos_freq_counter={}
feature_xpos_counter={}
for a,b in items:
  word0,main_pos_tags=a,b
  word_features=extract_word_features(word0)
  cur_xpos_dict=word_xpos_dict[word0]
  total_count=sum(cur_xpos_dict.values())
  for ft0 in word_features:
    word_feature_counter[ft0]=word_feature_counter.get(ft0,0)+total_count
  for key0,val0 in cur_xpos_dict.items():
    xpos_freq_counter[key0]=xpos_freq_counter.get(key0,0)+val0
    for ft0 in word_features:
      pair0=(ft0,key0)
      feature_xpos_counter[pair0]=feature_xpos_counter.get(pair0,0)+val0

  # print(a,total_count,word_features)
  # print(main_pos_tags)
  # print(word_xpos_dict[a])
  # print(word_feat_dict[a])
  # print("------")

# ft_items=list(word_feature_counter.items())
# ft_items.sort(key=lambda x:-x[-1])
# for a,b in ft_items[:100]: print(a,b)
# print("ft_items",len(ft_items))

# xpos_items=list(xpos_freq_counter.items())
# xpos_items.sort(key=lambda x:-x[-1])
# for a,b in xpos_items[:100]: print(a,b)
# print("xpos_items",len(xpos_items))

xpos_ft_items=list(feature_xpos_counter.items())
xpos_ft_items.sort(key=lambda x:-x[-1])
final_ft_xpos_wt_dict={}
for pair0,freq0 in xpos_ft_items:
  ft0,xpos0=pair0
  ft_freq0=word_feature_counter.get(ft0,0)
  xpos_freq0=xpos_freq_counter.get(xpos0,0)
  union_freq0=ft_freq0+xpos_freq0-freq0
  wt0=freq0/union_freq0
  found_dict=final_ft_xpos_wt_dict.get(ft0,{})
  found_dict[xpos0]=round(wt0,4)
  final_ft_xpos_wt_dict[ft0]=found_dict
  #print(pair0,freq0,round(wt0,4))
  #final_ft_xpos_wt_dict[pair0]=wt0
print("xpos_ft_items",len(xpos_ft_items))

xpos_ft_wt_items=list(final_ft_xpos_wt_dict.items())
#xpos_ft_wt_items.sort(key=lambda x:-x[-1])
for a,b in xpos_ft_wt_items[:100]:
  print(a,b)




NameError: name 'word_upos_dict' is not defined

In [ ]:
word="strategizing"
word="natural"
word="sweet"
features=extract_word_features(word,params={})
#features
for ft0 in features:
  print(ft0)
  cur_wt_dict=final_ft_xpos_wt_dict.get(ft0,{})
  cur_wt_dict_items=list(cur_wt_dict.items())
  cur_wt_dict_items.sort(key=lambda x:-x[-1])
  print(cur_wt_dict_items)
  print("-----")

len:5
[('NN', 0.1018), ('NNP', 0.0677), ('JJ', 0.0633), ('NNS', 0.0474), ('RB', 0.0427), ('VB', 0.0377), ('_', 0.0303), ('MD', 0.0289), ('SG-NOM', 0.0249), ('VBG', 0.024), ('IN', 0.0212), ('VBZ', 0.0211), ('VBP', 0.0187), ('EX', 0.0183), ('S', 0.0165), ('WRB', 0.0165), ('VBN', 0.0121), ('DT', 0.0119), ('VBD', 0.0118), ('WDT', 0.0108), ('CD', 0.0104), ('PRP$', 0.0102), ('POS', 0.01), ('PL-NOM', 0.0087), ('V', 0.0073), ('A', 0.0064), ('PAST-AUX', 0.0047), ('SP', 0.004), ('PAST', 0.0037), ('INF', 0.0036), ('B', 0.0027), ('VM', 0.0027), ('PR', 0.0024), ('UH', 0.0021), ('PRES', 0.0021), ('JJS', 0.0021), ('E', 0.002), ('ING', 0.0019), ('WH-REL', 0.0019), ('JJR', 0.0018), ('P3PL-GEN', 0.0017), ('NNPS', 0.0015), ('RBR', 0.0015), ('AP', 0.0015), ('PASS', 0.0015), ('WP$', 0.0012), ('DEM-PL', 0.001), ('DD', 0.001), ('NO', 0.001), ('PERF', 0.0009), ('PRP', 0.0007), ('ORD', 0.0007), ('NEG', 0.0007), ('CS', 0.0006), ('SG', 0.0006), ('CMP', 0.0006), ('PD', 0.0005), ('FW', 0.0005), ('GW', 0.0005), ('C

In [ ]:
word="so"
word_upos_dict[word], word_xpos_dict[word], word_feat_dict[word]


({'ADV': 1125, 'SCONJ': 191, 'ADP': 1, 'CCONJ': 45, 'INTJ': 28},
 {'RB': 1005, 'IN': 149, '_': 123, 'CC': 45, 'UH': 28, 'B': 27, 'CS': 13},
 {'_': 1305, 'ExtPos=SCONJ': 82, 'Degree=Pos': 2, 'Typo=Yes': 1})

In [ ]:
!head -150 ../data/UD/UD_English-EWT/en_ewt-ud-train.conllu

# newdoc id = weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000
# sent_id = weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000-0001
# newpar id = weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000-p0001
# text = Al-Zaman : American forces killed Shaikh Abdullah al-Ani, the preacher at the mosque in the town of Qaim, near the Syrian border.
1	Al	Al	PROPN	NNP	Number=Sing	0	root	0:root	SpaceAfter=No
2	-	-	PUNCT	HYPH	_	3	punct	3:punct	SpaceAfter=No
3	Zaman	Zaman	PROPN	NNP	Number=Sing	1	flat	1:flat	_
4	:	:	PUNCT	:	_	7	punct	7:punct	_
5	American	American	ADJ	JJ	Degree=Pos	6	amod	6:amod	_
6	forces	force	NOUN	NNS	Number=Plur	7	nsubj	7:nsubj	_
7	killed	kill	VERB	VBD	Mood=Ind|Number=Plur|Person=3|Tense=Past|VerbForm=Fin	1	parataxis	1:parataxis	_
8	Shaikh	Shaikh	PROPN	NNP	Number=Sing	7	obj	7:obj	_
9	Abdullah	Abdullah	PROPN	NNP	Number=Sing	8	flat	8:flat	_
10	al	al	PROPN	NNP	Number=Sing	8	flat	8:flat	SpaceAfter=No
11	-	-	PUNCT	HYPH	_	12	punct	12:punct	SpaceAf

In [ ]:
fpath="../data/UD/UD_English-EWT/en_ewt-ud-train.conllu"

word_xpos_list=[]
word_upos_list=[]

word_upos_dict={}
word_xpos_dict={}
word_feat_dict={}


fopen=open(fpath)
cur_conll_str=""
for i0,line0 in enumerate(fopen):
  if line0.startswith("#"): continue
  if line0.strip()=="":
    #print(cur_conll_str)
    #print("======")
    cur_conll_str=""
  else:
    cur_conll_str+=line0
    line_split0=line0.strip().split("\t")
    id0,form0,lemma0,upos0,xpos0,feats0,head0,deprel0,deps0,misc0=line_split0[:10]
    if not id0.isdigit(): continue
    temp_word_upos_dict=word_upos_dict.get(form0,{})
    temp_word_upos_dict[upos0]=temp_word_upos_dict.get(upos0,0)+1
    word_upos_dict[form0]=temp_word_upos_dict

    temp_word_xpos_dict=word_xpos_dict.get(form0,{})
    temp_word_xpos_dict[xpos0]=temp_word_xpos_dict.get(xpos0,0)+1
    word_xpos_dict[form0]=temp_word_xpos_dict

    temp_word_feat_dict=word_feat_dict.get(form0,{})
    temp_word_feat_dict[feats0]=temp_word_feat_dict.get(feats0,0)+1
    word_feat_dict[form0]=temp_word_feat_dict

    #word_feat_dict


    #print(line_split0)
  #print(line0.strip())
  #if i0>300: break
fopen.close()

items=list(word_upos_dict.items())
items.sort(key=lambda x:-sum(x[1].values()))
for a,b in items[:300]:
  print(a,b,word_xpos_dict[a],word_feat_dict[a])

. {'PUNCT': 8640} {'.': 8632, ',': 8} {'_': 8640}
the {'DET': 8141, 'PRON': 7, 'ADP': 2, 'PART': 1} {'DT': 8141, 'WDT': 1, 'PRP': 6, 'IN': 2, 'TO': 1} {'Definite=Def|PronType=Art': 8141, 'PronType=Rel|Typo=Yes': 1, 'Case=Nom|Number=Plur|Person=3|PronType=Prs|Typo=Yes': 6, 'Typo=Yes': 3}
, {'PUNCT': 7019, 'X': 2} {',': 7018, 'GW': 2, '.': 1} {'_': 7021}
to {'PART': 3227, 'ADP': 1785, 'SCONJ': 54, 'ADV': 6, 'VERB': 1, 'DET': 1, 'X': 1, 'NOUN': 1} {'TO': 3227, 'IN': 1840, 'RB': 6, 'VB': 1, 'DT': 1, 'NN': 1} {'_': 5067, 'Typo=Yes': 6, 'Typo=Yes|VerbForm=Inf': 1, 'Definite=Def|PronType=Art|Typo=Yes': 1, 'Number=Sing': 1}
and {'CCONJ': 4843, 'DET': 4, 'X': 6, 'ADP': 2} {'CC': 4843, 'DT': 4, 'GW': 6, 'IN': 2} {'_': 4849, 'Definite=Ind|PronType=Art|Typo=Yes': 4, 'Typo=Yes': 2}
a {'DET': 3589, 'NUM': 4, 'ADP': 7, 'NOUN': 4, 'ADV': 2, 'CCONJ': 1, 'PART': 1, 'AUX': 1} {'DT': 3589, 'LS': 4, 'IN': 7, 'NN': 4, 'RB': 2, 'CC': 1, 'TO': 1, 'MD': 1} {'Definite=Ind|PronType=Art': 3589, '_': 5, 'Abbr=Yes'

In [ ]:
treebank_fpath="../data/UD/UD_English-EWT/en_ewt-ud-train.conllu"
cur_iterator=iter_treebank(treebank_fpath)

for i0,a0 in enumerate(cur_iterator):
  print(a0)
  print("========")
  if i0>20: break

1	Aesthetic	aesthetic	ADJ	JJ	Degree=Pos	2	amod	2:amod	Discourse=organization-heading:1->57:8:grf-ly-_-_+sem-lxchn-1,671-_+sem-lxchn-4,619-_+sem-lxchn-5,620-_|Entity=(1-abstract-new-nnnnn-cf1-2-sgl|MSeg=Aesthet-ic
2	Appreciation	appreciation	NOUN	NN	Number=Sing	0	root	0:root	Entity=1)|MSeg=Appreciat-ion
3	and	and	CCONJ	CC	_	5	cc	5:cc	_
4	Spanish	Spanish	ADJ	JJ	Degree=Pos	5	amod	5:amod	Entity=(2-abstract-new-snssn-cf2-2-coref|MSeg=Span-ish
5	Art	art	NOUN	NN	Number=Sing	2	conj	2:conj:and	Entity=2)|SpaceAfter=No
6	:	:	PUNCT	:	_	2	punct	2:punct	_

1	Insights	insight	NOUN	NNS	Number=Plur	0	root	0:root	Discourse=elaboration-additional:2->1:0:grf-col-6-gold|Entity=(3-abstract-new-nnnns-cf1-1-coref|MSeg=In-sigh-t-s
2	from	from	ADP	IN	_	5	case	5:case	_
3	Eye	eye	NOUN	NN	Number=Sing	5	compound	5:compound	Entity=(4-abstract-new-sssss-cf2-3-coref(5-object-new-sssss-cf3-1-coref)|SpaceAfter=No|XML=<w>
4	-	-	PUNCT	HYPH	_	3	punct	3:punct	SpaceAfter=No
5	Tracking	tracking	NOUN	NN	Number=Sing	1	nmod	1:nm

In [ ]:
#!python -m spacy download en_core_web_md

!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 2.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


#Other Parsers

##spaCy

In [ ]:
from code_utils.parsing_lib import *
from code_utils.dep_lib import *
import spacy
import shelve, time, json

from spacy.tokenizer import Tokenizer
#nlp = spacy.load("en_core_web_sm")
#nlp = spacy.load("en_core_web_md")
#nlp = spacy.load("en_core_web_lg")

nlp.tokenizer = Tokenizer(nlp.vocab, token_match=re.compile(r'\S+').match)

#21 December 2025
def spacy_parse(tokens,strip_underscore=False):
  if strip_underscore: tokens=[v.strip("_") for v in tokens]
  joined_sent=" ".join(tokens)
  doc = nlp(joined_sent)
  conll_lines=[]
  for token in doc:
      token_i = token.i+1
      if token.i==token.head.i: head_i=0
      else: head_i = token.head.i+1
      items=[token_i,token.text, token.lemma_, token.tag_, token.pos_, "_", head_i, token.dep_,"_","_"]
      line0="\t".join([str(v) for v in items])
      conll_lines.append(line0)
  return "\n".join(conll_lines)




#spacy.load()
res_file_fpath="spacy_out_res.tsv"
res_file_fopen=open(res_file_fpath,"w")

parser_name="spacy_sm"
parser_output_main_dir="../output/parsing"
cur_parser_dir=os.path.join(parser_output_main_dir,parser_name)
if not os.path.exists(cur_parser_dir): os.makedirs(cur_parser_dir)


cur_parser_temp_shelve_path=os.path.join(cur_parser_dir,"temp_output_json_2.shelve")
cur_parser_temp_shelve=shelve.open(cur_parser_temp_shelve_path)

used_dict={}
parsed_dict={}
for a,b in cur_parser_temp_shelve.items():
  used_dict[a]=True
  parsed_dict[a]=json.loads(b)
cur_parser_temp_shelve.close()



#doc = nlp("Barack Obama was born in Hawaii.") # run annotation over a sentence
train_files,dev_files,test_files=[],[],[]
ud_dir="../data/UD"
#excluded_tb_list=['UD_English-GENTLE']
treebank_dir_list=os.listdir(ud_dir)
for tb0 in treebank_dir_list:
  tb_full_path=os.path.join(ud_dir,tb0)
  tb_files=os.listdir(tb_full_path)
  for tbf0 in tb_files:
    cur_file=os.path.join(tb_full_path,tbf0)
    if "train" in tbf0.lower() and "conllu" in tbf0.lower(): train_files.append(cur_file)
    elif "dev" in tbf0.lower() and "conllu" in tbf0.lower(): dev_files.append(cur_file)
    elif "test" in tbf0.lower() and "conllu" in tbf0.lower(): test_files.append(cur_file)


#print("dev_files",dev_files[0])
cur_dev_file=dev_files[0]

#for cur_file in dev_files[:1]:
for cur_file in dev_files+test_files:
  cur_file_second_part=cur_file.replace(ud_dir,"")
  new_path=cur_parser_dir+cur_file_second_part#os.path.join(cur_parser_dir,cur_dev_file_second_part)
  #print(new_path)
  if "en_gumreddit" in cur_file: continue

  tmp_out_dir=os.path.split(new_path)[0]
  if not os.path.exists(tmp_out_dir): os.makedirs(tmp_out_dir)

  out_fopen=open(new_path,"w")

  total_correct,total_tokens=0,0
  counter=0
  n_sents=0
  iter0=iter_conll_treebank(cur_file)
  for i0,conll0 in enumerate(iter0):
    counter+=1
    n_sents+=1
    if i0%100==0: print(cur_file,i0)
    conll_obj=conll2obj(conll0)
    toks=[v["form"] for v in conll_obj]
    #check_already_parsed=used_dict.get(str(toks),False)
    parsed_dict_info=parsed_dict.get(str(toks))
    if parsed_dict_info!=None:
      spacy_conll=parsed_dict_info["conll"]
    # if check_already_parsed==True:
    #   #continue
    #   cur_parser_temp_shelve=shelve.open(cur_parser_temp_shelve_path)
    #   spacy_output_obj_json=cur_parser_temp_shelve[str(toks)]
    #   spacy_output_obj=json.loads(spacy_output_obj_json)
    #   cur_parser_temp_shelve.close()
    #   spacy_conll=spacy_output_obj["conll"]

    else:
      t0=time.time()
      spacy_conll=spacy_parse(toks,strip_underscore=False)
      t1=time.time()
      elapsed=t1-t0
      spacy_output_obj={"conll":spacy_conll,"elapsed":elapsed}
      cur_parser_temp_shelve=shelve.open(cur_parser_temp_shelve_path)
      cur_parser_temp_shelve[str(toks)]=json.dumps(spacy_output_obj)
      cur_parser_temp_shelve.close()


    spacy_obj=conll2obj(spacy_conll)
    gold_heads=[v["head"] for v in conll_obj]
    spacy_heads=[v["head"] for v in spacy_obj]
    matching=[1 if v==w else 0 for v,w in zip(gold_heads,spacy_heads)]
    total_tokens+=len(matching)
    total_correct+=len([v for v in matching if v==1])

    if len(toks)!=len(spacy_obj):
      print("Something is wrong")
      print(conll0)
      print(toks)

    if not spacy_conll:
      # print("Something is wrong")
      # print(conll0)
      # print(toks)
      conll_obj_dict_list=[]
      for i0,tk0 in enumerate(toks): conll_obj_dict_list.append({"id":i0+1,"form":tk0,"head":0})
      #spacy_conll="\n".join([f"{i0+1}\t{tk0}" for i0,tk0 in enumerate(toks)])
      spacy_conll=obj2conll(conll_obj_dict_list)
      spacy_conll=spacy_conll.strip()
      #print(spacy_conll)

    out_fopen.write(spacy_conll+"\n\n")


  out_fopen.close()

  #uas_score=n_correct/n_total
  #print(cur_dev_file_second_part,"n_sentences",counter, "uas_score",round(uas_score,4))



  avg_sent_n_tokens=total_tokens/n_sents
  print(cur_file)
  print("n_sents",n_sents,"avg_sent_n_tokens",round(avg_sent_n_tokens,1))
  print("total_correct",total_correct)
  print("total_tokens",total_tokens)
  uas_score=total_correct/total_tokens
  print("uas_score",uas_score)
  line0=f"{cur_file}\t{n_sents}\t{round(avg_sent_n_tokens,1)}\t{total_tokens}\t{total_correct}\t{round(uas_score,2)}\n"
  res_file_fopen.write(line0)
res_file_fopen.close()

#print(params0)
print("results saved to:", res_file_fpath)
# sent="To support this integration, my Office has developed practical tools, including the newly launched third edition of the Aide-memoire on options for reflecting weapons and ammunition management in decisions of the Security Council and an online dashboard, which provide guidance for the drafting of resolutions to effectively include small arms and ammunition considerations."
# sent_=re.sub(r"(\W)",r" \1 ",sent)
# toks=sent_.strip().split()
# print(toks)
# spacy_conll=spacy_parse(toks)
# print(spacy_conll)
# doc = nlp(sent) # run annotation over a sentence

# print(doc)
#print(doc.entities)

../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 0
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 100
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 200
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 300
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 400
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 500
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 600
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 700
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 800
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 900
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 1000
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 1100
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 1200
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 1300
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 1400
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 1500
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 1600
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 1700
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 1800
../data/UD/UD_English-EW

###Evaluate Spacy

In [ ]:
from udtools.udeval import load_conllu_file, evaluate, build_evaluation_table

gold_dir="../data/UD/"
system_dir="../output/parsing/spacy_sm/"
# dev_file_pairs=[]
# test_file_pairs=[]
all_results=[]
for dir0 in os.listdir(system_dir):
  #print(dir0)
  system_corpus_path=os.path.join(system_dir,dir0)
  if not os.path.isdir(system_corpus_path): continue
  files=os.listdir(system_corpus_path)
  print(system_corpus_path)
  for file0 in files:
    print(file0)
    label=""
    if "dev" in file0.lower(): label="dev"
    if "test" in file0.lower(): label="test"
    cur_system_fpath=os.path.join(system_corpus_path,file0)
    cur_gold_fpath=os.path.join(gold_dir,dir0,file0)

    gold_ud = load_conllu_file(cur_gold_fpath,treebank_type={'multiple_roots_okay':True})
    system_ud = load_conllu_file(cur_system_fpath,treebank_type={'multiple_roots_okay':True})
    print(cur_system_fpath)
    print(cur_gold_fpath)

    evaluation = evaluate(gold_ud, system_ud)
    uas0=evaluation["UAS"].f1
    las0=evaluation["LAS"].f1
    all_results.append((label,file0,uas0,las0))
    #print(,evaluation["LAS"].f1)


all_results.sort()
for ar in all_results:
  print("\t".join([str(v) for v in ar]))



../output/parsing/spacy_sm/UD_English-EWT
en_ewt-ud-dev.conllu
../output/parsing/spacy_sm/UD_English-EWT/en_ewt-ud-dev.conllu
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu
en_ewt-ud-test.conllu
../output/parsing/spacy_sm/UD_English-EWT/en_ewt-ud-test.conllu
../data/UD/UD_English-EWT/en_ewt-ud-test.conllu
../output/parsing/spacy_sm/UD_English-CHILDES
en_childes-ud-dev.conllu
../output/parsing/spacy_sm/UD_English-CHILDES/en_childes-ud-dev.conllu
../data/UD/UD_English-CHILDES/en_childes-ud-dev.conllu
en_childes-ud-test.conllu
../output/parsing/spacy_sm/UD_English-CHILDES/en_childes-ud-test.conllu
../data/UD/UD_English-CHILDES/en_childes-ud-test.conllu
../output/parsing/spacy_sm/UD_English-Atis
en_atis-ud-dev.conllu
../output/parsing/spacy_sm/UD_English-Atis/en_atis-ud-dev.conllu
../data/UD/UD_English-Atis/en_atis-ud-dev.conllu
en_atis-ud-test.conllu
../output/parsing/spacy_sm/UD_English-Atis/en_atis-ud-test.conllu
../data/UD/UD_English-Atis/en_atis-ud-test.conllu
../output/parsing/spacy_

In [ ]:
!sed -n '8580,8608p' '../output/parsing/spacy_sm/UD_English-GENTLE/en_gentle-ud-test.conllu'


1	His	his	PRP$	PRON	_	2	poss	_	_
2	range	range	NN	NOUN	_	5	nsubj	_	_
3	of	of	IN	ADP	_	2	prep	_	_
4	motion	motion	NN	NOUN	_	3	pobj	_	_
5	is	be	VBZ	AUX	_	0	ROOT	_	_
6	10	10	CD	NUM	_	8	quantmod	_	_
7	to	to	TO	PART	_	8	quantmod	_	_
8	105	105	CD	NUM	_	9	nummod	_	_
9	degrees	degree	NNS	NOUN	_	5	attr	_	_
10	.	.	.	PUNCT	_	5	punct	_	_

1	He	he	PRP	PRON	_	2	nsubj	_	_
2	has	have	VBZ	VERB	_	0	ROOT	_	_
3	no	no	DT	DET	_	4	det	_	_
4	pain	pain	NN	NOUN	_	2	dobj	_	_
5	or	or	CC	CCONJ	_	4	cc	_	_
6	crepitance	crepitance	NN	NOUN	_	4	conj	_	_
7	.	.	.	PUNCT	_	2	punct	_	_

1	On	on	IN	ADP	_	9	prep	_	_
2	examination	examination	NN	NOUN	_	1	pobj	_	_
3	of	of	IN	ADP	_	2	prep	_	_
4	his	his	PRP$	PRON	_	6	poss	_	_
5	right	right	JJ	ADJ	_	6	amod	_	_
6	foot	foot	NN	NOUN	_	3	pobj	_	_
7	,	,	,	PUNCT	_	9	punct	_	_
8	he	he	PRP	PRON	_	9	nsubj	_	_
9	is	be	VBZ	AUX	_	0	ROOT	_	_


In [ ]:
!sed -n '5970,5990p' '../output/parsing/spacy_sm/UD_English-GENTLE/en_gentle-ud-test.conllu'

45	and	and	CC	CCONJ	_	41	cc	_	_
46	Solutions	Solutions	NNPS	PROPN	_	47	compound	_	_
47	Vending	Vending	NNP	PROPN	_	48	compound	_	_
48	International	International	NNP	PROPN	_	38	appos	_	_
49	,	,	,	PUNCT	_	50	punct	_	_
50	Inc	Inc	NNP	PROPN	_	53	nmod	_	_
51	a	a	DT	DET	_	53	det	_	_
52	DE	DE	NNP	PROPN	_	53	compound	_	_
53	corporation	corporation	NN	NOUN	_	30	pobj	_	_
54	(	(	-LRB-	PUNCT	_	56	punct	_	_
55	"	"	``	PUNCT	_	56	punct	_	_
56	Customer	customer	NN	NOUN	_	53	appos	_	_
57	"	"	''	PUNCT	_	56	punct	_	_
58	or	or	CC	CCONJ	_	56	cc	_	_
59	"	"	``	PUNCT	_	60	punct	_	_
60	you	you	PRP	PRON	_	56	conj	_	_
61	"	"	''	PUNCT	_	60	punct	_	_
62	)	)	-RRB-	PUNCT	_	56	punct	_	_
63	.	.	.	PUNCT	_	15	punct	_	_

1	1.	1.	LS	X	_	2	punct	_	_


In [ ]:
toks=['__________________________________________________']
spacy_conll=spacy_parse(toks,strip_underscore=False)
spacy_conll

'1\t__________________________________________________\t__________________________________________________\t.\tPUNCT\t_\t0\tROOT\t_\t_'

In [ ]:
conll0="""1	This	this	DET	DT	Number=Sing|PronType=Dem	3	det	3:det	Discourse=context-background:3->13:6:_|Entity=(3-abstract-new-sssss-cf3-2,3-coref-Contract
2	Services	service	PROPN	NNP	Number=Sing	3	compound	3:compound	Entity=(4-abstract-new-snnnn-cf5-1-coref-Service_%28business%29)
3	Agreement	Agreement	PROPN	NNP	Number=Sing	15	nsubj:pass	15:nsubj:pass	Entity=3)|MSeg=Agree-ment
4	(	(	PUNCT	-LRB-	_	7	punct	7:punct	Discourse=restatement-partial:4->3:0:_|SpaceAfter=No
5	"	"	PUNCT	``	_	7	punct	7:punct	SpaceAfter=No|XML=<q>
6	Services	service	PROPN	NNPS	Number=Plur	7	compound	7:compound	Entity=(3-abstract-giv:act-sssss-cf3-1,2-appos-Contract(4-abstract-giv:act-snnnn-cf5-1-coref-Service_%28business%29)|MSeg=Service-s
7	Agreement	Agreement	PROPN	NNP	Number=Sing	3	appos	3:appos	Entity=3)|MSeg=Agree-ment|SpaceAfter=No
8	"	"	PUNCT	''	_	7	punct	7:punct	XML=</q>
9	or	or	CCONJ	CC	_	11	cc	11:cc	_
10	"	"	PUNCT	``	_	11	punct	11:punct	SpaceAfter=No|XML=<q>
11	Agreement	Agreement	PROPN	NNP	Number=Sing	7	conj	3:appos|7:conj:or	Entity=(3-abstract-giv:act-sssss-cf3-1-coref-Contract)|MSeg=Agree-ment|SpaceAfter=No
12	"	"	PUNCT	''	_	11	punct	11:punct	SpaceAfter=No|XML=</q>
13	)	)	PUNCT	-RRB-	_	7	punct	7:punct	_
14	is	be	AUX	VBZ	Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin	15	aux:pass	15:aux:pass	Discourse=same-unit_m:5->3:1:_
15	entered	enter	VERB	VBN	Tense=Past|VerbForm=Part|Voice=Pass	0	root	0:root	MSeg=enter-ed
16	into	into	ADP	IN	_	15	obl	15:obl	MSeg=in-to
17	as	as	ADP	IN	ExtPos=ADP	20	case	20:case	_
18	of	of	ADP	IN	_	17	fixed	17:fixed	_
19	the	the	DET	DT	Definite=Def|PronType=Art	20	det	20:det	Entity=(5-time-new-nnnnn-cf2-2-coref
20	date	date	NOUN	NN	Number=Sing	15	obl	15:obl:as_of	_
21	noted	note	VERB	VBN	Tense=Past|VerbForm=Part|Voice=Pass	20	acl	20:acl	Discourse=elaboration-attribute:6->5:0:_|MSeg=not-ed
22	below	below	ADV	RB	Degree=Pos	21	advmod	21:advmod	Entity=5)|MSeg=be-low
23	(	(	PUNCT	-LRB-	_	27	punct	27:punct	Discourse=elaboration-additional:7->6:0:_|SpaceAfter=No
24	the	the	DET	DT	Definite=Def|PronType=Art	27	det	27:det	Entity=(5-time-giv:act-nnnnn-cf2-3,4-appos
25	"	"	PUNCT	``	_	27	punct	27:punct	SpaceAfter=No|XML=<q>
26	Effective	effective	ADJ	NNP	Degree=Pos	27	amod	27:amod	MSeg=Effect-ive
27	Date	Date	PROPN	NNP	Number=Sing	20	appos	20:appos	SpaceAfter=No
28	"	"	PUNCT	''	_	27	punct	27:punct	Entity=5)|SpaceAfter=No|XML=</q>
29	)	)	PUNCT	-RRB-	_	27	punct	27:punct	_
30	between	between	ADP	IN	_	32	case	32:case	Discourse=same-unit_m:8->3:1:_|MSeg=be-tween
31	StartEngine	StartEngine	PROPN	NNP	Number=Sing	32	compound	32:compound	Entity=(6-organization-new-sssss-cf4-1,2,4-coref|MSeg=Start-Engine
32	Crowdfunding	Crowdfunding	PROPN	NNP	Number=Sing	15	obl	15:obl:between	MSeg=Crowd-fund-ing|SpaceAfter=No
33	,	,	PUNCT	,	_	34	punct	34:punct	_
34	Inc.	Inc.	PROPN	NNP	Abbr=Yes|Number=Sing	32	nmod:desc	32:nmod:desc	Entity=6)|SpaceAfter=No
35	,	,	PUNCT	,	_	38	punct	38:punct	_
36	a	a	DET	DT	Definite=Ind|PronType=Art	38	det	38:det	Entity=(6-organization-giv:act-sssss-cf4-3-appos
37	Delaware	Delaware	PROPN	NNP	Number=Sing	38	compound	38:compound	Entity=(7-place-new-nnnnn-cf6-1-sgl-Delaware)
38	corporation	corporation	NOUN	NN	Number=Sing	32	appos	32:appos	Entity=6)|MSeg=corpor-ation
39	(	(	PUNCT	-LRB-	_	41	punct	41:punct	Discourse=restatement-partial:9->8:0:_|SpaceAfter=No
40	"	"	PUNCT	``	_	41	punct	41:punct	SpaceAfter=No|XML=<q>
41	Company	Company	PROPN	NNP	Number=Sing	32	appos	32:appos	Entity=(6-organization-giv:act-sssss-cf4-1-coref)|SpaceAfter=No
42	"	"	PUNCT	''	_	41	punct	41:punct	SpaceAfter=No|XML=</q>
43	)	)	PUNCT	-RRB-	_	41	punct	41:punct	SpaceAfter=No
44	,	,	PUNCT	,	_	47	punct	47:punct	_
45	and	and	CCONJ	CC	_	47	cc	47:cc	Discourse=same-unit_m:10->3:1:_
46	Solutions	Solution	PROPN	NNPS	Number=Plur	47	compound	47:compound	Entity=(8-organization-new-sssss-cf1-1,2,3,5-coref|MSeg=Solu-tion-s
47	Vending	Vending	PROPN	NNP	Number=Sing	32	conj	15:obl:between|32:conj:and	MSeg=Vend-ing
48	International	International	ADJ	NNP	Degree=Pos	47	amod	47:amod	MSeg=Inter-nation-al|SpaceAfter=No
49	,	,	PUNCT	,	_	50	punct	50:punct	_
50	Inc	Inc	PROPN	NNP	Number=Sing	47	nmod:desc	47:nmod:desc	_
51	a	a	DET	DT	Definite=Ind|PronType=Art	53	det	53:det	_
52	___DE	___DE	ADJ	JJ	Degree=Pos	53	compound	53:compound	_
53	corporation	corporation	NOUN	NN	Number=Sing	47	appos	47:appos	Entity=8)|MSeg=corpor-ation
54	(	(	PUNCT	-LRB-	_	56	punct	56:punct	Discourse=restatement-partial:11->10:0:_|SpaceAfter=No
55	"	"	PUNCT	``	_	56	punct	56:punct	SpaceAfter=No|XML=<q>
56	Customer	customer	NOUN	NN	Number=Sing	47	appos	47:appos	Entity=(8-organization-giv:act-sssss-cf1-1-appos)|MSeg=Custom-er|SpaceAfter=No
57	"	"	PUNCT	''	_	56	punct	56:punct	XML=</q>
58	or	or	CCONJ	CC	_	60	cc	60:cc	_
59	"	"	PUNCT	``	_	60	punct	60:punct	SpaceAfter=No|XML=<q>
60	you	you	PRON	PRP	Case=Acc|Number=Sing|Person=2|PronType=Prs	56	conj	47:appos|56:conj:or	Entity=(8-organization-giv:act-sssss-cf1-1-ana)|SpaceAfter=No
61	"	"	PUNCT	''	_	60	punct	60:punct	SpaceAfter=No|XML=</q>
62	)	)	PUNCT	-RRB-	_	56	punct	56:punct	SpaceAfter=No
63	.	.	PUNCT	.	_	15	punct	15:punct	_
"""
obj0=conll2obj(conll0)

[{'id': '1',
  'form': 'This',
  'lemma': 'this',
  'upos': 'DET',
  'xpos': 'DT',
  'feats': 'Number=Sing|PronType=Dem',
  'head': '3',
  'deprel': 'det',
  'deps': '3:det',
  'misc': 'Discourse=context-background:3->13:6:_|Entity=(3-abstract-new-sssss-cf3-2,3-coref-Contract'},
 {'id': '2',
  'form': 'Services',
  'lemma': 'service',
  'upos': 'PROPN',
  'xpos': 'NNP',
  'feats': 'Number=Sing',
  'head': '3',
  'deprel': 'compound',
  'deps': '3:compound',
  'misc': 'Entity=(4-abstract-new-snnnn-cf5-1-coref-Service_%28business%29)'},
 {'id': '3',
  'form': 'Agreement',
  'lemma': 'Agreement',
  'upos': 'PROPN',
  'xpos': 'NNP',
  'feats': 'Number=Sing',
  'head': '15',
  'deprel': 'nsubj:pass',
  'deps': '15:nsubj:pass',
  'misc': 'Entity=3)|MSeg=Agree-ment'},
 {'id': '4',
  'form': '(',
  'lemma': '(',
  'upos': 'PUNCT',
  'xpos': '-LRB-',
  'feats': '_',
  'head': '7',
  'deprel': 'punct',
  'deps': '7:punct',
  'misc': 'Discourse=restatement-partial:4->3:0:_|SpaceAfter=No'},
 {'id'

In [ ]:
toks

['However',
 ',',
 'you',
 'might',
 'attract',
 'other',
 'types',
 'of',
 'owls',
 'or',
 'birds',
 'of',
 'prey',
 'with',
 'your',
 'nesting',
 'box',
 '.']

In [ ]:
load_conllu_file("../output/parsing/spacy_sm/UD_English-EWT/en_ewt-ud-dev.conllu",treebank_type={'multiple_roots_okay':True})

UDError: There is a sentence with 0 tokens (possibly a double blank line) at line 6389

In [ ]:
cur_system_fpath

'../output/parsing/spacy_sm/UD_English-EWT/en_ewt-ud-dev.conllu'

In [ ]:
!head ../output/parsing/spacy_sm/UD_English-EWT/en_ewt-ud-dev.conllu


1	From	from	IN	ADP	_	4	prep	_	_
2	the	the	DT	DET	_	3	det	_	_
3	AP	AP	NNP	PROPN	_	1	pobj	_	_
4	comes	come	VBZ	VERB	_	0	ROOT	_	_
5	this	this	DT	DET	_	6	det	_	_
6	story	story	NN	NOUN	_	4	dobj	_	_
7	:	:	:	PUNCT	_	4	punct	_	_

1	President	President	NNP	PROPN	_	2	compound	_	_
2	Bush	Bush	NNP	PROPN	_	5	nsubj	_	_


In [ ]:
#!tail -6389 ../output/parsing/spacy_sm/UD_English-EWT/en_ewt-ud-dev.conllu
#!sed -n '6380,6489p' ../output/parsing/spacy_sm/UD_English-EWT/en_ewt-ud-dev.conllu
!sed -n '6380,6398p' ../output/parsing/spacy_sm/UD_English-EWT/en_ewt-ud-dev.conllu
6391


1	?	?	.	PUNCT	_	0	ROOT	_	_

1	Best	good	JJS	ADJ	_	2	amod	_	_
2	regards	regard	NNS	NOUN	_	0	ROOT	_	_
3	.	.	.	PUNCT	_	2	punct	_	_

1	?	?	.	PUNCT	_	0	ROOT	_	_

1	______________________	_	_	_	_	0	_	_	_


1	John	John	NNP	PROPN	_	4	compound	_	_
2	Staikos	Staikos	NNP	PROPN	_	4	compound	_	_
3	Direct	Direct	NNP	PROPN	_	4	compound	_	_
4	Dial	Dial	NNP	PROPN	_	0	ROOT	_	_
5	:?	:?	:	PUNCT	_	4	punct	_	_
6	203.961.7523	203.961.7523	CD	NUM	_	8	nummod	_	_
7	Direct	Direct	NNP	PROPN	_	8	compound	_	_


6391

In [ ]:
#print(res_file_fpath)
!cat spacy_out_res.tsv



../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu	2001	12.6	25144	14108	0.56
../data/UD/UD_English-CHILDES/en_childes-ud-dev.conllu	3860	6.3	24310	16205	0.67
../data/UD/UD_English-Atis/en_atis-ud-dev.conllu	572	11.6	6644	2768	0.42
../data/UD/UD_English-ESLSpok/en_eslspok-ud-dev.conllu	232	9.3	2167	1324	0.61
../data/UD/UD_English-ParTUT/en_partut-ud-dev.conllu	156	17.4	2716	1485	0.55
../data/UD/UD_English-LinES/en_lines-ud-dev.conllu	1118	19.4	21637	12141	0.56
../data/UD/UD_English-GUM/en_gum-ud-dev.conllu	1575	17.9	28119	14992	0.53
../data/UD/UD_English-GENTLE/en_gentle-ud-test.conllu	1334	13.3	17799	9282	0.52
../data/UD/UD_English-EWT/en_ewt-ud-test.conllu	2077	12.1	25091	14217	0.57
../data/UD/UD_English-CTeTex/en_ctetex-ud-test.conllu	276	33.6	9273	4868	0.52
../data/UD/UD_English-CHILDES/en_childes-ud-test.conllu	9591	6.0	57400	38783	0.68
../data/UD/UD_English-Pronouns/en_pronouns-ud-test.conllu	285	6.0	1705	1086	0.64
../data/UD/UD_English-Atis/en_atis-ud-test.conllu	586	11.2	6580	2625	

##Stanza parser

In [ ]:
#https://stanfordnlp.github.io/stanza/#getting-started
!pip install stanza

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 31.2 MB/s eta 0:00:00


In [ ]:
import stanza
stanza.download('en') # download English model
stanza_nlp = stanza.Pipeline(lang='en', processors='tokenize,pos,lemma,depparse', tokenize_pretokenized=True, tokenize_no_ssplit=True)
#nlp = stanza.Pipeline('en') # initialize English neural pipeline
#stanza_nlp = stanza.Pipeline(lang='en', tokenize_pretokenized=True)
#stanza_nlp = stanza.Pipeline(lang='en', tokenize_pretokenized=True, tokenize_no_ssplit=True)

#depparse


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: en (English) ...


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/en/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |
| depparse  | combined_charlm   |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Done loading processors!


###Run Stanza

In [ ]:
from code_utils.dep_lib import *
import shelve, time, json
res_file_fpath="stanza_out_res.tsv"
res_file_fopen=open(res_file_fpath,"w")

parser_name="stanza_1"
parser_output_main_dir="../output/parsing"
cur_parser_dir=os.path.join(parser_output_main_dir,parser_name)

cur_parser_temp_shelve_path=os.path.join(cur_parser_dir,"temp_output_json.shelve")
cur_parser_temp_shelve=shelve.open(cur_parser_temp_shelve_path)

used_dict={}
for a in cur_parser_temp_shelve.keys(): used_dict[a]=True
cur_parser_temp_shelve.close()

stanza_headers=["id","text","lemma","upos","xpos","feats","head","deprel","deps","misc"]

#doc = nlp("Barack Obama was born in Hawaii.") # run annotation over a sentence
train_files,dev_files,test_files=[],[],[]
ud_dir="../data/UD"
#excluded_tb_list=['UD_English-GENTLE']
treebank_dir_list=os.listdir(ud_dir)
for tb0 in treebank_dir_list:
  tb_full_path=os.path.join(ud_dir,tb0)
  tb_files=os.listdir(tb_full_path)
  for tbf0 in tb_files:
    cur_file=os.path.join(tb_full_path,tbf0)
    if "train" in tbf0.lower() and "conllu" in tbf0.lower(): train_files.append(cur_file)
    elif "dev" in tbf0.lower() and "conllu" in tbf0.lower(): dev_files.append(cur_file)
    elif "test" in tbf0.lower() and "conllu" in tbf0.lower(): test_files.append(cur_file)


#print("dev_files",dev_files[0])
cur_dev_file=dev_files[0]
cur_files=dev_files+test_files

for cur_file in cur_files:
#for cur_file in dev_files[:1]:
#for cur_file in dev_files+test_files:
  cur_file_second_part=cur_file.replace(ud_dir,"")
  new_path=cur_parser_dir+cur_file_second_part#os.path.join(cur_parser_dir,cur_dev_file_second_part)
  #print(new_path)
  if "en_gumreddit" in cur_file: continue

  tmp_out_dir=os.path.split(new_path)[0]
  if not os.path.exists(tmp_out_dir): os.makedirs(tmp_out_dir)

  out_fopen=open(new_path,"w")

  total_correct,total_tokens=0,0
  counter=0
  n_sents=0
  iter0=iter_conll_treebank(cur_file)
  for i0,conll0 in enumerate(iter0):
    counter+=1
    n_sents+=1
    if i0%500==0: print(cur_file,i0)
    #if i0==50: break
    conll_obj=conll2obj(conll0)
    toks=[v["form"] for v in conll_obj]
    #gold_heads=[v["form"] for v in conll_obj]
    gold_heads=[v["head"] for v in conll_obj]

    check_already_parsed=used_dict.get(str(toks),False)
    if check_already_parsed==True:
      cur_parser_temp_shelve=shelve.open(cur_parser_temp_shelve_path)
      stanza_output_obj_json=cur_parser_temp_shelve[str(toks)]
      stanza_output_obj=json.loads(stanza_output_obj_json)
      cur_parser_temp_shelve.close()
      stanza_dep_obj=stanza_output_obj["raw"]
      stanza_conll_str=stanza_output_obj["conll"]

    else:
      t0=time.time()
      stanza_doc = stanza_nlp([toks])
      stanza_obj_list=stanza_doc.to_dict()
      t1=time.time()
      elapsed=t1-t0
      #print("stanza_obj_list",stanza_obj_list)
      stanza_dep_obj=stanza_obj_list[0]
      #stanza_conll_str=obj2conll(stanza_obj_list[0])
      stanza_conll_str=obj2conll(stanza_dep_obj,params={"headers":stanza_headers})

      print(cur_file_second_part, i0, "elapsed",elapsed)
      #print([stanza_conll_str])
      stanza_output_obj={"conll":stanza_conll_str,"raw":stanza_dep_obj,"elapsed":elapsed}


      cur_parser_temp_shelve=shelve.open(cur_parser_temp_shelve_path)
      cur_parser_temp_shelve[str(toks)]=json.dumps(stanza_output_obj)
      cur_parser_temp_shelve.close()

    stanza_heads=[str(v["head"]) for v in stanza_dep_obj]
    matching=[1 if v==w else 0 for v,w in zip(gold_heads,stanza_heads)]
    total_tokens+=len(matching)
    total_correct+=len([v for v in matching if v==1])


    #out_fopen.write(stanza_conll_str+"\n\n")
    out_fopen.write(stanza_conll_str+"\n")


  out_fopen.close()

  #uas_score=n_correct/n_total
  #print(cur_dev_file_second_part,"n_sentences",counter, "uas_score",round(uas_score,4))



  avg_sent_n_tokens=total_tokens/n_sents
  print(cur_file)
  print("n_sents",n_sents,"avg_sent_n_tokens",round(avg_sent_n_tokens,1))
  print("total_correct",total_correct)
  print("total_tokens",total_tokens)
  if total_tokens==0: uas_score=0
  else: uas_score=total_correct/total_tokens
  print("uas_score",uas_score)
  line0=f"{cur_file}\t{n_sents}\t{round(avg_sent_n_tokens,1)}\t{total_tokens}\t{total_correct}\t{round(uas_score,2)}\n"
  res_file_fopen.write(line0)
res_file_fopen.close()

#cur_parser_temp_shelve.close()

#print(params0)
print("results saved to:", res_file_fpath)
print("cur_parser_temp_shelve_path",cur_parser_temp_shelve_path)

../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 0
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 500
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 1000
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 1500
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu 2000
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu
n_sents 2001 avg_sent_n_tokens 12.6
total_correct 22837
total_tokens 25147
uas_score 0.9081401360003182
../data/UD/UD_English-CHILDES/en_childes-ud-dev.conllu 0
../data/UD/UD_English-CHILDES/en_childes-ud-dev.conllu 500
../data/UD/UD_English-CHILDES/en_childes-ud-dev.conllu 1000
../data/UD/UD_English-CHILDES/en_childes-ud-dev.conllu 1500
../data/UD/UD_English-CHILDES/en_childes-ud-dev.conllu 2000
../data/UD/UD_English-CHILDES/en_childes-ud-dev.conllu 2500
../data/UD/UD_English-CHILDES/en_childes-ud-dev.conllu 3000
../data/UD/UD_English-CHILDES/en_childes-ud-dev.conllu 3500
../data/UD/UD_English-CHILDES/en_childes-ud-dev.conllu
n_sents 3860 avg_sent_n_tokens 6.3
total_correct 22031
total_t

### Evaluate Stanza

In [ ]:
from udtools.udeval import load_conllu_file, evaluate, build_evaluation_table

gold_dir="../data/UD/"
system_dir="../output/parsing/stanza_1/"
# dev_file_pairs=[]
# test_file_pairs=[]
all_results=[]
for dir0 in os.listdir(system_dir):
  #print(dir0)
  system_corpus_path=os.path.join(system_dir,dir0)
  if not os.path.isdir(system_corpus_path): continue
  files=os.listdir(system_corpus_path)
  print(system_corpus_path)
  for file0 in files:
    print(file0)
    label=""
    if "dev" in file0.lower(): label="dev"
    if "test" in file0.lower(): label="test"
    cur_system_fpath=os.path.join(system_corpus_path,file0)
    cur_gold_fpath=os.path.join(gold_dir,dir0,file0)

    gold_ud = load_conllu_file(cur_gold_fpath)
    system_ud = load_conllu_file(cur_system_fpath)

    evaluation = evaluate(gold_ud, system_ud)
    uas0=evaluation["UAS"].f1
    las0=evaluation["LAS"].f1
    all_results.append((label,file0,uas0,las0))
    #print(,evaluation["LAS"].f1)


all_results.sort()
for ar in all_results:
  print("\t".join([str(v) for v in ar]))

    #print(cur_system_fpath, cur_gold_fpath)

# gold_path="../data/UD/UD_English-LinES/en_lines-ud-test.conllu"
# system_path="../output/parsing/stanza_1/UD_English-LinES/en_lines-ud-test.conllu"

# # gold_ud = load_conllu_file('gold.conllu')
# # system_ud = load_conllu_file('system.conllu')
# gold_ud = load_conllu_file(gold_path)
# system_ud = load_conllu_file(system_path)

# evaluation = evaluate(gold_ud, system_ud)
# results = build_evaluation_table(evaluation, verbose=True)
# print(results)

../output/parsing/stanza_1/UD_English-EWT
en_ewt-ud-dev.conllu
en_ewt-ud-test.conllu
../output/parsing/stanza_1/UD_English-CHILDES
en_childes-ud-dev.conllu
en_childes-ud-test.conllu
../output/parsing/stanza_1/UD_English-Atis
en_atis-ud-dev.conllu
en_atis-ud-test.conllu
../output/parsing/stanza_1/UD_English-ESLSpok
en_eslspok-ud-dev.conllu
en_eslspok-ud-test.conllu
../output/parsing/stanza_1/UD_English-ParTUT
en_partut-ud-dev.conllu
en_partut-ud-test.conllu
../output/parsing/stanza_1/UD_English-LinES
en_lines-ud-dev.conllu
en_lines-ud-test.conllu
../output/parsing/stanza_1/UD_English-GUM
en_gum-ud-dev.conllu
en_gum-ud-test.conllu
../output/parsing/stanza_1/UD_English-GENTLE
en_gentle-ud-test.conllu
../output/parsing/stanza_1/UD_English-CTeTex
en_ctetex-ud-test.conllu
../output/parsing/stanza_1/UD_English-Pronouns
en_pronouns-ud-test.conllu
../output/parsing/stanza_1/UD_English-LittlePrince
en_littleprince-ud-test.conllu
../output/parsing/stanza_1/UD_English-PUD
en_pud-ud-test.conllu
dev

In [ ]:
for a,b in evaluation.items():
  print(a,b)

Tokens <udtools.udeval.evaluate.<locals>.Score object at 0x79d937b56360>
Sentences <udtools.udeval.evaluate.<locals>.Score object at 0x79d937b564e0>
Words <udtools.udeval.evaluate.<locals>.Score object at 0x79d937b56450>
UPOS <udtools.udeval.evaluate.<locals>.Score object at 0x79d937b564b0>
XPOS <udtools.udeval.evaluate.<locals>.Score object at 0x79d937b563f0>
UFeats <udtools.udeval.evaluate.<locals>.Score object at 0x79d937b56510>
AllTags <udtools.udeval.evaluate.<locals>.Score object at 0x79d937b56630>
Lemmas <udtools.udeval.evaluate.<locals>.Score object at 0x79d937b56690>
UAS <udtools.udeval.evaluate.<locals>.Score object at 0x79d937b565a0>
LAS <udtools.udeval.evaluate.<locals>.Score object at 0x79d937b565d0>
ELAS <udtools.udeval.evaluate.<locals>.Score object at 0x79d937b56420>
EULAS <udtools.udeval.evaluate.<locals>.Score object at 0x79d937b56540>
CLAS <udtools.udeval.evaluate.<locals>.Score object at 0x79d937b56660>
MLAS <udtools.udeval.evaluate.<locals>.Score object at 0x79d937

In [ ]:
b.f1

0.7594103711503027

In [ ]:
evaluation["LAS"].f1, evaluation["UAS"].f1

(0.8418234587670136, 0.8780524419535628)

In [ ]:
!head -50 "../data/UD/UD_English-LinES/en_lines-ud-test.conllu"

# newdoc id = 1
# sent_id = en_lines-ud-test-doc1-4209
# text = If you plot multiple charts, you will see a drop area for multi-chart fields.
1	If	if	SCONJ	_	_	3	mark	_	_
2	you	you	PRON	PERS-P2	Case=Nom|Person=2|PronType=Prs	3	nsubj	_	_
3	plot	plot	VERB	PRES	Mood=Ind|Tense=Pres|VerbForm=Fin	9	advcl	_	_
4	multiple	multiple	ADJ	POS	Degree=Pos	5	amod	_	_
5	charts	chart	NOUN	PL-NOM	Number=Plur	3	obj	_	SpaceAfter=No
6	,	,	PUNCT	Comma	_	3	punct	_	_
7	you	you	PRON	PERS-P2	Case=Nom|Person=2|PronType=Prs	9	nsubj	_	_
8	will	will	AUX	PRES-AUX	Mood=Ind|VerbForm=Fin	9	aux	_	_
9	see	see	VERB	INF	VerbForm=Inf	0	root	_	_
10	a	a	DET	IND-SG	Definite=Ind|PronType=Art	12	det	_	_
11	drop	drop	VERB	INF	VerbForm=Inf	12	compound	_	_
12	area	area	NOUN	SG-NOM	Number=Sing	9	obj	_	_
13	for	for	ADP	_	_	15	case	_	_
14	multi-chart	multichart	NOUN	SG-NOM	Number=Sing	15	compound	_	_
15	fields	field	NOUN	PL-NOM	Number=Plur	12	nmod	_	SpaceAfter=No
16	.	.	PUNCT	Period	_	9	punct	_	_

# sent_id = en_lines-ud-test-doc1-4210

In [ ]:
!head -50 ../output/parsing/stanza_1/UD_English-GUM/en_gum-ud-test.conllu

1	The	the	DET	DT	Definite=Def|PronType=Art	2	det	_	
2	prevalence	prevalence	NOUN	NN	Number=Sing	0	root	_	
3	of	of	ADP	IN	_	4	case	_	
4	discrimination	discrimination	NOUN	NN	Number=Sing	2	nmod	_	
5	across	across	ADP	IN	_	7	case	_	
6	racial	racial	ADJ	JJ	Degree=Pos	7	amod	_	
7	groups	group	NOUN	NNS	Number=Plur	4	nmod	_	
8	in	in	ADP	IN	_	10	case	_	
9	contemporary	contemporary	ADJ	JJ	Degree=Pos	10	amod	_	
10	America	America	PROPN	NNP	Number=Sing	7	nmod	_	
11	:	:	PUNCT	:	_	2	punct	_	SpaceAfter=No

1	Results	result	NOUN	NNS	Number=Plur	0	root	_	
2	from	from	ADP	IN	_	6	case	_	
3	a	a	DET	DT	Definite=Ind|PronType=Art	6	det	_	
4	nationally	nationally	ADV	RB	_	5	advmod	_	
5	representative	representative	ADJ	JJ	Degree=Pos	6	amod	_	
6	sample	sample	NOUN	NN	Number=Sing	1	nmod	_	
7	of	of	ADP	IN	_	8	case	_	
8	adults	adult	NOUN	NNS	Number=Plur	6	nmod	_	SpaceAfter=No

1	Introduction	introduction	NOUN	NN	Number=Sing	0	root	_	
2	.	.	PUNCT	.	_	1	punct	_	SpaceAfter=No

1	Personal	personal	ADJ	JJ	Degree=Pos	

In [ ]:
new_path

'../output/parsing/stanza_1/UD_English-GUM/en_gum-ud-test.conllu'

In [ ]:
stanza_output_obj

{'conll': '1\t(\t(\tPUNCT\t-LRB-\t_\t2\tpunct\t_\t\n2\tSee\tsee\tVERB\tVB\tMood=Imp|VerbForm=Fin\t0\troot\t_\t\n3\tattached\tattach\tVERB\tVBN\tTense=Past|VerbForm=Part|Voice=Pass\t4\tamod\t_\t\n4\tfile\tfile\tNOUN\tNN\tNumber=Sing\t2\tobj\t_\t\n5\t:\t:\tPUNCT\t:\t_\t6\tpunct\t_\t\n6\tConstellation\tconstellation\tX\tGW\t_\t4\tappos\t_\t\n7\tPower\tpower\tX\tGW\t_\t6\tflat\t_\t\n8\t(\t(\tPUNCT\t-LRB-\t_\t9\tpunct\t_\t\n9\tGISB\tgisb\tX\tGW\t_\t6\tflat\t_\t\n10\tdraft\tdraft\tX\tGW\t_\t9\tflat\t_\t\n11\t)\t)\tX\tGW\t_\t9\tflat\t_\t\n12\t.doc\t.doc\tX\tNN\t_\t9\tflat\t_\t\n13\t)\t)\tPUNCT\t-RRB-\t_\t9\tpunct\t_\t\n14\t(\t(\tPUNCT\t-LRB-\t_\t15\tpunct\t_\t\n15\tSee\tsee\tVERB\tVB\tMood=Imp|VerbForm=Fin\t2\tparataxis\t_\t\n16\tattached\tattach\tVERB\tVBN\tTense=Past|VerbForm=Part|Voice=Pass\t17\tamod\t_\t\n17\tfile\tfile\tNOUN\tNN\tNumber=Sing\t15\tobj\t_\t\n18\t:\t:\tPUNCT\t:\t_\t19\tpunct\t_\t\n19\tSam3102.doc\tsam3102.doc\tNOUN\tNN\tNumber=Sing\t17\tappos\t_\t\n20\t)\t)\tPUNCT\t-RRB-\t_

In [ ]:
shelve_open=shelve.open(cur_parser_temp_shelve_path)
for a,b in shelve_open.items():
  print(a)
  b_dict=json.loads(b)
  for a0,b0 in b_dict.items():
    print(a0,b0)
  print("----")
shelve_open.close()

['"', 'They', 'can', 'freely', 'write', 'anything', 'they', 'like', 'about', 'our', 'prophet', ',', 'but', 'if', 'one', 'raises', 'doubts', 'about', 'the', 'Holocaust', 'he', 'is', 'either', 'fined', 'or', 'sent', 'to', 'prison', ',', '"', 'he', 'added', '.']
conll 1	"	"	PUNCT	``	_	5	punct	_	
2	They	they	PRON	PRP	Case=Nom|Number=Plur|Person=3|PronType=Prs	5	nsubj	_	
3	can	can	AUX	MD	VerbForm=Fin	5	aux	_	
4	freely	freely	ADV	RB	_	5	advmod	_	
5	write	write	VERB	VB	VerbForm=Inf	32	ccomp	_	
6	anything	anything	PRON	NN	Number=Sing|PronType=Ind	5	obj	_	
7	they	they	PRON	PRP	Case=Nom|Number=Plur|Person=3|PronType=Prs	8	nsubj	_	
8	like	like	VERB	VBP	Mood=Ind|Number=Plur|Person=3|Tense=Pres|VerbForm=Fin	6	acl:relcl	_	
9	about	about	ADP	IN	_	11	case	_	
10	our	our	PRON	PRP$	Case=Gen|Number=Plur|Person=1|Poss=Yes|PronType=Prs	11	nmod:poss	_	
11	prophet	prophet	NOUN	NN	Number=Sing	8	obl	_	
12	,	,	PUNCT	,	_	24	punct	_	
13	but	but	CCONJ	CC	_	24	cc	_	
14	if	if	SCONJ	IN	_	16	mark	_	
15	one	one	PRON	PRP

In [ ]:
#28 January 2026
def obj2conll(dep_obj,params={}):
    final_conll_str=""
    ignored_cols=params.get("ignored_cols",["deps"])
    temp_col_headers=["id","form","lemma","upos","xpos","feats","head","deprel","deps","misc"]
    col_headers=params.get("headers",temp_col_headers)
    for obj0 in dep_obj:
        cur_cells=[]
        for col0 in col_headers:
            cur_val=obj0.get(col0,"_")
            if col0 in ignored_cols: cur_val="_"
            cur_cells.append(cur_val)
        line="\t".join([str(v) for v in cur_cells])+"\n"
        final_conll_str+=line
    return final_conll_str


stanza_obj_list[0]
stanza_headers=["id","text","lemma","upos","xpos","feats","head","deprel","deps","misc"]
out=obj2conll(stanza_obj_list[0],params={"headers":stanza_headers})
print(out)

1	From	from	ADP	IN	_	3	case	_	
2	the	the	DET	DT	Definite=Def|PronType=Art	3	det	_	
3	AP	AP	NOUN	NN	Number=Sing	4	obl	_	
4	comes	come	VERB	VBZ	Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin	0	root	_	
5	this	this	DET	DT	Number=Sing|PronType=Dem	6	det	_	
6	story	story	NOUN	NN	Number=Sing	4	obj	_	
7	:	:	PUNCT	:	_	4	punct	_	SpaceAfter=No



In [ ]:
for a in stanza_obj_list[0]:
  print(a)

{'id': 1, 'text': 'From', 'lemma': 'from', 'upos': 'ADP', 'xpos': 'IN', 'head': 3, 'deprel': 'case', 'misc': '', 'start_char': 0, 'end_char': 4}
{'id': 2, 'text': 'the', 'lemma': 'the', 'upos': 'DET', 'xpos': 'DT', 'feats': 'Definite=Def|PronType=Art', 'head': 3, 'deprel': 'det', 'misc': '', 'start_char': 5, 'end_char': 8}
{'id': 3, 'text': 'AP', 'lemma': 'AP', 'upos': 'NOUN', 'xpos': 'NN', 'feats': 'Number=Sing', 'head': 4, 'deprel': 'obl', 'misc': '', 'start_char': 9, 'end_char': 11}
{'id': 4, 'text': 'comes', 'lemma': 'come', 'upos': 'VERB', 'xpos': 'VBZ', 'feats': 'Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin', 'head': 0, 'deprel': 'root', 'misc': '', 'start_char': 12, 'end_char': 17}
{'id': 5, 'text': 'this', 'lemma': 'this', 'upos': 'DET', 'xpos': 'DT', 'feats': 'Number=Sing|PronType=Dem', 'head': 6, 'deprel': 'det', 'misc': '', 'start_char': 18, 'end_char': 22}
{'id': 6, 'text': 'story', 'lemma': 'story', 'upos': 'NOUN', 'xpos': 'NN', 'feats': 'Number=Sing', 'head': 4, 

In [ ]:
import re
sent="Barack Obama was born in Hawaii."
sent_v2=re.sub(r"(\W)",r" \1 ",sent)
tokens=re.split(r"\s+",sent_v2.strip())
print(tokens)
doc = stanza_nlp([tokens])

['Barack', 'Obama', 'was', 'born', 'in', 'Hawaii', '.']


In [ ]:
stanza_obj_list

[[{'id': 1,
   'text': 'In',
   'lemma': 'in',
   'upos': 'ADP',
   'xpos': 'IN',
   'head': 5,
   'deprel': 'case',
   'misc': '',
   'start_char': 0,
   'end_char': 2},
  {'id': 2,
   'text': 'an',
   'lemma': 'a',
   'upos': 'DET',
   'xpos': 'DT',
   'feats': 'Definite=Ind|PronType=Art',
   'head': 5,
   'deprel': 'det',
   'misc': '',
   'start_char': 3,
   'end_char': 5},
  {'id': 3,
   'text': 'apparently',
   'lemma': 'apparently',
   'upos': 'ADV',
   'xpos': 'RB',
   'head': 4,
   'deprel': 'advmod',
   'misc': '',
   'start_char': 6,
   'end_char': 16},
  {'id': 4,
   'text': 'unrelated',
   'lemma': 'unrelated',
   'upos': 'ADJ',
   'xpos': 'JJ',
   'feats': 'Degree=Pos',
   'head': 5,
   'deprel': 'amod',
   'misc': '',
   'start_char': 17,
   'end_char': 26},
  {'id': 5,
   'text': 'incidents',
   'lemma': 'incident',
   'upos': 'NOUN',
   'xpos': 'NNS',
   'feats': 'Number=Plur',
   'head': 11,
   'deprel': 'obl',
   'misc': '',
   'start_char': 27,
   'end_char': 36},
 

In [ ]:
doc.to_dict()

[[{'id': 1,
   'text': 'Barack',
   'lemma': 'Barack',
   'upos': 'PROPN',
   'xpos': 'NNP',
   'feats': 'Number=Sing',
   'head': 4,
   'deprel': 'nsubj:pass',
   'misc': '',
   'start_char': 0,
   'end_char': 6},
  {'id': 2,
   'text': 'Obama',
   'lemma': 'Obama',
   'upos': 'PROPN',
   'xpos': 'NNP',
   'feats': 'Number=Sing',
   'head': 1,
   'deprel': 'flat',
   'misc': '',
   'start_char': 7,
   'end_char': 12},
  {'id': 3,
   'text': 'was',
   'lemma': 'be',
   'upos': 'AUX',
   'xpos': 'VBD',
   'feats': 'Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin',
   'head': 4,
   'deprel': 'aux:pass',
   'misc': '',
   'start_char': 13,
   'end_char': 16},
  {'id': 4,
   'text': 'born',
   'lemma': 'bear',
   'upos': 'VERB',
   'xpos': 'VBN',
   'feats': 'Tense=Past|VerbForm=Part|Voice=Pass',
   'head': 0,
   'deprel': 'root',
   'misc': '',
   'start_char': 17,
   'end_char': 21},
  {'id': 5,
   'text': 'in',
   'lemma': 'in',
   'upos': 'ADP',
   'xpos': 'IN',
   'head': 6,
   

In [ ]:
list1=doc.to_dict()
list1[0][0]

{'id': 1,
 'text': 'Barack',
 'lemma': 'Barack',
 'upos': 'PROPN',
 'xpos': 'NNP',
 'feats': 'Number=Sing',
 'head': 4,
 'deprel': 'nsubj:pass',
 'misc': '',
 'start_char': 0,
 'end_char': 6,
 'ner': 'B-PERSON',
 'multi_ner': ('B-PERSON',)}

In [ ]:
#https://lindat.mff.cuni.cz/services/udpipe/
#https://ufal.mff.cuni.cz/udpipe/2


In [ ]:
!ls code_utils/

ai_utils.py		   extract_docx.py		   recording_lib.py
align_utils_OLD.py	   file_utils.py		   rnn_utils_OLD.py
align_utils.py		   forced_align_utils.py	   rnn_utils.py
arabic_lib.py		   general.py			   scrape_lib_OLD.py
audio_features_utils.py    geo_utils.py			   script.js
audio_processing_utils.py  h5py_utils.py		   semantic_utils.py
audio_txt_utils.py	   index.html			   socket-server.py
backend_utils.py	   indexing_lib.py		   soup_OLD.py
bert_walign.ipynb	   info.txt			   sqld_utils.py
bitext_lib.py		   load_git.sh			   subword_utils.py
caching_utils.py	   make_trie.py			   textgrid_lib.py
cat_utils.py		   mongo_utils.py		   tika_utils.py
cdx-index-client.py	   multiprocessing_lib.py	   treebank.py
cgi-client.py		   np_lstm.py			   update_git.sh
CGIHTTPServer.py	   pandas_utils.py		   verbs.shelve.bak
char_ipa_utils.py	   parsing_lib.py		   verbs.shelve.dir
classification_utils.py    pdf_qr.py			   walign2022.ipynb
dep_lib.py		   phrase_extraction_unterm.ipynb  walign_mt.ipynb
docx_u

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 61.3 MB/s eta 0:00:00


#RNN POS Tagger

##Training

In [ ]:
from code_utils.rnn_utils import *
from code_utils.parsing_lib import *
from code_utils.dep_lib import *

#parameters
featex_params={"len":True,"caps":True,"n_chars":4,"norm_digit":True,"check_hyphen":True,"check_dot":True, "max_len":15}
cur_label_dict={"upos":upos_list,"xpos":xpos_list}
hidden_size=64
n_layers=3
num_epochs = 100
cur_lr=0.0001
max_n_items=None #500
exp_name="H64_L3_LR001_updated_xpos_loading"



#Functions
def conll2nn_pos(conll_str,ft_obj,lb_obj):
  conll_obj_list=conll2obj(conll_str)
  input_ft_tensor=[]
  input_lb_tensor=[]
  for a in conll_obj_list:
    ft_vec0=wd_ft_obj.to_vec(a["form"])
    lb_vec0=label_proc_obj.encode(a)
    #input_ft_tensor.append(ft_vec0.tolist())
    input_ft_tensor.append(ft_vec0)
    input_lb_tensor.append(lb_vec0.tolist())
  final_ft_tensor=torch.tensor(input_ft_tensor)
  final_lb_tensor=torch.tensor(input_lb_tensor)
  return final_ft_tensor,final_lb_tensor

def words2ft_tensor(words,ft_obj):
  input_ft_tensor=[]
  for word0 in words:
    ft_vec0=wd_ft_obj.to_vec(word0)
    input_ft_tensor.append(ft_vec0)
  final_ft_tensor=torch.tensor(input_ft_tensor)
  return final_ft_tensor


#Data
train_files,dev_files,test_files=[],[],[]
ud_dir="../data/UD"
#excluded_tb_list=['UD_English-GENTLE']
treebank_dir_list=os.listdir(ud_dir)
for tb0 in treebank_dir_list:
  tb_full_path=os.path.join(ud_dir,tb0)
  tb_files=os.listdir(tb_full_path)
  for tbf0 in tb_files:
    cur_file=os.path.join(tb_full_path,tbf0)
    if "train" in tbf0.lower() and "conllu" in tbf0.lower(): train_files.append(cur_file)
    elif "dev" in tbf0.lower() and "conllu" in tbf0.lower(): dev_files.append(cur_file)
    elif "test" in tbf0.lower() and "conllu" in tbf0.lower(): test_files.append(cur_file)


wd_ft_obj=wd_ft(featex_params)
label_proc_obj=label_proc(cur_label_dict)
input_size=wd_ft_obj.n_features
output_size=label_proc_obj.n_labels

training_params={"n_input":input_size,"n_output":output_size,"n_hidden":hidden_size,"LR":cur_lr,"n_layers":n_layers}




# cur_train_file=train_files[0]
# cur_dev_file=dev_files[0]

lowest_dev_loss=None

model_dir="../models/pos_models"
model_fname=f"{exp_name}.model"
model_fpath=os.path.join(model_dir,model_fname)

last_epoch=None
if os.path.exists(model_fpath):
  loaded_model_dict=torch.load(model_fpath)
  training_params=loaded_model_dict["training_params"]
  state_dict=loaded_model_dict.get("state_dict")
  last_epoch=loaded_model_dict.get("epoch")
  rnn=RNN(training_params["n_input"],training_params["n_hidden"],training_params["n_output"],training_params["n_layers"],True,True)
  cur_lr=training_params["LR"]
  if state_dict!=None: rnn.load_state_dict(state_dict)
  print("loaded model",model_fpath)
else:
  rnn=RNN(input_size,hidden_size,output_size,n_layers,True,True)

#rnn=RNN(input_size,hidden_size,output_size,n_layers,True,True)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(rnn.parameters(), lr=cur_lr)



for epoch in range(num_epochs):
  if last_epoch!=None and epoch<=last_epoch:
    print("skip epoch", epoch)
    continue
  rnn.train()
  epoch_train_loss,epoch_dev_loss = 0,0
  n_train,n_dev=0,0
  for cur_train_file in train_files:
    #print(cur_train_file)
    train_iter0=iter_conll_treebank(cur_train_file)
    for i0,conll0 in enumerate(train_iter0):
      if i0==max_n_items: break
      n_train+=1
      if n_train%1000==0: print("Train:", n_train, cur_train_file)
      final_ft,final_lb=conll2nn_pos(conll0,wd_ft_obj,label_proc_obj)
      output=rnn(final_ft)
      loss = criterion(output, final_lb)
      epoch_train_loss += loss.item()

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

  for cur_dev_file in dev_files:
    #print(cur_dev_file)
    dev_iter0=iter_conll_treebank(cur_dev_file)
    for i0,conll0 in enumerate(dev_iter0):
      if i0==max_n_items: break
      n_dev+=1
      if n_dev%1000==0: print("Dev:", n_dev, cur_dev_file)

      final_ft,final_lb=conll2nn_pos(conll0,wd_ft_obj,label_proc_obj)
      output=rnn(final_ft)
      loss = criterion(output, final_lb)
      epoch_dev_loss += loss.item()




  print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_train_loss / n_train:.4f}, Dev Loss: {epoch_dev_loss / n_dev:.4f}')
  avg_dev_loss=epoch_dev_loss / n_dev
  if lowest_dev_loss==None or lowest_dev_loss>avg_dev_loss:
    lowest_dev_loss=avg_dev_loss
    model_data_dict={}
    model_data_dict["training_params"]=training_params
    model_data_dict["label_params"]=cur_label_dict
    model_data_dict["featex_params"]=featex_params
    model_data_dict["best_dev_loss"]=lowest_dev_loss
    model_data_dict["epoch"]=epoch
    model_data_dict["state_dict"]=rnn.state_dict()

    torch.save(model_data_dict, model_fpath)
    print(f"best dev loss {lowest_dev_loss} - model saved to {model_fpath}")
      #save
    #print(f'Epoch [{epoch+1}/{num_epochs}], Dev Loss: {epoch_dev_loss / n_dev:.4f}')
    #print("epoch",epoch, "epoch_train_loss",epoch_train_loss)


# random_uniform = torch.rand(3,input_size)
# #random_uniform

# output=rnn(random_uniform)
# for item0 in output:
#   #item0=item0.flatten()
#   print(item0.shape)
#   decoded_dict=label_proc_obj.decode(item0)
#   for a,b in decoded_dict.items():
#     print(a,b)
#   print("------")
#   #print(decoded)
# #output.shape


#best dev loss 9.377240737874187 - model saved to ../models/pos_models/H16_L2_LR001.model

# Epoch [7/100], Train Loss: 9.1720, Dev Loss: 9.3583
# best dev loss 9.35828448630936 - model saved to ../models/pos_models/H32_L2_LR001.model


Train: 1000 ../data/UD/UD_English-EWT/en_ewt-ud-train.conllu
Train: 2000 ../data/UD/UD_English-EWT/en_ewt-ud-train.conllu
Train: 3000 ../data/UD/UD_English-EWT/en_ewt-ud-train.conllu
Train: 4000 ../data/UD/UD_English-EWT/en_ewt-ud-train.conllu
Train: 5000 ../data/UD/UD_English-EWT/en_ewt-ud-train.conllu
Train: 6000 ../data/UD/UD_English-EWT/en_ewt-ud-train.conllu
Train: 7000 ../data/UD/UD_English-EWT/en_ewt-ud-train.conllu
Train: 8000 ../data/UD/UD_English-EWT/en_ewt-ud-train.conllu
Train: 9000 ../data/UD/UD_English-EWT/en_ewt-ud-train.conllu
Train: 10000 ../data/UD/UD_English-EWT/en_ewt-ud-train.conllu
Train: 11000 ../data/UD/UD_English-EWT/en_ewt-ud-train.conllu
Train: 12000 ../data/UD/UD_English-EWT/en_ewt-ud-train.conllu
Train: 13000 ../data/UD/UD_English-CHILDES/en_childes-ud-train.conllu
Train: 14000 ../data/UD/UD_English-CHILDES/en_childes-ud-train.conllu
Train: 15000 ../data/UD/UD_English-CHILDES/en_childes-ud-train.conllu
Train: 16000 ../data/UD/UD_English-CHILDES/en_childes-u

In [ ]:
cur_dev_file=dev_files[0]
dev_items0=list(iter_conll_treebank(cur_dev_file))
cur_dev_conll=dev_items0[2]
ft_tensor0,lb_tensor0=conll2nn_pos(cur_dev_conll,wd_ft_obj,label_proc_obj)
conll_obj=conll2obj(cur_dev_conll)

out1=rnn(ft_tensor0)
for o1,c1 in zip(out1,conll_obj):
  label_out=label_proc_obj.decode(o1)
  #print(c1)
  print(c1["form"], label_out)
#out0.shape

Bush {'upos': [('NOUN', 1.0), ('ADJ', 0.9999697208404541), ('PROPN', 0.9620722532272339), ('INTJ', 0.8942938446998596), ('VERB', 0.6207646727561951), ('ADV', 0.00016960874199867249), ('NUM', 1.947719465533737e-05), ('DET', 3.7380259527708404e-06), ('PRON', 3.932341030576936e-07), ('X', 7.952561525970125e-11), ('AUX', 7.647957787904058e-12), ('SYM', 3.07580653655104e-12), ('CCONJ', 1.211608667128078e-13), ('SCONJ', 9.832607140601232e-14), ('PART', 7.174594056370538e-16), ('ADP', 5.773242234353832e-17), ('PUNCT', 4.846445418683901e-18)], 'xpos': [('NN', 1.0), ('JJ', 0.9999741315841675), ('VB', 0.9973527193069458), ('NNP', 0.9963460564613342), ('UH', 0.907490611076355), ('VBP', 0.038626883178949356), ('RB', 0.0026242102030664682), ('CD', 0.00010011401900555938), ('WP$', 9.195577490572759e-07), ('VBN', 4.024062150165264e-07), ('WP', 3.1240301723300945e-07), ('NNS', 1.3474887339270936e-07), ('PAST', 7.423649606153049e-08), ('VBD', 6.070619917863951e-08), ('DT', 3.2350406797831965e-08), ('V'

In [ ]:
def words2ft_tensor(words,ft_obj):
  input_ft_tensor=[]
  for word0 in words:
    ft_vec0=wd_ft_obj.to_vec(word0)
    input_ft_tensor.append(ft_vec0)
  final_ft_tensor=torch.tensor(input_ft_tensor)
  return final_ft_tensor

model_fpath="../models/pos_models/H16_L1_E10_LR001-TEST.model"
loaded_model_dict=torch.load(model_fpath)

words0=["The","president","asked","a","question","."]
input_tensor=words2ft_tensor(words0,wd_ft_obj)



out1=rnn(input_tensor)
for o1,c1 in zip(out1,words0):
  label_out=label_proc_obj.decode(o1)
  #print(c1)
  print(c1, label_out)


The {'upos': [('NOUN', 0.9830068349838257), ('VERB', 0.982347846031189), ('PUNCT', 0.9773704409599304), ('DET', 0.9717513918876648), ('PROPN', 0.969980776309967), ('ADP', 0.9694361090660095), ('PRON', 0.9553362727165222), ('ADJ', 0.9457358717918396), ('AUX', 0.9377699494361877), ('CCONJ', 0.9280815720558167), ('PART', 0.9028696417808533), ('ADV', 0.8568549156188965), ('NUM', 0.5889342427253723), ('INTJ', 0.2844289243221283), ('SCONJ', 0.21264000236988068), ('X', 0.031357843428850174), ('SYM', 0.015921393409371376)], 'xpos': [('NN', 0.9611391425132751), ('PRP', 0.9482163190841675), ('.', 0.9399795532226562), ('IN', 0.9311474561691284), ('VB', 0.928742527961731), ('_', 0.9161356687545776), ('DT', 0.9006926417350769), ('NNP', 0.900516152381897), ('JJ', 0.8686551451683044), ('RB', 0.8497112989425659), ('NNS', 0.8448314070701599), (',', 0.8400357365608215), ('S', 0.8183004856109619), ('VBP', 0.8142459988594055), ('CC', 0.8081426620483398), ('SG-NOM', 0.7760902047157288), ('VBD', 0.768095374

##Load RNN POS model

In [ ]:
from code_utils.rnn_utils import *
from code_utils.parsing_lib import *
from code_utils.dep_lib import *

#model_fpath="../models/pos_models/H16_L1_E10_LR001-TEST.model"
#print("loaded_model_dict",list(loaded_model_dict.keys()))
#loaded_model_dict ['training_params', 'label_params', 'featex_params', 'best_dev_loss', 'epoch', 'state_dict_best_loss']

#inspect models
models_dir="../models/pos_models/"
for mod0 in os.listdir(models_dir):
  cur_model_fpath=os.path.join(models_dir,mod0)
  cur_loaded_model_dict=torch.load(cur_model_fpath)
  print(cur_model_fpath,cur_loaded_model_dict.get("best_dev_loss"))

#model_fpath="../models/pos_models/H64_L2_LR001.model"
#model_fpath="../models/pos_models/H16_L2_LR001.model"
model_fpath="../models/pos_models/H16_L1_E10_LR001.model"
#model_fpath="../models/pos_models/H64_L3_LR001_updated_xpos_w_loading-TEST.model"
loaded_model_dict=torch.load(model_fpath)
featex_params=loaded_model_dict["featex_params"]
training_params=loaded_model_dict["training_params"]
label_params=loaded_model_dict["label_params"]

print(list(loaded_model_dict.keys()))

state_dict=loaded_model_dict.get("state_dict")
if state_dict==None: state_dict=loaded_model_dict.get("state_dict_best_loss")
# training_params={"n_input":input_size,"n_output":output_size,"n_hidden":hidden_size,"LR":cur_lr,"n_layers":n_layers}
rnn=RNN(training_params["n_input"],training_params["n_hidden"],training_params["n_output"],training_params["n_layers"],True,True)
rnn.load_state_dict(state_dict)
rnn.eval()

wd_ft_obj=wd_ft(featex_params)
label_proc_obj=label_proc(label_params)

# wd_ft_obj=wd_ft(loaded_model_dict["featex_params"])
# label_proc_obj=label_proc(cur_label_dict)
# input_size=wd_ft_obj.n_features
# output_size=label_proc_obj.n_labels



words0=["The","president","asked","a","question","."]

sent="what also happened in Germany"
words0=sent.split()
input_tensor=words2ft_tensor(words0,wd_ft_obj)

out1=rnn(input_tensor)
for o1,c1 in zip(out1,words0):
  label_out=label_proc_obj.decode(o1)
  #print(c1)
  print(c1)
  for a,b in label_out.items(): print(a,b)

../models/pos_models/H16_L1_E10_LR001.model 9.36521951164171
../models/pos_models/H16_L1_E10_LR001-TEST.model 9.576410300170453
../models/pos_models/H16_L2_LR001.model 9.377240737874187
../models/pos_models/H32_L2_LR001.model 9.356027877809204
../models/pos_models/H64_L2_LR001.model 9.331363994382627
../models/pos_models/H64_L3_LR001_updated_xpos.model 7.942426484497254
../models/pos_models/H64_L3_LR001_updated_xpos_w_loading-TEST.model 8.419914465537046
['training_params', 'label_params', 'featex_params', 'best_dev_loss', 'epoch', 'state_dict_best_loss']
what
upos [('PRON', 1.0), ('DET', 0.9935404658317566), ('INTJ', 0.004166592378169298), ('NOUN', 0.002093130024150014), ('AUX', 0.0010536661138758063), ('VERB', 0.0008464443380944431), ('ADJ', 0.0006051063537597656), ('SCONJ', 0.00025477627059444785), ('ADV', 3.76001054291919e-07), ('PUNCT', 2.642295839905273e-07), ('PROPN', 3.134940840254785e-08), ('X', 4.892684035695538e-09), ('PART', 3.580586660500984e-10), ('SYM', 1.920194420135246

In [ ]:
import time
from code_utils.parsing_lib import *

rnn_model_fpath="../models/pos_models/H64_L3_LR001_updated_xpos_loading.model"
#rnn_model_fpath=None
pos_obj=POS(rnn_model_fpath)

sent="what also happened in Germany"
sent="the old man also asked the same question"
sent="the old man has also asked the same questions"
sent="however , we didn't have any good answers"
sent="I will probably ask him to come some time later"
#sent="Ecuador also supports initiatives for the protection, conservation and sustainable use of the ocean, such as the prompt entry into force of the Agreement under the United Nations Convention on the Law of the Sea on the Conservation and Sustainable Use of Marine Biological Diversity of Areas beyond National Jurisdiction, and the holding of the third United Nations Conference on Oceans in 2025."
sent="Our commitment is also reflected in the efforts made by the Ecuadorian presidency of the Committee in charge of developing a legally binding instrument to eliminate plastics pollution."

sent=re.sub(r"(\W)",r" \1 ",sent)
words0=sent.strip().split()

t0=time.time()
for _ in range(100):
  outcome=pos_obj.tag_words(words0,min_wt=0.1)
t1=time.time()

for a in outcome: print(a)
print(t1-t0)

{'word': 'Our', 'upos': [('PRON', 0.9999985694885254), ('DET', 0.35972481966018677)], 'xpos': [('PRP$', 0.9997376799583435)]}
{'word': 'commitment', 'upos': [('NOUN', 1.0), ('ADJ', 0.9986128807067871), ('VERB', 0.9704145789146423)], 'xpos': [('NN', 1.0), ('JJ', 0.9969983100891113), ('VBG', 0.1123284325003624)]}
{'word': 'is', 'upos': [('AUX', 0.999998927116394)], 'xpos': [('VBZ', 0.9999921321868896)]}
{'word': 'also', 'upos': [('ADV', 0.9999994039535522)], 'xpos': [('RB', 0.9999996423721313)]}
{'word': 'reflected', 'upos': [('VERB', 1.0)], 'xpos': [('VBN', 0.9992351531982422), ('VB', 0.8263813853263855), ('VBD', 0.28955093026161194)]}
{'word': 'in', 'upos': [('ADP', 0.9999995231628418), ('ADV', 0.9909136295318604)], 'xpos': [('IN', 0.9999963045120239), ('RP', 0.9533300995826721), ('RB', 0.9520432353019714)]}
{'word': 'the', 'upos': [('DET', 0.9999979734420776)], 'xpos': [('DT', 0.9999988079071045)]}
{'word': 'efforts', 'upos': [('NOUN', 0.9999988079071045)], 'xpos': [('NNS', 0.99999344

In [ ]:
#from code_utils.rnn_utils import *
from code_utils.parsing_lib import *
from code_utils.dep_lib import *
from code_utils import rnn_utils
import time

#given the RNN model for POS tagger, initiate and run the tagging for any given sequence of words
class POS:
  def __init__(self,rnn_model_path,params={}) -> None:
    self.params=params
    self.loaded_model_dict=torch.load(rnn_model_path) #first we load the RNN POS model
    self.featex_params=self.loaded_model_dict["featex_params"]
    self.training_params=self.loaded_model_dict["training_params"]
    self.label_params=self.loaded_model_dict["label_params"]
    self.state_dict=self.loaded_model_dict.get("state_dict")
    if self.state_dict==None: self.state_dict=self.loaded_model_dict.get("state_dict_best_loss")

    self.wd_ft_obj=wd_ft(self.featex_params) #loaded from word features class from parsing lib
    self.label_proc_obj=rnn_utils.label_proc(self.label_params) #loaded from label processing class from RNN lib
    #last two true params are sigmoid & matching input output size
    self.rnn=rnn_utils.RNN(self.training_params["n_input"],self.training_params["n_hidden"],self.training_params["n_output"],self.training_params["n_layers"],True,True)
    self.rnn.load_state_dict(self.state_dict)
    self.rnn.eval()

  def tag_words(self,words,min_wt=None):
    input_tensor=words2ft_tensor(words,self.wd_ft_obj)
    out1=self.rnn(input_tensor)
    final_pos_obj_list=[]
    for o1,w1 in zip(out1,words):
      label_out=self.label_proc_obj.decode(o1)
      #print(w1)
      cur_obj_list={"word":w1}
      for a,b in label_out.items():
        tag_wt_list=b
        if min_wt!=None: tag_wt_list=[(tg0,wt0) for tg0,wt0 in tag_wt_list if wt0>min_wt]
        #print(a,tag_wt_list)
        cur_obj_list[a]=tag_wt_list
      final_pos_obj_list.append(cur_obj_list)
    return final_pos_obj_list

rnn_model_fpath="../models/pos_models/H64_L3_LR001_updated_xpos_loading.model"
pos_obj=POS(rnn_model_fpath)

sent="what also happened in Germany"
sent="the old man also asked the same question"
sent="the old man has also asked the same questions"
sent="however , we didn't have any good answers"
sent="I will probably ask him to come some time later"
sent="Ecuador also supports initiatives for the protection, conservation and sustainable use of the ocean, such as the prompt entry into force of the Agreement under the United Nations Convention on the Law of the Sea on the Conservation and Sustainable Use of Marine Biological Diversity of Areas beyond National Jurisdiction, and the holding of the third United Nations Conference on Oceans in 2025."


#words0=sent.split()
sent=re.sub(r"(\W)",r" \1 ",sent)
words0=sent.split()

t0=time.time()
for _ in range(1000):
  outcome=pos_obj.tag_words(words0,min_wt=0.1)
t1=time.time()

for a in outcome: print(a)
print(t1-t0)




{'word': 'Ecuador', 'upos': [('NOUN', 1.0), ('PROPN', 0.9914193153381348)], 'xpos': [('NN', 0.9999328851699829), ('NNP', 0.5867447853088379)]}
{'word': 'also', 'upos': [('ADV', 1.0)], 'xpos': [('RB', 1.0)]}
{'word': 'supports', 'upos': [('VERB', 0.9999978542327881), ('NOUN', 0.9831721782684326)], 'xpos': [('VBZ', 0.999607264995575), ('NNS', 0.8740209341049194)]}
{'word': 'initiatives', 'upos': [('NOUN', 0.9999953508377075), ('VERB', 0.6052702069282532)], 'xpos': [('NNS', 0.9999417066574097)]}
{'word': 'for', 'upos': [('ADP', 0.9999997615814209), ('SCONJ', 0.9998799562454224)], 'xpos': [('IN', 0.9999998807907104)]}
{'word': 'the', 'upos': [('DET', 0.9999979734420776)], 'xpos': [('DT', 0.9999959468841553)]}
{'word': 'protection', 'upos': [('NOUN', 1.0)], 'xpos': [('NN', 0.9999990463256836)]}
{'word': ',', 'upos': [('PUNCT', 0.9999995231628418)], 'xpos': [(',', 0.9999215602874756)]}
{'word': 'conservation', 'upos': [('NOUN', 1.0), ('VERB', 0.5211946964263916)], 'xpos': [('NN', 0.999999880

#Analyzing UD Data
Identify train/dev/test data

In [ ]:
train_files,dev_files,test_files=[],[],[]
ud_dir="../data/UD"
#excluded_tb_list=['UD_English-GENTLE']
treebank_dir_list=os.listdir(ud_dir)
for tb0 in treebank_dir_list:
  tb_full_path=os.path.join(ud_dir,tb0)
  tb_files=os.listdir(tb_full_path)
  for tbf0 in tb_files:
    cur_file=os.path.join(tb_full_path,tbf0)
    if "train" in tbf0.lower() and "conllu" in tbf0.lower(): train_files.append(cur_file)
    elif "dev" in tbf0.lower() and "conllu" in tbf0.lower(): dev_files.append(cur_file)
    elif "test" in tbf0.lower() and "conllu" in tbf0.lower(): test_files.append(cur_file)

print("train_files",len(train_files))
for a in train_files: print(a)
print("dev_files",len(dev_files))
for a in dev_files: print(a)
print("test_files",len(test_files))
for a in test_files: print(a)


train_files 8
../data/UD/UD_English-EWT/en_ewt-ud-train.conllu
../data/UD/UD_English-CHILDES/en_childes-ud-train.conllu
../data/UD/UD_English-Atis/en_atis-ud-train.conllu
../data/UD/UD_English-ESLSpok/en_eslspok-ud-train.conllu
../data/UD/UD_English-ParTUT/en_partut-ud-train.conllu
../data/UD/UD_English-LinES/en_lines-ud-train.conllu
../data/UD/UD_English-GUMReddit/en_gumreddit-ud-train.conllu
../data/UD/UD_English-GUM/en_gum-ud-train.conllu
dev_files 8
../data/UD/UD_English-EWT/en_ewt-ud-dev.conllu
../data/UD/UD_English-CHILDES/en_childes-ud-dev.conllu
../data/UD/UD_English-Atis/en_atis-ud-dev.conllu
../data/UD/UD_English-ESLSpok/en_eslspok-ud-dev.conllu
../data/UD/UD_English-ParTUT/en_partut-ud-dev.conllu
../data/UD/UD_English-LinES/en_lines-ud-dev.conllu
../data/UD/UD_English-GUMReddit/en_gumreddit-ud-dev.conllu
../data/UD/UD_English-GUM/en_gum-ud-dev.conllu
test_files 13
../data/UD/UD_English-GENTLE/en_gentle-ud-test.conllu
../data/UD/UD_English-EWT/en_ewt-ud-test.conllu
../data/UD

In [ ]:
from code_utils.dep_lib import *

def conll2nn_pos(conll_str,ft_obj,lb_obj):
  conll_obj_list=conll2obj(conll_str)
  input_ft_tensor=[]
  input_lb_tensor=[]
  for a in conll_obj_list:
    ft_vec0=wd_ft_obj.to_vec(a["form"])
    lb_vec0=label_proc_obj.encode(a)
    input_ft_tensor.append(ft_vec0.tolist())
    input_lb_tensor.append(lb_vec0.tolist())
  final_ft_tensor=torch.tensor(input_ft_tensor)
  final_lb_tensor=torch.tensor(input_lb_tensor)
  return final_ft_tensor,input_lb_tensor


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)



cur_train_file=train_files[0]
iter0=iter_conll_treebank(cur_train_file)
for i0,conll0 in enumerate(iter0):


  final_ft,final_lb=conll2nn_pos(conll0,wd_ft_obj,label_proc_obj)
  output=rnn(final_ft)
  print(output.shape)

    #print(tensor0.shape, tensor0)
  print("----")
  if i0>50: break


torch.Size([29, 252])
----
torch.Size([18, 252])
----
torch.Size([17, 252])
----
torch.Size([16, 252])
----
torch.Size([36, 252])
----
torch.Size([13, 252])
----
torch.Size([13, 252])
----
torch.Size([16, 252])
----
torch.Size([35, 252])
----
torch.Size([20, 252])
----
torch.Size([19, 252])
----
torch.Size([29, 252])
----
torch.Size([19, 252])
----
torch.Size([17, 252])
----
torch.Size([27, 252])
----
torch.Size([26, 252])
----
torch.Size([27, 252])
----
torch.Size([11, 252])
----
torch.Size([20, 252])
----
torch.Size([9, 252])
----
torch.Size([15, 252])
----
torch.Size([16, 252])
----
torch.Size([17, 252])
----
torch.Size([40, 252])
----
torch.Size([49, 252])
----
torch.Size([51, 252])
----
torch.Size([13, 252])
----
torch.Size([10, 252])
----
torch.Size([9, 252])
----
torch.Size([30, 252])
----
torch.Size([20, 252])
----
torch.Size([26, 252])
----
torch.Size([28, 252])
----
torch.Size([16, 252])
----
torch.Size([13, 252])
----
torch.Size([12, 252])
----
torch.Size([19, 252])
----
tor

In [ ]:
x = torch.tensor([1, 2, 3])

isinstance(x, torch.Tensor)

True

In [ ]:
y = [1, 2, 3]

isinstance(y, torch.Tensor)

False

#UD Tools - Eval

In [ ]:
!git clone https://github.com/UniversalDependencies/tools.git ud_tools

fatal: destination path 'ud_tools' already exists and is not an empty directory.


In [ ]:
!pip install udapi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 7.3 MB/s eta 0:00:00


In [ ]:
os.chdir(cwd)
os.chdir("ud_tools")
!python eval.py

#!python eval.py -v goldstandard.conllu systemoutput.conllu
#!python eval.py -v goldstandard.conllu systemoutput.conllu

"output/parsing/spacy/UD_English-LinES/en_lines-ud-dev.conllu"
"data/UD/UD_English-LinES/en_lines-ud-dev.txt"

!python eval.py -v "../../data/UD/UD_English-LinES/en_lines-ud-dev.conllu" "../../output/parsing/spacy/UD_English-LinES/en_lines-ud-dev.conllu" --multiple-roots-okay
os.chdir(cwd)

#!ls ud_tools

usage: eval.py [-h] [--verbose] [--counts] [--no-enhanced]
               [--enhancements ENHANCEMENTS] [--no-empty-nodes NO_EMPTY_NODES]
               [--multiple-roots-okay]
               gold_file system_file
eval.py: error: the following arguments are required: gold_file, system_file
Metric     | Precision |    Recall |  F1 Score | AligndAcc
-----------+-----------+-----------+-----------+-----------
Tokens     |     97.69 |     98.83 |     98.26 |
Sentences  |    100.00 |    100.00 |    100.00 |
Words      |    100.00 |    100.00 |    100.00 |
UPOS       |      0.00 |      0.00 |      0.00 |      0.00
XPOS       |      0.08 |      0.08 |      0.08 |      0.08
UFeats     |     35.12 |     35.12 |     35.12 |     35.12
AllTags    |      0.00 |      0.00 |      0.00 |      0.00
Lemmas     |     95.49 |     95.49 |     95.49 |     95.49
UAS        |     56.11 |     56.11 |     56.11 |     56.11
LAS        |     37.54 |     37.54 |     37.54 |     37.54
CLAS       |     64.45 |     3

In [ ]:
!ls ../data/UD/UD_English-LinES/en_lines-ud-dev.conllu

en_lines-ud-dev.conllu	 en_lines-ud-test.txt	   LICENSE.txt
en_lines-ud-dev.txt	 en_lines-ud-train.conllu  README.txt
en_lines-ud-test.conllu  en_lines-ud-train.txt	   stats.xml


In [ ]:
from udtools.udeval import load_conllu_file, evaluate, build_evaluation_table

gold_path="../data/UD/UD_English-LinES/en_lines-ud-test.conllu"
system_path="../output/parsing/stanza_1/UD_English-LinES/en_lines-ud-test.conllu"

# gold_ud = load_conllu_file('gold.conllu')
# system_ud = load_conllu_file('system.conllu')
gold_ud = load_conllu_file(gold_path)
system_ud = load_conllu_file(system_path)

evaluation = evaluate(gold_ud, system_ud)
results = build_evaluation_table(evaluation, verbose=True)
print(results)

Metric     | Precision |    Recall |  F1 Score | AligndAcc
-----------+-----------+-----------+-----------+-----------
Tokens     |     97.72 |     98.85 |     98.28 |
Sentences  |    100.00 |    100.00 |    100.00 |
Words      |    100.00 |    100.00 |    100.00 |
UPOS       |     94.25 |     94.25 |     94.25 |     94.25
XPOS       |      0.28 |      0.28 |      0.28 |      0.28
UFeats     |     84.87 |     84.87 |     84.87 |     84.87
AllTags    |      0.28 |      0.28 |      0.28 |      0.28
Lemmas     |     95.88 |     95.88 |     95.88 |     95.88
UAS        |     87.81 |     87.81 |     87.81 |     87.81
LAS        |     84.18 |     84.18 |     84.18 |     84.18
CLAS       |     80.62 |     80.49 |     80.56 |     80.49
MLAS       |     59.21 |     59.12 |     59.16 |     59.12
BLEX       |     76.00 |     75.88 |     75.94 |     75.88


In [ ]:
!ls ud_tools/udtools/src/udtools/

argparser.py  incident.py  level2.py  level5.py    state.py   validator.py
cli.py	      __init__.py  level3.py  level6.py    udeval.py
data.py       level1.py    level4.py  __pycache__  utils.py


In [ ]:
!ls ud_tools

add_parallel_to_all_treebanks.pl      find_duplicate_sentences.pl
check_files.pl			      fix_format_of_deps.pl
check_overlaps.pl		      fix-space-after-paragraph.pl
check_release.pl		      generate_comparison_of_treebanks.pl
check_sentence_ids.pl		      generate_treebank_hub.pl
check-space-after-paragraph.pl	      Graph.pm
check_text_wosp_match.sh	      klcpos3.pl
collect_propn_sequences.pl	      LICENSE.txt
conll_convert_tags_to_uposf.pl	      mergept.pl
conllu_align_tokens.pl		      mwtoken-stats.pl
conllu_break_cycles.pl		      Node.pm
conllu_copy_annotation.pl	      normalize_unicode.pl
conllu_copy_basic_to_enhanced.pl      overlap.py
conllu_copy_sentence_segmentation.pl  package_st_data.sh
conllu_copy_tokenization.pl	      package_ud_release.sh
conllu_cut.pl			      README.md
conllu-dependency-stats.pl	      remove_duplicate_sentences.pl
conllu-formconvert.py		      remove_sense_suffixes_from_lemmas.pl
conllu_quick_fix_id_sequence.pl       requirements.txt
conllu_quick_fix.pl		   

In [ ]:
!ls ud_tools/udtools/

LICENSE.txt  MANIFEST.in  pyproject.toml  README.md  src  tests


In [ ]:
!cat ud_tools/udtools/src/udtools/validator.py

#! /usr/bin/env python3
# Original code (2015) by Filip Ginter and Sampo Pyysalo.
# DZ 2018-11-04: Porting the validator to Python 3.
# DZ: Many subsequent changes. See the git history.
import sys
import io
import argparse
# Once we know that the low-level CoNLL-U format is OK, we will be able to use
# the Udapi library to access the data and perform the tests at higher levels.
import udapi.block.read.conllu
# Allow using this module from the root folder of tools even if it is not
# installed as a package: use the relative path validator/src/validator for
# submodules. If the path is not available, try the standard qualification,
# assuming that the user has installed udtools from PyPI and then called
# from udtools import Validator.
try:
    import udtools.src.udtools.utils as utils
    from udtools.src.udtools.incident import Error, TestClass
    from udtools.src.udtools.state import State
    import udtools.src.udtools.data as data
    from udtools.src.udtools.level6 import Level6
e

In [ ]:
!cat ud_tools/validate.py

!cat ud_tools/eval.py

#! /usr/bin/env python3
# Original code (2015) by Filip Ginter and Sampo Pyysalo.
# DZ 2018-11-04: Porting the validator to Python 3.
# DZ: Many subsequent changes. See the git history.
import sys
# Import the Validator class from the package subfolder regardless whether it
# is installed as a package.
from udtools.src.udtools.validator import Validator
from udtools.src.udtools.argparser import parse_args_validator



#==============================================================================
# The main function.
#==============================================================================



def main():
    args = parse_args_validator()
    validator = Validator(lang=args.lang, level=args.level, args=args)
    state = validator.validate_files(args.input)
    # Summarize the warnings and errors.
    summary = str(state)
    if not args.quiet:
        print(summary, file=sys.stderr)
    if state.passed():
        return 0
    else:
        return 1

if __name__=="__main__":
    er